# Etapa 01 — Cálculo do Potencial de Retorno Financeiro

Objetivo: conectar o MAPE do futuro modelo de precificação a um retorno financeiro
esperado (EV) para a plataforma, usando como preço de referência a mediana
Leave-One-Out (LOO) dos comparáveis e o modelo de ocupação do InsideAirbnb.

Dataset: InsideAirbnb — Amsterdam (`listings_details.csv`). Moeda: euro (€).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import os

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_rows', None) 

## 1.1 Registro de premissas

In [ ]:
# Premissas de negócio e do modelo, centralizadas para reuso em toda a Etapa 01

PREMISSAS = {
    # Receita da plataforma
    'take_rate': 0.15,            # fração da diária capturada pela plataforma por reserva

    # Modelo de ocupação do InsideAirbnb ("San Francisco Model")
    'review_rate': 0.50,          # fração de hóspedes que deixam review
    'avg_length_of_stay': 3,      # noites por reserva (padrão InsideAirbnb p/ Amsterdam)
    'occupancy_cap': 0.70,        # teto de ocupação anual do modelo

    # Preço de referência (mediana Leave-One-Out)
    'GROUP_COLS': ['neighbourhood_cleansed', 'room_type', 'accommodates'],
    'n_min': 30,                  # tamanho mínimo de célula antes de aplicar fallback
    'winsorize_bounds': (0.01, 0.99),  # percentis de clip do price (p1, p99)

    # Varredura de MAPE para a curva EV(MAPE)
    'mape_grid': [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40],

    # Fatores de realismo (opcionais, registrados para a análise de sensibilidade)
    'taxa_adocao': 1.0,           # fração dos anfitriões subprecificados que de fato ajusta o preço
    'desconto_elasticidade': 1.0, # fator de desconto sobre o uplift (efeito de elasticidade-preço)
}

## 1.2. Carga e limpeza mínima dos dados

In [ ]:
# Carga apenas das colunas necessárias para o cálculo do retorno financeiro
USE_COLS = ['price', 'neighbourhood_cleansed', 'room_type', 'accommodates', 'reviews_per_month']

df = pd.read_csv('../data/listings_details.csv', usecols=USE_COLS)

# price vem como texto com "$" por herança do schema — os valores são em euro
df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

print(f'Total de anúncios carregados: {len(df):,}')

## 1.3. Estimativa de noites reservadas

In [ ]:
# Modelo de ocupação do InsideAirbnb ("San Francisco Model")
# reviews_per_month ausente -> ocupação desconhecida -> noites_reservadas = 0 (conservador)

qtd_sem_reviews = df['reviews_per_month'].isna().sum()

df['bookings_per_month'] = df['reviews_per_month'] / PREMISSAS['review_rate']
df['nights_per_month'] = df['bookings_per_month'] * PREMISSAS['avg_length_of_stay']
df['noites_reservadas'] = np.minimum(
    df['nights_per_month'] * 12,
    365 * PREMISSAS['occupancy_cap']
)
df['noites_reservadas'] = df['noites_reservadas'].fillna(0.0)

print(f'Anúncios sem reviews_per_month (noites_reservadas = 0): {qtd_sem_reviews:,} '
      f'({qtd_sem_reviews / len(df):.1%} do total)')

## 1.4. Preço de referência

In [ ]:
# Winsorize de price nos percentis definidos nas premissas (p1, p99)
p_low, p_high = df['price'].quantile(PREMISSAS['winsorize_bounds'][0]), df['price'].quantile(PREMISSAS['winsorize_bounds'][1])

df['price_wz'] = df['price'].clip(lower=p_low, upper=p_high)

print(f'Bounds de winsorize: p{int(PREMISSAS["winsorize_bounds"][0]*100)}=€{p_low:.2f} | '
      f'p{int(PREMISSAS["winsorize_bounds"][1]*100)}=€{p_high:.2f}')

In [ ]:
# Nível 1 (mais granular): (neighbourhood_cleansed, room_type, accommodates)
GROUP_COLS = PREMISSAS['GROUP_COLS']
n_min = PREMISSAS['n_min']

df['group_size_l1'] = df.groupby(GROUP_COLS)['price_wz'].transform('size')

# Mediana Leave-One-Out exata e inline: para cada grupo, a mediana da série sem o próprio índice
df['ref_price_l1'] = df.groupby(GROUP_COLS)['price_wz'].transform(
    lambda s: pd.Series(
        [s.drop(idx).median() for idx in s.index], index=s.index
    )
)

print(f'Grupos no nível 1: {df.groupby(GROUP_COLS).ngroups:,}')
print(f'Anúncios em grupos com tamanho >= n_min ({n_min}): '
      f'{(df["group_size_l1"] >= n_min).sum():,} ({(df["group_size_l1"] >= n_min).mean():.1%})')

In [ ]:
# Níveis de fallback hierárquico (mediana LOO em grupos progressivamente mais amplos)

# Nível 2: (bairro, room_type)
df['group_size_l2'] = df.groupby(['neighbourhood_cleansed', 'room_type'])['price_wz'].transform('size')
df['ref_price_l2'] = df.groupby(['neighbourhood_cleansed', 'room_type'])['price_wz'].transform(
    lambda s: pd.Series([s.drop(idx).median() for idx in s.index], index=s.index)
)

# Nível 3: (room_type, accommodates)
df['group_size_l3'] = df.groupby(['room_type', 'accommodates'])['price_wz'].transform('size')
df['ref_price_l3'] = df.groupby(['room_type', 'accommodates'])['price_wz'].transform(
    lambda s: pd.Series([s.drop(idx).median() for idx in s.index], index=s.index)
)

# Nível 4: (room_type)
df['group_size_l4'] = df.groupby(['room_type'])['price_wz'].transform('size')
df['ref_price_l4'] = df.groupby(['room_type'])['price_wz'].transform(
    lambda s: pd.Series([s.drop(idx).median() for idx in s.index], index=s.index)
)

# Nível 5: cidade inteira
df['group_size_l5'] = len(df)
df['ref_price_l5'] = pd.Series(
    [df['price_wz'].drop(idx).median() for idx in df.index], index=df.index
)

print('Níveis de fallback calculados.')

In [ ]:
# Seleção hierárquica: usa o nível mais granular com group_size >= n_min (e ref_price não-nulo);
# senão desce para o próximo nível da cascata, até a cidade inteira (nível 5, sempre válido)

df['ref_price'] = np.where(
    (df['group_size_l1'] >= n_min) & df['ref_price_l1'].notna(), df['ref_price_l1'],
    np.where(
        (df['group_size_l2'] >= n_min) & df['ref_price_l2'].notna(), df['ref_price_l2'],
        np.where(
            (df['group_size_l3'] >= n_min) & df['ref_price_l3'].notna(), df['ref_price_l3'],
            np.where(
                (df['group_size_l4'] >= n_min) & df['ref_price_l4'].notna(), df['ref_price_l4'],
                df['ref_price_l5']
            )
        )
    )
)

df['ref_price_level'] = np.where(
    (df['group_size_l1'] >= n_min) & df['ref_price_l1'].notna(), 'L1_bairro_tipo_capacidade',
    np.where(
        (df['group_size_l2'] >= n_min) & df['ref_price_l2'].notna(), 'L2_bairro_tipo',
        np.where(
            (df['group_size_l3'] >= n_min) & df['ref_price_l3'].notna(), 'L3_tipo_capacidade',
            np.where(
                (df['group_size_l4'] >= n_min) & df['ref_price_l4'].notna(), 'L4_tipo',
                'L5_cidade'
            )
        )
    )
)

## 1.5. Varredura de MAPE e curva EV(MAPE)

In [ ]:
# Varredura de MAPE: para cada valor do grid, calcula a banda, classifica zonas e o EV
# Observação: a comparação usa o price ORIGINAL do anúncio (não o winsorizado),
# pois é o preço efetivamente praticado pelo anfitrião; o winsorize serviu apenas
# para robustecer o cálculo do ref_price.

resultados_mape = []

for mape in PREMISSAS['mape_grid']:
    L = df['ref_price'] * (1 - mape)
    U = df['ref_price'] * (1 + mape)

    subprecificado = df['price'] < L
    superprecificado = df['price'] > U
    zona_morta = ~subprecificado & ~superprecificado

    uplift = (L - df['price']).where(subprecificado, 0.0)

    ev = (
        PREMISSAS['take_rate']
        * (uplift * df['noites_reservadas'] * PREMISSAS['taxa_adocao'] * PREMISSAS['desconto_elasticidade']).sum()
    )

    resultados_mape.append({
        'mape': mape,
        'qtd_subprecificado': int(subprecificado.sum()),
        'qtd_zona_morta': int(zona_morta.sum()),
        'qtd_superprecificado': int(superprecificado.sum()),
        'uplift_medio_subprecificado': uplift[subprecificado].mean() if subprecificado.any() else 0.0,
        'EV': ev,
    })

df_ev_mape = pd.DataFrame(resultados_mape)

In [ ]:


fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_ev_mape['mape'], df_ev_mape['EV'], marker='o', color='#1f77b4', linewidth=2)
ax.set_xlabel('MAPE do modelo')
ax.set_ylabel('EV — Retorno financeiro esperado (€/ano)')
ax.set_title('Curva EV(MAPE) — Potencial de Retorno Financeiro')
ax.yaxis.set_major_formatter(lambda x, _: f'€{x:,.0f}')
ax.xaxis.set_major_formatter(lambda x, _: f'{x:.0%}')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 1.6. Cenários (otimista / base / pessimista)

In [ ]:
# Cenários ancorados em níveis plausíveis de MAPE para um modelo de precificação de imóveis
CENARIOS_MAPE = {
    'otimista': 0.10,
    'base': 0.20,
    'pessimista': 0.30,
}

df_cenarios = df_ev_mape[df_ev_mape['mape'].isin(CENARIOS_MAPE.values())].copy()
df_cenarios['cenario'] = df_cenarios['mape'].map({v: k for k, v in CENARIOS_MAPE.items()})
df_cenarios = df_cenarios[['cenario', 'mape', 'qtd_subprecificado', 'qtd_zona_morta',
                            'qtd_superprecificado', 'uplift_medio_subprecificado', 'EV']]
df_cenarios = df_cenarios.set_index('cenario').loc[['otimista', 'base', 'pessimista']]

print('Cenários de retorno financeiro (EV) por nível de MAPE:')
df_cenarios

# Etapa 02 — Data Quality

### 2.1 Join final entre as tabelas

In [ ]:
# Diagnóstico: listings_details.csv (base principal)
df_details = pd.read_csv('../data/listings_details.csv', low_memory=False)

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.10

# Split sobre os dados BRUTOS, antes de qualquer tratamento
df_train, df_test = train_test_split(
    df_details,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

# Persistência: df_test fica congelado até a Etapa 09
os.makedirs('../data/processed', exist_ok=True)
df_test.to_csv('../data/processed/test_raw.csv', index=False)
df_train.to_csv('../data/processed/train_raw.csv', index=False)

print(f'Dataset original: {df_details.shape}')
print(f'Treino:           {df_train.shape}  ({(1-TEST_SIZE)*100:.0f}%)')
print(f'Teste (hold-out): {df_test.shape}  ({TEST_SIZE*100:.0f}%)')
print('Salvos em ../data/processed/')

In [ ]:
# Join final: como nenhuma tabela auxiliar acrescenta coluna nova, df_final = df_train
# A partir daqui todo o pipeline opera apenas sobre o conjunto de TREINO.
df_final = df_train.copy()
print(df_final.shape)

Formalizamos aqui o resultado do Join entre todas as tabelas do projeto, decidido nos
Blocos 1-4:

- **`listings_details.csv` (limpo, `df_clean`)** — base, já no nível `listing_id`.
- **`calendar.csv`** — fora de escopo (decisão de modelagem estática, sem dimensão
  temporal).
- **`listings.csv`** — usado só para validação cruzada de `price` (Bloco 2.3); sua única
  coluna exclusiva (`neighbourhood_group`) é 100% nula, sem ganho de informação.
- **`neighbourhoods.csv`** — usado só para validação cruzada de `neighbourhood_cleansed`
  (Bloco 2.11); não tem chave numérica para juntar e não traz colunas novas.
- **`reviews_details.csv` (agregado)** — checagem cruzada (Bloco 3) mostrou que é
  redundante com `number_of_reviews`/`first_review`/`last_review` já existentes em
  `df_clean`; não é unido, para evitar colunas duplicadas.

Como nenhuma tabela auxiliar acrescenta coluna nova útil, o Join resulta no próprio
`df_clean`. Explicitamos isso como `df_final`, o nome oficial da tabela usada nas
próximas etapas.

In [ ]:
# Parse de colunas monetárias (string "$X,XXX.XX" -> float) e mapeamento de flags booleanas f/t -> {0,1}
COLS_MONETARIAS = ['price', 'security_deposit', 'cleaning_fee', 'extra_people']
for col in COLS_MONETARIAS:
    df_final[col] = df_final[col].str.replace(r'[\$,]', '', regex=True).astype(float)

COLS_FLAGS = [
    'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
    'is_location_exact', 'instant_bookable', 'require_guest_profile_picture',
    'require_guest_phone_verification',
]
for col in COLS_FLAGS:
    df_final[col] = df_final[col].map({'f': 0, 't': 1})

print(df_final[COLS_MONETARIAS].dtypes)
print(df_final[COLS_FLAGS].dtypes)

## 2.2 Estatística descritiva das features

### 2.2.1 Features numéricas

In [ ]:
# Flags booleanas (mapeadas para {0,1} no Bloco 2.10) são tratadas como categóricas,
# não entram na tabela de estatística numérica — vão para o boxplot categórico (7.2)
COLS_BOOLEANAS = [
    'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
    'is_location_exact', 'instant_bookable', 'require_guest_profile_picture',
    'require_guest_phone_verification',
]

num_cols = df_final.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c not in COLS_BOOLEANAS]

stats_numericas = pd.DataFrame({
    'minimo': df_final[num_cols].min(),
    'maximo': df_final[num_cols].max(),
    'intervalo': df_final[num_cols].max() - df_final[num_cols].min(),
    'media': df_final[num_cols].mean(),
    'mediana': df_final[num_cols].median(),
    'desvio_padrao': df_final[num_cols].std(),
    'coef_variacao': df_final[num_cols].std() / df_final[num_cols].mean(),
    'skew': df_final[num_cols].skew(),
    'kurtosis': df_final[num_cols].kurt(),
})

print(f'Total de features numéricas: {len(num_cols)}')
stats_numericas

**Análise da tabela — features numéricas**

Lendo a tabela célula a célula, agrupamos os problemas em 4 categorias. As colunas
marcadas com ❌ devem ser **excluídas** antes da Etapa 03; as marcadas com ⚠️ exigem
tratamento (não exclusão).

**1) Identificadores — não são features, são chaves/metadados**
- ❌ `scrape_id` — constante (`min == max == 20.181.206.172.549`, desvio padrão 0,
  skew/kurtosis 0). Não carrega nenhuma informação, é o mesmo valor em 100% das linhas.
- ❌ `host_id` — identificador único do anfitrião. Média/mediana/desvio-padrão não têm
  significado de negócio (é um código, não uma quantidade).
- `id` é mantido (chave primária do dataset, necessária para índice/merge — não é
  usada como feature de entrada do modelo).

**2) Colunas 100% nulas — sem nenhuma informação**
- ❌ `thumbnail_url`, `medium_url`, `xl_picture_url`, `host_acceptance_rate`,
  `neighbourhood_group_cleansed` — todas aparecem com `min`/`max`/`media`/`desvio_padrao`
  = `NaN`, ou seja, 100% de valores faltantes. Não há o que imputar ou aproveitar;
  devem ser descartadas.

**3) Coluna duplicada**
- ❌ `host_total_listings_count` — estatísticas **idênticas** a `host_listings_count`
  em todas as métricas (min 0, max 698, média 6,11, mediana 1, desvio 30,39, mesma
  assimetria/curtose). É cópia de `host_listings_count`; manter só uma das duas.

**4) Outliers por erro de digitação / valores fora do domínio físico ou do contrato**
- ❌ `bathrooms` — máximo de **100,5** banheiros. Fisicamente impossível para um
  imóvel residencial; é erro de digitação (kurtosis extrema de 12.264, confirma o
  outlier isolado). A linha correspondente deve ser removida ou o valor corrigido, não
  a coluna descartada.
- ⚠️ `minimum_nights` — máximo de **1.001** noites (~2,7 anos de estadia mínima).
  Incoerente com short/long-term rental do Airbnb; são poucos registros isolados
  (skew 56, kurtosis 4.130) — candidatos a remoção de linha, não de coluna.
- ⚠️ `maximum_nights` — máximo de **9.999**, porém a **mediana é 1.125**, valor que se
  repete em mais da metade da base (sentinela de "sem limite definido pelo anfitrião",
  não uma estadia real). Não remover a coluna, mas tratar `1125`/`9999` como
  categoria "sem limite" e não como número contínuo na modelagem.
- `beds` = 32 no máximo é um valor extremo, mas plausível para hostels/barcos
  (`accommodates` também chega a 17) — manter, não é erro de digitação.

**5) Pegadinhas de escala já documentadas (mantidas, sem remoção)**
- `review_scores_rating` está corretamente em escala **0–100** (min 20, max 100) — não
  reescalar para 0–5.
- `availability_365` (min 0, max 365) é disponibilidade futura no calendário, **não**
  ocupação — não usar como proxy de ocupação (reforça Seção 4 do CLAUDE.md).

**Resumo — colunas a excluir nesta etapa:**
`scrape_id`, `host_id`, `thumbnail_url`, `medium_url`, `xl_picture_url`,
`host_acceptance_rate`, `neighbourhood_group_cleansed`, `host_total_listings_count`.

**Linhas a investigar/remover:** registros com `bathrooms` e `minimum_nights` fora do
domínio plausível (erros de digitação).

*Observação: esta tabela foi gerada em cima de `df_final = df_details.copy()` (dados
ainda crus — `price` como string, flags `f`/`t` não mapeadas). A limpeza de colunas e
linhas listada acima ainda precisa ser implementada nas próximas células desta seção.*

In [ ]:
# Limpeza de colunas e linhas recomendada na análise da estatística descritiva numérica

# 1) Colunas: IDs irrelevantes, 100% nulas e coluna duplicada
COLS_PARA_REMOVER = [
    'scrape_id',                    # constante (mesmo valor em 100% das linhas)
    'host_id',                      # identificador único, sem significado de negócio
    'thumbnail_url',                # 100% nula
    'medium_url',                   # 100% nula
    'xl_picture_url',               # 100% nula
    'host_acceptance_rate',         # 100% nula
    'neighbourhood_group_cleansed', # 100% nula
    'host_total_listings_count',    # duplicada de host_listings_count
]

shape_antes = df_final.shape
df_final = df_final.drop(columns=COLS_PARA_REMOVER, errors='ignore')
print(f'Colunas removidas: {len(COLS_PARA_REMOVER)}')
print(f'Shape antes: {shape_antes} -> depois: {df_final.shape}')

# 2) Linhas: outliers por erro de digitação (fora do domínio físico/contratual)
mask_bathrooms_invalido = df_final['bathrooms'] > 10        # 100.5 banheiros é impossível
mask_min_nights_invalido = df_final['minimum_nights'] > 365  # 1001 noites mínimas não faz sentido

print(f"Linhas com bathrooms inválido: {mask_bathrooms_invalido.sum()}")
print(f"Linhas com minimum_nights inválido: {mask_min_nights_invalido.sum()}")

shape_antes_linhas = df_final.shape
df_final = df_final[~(mask_bathrooms_invalido | mask_min_nights_invalido)]
print(f'Shape antes: {shape_antes_linhas} -> depois: {df_final.shape}')

### 2.2.2 Features categóricas

In [ ]:
import matplotlib.pyplot as plt

# Flags booleanas (0/1) + categóricas de baixa cardinalidade — todas tratadas como
# categóricas para o boxplot de price por categoria
CAT_COLS_BOXPLOT = [
    'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
    'is_location_exact', 'instant_bookable', 'require_guest_profile_picture',
    'require_guest_phone_verification',
    'host_response_time', 'room_type', 'bed_type', 'cancellation_policy',
    'market', 'property_type', 'neighbourhood_cleansed',
]
print(f'Total de features categóricas no boxplot: {len(CAT_COLS_BOXPLOT)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_final.boxplot(column='price', by='host_is_superhost', ax=axes[0])
axes[0].set_title('price por host_is_superhost')
df_final.boxplot(column='price', by='host_has_profile_pic', ax=axes[1])
axes[1].set_title('price por host_has_profile_pic')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_final.boxplot(column='price', by='host_identity_verified', ax=axes[0])
axes[0].set_title('price por host_identity_verified')
df_final.boxplot(column='price', by='is_location_exact', ax=axes[1])
axes[1].set_title('price por is_location_exact')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_final.boxplot(column='price', by='instant_bookable', ax=axes[0])
axes[0].set_title('price por instant_bookable')
df_final.boxplot(column='price', by='require_guest_profile_picture', ax=axes[1])
axes[1].set_title('price por require_guest_profile_picture')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_final.boxplot(column='price', by='require_guest_phone_verification', ax=axes[0])
axes[0].set_title('price por require_guest_phone_verification')
df_final.boxplot(column='price', by='host_response_time', ax=axes[1])
axes[1].set_title('price por host_response_time')
axes[1].tick_params(axis='x', rotation=30)
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_final.boxplot(column='price', by='room_type', ax=axes[0])
axes[0].set_title('price por room_type')
axes[0].tick_params(axis='x', rotation=20)
df_final.boxplot(column='price', by='bed_type', ax=axes[1])
axes[1].set_title('price por bed_type')
axes[1].tick_params(axis='x', rotation=20)
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_final.boxplot(column='price', by='cancellation_policy', ax=axes[0])
axes[0].set_title('price por cancellation_policy')
axes[0].tick_params(axis='x', rotation=20)
df_final.boxplot(column='price', by='market', ax=axes[1])
axes[1].set_title('price por market')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df_final.boxplot(column='price', by='neighbourhood_cleansed', ax=ax)
ax.set_title('price por neighbourhood_cleansed')
ax.tick_params(axis='x', rotation=90)
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df_final.boxplot(column='price', by='property_type', ax=ax)
ax.set_title('price por property_type')
ax.tick_params(axis='x', rotation=90)
plt.suptitle('')
plt.tight_layout()
plt.show()

**Análise dos boxplots categóricos — candidatas a remoção**

Avaliando cada boxplot em conjunto com a distribuição de frequência das categorias:

**1) Flags quase constantes (desbalanceamento extremo, sem poder discriminativo)**
- ❌ `host_has_profile_pic` — 99,9% dos anúncios têm valor `1`; a categoria `0` tem
  representação estatisticamente insignificante (~0,1%) e o boxplot da classe
  minoritária é instável. Não separa preço.
- ❌ `require_guest_profile_picture` — 98,8% em `0`; a caixa de `1` é estreita mas
  baseada em amostra muito pequena — risco de espúrio, não de sinal real.
- ❌ `require_guest_phone_verification` — mesmo padrão (98,4% em `0`). Nenhuma das
  duas flags "require_guest_*" mostrou separação de mediana robusta o suficiente
  para justificar manter dado o desbalanceamento.
- ❌ `bed_type` — 99% dos registros são `Real Bed`; as demais categorias (Futon,
  Pull-out Sofa, Airbed, Couch) têm poucas dezenas de observações cada. O boxplot
  praticamente reproduz a distribuição geral de preço, não a de cada tipo de cama.
- ❌ `market` — 100% (ou virtualmente) dos registros são `Amsterdam`, valor único
  esperado já que a decisão de limpeza anterior removeu os registros fora dessa
  bounding box geográfica. Coluna constante remanescente, sem informação.

**2) Flags binárias com poder discriminativo real (manter)**
- ✅ `host_is_superhost` (84%/16%) — mostrou menos outliers extremos entre
  superhosts, sinal consistente com preços mais controlados.
- ✅ `host_identity_verified` (59%/41%) — balanceamento razoável, mesmo sem diferença
  forte de mediana; mantido por plausibilidade de negócio (verificação de identidade
  pode correlacionar com profissionalização do anfitrião).
- ✅ `is_location_exact` (70%/30%) — balanceamento adequado.
- ✅ `instant_bookable` (76%/24%) — balanceamento adequado, quantidade suficiente em
  ambas as classes para uma leitura confiável do boxplot.

**3) Categóricas de baixa cardinalidade com sinal claro (manter)**
- ✅ `room_type` — forte separação de mediana/cauda entre "Entire home/apt",
  "Private room" e "Shared room". Uma das features mais relevantes identificadas
  até agora.
- ✅ `cancellation_policy` — 4 categorias razoavelmente balanceadas (24–39% cada),
  outliers mais extremos concentrados em "moderate" e "strict_14_with_grace_period".
- ✅ `host_response_time` — 4 categorias balanceadas (23–49%), mantida apesar de
  medianas próximas entre si — combinação com outras features pode revelar
  interação relevante mais adiante (Etapa 03/04).

**4) Alta cardinalidade — manter, mas sinalizar tratamento futuro**
- ⚠️ `property_type` (31 categorias) — `Apartment` domina com 77,8%; categorias como
  `Barn`, `Tent`, `Nature lodge` têm poucos registros e caixas de boxplot pouco
  confiáveis. Não remover agora (há sinal real em categorias como House/Apartment),
  mas será candidata a agrupamento (ex.: agrupar categorias raras em "Other") na
  Etapa 03 (Feature Engineering) antes de qualquer encoding.
- ⚠️ `neighbourhood_cleansed` (22 categorias) — mais equilibrada que `property_type`
  (máximo de 17,5% em uma única categoria), com sinal visível de bairros centrais
  concentrando os maiores outliers. Manter sem agrupamento.

**Resumo — colunas categóricas a remover nesta etapa:**
`host_has_profile_pic`, `require_guest_profile_picture`,
`require_guest_phone_verification`, `bed_type`, `market`.

*Motivo comum: desbalanceamento extremo (uma categoria com >98% das observações),
tornando o boxplot da(s) categoria(s) minoritária(s) estatisticamente pouco
confiável e sem poder discriminativo prático para o preço.*

In [ ]:
# Remoção das colunas categóricas com desbalanceamento extremo identificadas na análise.
# Apenas essas 5 são dropadas — as demais colunas (inclusive texto livre e URLs) são
# mantidas para uso posterior na Etapa 03 (Feature Engineering).
COLS_CATEGORICAS_REMOVER = [
    'host_has_profile_pic',
    'require_guest_profile_picture',
    'require_guest_phone_verification',
    'bed_type',
    'market',
]

shape_antes = df_final.shape
df_final = df_final.drop(columns=COLS_CATEGORICAS_REMOVER, errors='ignore')

print(f'Colunas categóricas removidas: {len(COLS_CATEGORICAS_REMOVER)}')
print(f'Shape antes: {shape_antes} -> depois: {df_final.shape}')

In [ ]:
# Remoção da linha com maximum_nights = 9999 (valor sentinela isolado, fora do domínio plausível)
mask_max_nights_invalido = df_final['maximum_nights'] == 9999

print(f'Linhas com maximum_nights = 9999: {mask_max_nights_invalido.sum()}')

shape_antes = df_final.shape
df_final = df_final[~mask_max_nights_invalido]
print(f'Shape antes: {shape_antes} -> depois: {df_final.shape}')

In [ ]:
# Remoção dos anúncios com price == 0.
#
# Motivo: o MAPE — métrica-objetivo do projeto — é |y - ŷ| / |y|. Com y = 0 o termo
# diverge (divisão por zero), tornando a métrica incalculável para o dataset inteiro.
# Além disso, uma diária de € 0,00 não representa uma oferta real de mercado.
_linhas_antes = len(df_final)
df_final = df_final[df_final['price'] > 0].copy()
_removidas = _linhas_antes - len(df_final)

print(f'Anúncios com price == 0 removidos: {_removidas}')
print(f'Linhas: {_linhas_antes} -> {len(df_final)}')
print(f"Novo price mínimo: € {df_final['price'].min():.2f}")

In [ ]:
df_final.shape

# Etapa 03 — Feature Engineering

Implementação do planejamento em `docs/plans/03_feature_engineering.md`.

Partimos de `df_final` (saída da Etapa 02) e construímos `df03` com ~50 novas features
organizadas em 8 blocos temáticos.

**Regra crítica:** nenhuma feature pode derivar de `price`. `weekly_price` e
`monthly_price` são dropadas logo no início por vazamento.

In [ ]:
import numpy as np
import re

# Ponto de partida: cópia da saída da Etapa 02
df03 = df_final.copy()
shape_inicial = df03.shape

# PASSO 1 — Drop das colunas de VAZAMENTO (antes de tudo)
# weekly_price e monthly_price derivam de price -> contaminam o alvo
COLS_VAZAMENTO = ['weekly_price', 'monthly_price']
df03 = df03.drop(columns=COLS_VAZAMENTO, errors='ignore')

# Data de referência para cálculos temporais (data do scrape)
DATA_REF = pd.Timestamp('2018-12-06')

print(f'df_final (entrada): {shape_inicial}')
print(f'Após drop de vazamento ({COLS_VAZAMENTO}): {df03.shape}')
print(f'Data de referência: {DATA_REF.date()}')

In [ ]:
# BLOCO C — Features temporais

# Conversão das datas
for col in ['host_since', 'first_review', 'last_review']:
    df03[col] = pd.to_datetime(df03[col], errors='coerce')

# Experiência do anfitrião
# Nota: mantemos apenas a versão em dias. A versão em anos seria host_experiencia_dias
# / 365,25 — uma duplicata exata (ρ = 1,0000), que não acrescenta informação alguma
# ao modelo e apenas polui a matriz de correlação da EDA.
df03['host_experiencia_dias'] = (DATA_REF - df03['host_since']).dt.days

# Idade do anúncio e atividade
df03['dias_desde_primeira_review'] = (DATA_REF - df03['first_review']).dt.days
df03['dias_desde_ultima_review'] = (DATA_REF - df03['last_review']).dt.days

# Anúncio ativo: review nos últimos 180 dias (NaN -> 0, pois sem review não é ativo)
df03['anuncio_ativo'] = (df03['dias_desde_ultima_review'] <= 180).fillna(False).astype(int)


def parse_calendar_updated(texto):
    """Converte texto relativo do calendar_updated em número de dias."""
    if pd.isna(texto):
        return np.nan
    t = str(texto).lower().strip()
    if t == 'today':
        return 0
    if t == 'yesterday':
        return 1
    if t in ('a week ago', 'one week ago'):
        return 7
    if 'never' in t:
        return np.nan
    m = re.match(r'(\d+)\s+(day|week|month)s?\s+ago', t)
    if m:
        n, unidade = int(m.group(1)), m.group(2)
        return n * {'day': 1, 'week': 7, 'month': 30}[unidade]
    return np.nan


df03['calendar_atualizado_dias'] = df03['calendar_updated'].apply(parse_calendar_updated)

_bloco_c = ['host_experiencia_dias', 'dias_desde_primeira_review',
            'dias_desde_ultima_review', 'anuncio_ativo', 'calendar_atualizado_dias']
print(f'Bloco C — {len(_bloco_c)} features criadas')
print(df03[_bloco_c].describe().T[['count', 'mean', 'min', 'max']].round(2).to_string())
print()
print(f"Nulos em calendar_atualizado_dias: {df03['calendar_atualizado_dias'].isna().sum()}")

In [ ]:
# BLOCO A — Features geográficas

def haversine_km(lat1, lon1, lat2, lon2):
    """Distância em km entre dois pontos (fórmula de Haversine)."""
    R = 6371.0  # raio da Terra em km
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


# Polos turísticos de referência em Amsterdam
#
# Nota: Amsterdam Centraal (52.3791, 4.9003) foi descartada. Fica a apenas 0,86 km do
# Dam Square, e a distância resultante era quase colinear com dist_centro_km
# (Pearson +0,958; VIF 44,3). Nos testes, o par centro + museumplein prevê melhor
# (R2 0,087) que o par centro + centraal (R2 0,074) — a terceira distância agregava
# +0,006 de R2 ao custo de multicolinearidade severa.
POLOS = {
    'dist_centro_km':      (52.3731, 4.8922),  # Dam Square — centro histórico
    'dist_museumplein_km': (52.3579, 4.8816),  # Museumplein — polo cultural
}

for nome, (lat, lon) in POLOS.items():
    df03[nome] = haversine_km(df03['latitude'], df03['longitude'], lat, lon)

# Faixa de distância — captura não-linearidade
df03['dist_centro_bin'] = pd.cut(
    df03['dist_centro_km'],
    bins=[0, 1, 2, 3, 5, 8, np.inf],
    labels=['<1km', '1-2km', '2-3km', '3-5km', '5-8km', '>8km'],
)

_bloco_a = list(POLOS.keys()) + ['dist_centro_bin']
print(f'Bloco A — {len(_bloco_a)} features criadas')
print(df03[list(POLOS.keys())].describe().T[['mean', 'min', 'max']].round(2).to_string())
print()
print('Distribuição por faixa de distância:')
print(df03['dist_centro_bin'].value_counts(sort=False).to_string())

In [ ]:
# BLOCO B — Amenities

def parse_amenities(texto):
    """Extrai o conjunto de amenities da string no formato {A,"B C",D}."""
    if pd.isna(texto):
        return set()
    itens = re.findall(r'"([^"]+)"|([^{},]+)', str(texto))
    return {(a or b).strip() for a, b in itens if (a or b).strip()}


amen_sets = df03['amenities'].apply(parse_amenities)

# B.1 — Contagem total
df03['n_amenities'] = amen_sets.apply(len)

# B.2 — Flags binárias selecionadas (incidência entre 10% e 90% + relevância de negócio)
FLAGS_AMENITIES = {
    'tem_tv':                  ['TV'],
    'tem_washer':              ['Washer'],
    'tem_dryer':               ['Dryer'],
    'tem_family_kid_friendly': ['Family/kid friendly'],
    'tem_cable_tv':            ['Cable TV'],
    'tem_private_entrance':    ['Private entrance'],
    'tem_dishwasher':          ['Dishwasher'],
    'tem_patio_balcony':       ['Patio or balcony'],
    'tem_free_parking':        ['Free parking on premises', 'Free street parking'],
    'tem_elevator':            ['Elevator', 'Elevator in building'],
    'tem_step_free_access':    ['Step-free access'],
}

for flag, termos in FLAGS_AMENITIES.items():
    df03[flag] = amen_sets.apply(lambda s, t=termos: int(any(x in s for x in t)))

# B.3 — Índice composto de padrão do imóvel
COLS_LUXO = ['tem_dishwasher', 'tem_dryer', 'tem_elevator',
             'tem_patio_balcony', 'tem_free_parking', 'tem_private_entrance']
df03['n_amenities_luxo'] = df03[COLS_LUXO].sum(axis=1)

_bloco_b = ['n_amenities'] + list(FLAGS_AMENITIES.keys()) + ['n_amenities_luxo']
print(f'Bloco B — {len(_bloco_b)} features criadas')
print(f"n_amenities: media={df03['n_amenities'].mean():.1f}  min={df03['n_amenities'].min()}  max={df03['n_amenities'].max()}")
print()
print('Incidência das flags (%):')
for f in FLAGS_AMENITIES:
    print(f'  {f:26} {df03[f].mean()*100:5.1f}%')
print()
print(f"n_amenities_luxo: media={df03['n_amenities_luxo'].mean():.2f} (0 a {df03['n_amenities_luxo'].max()})")

In [ ]:
# BLOCO D — Host / Operação

def parse_verifications(texto):
    """Extrai a lista de verificações do formato "['email', 'phone', ...]"."""
    if pd.isna(texto):
        return []
    return re.findall(r"'([^']+)'", str(texto))


verif = df03['host_verifications'].apply(parse_verifications)

df03['n_host_verifications'] = verif.apply(len)
df03['host_tem_id_verificado'] = verif.apply(
    lambda lst: int('government_id' in lst or 'offline_government_id' in lst)
)

# Separa anfitrião casual de gestora profissional
# Nota: não criamos host_portfolio_size — seria uma cópia literal de
# calculated_host_listings_count (ρ = 1,0000). A coluna original já cumpre esse papel
# e permanece no dataset; duplicá-la só inflaria a contagem de features.
df03['host_eh_profissional'] = (df03['calculated_host_listings_count'] > 1).astype(int)

# Anfitrião local vs. investidor remoto
df03['host_eh_local'] = (
    df03['host_location'].fillna('').str.lower().str.contains('amsterdam').astype(int)
)

# host_response_rate: texto "95%" -> 0.95
df03['host_response_rate_num'] = (
    df03['host_response_rate'].str.rstrip('%').astype(float) / 100
)

_bloco_d = ['n_host_verifications', 'host_tem_id_verificado', 'host_eh_profissional',
            'host_eh_local', 'host_response_rate_num']
print(f'Bloco D — {len(_bloco_d)} features criadas')
print()
print(f"n_host_verifications: media={df03['n_host_verifications'].mean():.1f} (max {df03['n_host_verifications'].max()})")
print(f"host_tem_id_verificado: {df03['host_tem_id_verificado'].mean()*100:.1f}%")
print(f"host_eh_profissional:   {df03['host_eh_profissional'].mean()*100:.1f}%")
print(f"host_eh_local:          {df03['host_eh_local'].mean()*100:.1f}%")
print(f"host_response_rate_num: media={df03['host_response_rate_num'].mean():.3f}  nulos={df03['host_response_rate_num'].isna().sum()}")

In [ ]:
# BLOCO E — Estrutura do imóvel (razões)
# Zeros viram NaN antes da divisão para evitar infinitos

_bedrooms = df03['bedrooms'].replace(0, np.nan)
_beds = df03['beds'].replace(0, np.nan)
_accommodates = df03['accommodates'].replace(0, np.nan)

df03['pessoas_por_quarto'] = df03['accommodates'] / _bedrooms
df03['pessoas_por_cama'] = df03['accommodates'] / _beds
df03['banheiros_por_pessoa'] = df03['bathrooms'] / _accommodates
df03['camas_por_quarto'] = df03['beds'] / _bedrooms
df03['tem_multiplos_banheiros'] = (df03['bathrooms'] > 1).astype(int)

_bloco_e = ['pessoas_por_quarto', 'pessoas_por_cama', 'banheiros_por_pessoa',
            'camas_por_quarto', 'tem_multiplos_banheiros']
print(f'Bloco E — {len(_bloco_e)} features criadas')
print(df03[_bloco_e].describe().T[['count', 'mean', 'min', 'max']].round(2).to_string())
print()
n_inf = np.isinf(df03[_bloco_e].select_dtypes(include=[np.number])).sum().sum()
print(f'Valores infinitos gerados: {n_inf} (esperado: 0)')

In [ ]:
# BLOCO F — Política de reserva

# maximum_nights: 1125 é sentinela do Airbnb ("sem limite"), presente em ~51% dos dados
df03['sem_limite_maximo'] = (df03['maximum_nights'] >= 1125).astype(int)
df03['maximum_nights_cap'] = df03['maximum_nights'].clip(upper=365)

# Regimes de estadia mínima
df03['min_nights_bin'] = pd.cut(
    df03['minimum_nights'],
    bins=[0, 1, 3, 7, 30, np.inf],
    labels=['1', '2-3', '4-7', '8-30', '>30'],
)
df03['permite_estadia_curta'] = (df03['minimum_nights'] <= 3).astype(int)

# Custo de limpeza diluído na estadia mínima (não vaza: cleaning_fee != price)
df03['taxa_limpeza_por_noite'] = df03['cleaning_fee'] / df03['minimum_nights'].replace(0, np.nan)

# Nulo em cleaning_fee/security_deposit significa "sem cobrança"
df03['tem_taxa_limpeza'] = (df03['cleaning_fee'].fillna(0) > 0).astype(int)
df03['tem_deposito'] = (df03['security_deposit'].fillna(0) > 0).astype(int)

_bloco_f = ['sem_limite_maximo', 'maximum_nights_cap', 'min_nights_bin',
            'permite_estadia_curta', 'taxa_limpeza_por_noite', 'tem_taxa_limpeza', 'tem_deposito']
print(f'Bloco F — {len(_bloco_f)} features criadas')
print()
print(f"sem_limite_maximo:     {df03['sem_limite_maximo'].mean()*100:5.1f}%")
print(f"permite_estadia_curta: {df03['permite_estadia_curta'].mean()*100:5.1f}%")
print(f"tem_taxa_limpeza:      {df03['tem_taxa_limpeza'].mean()*100:5.1f}%")
print(f"tem_deposito:          {df03['tem_deposito'].mean()*100:5.1f}%")
print()
print(f"maximum_nights_cap: media={df03['maximum_nights_cap'].mean():.1f}  max={df03['maximum_nights_cap'].max()}")
print()
print('Distribuição de min_nights_bin:')
print(df03['min_nights_bin'].value_counts(sort=False).to_string())

In [ ]:
# BLOCO G — Reviews e reputação

df03['sem_reviews'] = (df03['number_of_reviews'] == 0).astype(int)

# As 6 subnotas estão em escala 0-10 (review_scores_rating está em 0-100 e fica de fora)
SUBNOTAS = ['review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
            'review_scores_communication', 'review_scores_location', 'review_scores_value']
df03['review_score_medio'] = df03[SUBNOTAS].mean(axis=1)

# Gaps: isolam dimensões específicas frente à avaliação geral
df03['gap_location_vs_geral'] = df03['review_scores_location'] - df03['review_score_medio']
df03['gap_value_vs_geral'] = df03['review_scores_value'] - df03['review_score_medio']

# Velocidade de ocupação normalizada pelo tempo de mercado
_meses_ativo = (df03['dias_desde_primeira_review'] / 30.44).replace(0, np.nan)
df03['reviews_por_mes_ativo'] = df03['number_of_reviews'] / _meses_ativo

_bloco_g = ['sem_reviews', 'review_score_medio', 'gap_location_vs_geral',
            'gap_value_vs_geral', 'reviews_por_mes_ativo']
print(f'Bloco G — {len(_bloco_g)} features criadas')
print()
print(f"sem_reviews: {df03['sem_reviews'].mean()*100:.1f}%")
print()
print(df03[['review_score_medio', 'gap_location_vs_geral', 'gap_value_vs_geral',
            'reviews_por_mes_ativo']].describe().T[['count', 'mean', 'min', 'max']].round(3).to_string())

In [ ]:
# BLOCO H — Texto livre (sinais de esforço do anfitrião, sem NLP)

CAMPOS_TEXTO = ['summary', 'space', 'description', 'neighborhood_overview', 'notes',
                'transit', 'access', 'interaction', 'house_rules', 'host_about']

df03['tem_descricao_completa'] = df03['description'].notna().astype(int)
df03['len_description'] = df03['description'].fillna('').str.len()
df03['len_house_rules'] = df03['house_rules'].fillna('').str.len()
df03['n_campos_texto_preenchidos'] = df03[CAMPOS_TEXTO].notna().sum(axis=1)

_bloco_h = ['tem_descricao_completa', 'len_description', 'len_house_rules',
            'n_campos_texto_preenchidos']
print(f'Bloco H — {len(_bloco_h)} features criadas')
print()
print(f"tem_descricao_completa: {df03['tem_descricao_completa'].mean()*100:.1f}%")
print(f"len_description: media={df03['len_description'].mean():.0f}  max={df03['len_description'].max()}")
print(f"len_house_rules: media={df03['len_house_rules'].mean():.0f}  max={df03['len_house_rules'].max()}")
print()
print('Distribuição de n_campos_texto_preenchidos (de 0 a 10):')
print(df03['n_campos_texto_preenchidos'].value_counts().sort_index().to_string())

In [ ]:
# Agrupamento de property_type: categorias com <1% viram "Other"
# Exceção: "Boat" é preservada por ser característica de Amsterdam (casas-barco)

LIMIAR = 0.01
freq = df03['property_type'].value_counts(normalize=True)
CATEGORIAS_MANTIDAS = set(freq[freq >= LIMIAR].index) | {'Boat'}

df03['property_type_agrupado'] = df03['property_type'].where(
    df03['property_type'].isin(CATEGORIAS_MANTIDAS), 'Other'
)

print(f'property_type: {df03["property_type"].nunique()} categorias -> '
      f'{df03["property_type_agrupado"].nunique()} após agrupamento')
print()
print('Distribuição resultante (%):')
print((df03['property_type_agrupado'].value_counts(normalize=True) * 100).round(2).to_string())

In [ ]:
# Drop final: colunas originais já consumidas pelas features criadas

COLS_DROP_FINAL = [
    # Inviáveis por nulidade
    'square_feet', 'license',
    # URLs — identificadores sem valor preditivo
    'listing_url', 'picture_url', 'host_url', 'host_thumbnail_url', 'host_picture_url',
    # Identificadores de texto livre — risco de overfitting
    'host_name', 'name',
    # Constantes (2 valores) — serviram apenas como data de referência
    'last_scraped', 'calendar_last_scraped',
    # Constantes — todos Amsterdam/NL
    'country', 'country_code', 'state', 'smart_location',
    'experiences_offered', 'has_availability', 'requires_license',
    'is_business_travel_ready', 'jurisdiction_names',
    # Bairro cru — neighbourhood_cleansed é a fonte de verdade
    'neighbourhood',
    # Alta cardinalidade sem ganho sobre neighbourhood_cleansed
    'zipcode', 'street', 'host_neighbourhood', 'city',
    # Consumidas pelos blocos de features
    'amenities', 'host_verifications', 'calendar_updated', 'host_location',
    'host_response_rate', 'property_type',
    'host_since', 'first_review', 'last_review',
]

# Campos de texto substituídos pelas features do Bloco H
COLS_DROP_FINAL += CAMPOS_TEXTO

shape_antes = df03.shape
df03 = df03.drop(columns=COLS_DROP_FINAL, errors='ignore')

n_removidas = shape_antes[1] - df03.shape[1]
print(f'Colunas removidas: {n_removidas}')
print(f'Shape: {shape_antes} -> {df03.shape}')

In [ ]:
# VALIDAÇÃO FINAL

novas_features = (_bloco_a + _bloco_b + _bloco_c + _bloco_d +
                  _bloco_e + _bloco_f + _bloco_g + _bloco_h + ['property_type_agrupado'])

print('=' * 62)
print('VALIDAÇÃO — ETAPA 03')
print('=' * 62)

# 1) Granularidade preservada
ok_linhas = len(df03) == len(df_final)
ok_gran = len(df03) == df03['id'].nunique()
print(f'\n1) Granularidade')
print(f'   Linhas preservadas: {len(df_final)} -> {len(df03)}  [{"OK" if ok_linhas else "FALHOU"}]')
print(f'   1 linha por listing_id: {"OK" if ok_gran else "FALHOU"}')

# 2) Sem infinitos
num = df03.select_dtypes(include=[np.number])
n_inf = np.isinf(num).sum().sum()
print(f'\n2) Valores infinitos: {n_inf}  [{"OK" if n_inf == 0 else "FALHOU"}]')

# 3) Anti-vazamento: correlação de Spearman com price
print(f'\n3) Anti-vazamento (|rho| > 0.95 = suspeito)')
feats_num = [f for f in novas_features if f in num.columns]
corr = df03[feats_num + ['price']].corr(method='spearman')['price'].drop('price')
suspeitas = corr[corr.abs() > 0.95]
if len(suspeitas) == 0:
    print(f'   Nenhuma feature suspeita  [OK]')
else:
    print(f'   SUSPEITAS: {suspeitas.to_dict()}  [FALHOU]')

print(f'\n   Top 8 correlações com price:')
for nome, v in corr.abs().sort_values(ascending=False).head(8).items():
    print(f'     {nome:28} {corr[nome]:+.3f}')

# 4) Resumo
print(f'\n4) Resumo')
print(f'   Features novas criadas: {len(novas_features)}')
print(f'   Shape final de df03: {df03.shape}')
print('=' * 62)

In [ ]:
# Inventário das features criadas e diagnóstico de nulos

blocos = {
    'A — Geográficas': _bloco_a,
    'B — Amenities': _bloco_b,
    'C — Temporais': _bloco_c,
    'D — Host': _bloco_d,
    'E — Estrutura': _bloco_e,
    'F — Política de reserva': _bloco_f,
    'G — Reviews': _bloco_g,
    'H — Texto': _bloco_h,
    'Agrupamento': ['property_type_agrupado'],
}

print('INVENTÁRIO DE FEATURES CRIADAS')
print('=' * 62)
for nome, feats in blocos.items():
    print(f'\n{nome} ({len(feats)})')
    for f in feats:
        print(f'  - {f}')

print('\n' + '=' * 62)
print('FEATURES NOVAS COM NULOS (todos herdados das colunas de origem)')
print('=' * 62)
nulos = df03[novas_features].isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
for f, n in nulos.items():
    print(f'  {f:30} {n:6} ({n/len(df03)*100:5.1f}%)')

print()
print('Nenhum desses nulos foi introduzido pelo feature engineering:')
print('  - host_response_rate_num  -> Airbnb só exibe taxa com histórico de mensagens')
print('  - *_review, reviews_*     -> 11,9% dos anúncios não têm review (categoria válida)')
print('  - pessoas_por_quarto etc. -> bedrooms/beds = 0 viraram NaN (evita infinito)')

In [ ]:
# Remoção de features com problemas identificados na validação

COLS_REMOVER_POS_VALIDACAO = {
    'host_response_rate_num': '47,4% de nulos — maior lacuna do conjunto',
    'tem_descricao_completa': '99,4% de incidência — quase-constante, não discrimina',
}

shape_antes = df03.shape
df03 = df03.drop(columns=list(COLS_REMOVER_POS_VALIDACAO), errors='ignore')

# Mantém as listas de blocos sincronizadas com o dataset
_bloco_d = [f for f in _bloco_d if f not in COLS_REMOVER_POS_VALIDACAO]
_bloco_h = [f for f in _bloco_h if f not in COLS_REMOVER_POS_VALIDACAO]
novas_features = [f for f in novas_features if f not in COLS_REMOVER_POS_VALIDACAO]

print('Features removidas:')
for col, motivo in COLS_REMOVER_POS_VALIDACAO.items():
    print(f'  - {col:24} {motivo}')
print()
print(f'Shape: {shape_antes} -> {df03.shape}')
print(f'Features novas: {len(novas_features)}')

# Etapa 04 — Análise Exploratória (EDA)

Entramos com `df03`: **18.020 linhas × 83 colunas**, granularidade de 1 linha por
`listing_id`.

**Isolamento do teste** (CLAUDE.md, Seção 5.3): os 2.003 anúncios do hold-out
(`test_raw.csv`) **não entram nesta análise**. Olhar o teste aqui contaminaria as decisões
de seleção (Etapa 05) e preparação (Etapa 06) — é vazamento por via analítica.

**Orientação da análise.** O MAPE é um erro *percentual*: errar € 20 num anúncio de € 60
custa 33%; errar os mesmos € 20 num anúncio de € 400 custa 5%. Por isso a EDA dá atenção
desproporcional à **cauda inferior de preços**, onde o modelo mais pode destruir a métrica.

## 4.1 Análise Univariada

Organizada em **7 grupos temáticos** (G1 a G7), cada um com no máximo ~17 colunas, para
que a leitura seja feita um grupo por vez em vez do dataset inteiro de uma só vez.

In [ ]:
# Definição dos 7 grupos temáticos da EDA.
# Fonte única de verdade: todas as análises seguintes leem daqui.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

GRUPOS = {
    'G1 — Capacidade e estrutura física': [
        'accommodates', 'bathrooms', 'bedrooms', 'beds', 'guests_included',
        'pessoas_por_quarto', 'pessoas_por_cama', 'banheiros_por_pessoa',
        'camas_por_quarto', 'tem_multiplos_banheiros', 'room_type',
        'property_type_agrupado',
    ],
    'G2 — Geografia': [
        'latitude', 'longitude', 'dist_centro_km', 'dist_museumplein_km',
        'dist_centro_bin', 'neighbourhood_cleansed',
    ],
    'G3 — Amenities e qualidade': [
        'n_amenities', 'n_amenities_luxo', 'tem_tv', 'tem_washer', 'tem_dryer',
        'tem_family_kid_friendly', 'tem_cable_tv', 'tem_private_entrance',
        'tem_dishwasher', 'tem_patio_balcony', 'tem_free_parking', 'tem_elevator',
        'tem_step_free_access',
    ],
    'G4 — Host e operação': [
        'host_is_superhost', 'host_identity_verified', 'host_response_time',
        'host_listings_count', 'calculated_host_listings_count',
        'host_eh_profissional', 'host_eh_local', 'host_tem_id_verificado',
        'n_host_verifications', 'host_experiencia_dias',
    ],
    'G5 — Política de reserva e custos': [
        'minimum_nights', 'maximum_nights', 'maximum_nights_cap', 'sem_limite_maximo',
        'min_nights_bin', 'permite_estadia_curta', 'cancellation_policy',
        'instant_bookable', 'is_location_exact', 'cleaning_fee',
        'taxa_limpeza_por_noite', 'tem_taxa_limpeza', 'security_deposit',
        'tem_deposito', 'extra_people', 'calendar_atualizado_dias',
    ],
    'G6 — Reviews e reputação': [
        'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
        'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
        'review_scores_value', 'review_score_medio', 'gap_location_vs_geral',
        'gap_value_vs_geral', 'number_of_reviews', 'reviews_per_month',
        'reviews_por_mes_ativo', 'sem_reviews', 'dias_desde_primeira_review',
        'dias_desde_ultima_review', 'anuncio_ativo',
    ],
    'G7 — Disponibilidade e texto': [
        'availability_30', 'availability_60', 'availability_90', 'availability_365',
        'len_description', 'len_house_rules', 'n_campos_texto_preenchidos',
    ],
}

TARGET = 'price'

# Classificação por tipo de tratamento visual
_num = df03.select_dtypes(include=[np.number]).columns
BINARIAS = {c for c in _num if df03[c].nunique() == 2}
CATEGORICAS = set(df03.select_dtypes(include=['object', 'category']).columns)


def tipo_coluna(col):
    """Define o tratamento visual: 'binaria', 'categorica' ou 'continua'."""
    if col in BINARIAS:
        return 'binaria'
    if col in CATEGORICAS:
        return 'categorica'
    return 'continua'


# Checagem de cobertura: os grupos devem particionar as explicativas sem sobra
_todas = [c for cols in GRUPOS.values() for c in cols]
_esperadas = set(df03.columns) - {'id', TARGET}

print('COBERTURA DOS GRUPOS')
print('=' * 62)
for nome, cols in GRUPOS.items():
    tipos = pd.Series([tipo_coluna(c) for c in cols]).value_counts()
    detalhe = ', '.join(f'{v} {k}' for k, v in tipos.items())
    print(f'{nome:38s} {len(cols):2d} cols  ({detalhe})')

print('=' * 62)
print(f'Soma dos grupos:          {len(_todas)}')
print(f'Sem duplicatas entre grupos: {len(_todas) == len(set(_todas))}')
print(f'No grupo mas não em df03: {sorted(set(_todas) - set(df03.columns)) or "nenhuma"}')
print(f'Em df03 mas sem grupo:    {sorted(_esperadas - set(_todas)) or "nenhuma"}')
print(f'\nTotal: {len(_todas)} explicativas + id + {TARGET} = {len(_todas) + 2} colunas')

### Análise 1.1 — O alvo `price`

Executada **primeiro e sozinha**, porque condiciona todo o resto da EDA: a escala do alvo
define como as correlações são medidas, e o tratamento de outliers define sobre quais
linhas todas as análises seguintes operam.

In [ ]:
# Análise 1.1 — estatística descritiva do alvo
p = df03[TARGET]


percentis = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
desc = p.describe(percentiles=percentis)

print('ESTATÍSTICA DESCRITIVA — price (€)')
print('=' * 46)
for k, v in desc.items():
    rotulo = 'contagem' if k == 'count' else k
    print(f'  {rotulo:>10s}  {v:>12,.2f}')

print()
print(f'  {"assimetria":>10s}  {p.skew():>12,.2f}')
print(f'  {"curtose":>10s}  {p.kurtosis():>12,.2f}')
print(f'  {"CV":>10s}  {p.std() / p.mean():>12,.2%}')

# Faixas de corte candidatas
print()
print('FAIXAS EXTREMAS')
print('=' * 46)
cortes = [
    ('price == 0',    (p == 0).sum()),
    ('price < 20',    (p < 20).sum()),
    ('price < 30',    (p < 30).sum()),
    ('price > 500',   (p > 500).sum()),
    ('price > 1.000', (p > 1000).sum()),
    ('price > 2.000', (p > 2000).sum()),
]
for rotulo, n in cortes:
    print(f'  {rotulo:<16s} {n:>5d}  ({n / len(p):>6.2%})')

print()
print(f'  p1  = € {p.quantile(0.01):>8,.2f}     p99 = € {p.quantile(0.99):>8,.2f}')
print(f'  IQR = € {p.quantile(0.75) - p.quantile(0.25):>8,.2f}')
lim_sup_tukey = p.quantile(0.75) + 1.5 * (p.quantile(0.75) - p.quantile(0.25))
print(f'  Limite superior de Tukey (Q3 + 1,5·IQR): € {lim_sup_tukey:,.2f}'
      f'  ->  {(p > lim_sup_tukey).sum()} anúncios acima ({(p > lim_sup_tukey).mean():.2%})')

In [ ]:
# Análise 1.1 — distribuição do alvo (4 painéis)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1) Histograma completo — evidencia a cauda
axes[0, 0].hist(p, bins=120, color='#4C72B0', edgecolor='none')
axes[0, 0].set_title('Escala completa (€ 8 a € 8.500)', fontweight='bold')
axes[0, 0].set_xlabel('price (€)')
axes[0, 0].set_ylabel('anúncios')

# 2) Histograma até p99 — onde a massa realmente está
p99 = p.quantile(0.99)
axes[0, 1].hist(p[p <= p99], bins=80, color='#55A868', edgecolor='none')
axes[0, 1].axvline(p.median(), color='#C44E52', ls='--', lw=2,
                   label=f'mediana € {p.median():,.0f}')
axes[0, 1].axvline(p.mean(), color='#8172B2', ls='--', lw=2,
                   label=f'média € {p.mean():,.0f}')
axes[0, 1].set_title(f'Até o p99 (€ {p99:,.0f}) — 99% dos anúncios', fontweight='bold')
axes[0, 1].set_xlabel('price (€)')
axes[0, 1].set_ylabel('anúncios')
axes[0, 1].legend()

# 3) Boxplot — magnitude dos outliers
axes[1, 0].boxplot(p, vert=False, widths=0.6,
                   flierprops=dict(marker='o', markersize=3, alpha=0.3))
axes[1, 0].set_title('Boxplot — escala completa', fontweight='bold')
axes[1, 0].set_xlabel('price (€)')
axes[1, 0].set_yticks([])

# 4) Zoom na cauda inferior — a faixa crítica para o MAPE
axes[1, 1].hist(p[p <= 100], bins=50, color='#DD8452', edgecolor='none')
axes[1, 1].axvline(20, color='#C44E52', ls='--', lw=2, label='€ 20')
axes[1, 1].set_title('Cauda inferior (≤ € 100) — faixa crítica do MAPE',
                     fontweight='bold')
axes[1, 1].set_xlabel('price (€)')
axes[1, 1].set_ylabel('anúncios')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f'Assimetria: {p.skew():.2f}   |   Curtose: {p.kurtosis():.2f}')
print(f'Média (€ {p.mean():,.2f}) supera a mediana (€ {p.median():,.2f}) '
      f'em {p.mean() / p.median() - 1:.1%} — efeito da cauda longa à direita.')

In [ ]:
# Análise 1.1 — inspeção dos extremos antes de decidir o tratamento
print('OS 10 ANÚNCIOS MAIS BARATOS')
print('=' * 78)
cols_ctx = ['price', 'room_type', 'accommodates', 'neighbourhood_cleansed',
            'number_of_reviews', 'minimum_nights']
print(df03.nsmallest(10, TARGET)[cols_ctx].to_string(index=False))

print()
print('OS 10 ANÚNCIOS MAIS CAROS')
print('=' * 78)
print(df03.nlargest(10, TARGET)[cols_ctx].to_string(index=False))

# Arredondamento psicológico: preço é decisão humana, não contínua
print()
print('ARREDONDAMENTO DE PREÇO (indício de precificação manual)')
print('=' * 78)
term_0 = (p % 10 == 0).mean()
term_5 = (p % 5 == 0).mean()
print(f'  Múltiplos de € 10: {term_0:.1%}')
print(f'  Múltiplos de €  5: {term_5:.1%}')
print()
print('  Os 8 preços mais frequentes:')
for valor, n in p.value_counts().head(8).items():
    print(f'    € {valor:>7,.2f}  {n:>5d} anúncios  ({n / len(p):>5.2%})')

**Observações — o alvo `price`**

- **Assimetria extrema.** `skew = 22,68` e curtose `1.016`. A média (€ 152) supera a mediana
  (€ 125) em 21,6% — a cauda direita arrasta a média para longe do anúncio típico.
- **Faixa central estreita, cauda longa.** O IQR é de apenas € 78 (€ 97 a € 175), mas o
  máximo chega a € 8.500 — **68× a mediana**.
- **Preço é uma decisão humana, não uma variável contínua.** 50,8% dos anúncios têm preço
  múltiplo de € 10 e 75,1% múltiplo de € 5. Os picos do histograma estão em € 100, € 120,
  € 150, € 200. Isso explica a aparência "serrilhada" da distribuição e impõe um piso ao
  MAPE: o modelo prevê um contínuo, mas o anfitrião arredonda.

**Sobre os extremos — a análise continua em 1.1.b.**

> **Nota:** a primeira leitura destes gráficos sugeriu manter todos os extremos. A
> inspeção caso a caso da **Análise 1.1.b** mostrou que parte deles não é preço de
> mercado, e sim erro de cadastro ou calendário bloqueado. A decisão final está lá.

Inspecionando os casos individualmente:

- **Cauda superior.** O anúncio de € 8.500 é um *Entire home/apt* para 2 pessoas com
  13 reviews — ou seja, um anúncio real e ocupado com preço implausível: é **erro de
  cadastro**. Já € 5.000 para um *Private room* com 0 reviews é o padrão clássico de
  anfitrião que inflaciona o preço para **bloquear o calendário** sem desativar o anúncio.
- **Cauda inferior.** Os 4 anúncios abaixo de € 20 são igualmente suspeitos (€ 8 para um
  apartamento inteiro de 4 pessoas), mas são **4 registros em 18.020** — 0,02%.

Optamos por **não winsorizar** nesta etapa, por três motivos:

1. **Volume irrelevante:** 22 anúncios acima de € 1.000 representam 0,12% do conjunto.
   O limite de Tukey (€ 292) marcaria 6,32% como outlier — cortar aí eliminaria imóveis
   de luxo legítimos, que são um segmento real do mercado de Amsterdam.
2. **O log resolve o problema.** A transformação avaliada na Análise 1.2 comprime a cauda
   sem descartar informação — preferível a remover linhas.
3. **Etapa errada.** A Etapa 01 winsoriza porque calcula uma *mediana de referência*, onde
   o extremo contamina a estatística do grupo. Aqui o alvo alimenta um modelo, e modelos
   de árvore são robustos a extremos. Se a Etapa 07 mostrar que os extremos prejudicam o
   MAPE, o corte volta à mesa com evidência.

**Premissa registrada:** `price` mantido integralmente (€ 8 a € 8.500), sem winsorize.

### Análise 1.1.b — Outliers de `price`: erro de cadastro ou preço de mercado?

O valor absoluto não distingue um imóvel de luxo de um erro de digitação: € 1.000/noite é
plausível para um apartamento de 16 pessoas no Centrum e implausível para um quarto
compartilhado em Osdorp. A pergunta correta não é *"o preço é alto?"*, e sim **"o preço é
alto em comparação com imóveis equivalentes?"**

**Critério adotado:** comparar cada anúncio com a **mediana do seu grupo**
`(neighbourhood_cleansed, room_type, accommodates)` — o mesmo `GROUP_COLS` da metodologia
travada da Etapa 01 (CLAUDE.md, Seção 3.1). A razão `price / mediana_do_grupo` mede o
desvio em relação aos comparáveis diretos.

A distribuição dessa razão mostra o que é normal no mercado:

| Percentil | Razão |
|---|---|
| p1 | 0,45× |
| p25 | 0,81× |
| p50 | 1,00× |
| p75 | 1,24× |
| p99 | 2,77× |
| p99,9 | 8,50× |

Ou seja: **99% dos anúncios ficam abaixo de 2,8× o preço dos seus comparáveis.** Um desvio
de 70× não é precificação — é erro.

In [ ]:
# Análise 1.1.b — diagnóstico dos outliers contra os comparáveis de mercado
GROUP_COLS = ['neighbourhood_cleansed', 'room_type', 'accommodates']

_g = df03.groupby(GROUP_COLS)[TARGET]
df03['ref_grupo'] = _g.transform('median')   # preço mediano dos comparáveis
df03['n_grupo'] = _g.transform('size')       # tamanho do grupo (confiança da referência)
df03['razao_mercado'] = df03[TARGET] / df03['ref_grupo']

print('DISTRIBUIÇÃO DA RAZÃO price / mediana_do_grupo')
print('=' * 52)
for q in [0.01, 0.25, 0.50, 0.75, 0.99, 0.999]:
    print(f'  p{q * 100:>5.1f}  {df03["razao_mercado"].quantile(q):>6.2f}x')

# Perfil de quem cobra muito acima dos comparáveis
_extremos = df03[df03['razao_mercado'] > 5]
print()
print(f'PERFIL — anúncios acima de 5x os comparáveis (n = {len(_extremos)})')
print('=' * 66)
print(f'{"indicador":<34s} {"extremos":>10s} {"dataset":>10s}')
print('-' * 66)
comp = [
    ('Sem nenhuma review', lambda d: (d['number_of_reviews'] == 0).mean()),
    ('Inativos (sem review há >1 ano)', lambda d: (d['dias_desde_ultima_review'] > 365).mean()),
]
for rotulo, fn in comp:
    print(f'{rotulo:<34s} {fn(_extremos):>9.1%} {fn(df03):>10.1%}')
print(f'{"Mediana de reviews":<34s} {_extremos["number_of_reviews"].median():>9.0f} '
      f'{df03["number_of_reviews"].median():>10.0f}')

print()
print('Anúncios muito acima dos comparáveis quase não recebem reserva:')
print('a taxa de "sem review" é 3x a do dataset. O preço não é praticado —')
print('é um anúncio inativo, com calendário bloqueado por preço proibitivo.')

In [ ]:
# Análise 1.1.b — teste do erro de digitação (dígito a mais / a menos)
# Hipótese: se o preço foi digitado com um zero extra (ou faltando), dividi-lo
# (ou multiplicá-lo) por 10 deve reaproximá-lo da mediana dos comparáveis.
_cols_ctx = ['price', 'ref_grupo', 'razao_mercado', 'n_grupo', 'room_type',
             'accommodates', 'neighbourhood_cleansed', 'number_of_reviews']

# Só avaliamos grupos com massa crítica: com n < 10 a mediana não é referência confiável
N_MIN_GRUPO = 10

print('CAUDA INFERIOR — razão < 0,25x, grupos com n >= 10')
print('=' * 96)
_baixos = df03[(df03['razao_mercado'] < 0.25) & (df03['n_grupo'] >= N_MIN_GRUPO)]
for _, r in _baixos.sort_values('razao_mercado').iterrows():
    _corr = r[TARGET] * 10
    print(f"  € {r[TARGET]:>7,.2f} ({r['razao_mercado']:>5.2f}x)  ->  x10 = € {_corr:>8,.2f} "
          f"= {_corr / r['ref_grupo']:>5.2f}x   [{r['room_type']}, {int(r['accommodates'])}p, "
          f"{r['neighbourhood_cleansed']}]")

print()
print('CAUDA SUPERIOR — razão > 5x, grupos com n >= 10')
print('=' * 96)
_altos = df03[(df03['razao_mercado'] > 5) & (df03['n_grupo'] >= N_MIN_GRUPO)]
for _, r in _altos.sort_values(TARGET, ascending=False).head(12).iterrows():
    _corr = r[TARGET] / 10
    print(f"  € {r[TARGET]:>8,.2f} ({r['razao_mercado']:>5.1f}x)  ->  /10 = € {_corr:>7,.2f} "
          f"= {_corr / r['ref_grupo']:>5.2f}x   [{int(r['number_of_reviews'])} reviews]")

print()
print('Leitura: na cauda INFERIOR, multiplicar por 10 devolve valores coerentes com o')
print('mercado (0,91x / 1,00x / 1,10x) — assinatura clara de dígito faltando.')
print('Na cauda SUPERIOR o teste não converge: dividir por 10 não reaproxima da mediana.')
print('Não são decimais deslocados — são preços deliberadamente proibitivos.')

**Decisão caso a caso**

A inspeção separa os extremos em **três categorias distintas**, que exigem tratamentos
diferentes:

**1. Erro de digitação — dígito faltando (cauda inferior).** Três casos são conclusivos:
€ 20, € 14 e € 12 viram **0,91× / 1,00× / 1,10×** a mediana dos comparáveis quando
multiplicados por 10. Essa convergência não é coincidência: é a assinatura de um zero
omitido no cadastro. O caso de € 8 (apartamento inteiro, 4 pessoas, 32 amenities, 20
reviews) é o mesmo fenômeno — nenhum anfitrião aluga um imóvel desses por € 8/noite.

**2. Preço proibitivo — calendário bloqueado (cauda superior).** Aqui o teste do dígito
**não converge**: dividir por 10 não reaproxima da mediana. Não são decimais deslocados.
O perfil desses anúncios revela o que realmente são:

| Indicador | Razão > 5× | Dataset |
|---|---|---|
| Sem nenhuma review | **38,9%** | 11,9% |
| Mediana de reviews | 2 | 8 |

A taxa de anúncios sem review é **3× maior**. Esse preço **não é praticado no mercado** —
é a prática conhecida de inflar o valor para bloquear o calendário sem despublicar o
anúncio. Incluí-los ensina o modelo uma relação preço/características que não existe.

**3. Luxo legítimo.** Os anúncios de € 2.000 e € 2.200 em Centrum-Oost (8 pessoas, 5
quartos, 5,5 banheiros) têm razão de **1,00× e 1,10×** — estão *na mediana* dos seus
comparáveis. São imóveis de alto padrão precificados coerentemente, e **devem permanecer**:
representam um segmento real do mercado que o recomendador precisa saber atender.

---

**Critério de remoção adotado**

Remover apenas anúncios cujo preço é **inconsistente com seus comparáveis diretos**:

- `razao_mercado > 5` — mais de 5× os comparáveis (p99 = 2,77×; o corte é ~2× o p99)
- **ou** `razao_mercado < 0,25` — menos de 1/4 dos comparáveis
- **e** `n_grupo >= 10` — só grupos com massa crítica, onde a mediana é referência confiável

A restrição de `n_grupo` é o que protege o luxo legítimo: em grupos pequenos a mediana é
instável, e uma razão alta pode ser artefato estatístico em vez de erro de cadastro.

**Por que remover em vez de winsorizar:** winsorize *substitui* o valor por um teto
arbitrário, mantendo a linha e ensinando ao modelo um preço que ninguém praticou. Como são
registros cujo preço não reflete o mercado, o correto é retirá-los — não fabricar um valor
plausível para eles.

In [ ]:
# Análise 1.1.b — remoção dos anúncios com preço inconsistente com o mercado
RAZAO_MAX = 5.00    # acima de 5x os comparáveis (p99 = 2,77x)
RAZAO_MIN = 0.25    # abaixo de 1/4 dos comparáveis
N_MIN_GRUPO = 10    # só grupos com massa crítica: mediana confiável

_mask_grupo_confiavel = df03['n_grupo'] >= N_MIN_GRUPO
_mask_caro_demais = df03['razao_mercado'] > RAZAO_MAX
_mask_barato_demais = df03['razao_mercado'] < RAZAO_MIN
_mask_remover = _mask_grupo_confiavel & (_mask_caro_demais | _mask_barato_demais)

print('REGISTROS A REMOVER')
print('=' * 58)
print(f'  Acima de {RAZAO_MAX:.0f}x os comparáveis ..... '
      f'{(_mask_grupo_confiavel & _mask_caro_demais).sum():>3d}')
print(f'  Abaixo de {RAZAO_MIN:.2f}x os comparáveis ... '
      f'{(_mask_grupo_confiavel & _mask_barato_demais).sum():>3d}')
print(f'  {"Total":<32s} {_mask_remover.sum():>3d}  ({_mask_remover.mean():.3%} do dataset)')

_shape_antes = df03.shape
_price_antes = df03[TARGET].copy()

df03 = df03[~_mask_remover].copy()

# As colunas de diagnóstico não são features — saem junto
df03 = df03.drop(columns=['ref_grupo', 'n_grupo', 'razao_mercado'], errors='ignore')

print()
print('EFEITO NO ALVO')
print('=' * 58)
print(f'{"":<16s} {"antes":>12s} {"depois":>12s}')
print('-' * 58)
print(f'{"linhas":<16s} {_shape_antes[0]:>12,d} {df03.shape[0]:>12,d}')
print(f'{"mínimo":<16s} € {_price_antes.min():>10,.2f} € {df03[TARGET].min():>10,.2f}')
print(f'{"máximo":<16s} € {_price_antes.max():>10,.2f} € {df03[TARGET].max():>10,.2f}')
print(f'{"mediana":<16s} € {_price_antes.median():>10,.2f} € {df03[TARGET].median():>10,.2f}')
print(f'{"média":<16s} € {_price_antes.mean():>10,.2f} € {df03[TARGET].mean():>10,.2f}')
print(f'{"assimetria":<16s} {_price_antes.skew():>12,.2f} {df03[TARGET].skew():>12,.2f}')
print(f'{"curtose":<16s} {_price_antes.kurtosis():>12,.2f} {df03[TARGET].kurtosis():>12,.2f}')

print()
print(f'Shape final: {df03.shape}')
print(f'Granularidade 1 linha por listing_id: {len(df03) == df03["id"].nunique()}')

### Análise 1.2 — Transformação do alvo

Com `skew = 22,68`, a questão é se o modelo deve ser treinado em `price` ou em
`log(price)`. A decisão afeta a Etapa 07 inteira, então precisa de evidência, não só do
número de assimetria.

In [ ]:
# Análise 1.2 — price vs. log1p(price)
# Lê df03 diretamente: reflete o alvo já limpo dos outliers (Análise 1.1.b)
from scipy import stats

p = df03[TARGET]
p_log = np.log1p(p)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Histogramas comparados
axes[0, 0].hist(p, bins=120, color='#4C72B0', edgecolor='none')
axes[0, 0].set_title(f'price — assimetria {p.skew():.2f}', fontweight='bold')
axes[0, 0].set_xlabel('price (€)')
axes[0, 0].set_ylabel('anúncios')

axes[0, 1].hist(p_log, bins=60, color='#55A868', edgecolor='none')
axes[0, 1].set_title(f'log1p(price) — assimetria {p_log.skew():.2f}', fontweight='bold')
axes[0, 1].set_xlabel('log1p(price)')
axes[0, 1].set_ylabel('anúncios')

# Q-Q plots — aderência à Normal
stats.probplot(p, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q plot — price', fontweight='bold')
axes[1, 0].get_lines()[0].set_markersize(2)
axes[1, 0].get_lines()[0].set_alpha(0.3)

stats.probplot(p_log, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q plot — log1p(price)', fontweight='bold')
axes[1, 1].get_lines()[0].set_markersize(2)
axes[1, 1].get_lines()[0].set_alpha(0.3)

plt.tight_layout()
plt.show()

# Comparação numérica
print('COMPARAÇÃO DAS ESCALAS')
print('=' * 58)
print(f'{"métrica":<22s} {"price":>15s} {"log1p(price)":>15s}')
print('-' * 58)
print(f'{"assimetria":<22s} {p.skew():>15,.2f} {p_log.skew():>15,.2f}')
print(f'{"curtose":<22s} {p.kurtosis():>15,.2f} {p_log.kurtosis():>15,.2f}')
print(f'{"CV":<22s} {p.std() / p.mean():>14.1%} {p_log.std() / p_log.mean():>14.1%}')

# Normalidade (D'Agostino em amostra — o teste é sensível a n grande)
amostra = p.sample(5000, random_state=42)
amostra_log = np.log1p(amostra)
_, pval_orig = stats.normaltest(amostra)
_, pval_log = stats.normaltest(amostra_log)
print(f'{"D Agostino (p-valor)":<22s} {pval_orig:>15.2e} {pval_log:>15.2e}')

print()
print('Nenhuma das duas é formalmente Normal (p < 0,05), o que é esperado com n = 18.020:')
print('o teste rejeita a normalidade por desvios mínimos quando a amostra é grande.')
print(f'O que importa é a redução da assimetria: {p.skew():.2f} -> {p_log.skew():.2f}.')

In [ ]:
# Análise 1.2 — teste decisivo: a escala log melhora o MAPE?
# A assimetria justifica o log em teoria. Como MAPE é a métrica do projeto,
# medimos o efeito diretamente em vez de supor.
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error

# Conjunto mínimo de features numéricas sem nulos, só para o teste comparativo
_feats = ['accommodates', 'bathrooms', 'bedrooms', 'beds', 'dist_centro_km',
          'dist_museumplein_km', 'n_amenities', 'minimum_nights']
X_teste = df03[_feats].fillna(df03[_feats].median())
y_real = df03[TARGET]

kf = KFold(n_splits=3, shuffle=True, random_state=42)
mape_orig, mape_log = [], []

for idx_tr, idx_va in kf.split(X_teste):
    X_tr, X_va = X_teste.iloc[idx_tr], X_teste.iloc[idx_va]
    y_tr, y_va = y_real.iloc[idx_tr], y_real.iloc[idx_va]

    # (a) treino na escala original
    m1 = RandomForestRegressor(n_estimators=60, max_depth=12, random_state=42, n_jobs=-1)
    m1.fit(X_tr, y_tr)
    mape_orig.append(mean_absolute_percentage_error(y_va, m1.predict(X_va)))

    # (b) treino em log1p, com MAPE medido na escala ORIGINAL (após expm1)
    m2 = RandomForestRegressor(n_estimators=60, max_depth=12, random_state=42, n_jobs=-1)
    m2.fit(X_tr, np.log1p(y_tr))
    pred_log = np.expm1(m2.predict(X_va))
    mape_log.append(mean_absolute_percentage_error(y_va, pred_log))

m_orig, m_log = np.mean(mape_orig), np.mean(mape_log)

print('MAPE POR ESCALA DE TREINO (RandomForest, CV=3)')
print('=' * 54)
print(f'  Treino em price ......... MAPE = {m_orig:.4f}  ({m_orig:.2%})')
print(f'  Treino em log1p(price) .. MAPE = {m_log:.4f}  ({m_log:.2%})')
print()
print(f'  Redução: {(m_orig - m_log) / m_orig:.1%} do MAPE')
print()
print('Nota: em ambos os casos o MAPE é medido na escala ORIGINAL (€), após expm1.')
print('Medir MAPE na escala log daria um número artificialmente baixo e sem sentido')
print('de negócio — o erro percentual só significa algo em euros.')

**Observações — transformação do alvo**

| Métrica | `price` | `log1p(price)` |
|---|---|---|
| Assimetria | 5,38 | **0,39** |
| Curtose | 90,67 | **1,21** |
| CV | 54,4% | 9,4% |

(Valores já sobre o alvo limpo da Análise 1.1.b.)

O Q-Q plot é o argumento visual mais claro: na escala original, os pontos abandonam a
linha teórica logo após o 2º quantil e disparam verticalmente. Em `log1p`, seguem a reta
ao longo de praticamente toda a distribuição, desviando só nos extremos.

**Decisão: treinar em `log1p(price)`.** Premissa registrada para a Etapa 07.

A evidência não é apenas a assimetria — é o próprio MAPE. Com um conjunto mínimo de 8
features, treinar em log reduziu o erro de **31,52% para 28,53%**, uma **queda de 9,5%**.

Vale notar que a remoção dos outliers (Análise 1.1.b) já havia baixado o MAPE de 33,86%
para 31,52% por si só. Os dois efeitos são **complementares**: remover registros cujo
preço não é praticado no mercado, e alinhar a função de perda à métrica.

O motivo é estrutural, e vale explicitar: o MAPE pondera erros de forma percentual, mas
treinar na escala original otimiza o **erro absoluto**. O modelo é então empurrado a
acertar os anúncios caros (onde os resíduos em euros são grandes) e a negligenciar os
baratos — exatamente o inverso do que o MAPE recompensa. Treinar em log alinha a função
de perda à métrica: no espaço logarítmico, minimizar o erro absoluto equivale
aproximadamente a minimizar o erro *relativo*.

**Dois cuidados obrigatórios para as Etapas 07 e 09:**

1. **O MAPE final deve ser sempre calculado na escala original**, após `expm1()`. Medi-lo
   na escala log produziria um número artificialmente baixo e sem significado de negócio —
   um erro percentual só é interpretável em euros.
2. **`expm1(mean(log(y)))` é a mediana, não a média.** A retrotransformação de uma
   predição em log devolve uma estimativa de mediana condicional. Para o recomendador da
   Etapa 09 isso é até desejável — a mediana é mais robusta e coerente com a metodologia
   travada da Etapa 01 (CLAUDE.md, Seção 3.1), que também usa mediana.

### Análises 1.3 a 1.9 — Distribuição por grupo (G1 a G7)

Cada grupo segue o mesmo roteiro: tabela descritiva → painéis de distribuição → observações.

**Critérios objetivos de sinalização**, aplicados uniformemente aos 7 grupos:

| Critério | Limiar | Ação sugerida |
|---|---|---|
| Variância nula | `nunique() == 1` | Descarte imediato |
| Quase-constância | >95% num único valor | Candidata a descarte |
| Nulos altos | >30% | Decidir imputação vs. descarte (Etapa 06) |
| Cauda extrema | p99 > 10× mediana | Candidata a clip/log (Etapa 06) |

A sinalização é **automática e uniforme** — a mesma regra para todas as 81 colunas, sem
julgamento caso a caso. O que cada sinal significa é discutido nas observações do grupo.

In [ ]:
# Funções reutilizáveis para o diagnóstico univariado dos grupos
LIM_QUASE_CONSTANTE = 0.95   # >95% num único valor
LIM_NULOS = 0.30             # >30% de nulos
LIM_CAUDA = 10               # p99 > 10x a mediana


def diagnosticar_grupo(nome_grupo):
    """Tabela descritiva + sinalização automática das colunas de um grupo."""
    cols = GRUPOS[nome_grupo]
    linhas = []

    for c in cols:
        s = df03[c]
        tipo = tipo_coluna(c)
        pct_nulos = s.isna().mean()

        # Concentração no valor/categoria mais frequente
        vc = s.value_counts(normalize=True, dropna=True)
        top_pct = vc.iloc[0] if len(vc) else np.nan
        top_val = vc.index[0] if len(vc) else None

        # Cauda: só faz sentido para contínuas positivas
        cauda = np.nan
        if tipo == 'continua':
            med = s.median()
            if pd.notna(med) and med > 0:
                cauda = s.quantile(0.99) / med

        # Sinalização automática, uniforme para todas as colunas
        sinais = []
        if s.nunique(dropna=True) <= 1:
            sinais.append('VARIANCIA-NULA')
        elif pd.notna(top_pct) and top_pct > LIM_QUASE_CONSTANTE:
            sinais.append('QUASE-CONSTANTE')
        if pct_nulos > LIM_NULOS:
            sinais.append('NULOS-ALTOS')
        if pd.notna(cauda) and cauda > LIM_CAUDA:
            sinais.append('CAUDA-EXTREMA')

        linhas.append({
            'coluna': c,
            'tipo': tipo,
            'nulos%': pct_nulos * 100,
            'nunique': s.nunique(dropna=True),
            'top_valor': str(top_val)[:22],
            'top%': top_pct * 100 if pd.notna(top_pct) else np.nan,
            'p99/mediana': cauda,
            'sinais': ' + '.join(sinais) if sinais else '',
        })

    tab = pd.DataFrame(linhas)
    print(f'{nome_grupo}  ({len(cols)} colunas)')
    print('=' * 116)
    print(tab.to_string(index=False, float_format=lambda v: f'{v:,.1f}'))

    alertas = tab[tab['sinais'] != '']
    print()
    if len(alertas):
        print(f'>> {len(alertas)} coluna(s) sinalizada(s):')
        for _, r in alertas.iterrows():
            print(f'   - {r["coluna"]:<32s} {r["sinais"]}')
    else:
        print('>> Nenhuma coluna sinalizada pelos critérios objetivos.')
    return tab


def plotar_grupo(nome_grupo, max_paineis=4):
    """Painéis de distribuição do grupo, no máximo 4 por figura."""
    cols = GRUPOS[nome_grupo]

    for ini in range(0, len(cols), max_paineis):
        bloco = cols[ini:ini + max_paineis]
        n = len(bloco)
        fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 3.4))
        if n == 1:
            axes = [axes]

        for ax, c in zip(axes, bloco):
            s = df03[c]
            tipo = tipo_coluna(c)

            if tipo == 'continua':
                dados = s.dropna()
                # Corta no p99 para a forma central não ficar esmagada pela cauda
                p99 = dados.quantile(0.99)
                if pd.notna(p99) and p99 > dados.min():
                    dados = dados[dados <= p99]
                ax.hist(dados, bins=40, color='#4C72B0', edgecolor='none')
                ax.set_title(f'{c}\n(até o p99)', fontsize=9, fontweight='bold')
            else:
                vc = s.value_counts(dropna=False).head(10)
                rotulos = [str(i)[:16] for i in vc.index]
                cor = '#DD8452' if tipo == 'binaria' else '#55A868'
                ax.barh(range(len(vc)), vc.values, color=cor)
                ax.set_yticks(range(len(vc)))
                ax.set_yticklabels(rotulos, fontsize=8)
                ax.invert_yaxis()
                ax.set_title(c, fontsize=9, fontweight='bold')

            ax.tick_params(labelsize=8)

        plt.tight_layout()
        plt.show()


print('Funções prontas: diagnosticar_grupo() e plotar_grupo()')
print(f'Limiares — quase-constante: >{LIM_QUASE_CONSTANTE:.0%} | '
      f'nulos: >{LIM_NULOS:.0%} | cauda: p99 > {LIM_CAUDA}x mediana')

In [ ]:
# Análise 1.3 — G1: Capacidade e estrutura física
_tab_g1 = diagnosticar_grupo('G1 — Capacidade e estrutura física')

In [ ]:
plotar_grupo('G1 — Capacidade e estrutura física')

**Observações — G1: Capacidade e estrutura física**

- **Nenhum alerta automático.** As 12 colunas passam nos quatro critérios: sem
  quase-constância, sem nulos altos, sem caudas extremas.
- **O mercado de Amsterdam é dominado por imóveis pequenos.** 54,1% dos anúncios acomodam
  2 pessoas, 59,3% têm 1 quarto e 81,6% têm exatamente 1 banheiro. A cidade é de casal e
  viajante solo — não de família.
- **79,2% são apartamentos inteiros**, contra 20% de quartos privativos. `Shared room` é
  quase inexistente, o que faz de `room_type` uma variável praticamente binária na prática.

**`bedrooms == 0` são estúdios, não dados faltantes.**

O histograma revela um pico em zero que a tabela descritiva não destaca: **1.006 anúncios
(5,6%)** têm `bedrooms = 0`. Investigando o perfil:

| Indicador | Valor |
|---|---|
| Entire home/apt | 737 (73,3%) |
| Private room | 269 (26,7%) |
| Mediana de `accommodates` | 2 pessoas |
| Mediana de `beds` | 1 cama |
| Mediana de `price` | € 100 |

São **estúdios e quartos sem divisão formal** — imóveis reais, com preço de mercado
coerente. O Airbnb registra "0 quartos" quando não há cômodo separado para dormir.

**Consequência: os nulos de `pessoas_por_quarto` (5,6%) e `camas_por_quarto` (5,7%) não são
dados ausentes** — são o resultado da proteção contra divisão por zero aplicada no Bloco E
da Etapa 03. O `NaN` aqui significa "não se aplica", não "não sabemos".

**Ação proposta para a Etapa 06:** criar a flag `eh_estudio` (`bedrooms == 0`) e imputar as
duas razões com um valor coerente, em vez de tratá-las como faltantes. A informação
"é estúdio" tem valor de negócio próprio e hoje está implícita num `NaN`.

In [ ]:
# Análise 1.4 — G2: Geografia
_tab_g2 = diagnosticar_grupo('G2 — Geografia')

# Checagem específica: pontos fora da bounding box de Amsterdam
LAT_MIN, LAT_MAX = 52.28, 52.43
LON_MIN, LON_MAX = 4.72, 5.07
_fora = ((df03['latitude'] < LAT_MIN) | (df03['latitude'] > LAT_MAX) |
         (df03['longitude'] < LON_MIN) | (df03['longitude'] > LON_MAX))
print()
print(f'Anúncios fora da bounding box de Amsterdam: {_fora.sum()}')
print(f'  latitude:  {df03["latitude"].min():.4f} a {df03["latitude"].max():.4f}')
print(f'  longitude: {df03["longitude"].min():.4f} a {df03["longitude"].max():.4f}')

print()
print('Anúncios por bairro (22 bairros):')
_vc = df03['neighbourhood_cleansed'].value_counts()
for bairro, n in _vc.items():
    print(f'  {bairro:<42s} {n:>5d}  ({n / len(df03):>5.1%})')

In [ ]:
# Análise 1.4 — G2: mapa geográfico e distribuição das distâncias
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# Mapa: densidade real dos anúncios
axes[0].scatter(df03['longitude'], df03['latitude'], s=2, alpha=0.12, color='#4C72B0')
axes[0].scatter([4.8922], [52.3731], s=110, color='#C44E52', marker='*',
                edgecolor='white', linewidth=0.8, label='Dam Square', zorder=5)
axes[0].scatter([4.8816], [52.3579], s=110, color='#DD8452', marker='*',
                edgecolor='white', linewidth=0.8, label='Museumplein', zorder=5)
axes[0].set_title('Distribuição geográfica dos anúncios', fontweight='bold')
axes[0].set_xlabel('longitude')
axes[0].set_ylabel('latitude')
axes[0].legend(fontsize=8)

axes[1].hist(df03['dist_centro_km'], bins=50, color='#55A868', edgecolor='none')
axes[1].set_title('dist_centro_km', fontweight='bold')
axes[1].set_xlabel('km do Dam Square')
axes[1].set_ylabel('anúncios')

_ordem = ['<1km', '1-2km', '2-3km', '3-5km', '5-8km', '>8km']
_vc_bin = df03['dist_centro_bin'].value_counts().reindex(_ordem)
axes[2].bar(range(len(_vc_bin)), _vc_bin.values, color='#8172B2')
axes[2].set_xticks(range(len(_vc_bin)))
axes[2].set_xticklabels(_ordem, rotation=45, ha='right', fontsize=8)
axes[2].set_title('dist_centro_bin', fontweight='bold')
axes[2].set_ylabel('anúncios')

plt.tight_layout()
plt.show()

print('CONCENTRAÇÃO POR BAIRRO')
print('=' * 52)
_vc = df03['neighbourhood_cleansed'].value_counts()
print(f'  Top 5 bairros concentram ....... {_vc.head(5).sum() / len(df03):>6.1%}')
print(f'  Bairros com < 1,5% do total .... {(_vc / len(df03) < 0.015).sum()} de 22')
print(f'  Menor bairro (Bijlmer-Oost) .... {_vc.min()} anúncios')
print()
print('Os 8 bairros menores somam apenas '
      f'{_vc.tail(8).sum() / len(df03):.1%} do dataset — cardinalidade alta com')
print('caudas finas é um risco para encoders na Etapa 06.')

**Observações — G2: Geografia**

- **Nenhum alerta automático**, e **nenhum anúncio fora da bounding box de Amsterdam**
  (lat. 52,29–52,42; lon. 4,76–5,03). As coordenadas estão limpas.
- **Adiantamento da hipótese H1 (preço cai com a distância).** Já é visível na univariada:

| Faixa | Densidade | Mediana de `price` |
|---|---|---|
| 0–1 km | 589 anúncios/km² | **€ 160** |
| 1–2 km | 478/km² | € 139 |
| 2–3 km | 379/km² | € 122 |
| 3–5 km | 87/km² | € 119 |
| 5–8 km | 8,5/km² | **€ 100** |

Tanto a densidade quanto o preço caem **monotonicamente**. A confirmação formal fica para
a Análise 2.2, mas o sinal é inequívoco.

- **A oferta forma um anel, não um disco.** O mapa mostra um vazio no centro exato: a
  faixa 0–1 km tem a maior densidade, mas apenas 1.850 anúncios em termos absolutos. O
  centro histórico de Amsterdam é ocupado por comércio, hotelaria tradicional e prédios
  tombados — sobra pouco espaço residencial para aluguel de curta duração.

**Risco identificado: cardinalidade alta com caudas finas em `neighbourhood_cleansed`.**

| Indicador | Valor |
|---|---|
| Top 5 bairros concentram | **53,4%** dos anúncios |
| Bairros com < 1,5% do total | **8 de 22** |
| Menor bairro (Bijlmer-Oost) | 90 anúncios (0,5%) |

Os 8 menores bairros somam apenas ~6% do dataset. Isso é um problema concreto para a
**Etapa 06**: um one-hot encoding criaria 22 colunas, várias delas com quase nenhum sinal,
e um target encoding sobre 90 observações produziria estimativas instáveis, com risco de
overfitting.

**Ação proposta para a Etapa 06:** usar *target encoding* com suavização bayesiana (que
puxa bairros pequenos na direção da média global) ou agrupar os bairros de cauda por
proximidade geográfica. A decisão pode ser tomada com o apoio da Análise 3.3, que
cruza bairro × room_type.

In [ ]:
# Análise 1.5 — G3: Amenities e qualidade
_tab_g3 = diagnosticar_grupo('G3 — Amenities e qualidade')

# As flags do Bloco B foram selecionadas por incidência entre 10% e 90%.
# Verificamos quais ainda respeitam esse critério após as remoções de linhas.
print()
print('INCIDÊNCIA DAS FLAGS vs. CRITÉRIO DO PLANO (10% a 90%)')
print('=' * 62)
_flags = [c for c in GRUPOS['G3 — Amenities e qualidade'] if c.startswith('tem_')]
for c in sorted(_flags, key=lambda x: -df03[x].mean()):
    inc = df03[c].mean()
    marca = '' if 0.10 <= inc <= 0.90 else '  <-- fora da faixa'
    print(f'  {c:<28s} {inc:>6.1%}{marca}')

In [ ]:
plotar_grupo('G3 — Amenities e qualidade')

# As duas flags abaixo do piso de 10% foram mantidas na Etapa 03 por argumento
# de negócio. Medimos se discriminam preço antes de decidir sobre elas.
from scipy.stats import mannwhitneyu

print('AS DUAS FLAGS DE BAIXA INCIDÊNCIA DISCRIMINAM PREÇO?')
print('=' * 74)
print(f'{"flag":<24s} {"n(1)":>6s} {"med(0)":>9s} {"med(1)":>9s} {"dif":>8s} {"p-valor":>10s}')
print('-' * 74)
for c in ['tem_elevator', 'tem_step_free_access', 'tem_free_parking']:
    g0 = df03.loc[df03[c] == 0, TARGET]
    g1 = df03.loc[df03[c] == 1, TARGET]
    _, pval = mannwhitneyu(g0, g1, alternative='two-sided')
    dif = (g1.median() / g0.median() - 1)
    print(f'{c:<24s} {len(g1):>6d} € {g0.median():>7.2f} € {g1.median():>7.2f} '
          f'{dif:>+7.1%} {pval:>10.2e}')

print()
print('Referência — a flag mais equilibrada do grupo:')
for c in ['tem_dryer']:
    g0 = df03.loc[df03[c] == 0, TARGET]
    g1 = df03.loc[df03[c] == 1, TARGET]
    _, pval = mannwhitneyu(g0, g1, alternative='two-sided')
    dif = (g1.median() / g0.median() - 1)
    print(f'{c:<24s} {len(g1):>6d} € {g0.median():>7.2f} € {g1.median():>7.2f} '
          f'{dif:>+7.1%} {pval:>10.2e}')

**Observações — G3: Amenities e qualidade**

- **Nenhum alerta automático.** A flag mais desbalanceada (`tem_step_free_access`, 93,7%
  em zero) fica logo abaixo do limiar de 95%.
- `n_amenities` tem distribuição bem-comportada (mediana 19, p99/mediana = 2,6). O
  histograma mostra picos em valores redondos, sugerindo que o Airbnb oferece "pacotes"
  de amenities pré-selecionados na interface de cadastro.

**Reavaliação das duas flags abaixo do piso de 10%**

A Etapa 03 declarou o critério "incidência entre 10% e 90%", mas manteve duas flags fora
da faixa por argumento de negócio, deixando a decisão para a EDA. Medindo a discriminação
de preço (Mann-Whitney):

| Flag | n(1) | Mediana(0) | Mediana(1) | Diferença | p-valor |
|---|---|---|---|---|---|
| `tem_elevator` | 1.743 | € 127 | € 115 | **−9,4%** | 1,8e-22 |
| `tem_step_free_access` | 1.131 | € 125 | € 130 | +4,0% | 1,9e-02 |
| `tem_free_parking` | 2.097 | € 125 | € 120 | −4,0% | 1,6e-16 |
| *(referência)* `tem_dryer` | 8.019 | € 119 | € 149 | **+25,2%** | 1,6e-203 |

**`tem_elevator` associa-se a preço MENOR — e isso é uma armadilha de confusão.**

O resultado parece absurdo: elevador deveria valorizar o imóvel. Investigando, a causa é
geográfica:

| | Com elevador | Sem elevador |
|---|---|---|
| Distância média ao centro | **3,69 km** | 2,57 km |

Prédios com elevador são construções mais modernas, concentradas na **periferia**. O
centro histórico de Amsterdam é feito de casas de canal estreitas e antigas, sem elevador —
e são justamente as mais caras. A flag está capturando *localização*, não *conforto*.

**Controlando por faixa de distância, o sinal se inverte:**

| Faixa | Sem elevador | Com elevador |
|---|---|---|
| < 1 km | € 159 | **€ 193** (+21%) |
| 1–2 km | € 139 | **€ 150** (+8%) |
| 2–3 km | € 120 | **€ 140** (+17%) |
| 3–5 km | € 120 | € 100 (−17%) |
| 5–8 km | € 115 | € 90 (−22%) |

**No centro, elevador vale mais.** Na periferia, a correlação inverte porque lá ele é
padrão construtivo, não diferencial.

**Decisão: manter as três flags.** Modelos de árvore capturam exatamente esse tipo de
interação (elevador × distância) que uma análise univariada não revela. Descartar
`tem_elevator` por causa de um sinal univariado negativo seria jogar fora informação real.
O Boruta (Etapa 05) decide com o modelo completo.

> **Alerta metodológico para a Análise 2.1.** Este caso mostra que o *ranking univariado de
> associação com o alvo* pode enganar quando há variáveis de confusão. Uma feature com
> correlação marginal fraca — ou até invertida — pode ser valiosa na presença de outras.
> O ranking da Análise 2.1 é um **mapa**, não um veredito.

In [ ]:
# Análise 1.6 — G4: Host e operação
_tab_g4 = diagnosticar_grupo('G4 — Host e operação')
plotar_grupo('G4 — Host e operação')

**Observações — G4 (Host e operação)**

**1. `host_response_time` — primeiro NULOS-ALTOS de toda a EDA (47,3%).**
É a contraparte categórica de `host_response_rate_num`, removida na Etapa 03 pelo mesmo
motivo (47,4% nulos). A causa é a mesma: o Airbnb só exibe métricas de resposta para
anfitriões com histórico de mensagens suficiente. **O nulo não é aleatório — é informativo:**

| | com resposta | sem resposta |
|---|---|---|
| price mediano | € 130,00 | € 120,00 |
| reviews (mediana) | 13 | 6 |
| % superhost | 22,3% | 8,8% |

O nulo marca o anfitrião de baixa atividade (metade das reviews, um quarto da taxa de
superhost). Descartar a coluna joga fora esse sinal. **Decisão:** manter e tratar o nulo
como **categoria explícita** (`"sem_histórico"`) na Etapa 06 — não imputar pela moda.
A confirmação do poder preditivo fica para a Análise 2.x.

**2. `host_listings_count` e `calculated_host_listings_count` — CAUDA-EXTREMA (p99/mediana ≈ 210).**
Não é erro de dado: é a estrutura real do mercado de Amsterdam. A mediana é **1 anúncio**
(anfitrião particular) e a cauda são gestoras profissionais:

| portfólio | host_listings_count | calculated_host_listings_count |
|---|---|---|
| > 1 anúncio | 24,4% | 21,1% |
| > 5 anúncios | 6,2% | 5,4% |
| > 20 anúncios | 3,3% | 2,9% |
| > 100 anúncios | 2,2% | 1,6% |
| máximo | 698 | 208 |

A distribuição é bimodal por natureza (particular vs. profissional), não uma cauda
contínua. As cinco maiores carteiras são todas `Entire home/apt`, nenhuma superhost,
preço € 73–90 — perfil de operação padronizada de baixo ticket, espalhada por bairros
diferentes. **Decisão:** não remover nem winsorizar. O corte de outliers da Análise 1.1.b
agiu sobre o **alvo**; aqui a cauda é uma característica legítima da explicativa. O
tratamento adequado é log ou binning na Etapa 06, e `host_eh_profissional` (78,9% zero)
já captura a mesma informação de forma binária.

**3. Redundância confirmada entre as duas colunas de portfólio.** ρ = 0,8743 — um dos
cinco pares |ρ| > 0,80 em aberto na Seção 2.3 do plano. A diferença é de escopo:
`host_listings_count` conta o portfólio global do anfitrião (máx. 698, inclui outras
cidades), `calculated_host_listings_count` conta só o que está nesta base de Amsterdam
(máx. 208). **Resolução adiada para a Análise 3.2**, com a hipótese de manter apenas
`calculated_host_listings_count` (escopo coerente com o dataset) mais `host_eh_profissional`.

**4. Flags do grupo — todas com variância utilizável, nenhuma quase-constante.**
`host_is_superhost` 84,1% zero · `host_identity_verified` 58,6% zero ·
`host_eh_profissional` 78,9% zero · `host_eh_local` 82,7% um ·
`host_tem_id_verificado` 72,6% zero. Nenhuma cruza o limite de 95%, então todas seguem
para a bivariada. `n_host_verifications` (mediana 4) e `host_experiencia_dias`
(2.797 valores distintos) são contínuas bem-comportadas, sem alerta.

**Pendência aberta:** `host_identity_verified` (58,6% zero) e `host_tem_id_verificado`
(72,6% zero) medem conceitos vizinhos e precisam de checagem de sobreposição na
Análise 3.2.

In [ ]:
# Análise 1.7 — G5: Política de reserva e custos
tab_g5 = diagnosticar_grupo('G5 — Política de reserva e custos')
display(tab_g5)

In [ ]:
plotar_grupo('G5 — Política de reserva e custos')

**Observações — G5 (Política de reserva e custos)**

**1. `security_deposit` — NULOS-ALTOS (30,9%), mas o nulo NÃO é "sem depósito".**
Esta é a descoberta mais importante do grupo. A coluna tem três estados distintos, e o
preço mediano separa os três:

| estado | anúncios | % | price mediano |
|---|---|---|---|
| NaN (campo não preenchido) | 5.554 | 30,9% | € 116,50 |
| 0,00 (declarado sem depósito) | 3.291 | 18,3% | € 125,00 |
| > 0 (cobra depósito) | 9.133 | 50,8% | € 135,00 |

O NaN não é "zero disfarçado": é € 8,50 **abaixo** do zero declarado. A gradação
NaN < zero < positivo sugere que o campo em branco marca o anúncio menos profissional
— quem não preencheu o formulário completo também cobra menos. **A decisão 3 da Etapa 02
(manter NaN, não assumir "NaN = sem cobrança") se confirma como correta.** Imputar zero
teria fundido dois grupos com € 8,50 de diferença mediana.

`cleaning_fee` (18,0% nulos) repete o padrão: NaN € 115,00 · zero € 125,00 · positivo
€ 130,00. Mesma conclusão. **Decisão:** tratar o nulo como categoria explícita na Etapa 06
em ambas as colunas. As flags `tem_deposito` e `tem_taxa_limpeza` já codificam
"positivo vs. resto" — a informação que falta é justamente a separação NaN/zero.

**2. `calendar_atualizado_dias` — CAUDA-EXTREMA (p99/mediana = 31,7), com implicação de negócio.**
Mediana de 35 dias, mas a cauda é enorme: p90 = 540 dias, máximo 1.620 dias (4,4 anos).

| sem atualizar | anúncios | % |
|---|---|---|
| > 7 dias | 11.734 | 65,3% |
| > 30 dias | 9.330 | 51,9% |
| > 180 dias | 4.379 | 24,4% |
| > 365 dias | 2.764 | 15,4% |

**15,4% dos anúncios não têm o calendário tocado há mais de um ano.** Isso é relevante
para o projeto além da modelagem: o preço de um anúncio abandonado há 4 anos é um preço
histórico, não uma decisão de mercado atual. Para a Etapa 01/10 (recomendador de preço
justo), recomendar um aumento a um anúncio inativo não gera EV — o anfitrião não está
operando. **Registrado como pendência para a Etapa 09/10**, não para tratamento aqui.
Para a modelagem, a coluna é uma explicativa legítima (recência de atualização é sinal
de atividade); o tratamento é log/binning na Etapa 06.

**3. `minimum_nights` — a marca da regulação de Amsterdam.**
Mediana 2 noites, com 40,4% exatamente em 2 e 24,8% em 3. Não é ruído: Amsterdam limita
aluguel de curta duração, e a política de 2–3 noites é o padrão do mercado
(`min_nights_bin` "2-3" = 65,2%). O máximo é 365 noites e apenas 116 anúncios (0,6%)
exigem ≥ 30 noites — aluguel mensal disfarçado de temporada. Volume pequeno demais para
justificar remoção; a flag `permite_estadia_curta` (81,0% um) já isola o grupo.

**4. `maximum_nights` e `maximum_nights_cap` — redundância por construção.**
51,1% dos anúncios têm `maximum_nights` = 1125 (o default do Airbnb, equivalente a
"sem limite"), e `sem_limite_maximo` = 1 para exatamente esses 51,1%. `maximum_nights_cap`
é a versão truncada em 365 (57,8% no teto). **As três colunas carregam a mesma informação
em três formatos.** Ambas com p99/mediana = 1,00 — nenhuma variação útil acima da mediana.
**Resolução na Análise 3.2:** manter provavelmente só `sem_limite_maximo` + `maximum_nights_cap`.

**5. Demais colunas — sem alerta, todas seguem para a bivariada.**
`cancellation_policy` (4 categorias, mais frequente `strict_14_with_grace_period` 39,3%) ·
`instant_bookable` (76,0% zero) · `is_location_exact` (69,8% um) ·
`extra_people` (65,4% em zero — a maioria não cobra por hóspede extra) ·
`taxa_limpeza_por_noite` (mediana € 10,00). Nenhuma cruza o limite de 95% de
quase-constância.

**Pendência aberta:** os pares `maximum_nights` / `maximum_nights_cap` / `sem_limite_maximo`
e `cleaning_fee` / `taxa_limpeza_por_noite` (esta é derivada daquela) precisam de
checagem de correlação na Análise 3.2.

In [ ]:
# Análise 1.8 — G6: Reviews e reputação
tab_g6 = diagnosticar_grupo('G6 — Reviews e reputação')
display(tab_g6)

In [ ]:
plotar_grupo('G6 — Reviews e reputação')

**Observações — G6 (Reviews e reputação)**

**1. Escala 0–100 confirmada em `review_scores_rating`.** min = 20, max = 100,
mediana = 97. A "pegadinha" da safra documentada na Etapa 02 (CLAUDE.md Seção 7) se
confirma: a coluna **não** está em 0–5. As 6 subnotas `review_scores_*` estão em 0–10.
**Nenhuma das duas deve ser reescalada** — registro explícito para a Etapa 06.

**2. Efeito teto severo — as notas quase não discriminam.**
Este é o achado central do grupo. As subnotas estão concentradas na nota máxima:

| subnota | % na nota 10 | mediana |
|---|---|---|
| `review_scores_communication` | 72,8% | 10 |
| `review_scores_checkin` | 69,8% | 10 |
| `review_scores_accuracy` | 65,2% | 10 |
| `review_scores_cleanliness` | 53,4% | 10 |
| `review_scores_location` | 45,8% | 10 |
| `review_scores_value` | 30,2% | 9 |

`communication` e `checkin` têm ~70% num único valor — perto do limite de 95% de
quase-constância, mas ainda abaixo. **Consequência prática:** o poder discriminante
dessas colunas é baixo. As duas mais informativas são `value` (30,2% no teto, única com
mediana 9) e `location` (45,8%) — justamente as que capturam trade-off, não cortesia.
As features `gap_location_vs_geral` e `gap_value_vs_geral`, criadas na Etapa 03,
existem exatamente para extrair sinal desse teto e são as candidatas mais promissoras
do grupo. Confirmação na Análise 2.x.

**3. Os nulos de score (13,1%) NÃO coincidem com `sem_reviews` (11,9%).**

| | anúncios |
|---|---|
| `sem_reviews` = 1 | 2.131 (11,9%) |
| `review_scores_rating` NaN | 2.355 (13,1%) |
| **tem review mas não tem nota** | **224** |

Os 224 são anúncios com poucas reviews (mediana de 1, máximo 19) — o Airbnb só publica
a nota agregada após um número mínimo de avaliações. **São dois nulos com causas
diferentes** e devem receber tratamento distinto na Etapa 06: `sem_reviews` = 1 é
"anúncio novo, sem histórico"; os 224 são "histórico insuficiente para nota". A flag
`sem_reviews` já isola o primeiro grupo.

**4. `number_of_reviews` — CAUDA-EXTREMA (p99/mediana = 25,7) e uma inversão de preço.**
Mediana 9 reviews, p99 = 231, máximo 695. Mas o dado relevante é o preço:

| | price mediano |
|---|---|
| sem review | € 139,00 |
| com review | € 125,00 |

**Anúncios sem review são € 14 mais caros.** Isso é coerente com o achado da Análise 1.1.b:
a ausência de review foi um dos marcadores do padrão de bloqueio de calendário (38,9%
dos outliers de preço não tinham review, contra 11,9% de base). Parte dos € 139 não é
preço de mercado — é preço de anúncio que não pretende ser reservado. Como sinal para o
modelo isso é útil (a flag separa os dois regimes), mas **reforça a pendência registrada
para a Etapa 09/10**: recomendar preço a anúncio inativo não gera EV.

**5. `anuncio_ativo` (57,8% ativo) separa bem duas populações.**
Mediana de `dias_desde_ultima_review`: 45 dias para ativo=1 contra **480 dias** para
ativo=0 — uma ordem de grandeza. O preço acompanha: € 130,00 vs € 120,00. A flag é
coerente com `calendar_atualizado_dias` do G5 e com a leitura de que ~40% da base está
em algum grau de inatividade.

**6. Demais CAUDA-EXTREMA — esperadas, sem ação.**
`reviews_per_month` (p99/mediana 10,4), `reviews_por_mes_ativo` (11,4) e
`dias_desde_ultima_review` (12,1). Todas são contagens/durações com assimetria natural
à direita — mesmo raciocínio aplicado às colunas de portfólio do G4. Tratamento por
log/binning na Etapa 06, sem remoção.

**Pendências abertas:** (a) as 6 subnotas + `review_score_medio` + `review_scores_rating`
formam um bloco provavelmente muito correlacionado — checagem obrigatória na Análise 3.2;
(b) `number_of_reviews` / `reviews_per_month` / `reviews_por_mes_ativo` medem intensidade
de avaliação em três escalas e também precisam de checagem de correlação.

In [ ]:
# Análise 1.9 — G7: Disponibilidade e texto
tab_g7 = diagnosticar_grupo('G7 — Disponibilidade e texto')
display(tab_g7)

In [ ]:
plotar_grupo('G7 — Disponibilidade e texto')

**Observações — G7 (Disponibilidade e texto)**

**1. `len_description` está truncada — não é uma medida real de tamanho.**
60,2% dos anúncios têm exatamente 1.000 caracteres, **zero** têm mais, e o máximo é
1.000. Isso não é distribuição natural: é o limite do campo `description` no Airbnb.
A coluna mede "atingiu o teto ou não" para a maioria da base, não o tamanho real do
texto. **Consequência:** o poder informativo é limitado e a interpretação "descrição
mais longa = anúncio mais elaborado" só vale abaixo do teto. Sugestão para a Etapa 06:
tratar como binária (`atingiu_limite_descricao`) ou binning, em vez de contínua.
`n_campos_texto_preenchidos` (0–10, mediana alta, 19,4% no máximo 10) é a medida de
completude mais confiável do grupo e não sofre desse problema.

**2. Bloco `availability_*` — quatro colunas, uma informação.**
Correlação de Spearman entre elas:

| | av_30 | av_60 | av_90 | av_365 |
|---|---|---|---|---|
| **av_30** | 1,00 | 0,92 | 0,88 | 0,77 |
| **av_60** | 0,92 | 1,00 | **0,97** | 0,86 |
| **av_90** | 0,88 | 0,97 | 1,00 | 0,90 |
| **av_365** | 0,77 | 0,86 | 0,90 | 1,00 |

São janelas aninhadas do mesmo calendário — a redundância é estrutural, não coincidência.
`availability_60` × `availability_90` chega a ρ = 0,97. **Resolução na Análise 3.2:**
manter provavelmente `availability_365` (janela mais ampla) e talvez `availability_30`
(sinal de curto prazo, ρ = 0,77 com a de 365 — a menos redundante do bloco).

**3. `availability_365` — CAUDA-EXTREMA (p99/mediana = 91,0), mas o problema real é o zero.**
A mediana é **4 dias** e 46,4% dos anúncios têm o calendário **totalmente fechado**
(`availability_365` = 0). A proporção cresce conforme a janela encurta: 57,4% em 30 dias,
53,1% em 60, 50,8% em 90.

| | price mediano | % sem review |
|---|---|---|
| calendário fechado | € 120,00 | 12,9% |
| com disponibilidade | € 135,00 | 11,0% |

**Reforço explícito da "pegadinha" da Etapa 02:** `availability_365` = 0 pode significar
*"totalmente reservado"* ou *"anfitrião fechou o calendário"* — são coisas opostas e a
coluna não distingue. É exatamente por isso que o CLAUDE.md Seção 4 proíbe usá-la como
proxy de ocupação, e o modelo do InsideAirbnb (via `reviews_per_month`) é usado em seu
lugar. **A coluna segue como explicativa** (disponibilidade declarada tem valor
preditivo), mas o uso como ocupação continua vedado.

**4. Convergência com G5 e G6 — os três grupos apontam para o mesmo fenômeno.**
`calendar_atualizado_dias` > 365 em 15,4% (G5), `anuncio_ativo` = 0 em 42,2% (G6) e
`availability_365` = 0 em 46,4% (G7) são leituras independentes de uma base em que uma
fração relevante dos anúncios **não está operando ativamente**. Para a modelagem isso é
sinal legítimo; para o recomendador de preço da Etapa 09/10 é a pendência já registrada:
o EV calculado sobre anúncios inativos é ilusório, porque não há reservas a capturar.
**Recomendação para a Etapa 10:** considerar um filtro de atividade no cálculo do EV,
ou ao menos reportar o EV segmentado entre ativos e inativos.

**5. `len_house_rules` — CAUDA-EXTREMA (p99/mediana = 17,5), sem ação.**
37,3% em zero (não preencheu regras) e um segundo pico em 1.000 (mesmo truncamento de
campo). Assimetria natural de contagem de caracteres. Tratamento por log/binning na
Etapa 06.

---

### Encerramento da Análise Univariada (1.1 – 1.9)

**Cobertura:** 81 colunas explicativas + alvo, distribuídas nos 7 grupos temáticos
(G1–G7) definidos no plano. Todas as colunas foram diagnosticadas e nenhuma ficou de fora.

**Resultado do alvo (`price`):** € 0 removidos (Etapa 02), 42 outliers removidos por
critério de razão de mercado (Análise 1.1.b), skew 5,38 → **0,39** com `log1p`,
MAPE de referência 31,52% → **28,53%**. Base final: **17.978 linhas × 83 colunas**,
1 linha por `listing_id`, faixa de preço € 19–3.142.

**Alertas automáticos por grupo:**

| grupo | colunas | VARIÂNCIA-NULA | QUASE-CONSTANTE | NULOS-ALTOS | CAUDA-EXTREMA |
|---|---|---|---|---|---|
| G4 — Host e operação | 10 | 0 | 0 | 1 | 2 |
| G5 — Política e custos | 16 | 0 | 0 | 1 | 1 |
| G6 — Reviews | 17 | 0 | 0 | 0 | 4 |
| G7 — Disponibilidade e texto | 7 | 0 | 0 | 0 | 2 |

**Nenhuma coluna foi sinalizada como VARIÂNCIA-NULA ou QUASE-CONSTANTE em todo o dataset** —
a limpeza da Etapa 02 (drop de constantes e de colunas 100% nulas) foi eficaz e não
sobrou coluna sem variabilidade útil.

**Três achados que alteram decisões de etapas seguintes:**
1. **NaN ≠ zero** em `security_deposit` e `cleaning_fee` (€ 8,50 de diferença mediana).
   Tratar o nulo como categoria explícita na Etapa 06, não imputar.
2. **Efeito teto severo** nas notas de review (até 72,8% na nota máxima). As features
   de *gap* criadas na Etapa 03 são as candidatas mais promissoras do grupo.
3. **~40–46% da base está inativa** por três medidas independentes. Impacto direto no
   cálculo de EV da Etapa 10, não na modelagem.

**Pendências abertas para a Análise 3.2 (multicolinearidade):** além dos 5 pares
|ρ| > 0,80 já listados na Seção 2.3 do plano, a univariada acrescentou os blocos
`availability_*` (ρ até 0,97), `maximum_nights` / `maximum_nights_cap` /
`sem_limite_maximo`, as 8 colunas de score de review, e o trio
`number_of_reviews` / `reviews_per_month` / `reviews_por_mes_ativo`.

## Consolidação da Análise Univariada — Quadro de Decisões

Esta célula reúne todas as observações das Análises 1.1 a 1.9 e as converte em três
listas de ação. **Nada aqui é executado ainda** — as remoções de coluna dependem da
Análise 3.2 (multicolinearidade) e da Etapa 05 (Boruta). É o registro consolidado que
fecha a univariada.

---

### Parte 1 — Anotações por grupo (resumo)

| Grupo | Cols | Alertas automáticos | Achado principal |
|---|---|---|---|
| **G0** — Alvo (`price`) | 1 | — | skew 5,38 → **0,39** com `log1p`; MAPE 31,52% → **28,53%** |
| **G1** — Capacidade | 12 | nenhum | `bedrooms == 0` são **estúdios** (1.006 anúncios), não dados faltantes |
| **G2** — Geografia | 6 | nenhum | Preço cai monotonicamente com a distância (€ 160 → € 100); `neighbourhood_cleansed` tem cauda fina (8 de 22 bairros < 1,5%) |
| **G3** — Amenities | 13 | nenhum | `tem_elevator` tem sinal **invertido** por confusão geográfica; controlado por distância, inverte de volta |
| **G4** — Host e operação | 10 | 1 NULOS-ALTOS<br>2 CAUDA-EXTREMA | Nulo de `host_response_time` é **informativo** (8,8% vs 22,3% de superhost) |
| **G5** — Política e custos | 16 | 1 NULOS-ALTOS<br>1 CAUDA-EXTREMA | **NaN ≠ zero** em `security_deposit` e `cleaning_fee` (€ 8,50 de diferença mediana) |
| **G6** — Reviews | 17 | 4 CAUDA-EXTREMA | **Efeito teto severo** (até 72,8% na nota máxima); 224 anúncios têm review mas não têm nota |
| **G7** — Disponib. e texto | 7 | 2 CAUDA-EXTREMA | `len_description` **truncada** em 1.000 chars; bloco `availability_*` com ρ até 0,97 |

**Nenhuma coluna** do dataset foi sinalizada como VARIÂNCIA-NULA ou QUASE-CONSTANTE.

---

### Parte 2 — Colunas a remover

**Removidas agora, por decisão da univariada: NENHUMA.**

Isto é uma conclusão, não uma omissão. Os quatro critérios automáticos
(variância nula, quase-constância, nulos altos, cauda extrema) **não condenaram nenhuma
coluna**:

- Os 2 alertas de **NULOS-ALTOS** viraram *manter e tratar*, porque o nulo provou ser
  informativo (evidência nos G4 e G5).
- Os 9 alertas de **CAUDA-EXTREMA** são assimetria natural de contagens e durações —
  resolve-se com transformação na Etapa 06, não com remoção.

**Candidatas a remoção, com decisão adiada para a Análise 3.2 (redundância):**

| Bloco | Colunas | Hipótese de resolução |
|---|---|---|
| Disponibilidade | `availability_30/60/90/365` (ρ até **0,97**) | manter `availability_365` + talvez `availability_30` (ρ = 0,77, a menos redundante) |
| Limite máximo | `maximum_nights`, `maximum_nights_cap`, `sem_limite_maximo` | manter `sem_limite_maximo` + `maximum_nights_cap` |
| Portfólio do host | `host_listings_count` × `calculated_host_listings_count` (ρ = **0,8743**) | manter `calculated_host_listings_count` (escopo coerente com a base) + `host_eh_profissional` |
| Scores de review | 6 subnotas + `review_score_medio` + `review_scores_rating` | manter as de maior variância (`value`, `location`) + os dois `gap_*` |
| Intensidade de review | `number_of_reviews`, `reviews_per_month`, `reviews_por_mes_ativo` | a definir por correlação |
| Verificação de identidade | `host_identity_verified` × `host_tem_id_verificado` | checar sobreposição conceitual |
| Taxa de limpeza | `cleaning_fee` × `taxa_limpeza_por_noite` (derivada) | a definir por correlação |

> **Por que não remover agora:** remover por correlação exige a matriz completa, que é o
> objeto da Análise 3.2. Além disso, o Boruta (Etapa 05) decide com o modelo completo —
> o caso `tem_elevator` do G3 mostra que julgar uma feature isoladamente engana.

---

### Parte 3 — Linhas a remover

**Removidas agora: NENHUMA linha adicional.**

Todas as remoções de linha já foram executadas nas etapas anteriores:

| Momento | Critério | Linhas | Base resultante |
|---|---|---|---|
| Etapa 02 | Hold-out de teste (10%, CLAUDE.md §5) | −2.003 | 18.027 |
| Etapa 02 | `price == 0` (MAPE indefinido) | −7 | 18.020 |
| Etapa 04 — Análise 1.1.b | Outliers por razão de mercado (>5× ou <0,25× os comparáveis, em grupos com n ≥ 10) | **−42** | **17.978** |

**Sobre os extremos de preço remanescentes (€ < 20 e € > 1.000):** a Seção 9 do plano
deixou esta pendência em aberto. **Recomendação: não remover.** A Análise 1.1.b já
atacou o problema pelo critério correto — desvio em relação aos *comparáveis de mercado*,
não valor absoluto. Um imóvel de € 1.500 que cobra 1,05× a mediana do seu grupo não é
outlier, é um imóvel caro num segmento caro. Cortar por limiar absoluto removeria imóveis
legítimos e manteria erros de digitação em grupos baratos.

**Anúncios inativos — não remover, mas segmentar no EV.** Três medidas independentes
apontam a mesma população:

| Medida | Grupo | % da base |
|---|---|---|
| `calendar_atualizado_dias` > 365 | G5 | 15,4% |
| `anuncio_ativo` = 0 | G6 | 42,2% |
| `availability_365` = 0 | G7 | 46,4% |

Para a **modelagem** são linhas válidas (o preço declarado é um dado real). Para o
**recomendador da Etapa 09/10** são um problema: recomendar aumento de preço a um
anúncio que não está operando não gera receita. Ver Parte 4, item E10-1.

---

### Parte 4 — Observações registradas para etapas futuras

#### Para a Etapa 06 (Data Preparation)

| # | Coluna(s) | Observação | Ação |
|---|---|---|---|
| E06-1 | `security_deposit`, `cleaning_fee` | NaN e 0,00 são estados **distintos** (€ 116,50 vs € 125,00 no depósito). A decisão 3 da Etapa 02 se confirma correta. | Nulo como **categoria explícita**. Nunca imputar zero. |
| E06-2 | `host_response_time` | 47,3% de nulos, mas o nulo marca anfitrião de baixa atividade (8,8% de superhost vs 22,3%). | Categoria `"sem_histórico"`. Não imputar pela moda. |
| E06-3 | `pessoas_por_quarto`, `camas_por_quarto` | Nulos (5,6% / 5,7%) significam **"não se aplica"** (estúdios), não "desconhecido". | Criar flag `eh_estudio` e imputar valor coerente. |
| E06-4 | `review_scores_*` | Dois nulos de **causas diferentes**: `sem_reviews` = 1 (2.131) vs "tem review, sem nota" (224). | Tratamento distinto para cada grupo. |
| E06-5 | `review_scores_rating` (0–100) e as 6 subnotas (0–10) | Escalas confirmadas: min 20, max 100, mediana 97. | **Não reescalar.** Pegadinha da safra (CLAUDE.md §7). |
| E06-6 | `neighbourhood_cleansed` | 22 categorias, 8 com < 1,5%; menor tem 90 anúncios. One-hot gera colunas vazias; target encoding puro superajusta. | Target encoding com **suavização bayesiana** ou agrupar cauda por proximidade. |
| E06-7 | `len_description` | Truncada em 1.000 chars (60,2% no teto, zero acima). Não mede tamanho real. | Binária (`atingiu_limite`) ou binning, não contínua. |
| E06-8 | 9 colunas com CAUDA-EXTREMA | Assimetria natural de contagens/durações, não erro. | Log ou binning. Sem remoção. |
| E06-9 | Todas as transformações com parâmetro | CLAUDE.md §5.4 — vazamento invalida a métrica final. | `fit` só no treino; `transform` no teste. |

#### Para a Análise 2.1 (bivariada) e Etapa 05 (Feature Selection)

| # | Observação |
|---|---|
| E05-1 | **Alerta metodológico do G3:** `tem_elevator` tem sinal univariado de −9,4%, mas controlado por distância **inverte para +21% no centro**. O ranking univariado da Análise 2.1 é um *mapa*, não um veredito. Não descartar feature por associação marginal fraca. |
| E05-2 | **Efeito teto nas reviews:** até 72,8% na nota máxima limita o poder discriminante. `gap_location_vs_geral` e `gap_value_vs_geral` existem para extrair sinal desse teto — verificar se cumprem o papel. |
| E05-3 | `room_type` é praticamente binária na prática (79,2% *Entire home/apt*, `Shared room` quase inexistente). |
| E05-4 | Anúncios **sem review são € 14 mais caros** (€ 139 vs € 125) — coerente com o padrão de bloqueio de calendário identificado na Análise 1.1.b. Parte desse preço não é preço de mercado. |

#### Para as Etapas 09 e 10 (Métricas, Recomendador e Resultado Final)

| # | Observação |
|---|---|
| **E10-1** | **Anúncios inativos distorcem o EV.** 15,4% sem atualizar o calendário há mais de um ano; 42,2% com `anuncio_ativo` = 0. Recomendar preço a quem não opera não gera receita. **Recomendação:** aplicar filtro de atividade no cálculo do EV, ou reportar o EV **segmentado** entre ativos e inativos. Afeta diretamente a comparação com a Etapa 01. |
| E09-1 | **Dívida registrada (CLAUDE.md §5.4):** `test_raw.csv` está cru (96 colunas). Toda a limpeza da Etapa 02 e todo o feature engineering da Etapa 03 precisam ser reaplicados **na mesma ordem e com os mesmos parâmetros**. |
| E09-2 | `availability_365` = 0 (46,4%) é ambíguo: *"totalmente reservado"* ou *"calendário fechado"*. Segue vedado como proxy de ocupação (CLAUDE.md §4). O modelo do InsideAirbnb via `reviews_per_month` permanece a fonte. |
| E09-3 | MAPE deve ser sempre calculado na **escala original**, após `expm1`. `expm1(mean(log(y)))` devolve a **mediana condicional** — coerente com a metodologia travada da Etapa 01 (CLAUDE.md §3.1). |

---

# Análise Bivariada (Seção 4 do plano)

**Objetivo:** relação de cada explicativa com o alvo `price`. Responde:
*"esta coluna ajuda a prever preço?"*

**Nota metodológica travada (Seção 4 do plano).** Todas as análises bivariadas usam
`price` na escala definida na Análise 1.2 — **`log1p`**. Correlações medidas na escala
original são dominadas pela cauda superior e não refletem a relação na faixa central,
justamente a que importa para o MAPE.

> **Lembrete de orientação (Contexto do plano).** O MAPE é erro *percentual*: errar € 20
> num anúncio de € 60 custa 33%; os mesmos € 20 num anúncio de € 400 custam 5%. A leitura
> de toda esta seção privilegia a **cauda inferior de preços** — é onde o modelo mais
> destrói a métrica e onde está a massa acionável do EV da Etapa 01.

**Advertência herdada da univariada (G3).** O caso `tem_elevator` mostrou que o ranking
univariado de associação **pode enganar na presença de confusão**: sinal marginal de
−9,4% que, controlado por distância, inverte para +21% no centro. Portanto:

> O ranking da Análise 2.1 é um **mapa**, não um veredito. Nenhuma feature é descartada
> aqui por associação marginal fraca — o portão formal é o Boruta (Etapa 05), que decide
> com o modelo completo.

**Roteiro desta seção:**

| Análise | Conteúdo | Plano |
|---|---|---|
| **2.1** | Ranking geral de associação (81 explicativas) | §4.1 |
| **2.2 – 2.8** | Detalhe por grupo (G1 → G7) + 8 hipóteses de negócio | §4.2 |
| **2.9** | Dispersão por faixa de preço — piso de MAPE alcançável | §4.3 |

In [ ]:
# Análise 2.1 — funções de associação com o alvo
# Cada tipo de coluna exige um teste diferente; o ranking final é unificado por |efeito|.
from scipy import stats

TARGET_LOG = 'price_log'
if TARGET_LOG not in df03.columns:
    df03[TARGET_LOG] = np.log1p(df03[TARGET])

def _assoc_continua(col):
    """Spearman (monotônica, escala original) + Pearson (linear, em log1p)."""
    _ok = df03[col].notna() & df03[TARGET_LOG].notna()
    if _ok.sum() < 30 or df03.loc[_ok, col].nunique() < 3:
        return None
    rho, p_rho = stats.spearmanr(df03.loc[_ok, col], df03.loc[_ok, TARGET])
    r, _ = stats.pearsonr(df03.loc[_ok, col], df03.loc[_ok, TARGET_LOG])
    return {
        'coluna': col, 'tipo': 'continua', 'n': int(_ok.sum()),
        'spearman': rho, 'pearson_log': r,
        'delta': abs(abs(rho) - abs(r)),   # divergência alta => relação não-linear
        'efeito': abs(rho), 'p_valor': p_rho,
    }

def _assoc_binaria(col):
    """Mann-Whitney + diferença percentual de mediana entre as classes 0 e 1."""
    g0 = df03.loc[df03[col] == 0, TARGET].dropna()
    g1 = df03.loc[df03[col] == 1, TARGET].dropna()
    if len(g0) < 30 or len(g1) < 30:
        return None
    u, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    med0, med1 = g0.median(), g1.median()
    # r de rank-biserial: efeito padronizado, comparável com |rho| das contínuas
    r_rb = 2 * u / (len(g0) * len(g1)) - 1
    return {
        'coluna': col, 'tipo': 'binaria', 'n': len(g0) + len(g1),
        'mediana_0': med0, 'mediana_1': med1,
        'dif_pct': (med1 - med0) / med0 * 100,
        'efeito': abs(r_rb), 'p_valor': p,
    }

def _assoc_categorica(col):
    """Kruskal-Wallis + eta^2 (razão de correlação) sobre price_log."""
    grupos = [g[TARGET_LOG].dropna().values
              for _, g in df03.groupby(col, observed=True) if len(g) >= 30]
    if len(grupos) < 2:
        return None
    h, p = stats.kruskal(*grupos)
    # eta^2 a partir da soma de quadrados entre/total
    _vals = np.concatenate(grupos)
    _media_geral = _vals.mean()
    ss_entre = sum(len(g) * (g.mean() - _media_geral) ** 2 for g in grupos)
    ss_total = ((_vals - _media_geral) ** 2).sum()
    eta2 = ss_entre / ss_total if ss_total > 0 else np.nan
    return {
        'coluna': col, 'tipo': 'categorica', 'n': len(_vals),
        'n_categorias': len(grupos), 'eta2': eta2,
        'efeito': np.sqrt(eta2),   # eta = correlação, mesma unidade de |rho|
        'p_valor': p,
    }

print('Funções de associação definidas.')
print(f'  {TARGET_LOG} criada: skew {df03[TARGET_LOG].skew():.2f} '
      f'(vs {df03[TARGET].skew():.2f} na escala original)')
print('  contínuas  -> Spearman + Pearson(log) + delta')
print('  binárias   -> Mann-Whitney + dif. de mediana + rank-biserial')
print('  categóricas-> Kruskal-Wallis + eta² (efeito = eta)')

In [ ]:
# Análise 2.1 — ranking unificado das 81 explicativas
_linhas, _puladas = [], []

for _grupo, _cols in GRUPOS.items():
    for _c in _cols:
        if _c not in df03.columns or _c in (TARGET, TARGET_LOG, 'id'):
            continue
        _t = tipo_coluna(_c)
        _f = {'continua': _assoc_continua, 'binaria': _assoc_binaria,
              'categorica': _assoc_categorica}.get(_t)
        _r = _f(_c) if _f else None
        if _r is None:
            _puladas.append(_c)
        else:
            _r['grupo'] = _grupo.split(' — ')[0]
            _linhas.append(_r)

ranking = (pd.DataFrame(_linhas)
           .sort_values('efeito', ascending=False)
           .reset_index(drop=True))
ranking.index += 1

print(f'Ranking de associação com o alvo — {len(ranking)} explicativas avaliadas')
if _puladas:
    print(f'  (puladas por n insuficiente: {_puladas})')
print('=' * 100)
print('\n>> TOP 20 — maior associação')
_cols_show = ['coluna', 'grupo', 'tipo', 'efeito', 'spearman', 'pearson_log',
              'delta', 'dif_pct', 'eta2', 'p_valor']
print(ranking.head(20)[_cols_show].to_string(
    float_format=lambda v: f'{v:.4f}' if abs(v) < 1000 else f'{v:.1f}'))

print('\n>> BOTTOM 10 — menor associação')
print(ranking.tail(10)[_cols_show].to_string(
    float_format=lambda v: f'{v:.4f}' if abs(v) < 1000 else f'{v:.1f}'))

In [ ]:
# Análise 2.1 — visualização do ranking + leitura de significância
_ALFA = 0.05
_cor_grupo = {'G1': '#4C72B0', 'G2': '#DD8452', 'G3': '#55A868', 'G4': '#C44E52',
              'G5': '#8172B3', 'G6': '#937860', 'G7': '#DA8BC3'}

fig, axes = plt.subplots(1, 2, figsize=(17, 9))

# --- painel 1: top 30
_top = ranking.head(30).iloc[::-1]
axes[0].barh(_top['coluna'], _top['efeito'],
             color=[_cor_grupo[g] for g in _top['grupo']])
axes[0].set_title('Top 30 — associação com price', fontweight='bold')
axes[0].set_xlabel('|efeito| (ρ de Spearman, rank-biserial ou η)')
axes[0].tick_params(labelsize=8)
axes[0].grid(axis='x', alpha=0.3)

# --- painel 2: efeito medio por grupo
_por_grupo = (ranking.groupby('grupo')['efeito']
              .agg(['mean', 'max', 'count'])
              .sort_values('mean', ascending=True))
axes[1].barh(_por_grupo.index, _por_grupo['mean'],
             color=[_cor_grupo[g] for g in _por_grupo.index], alpha=0.85)
axes[1].scatter(_por_grupo['max'], range(len(_por_grupo)),
                color='black', zorder=3, marker='D', s=45, label='máximo do grupo')
axes[1].set_title('Efeito médio e máximo por grupo temático', fontweight='bold')
axes[1].set_xlabel('|efeito|')
axes[1].legend()
axes[1].grid(axis='x', alpha=0.3)
for i, (_, r) in enumerate(_por_grupo.iterrows()):
    axes[1].text(r['mean'] + 0.004, i, f"n={int(r['count'])}", va='center', fontsize=8)

plt.tight_layout()
plt.show()

print('Significância estatística e força prática')
print('=' * 72)
_ns = ranking[ranking['p_valor'] >= _ALFA]
print(f'  Não significativas (p >= {_ALFA}): {len(_ns)} de {len(ranking)}')
if len(_ns):
    print('    ' + ', '.join(_ns['coluna'].tolist()))

print('\n  Distribuição da força do efeito (n significativas):')
_sig = ranking[ranking['p_valor'] < _ALFA]
for _rot, _lo, _hi in [('forte    (>= 0,30)', 0.30, 1.01),
                       ('moderado (0,15–0,30)', 0.15, 0.30),
                       ('fraco    (0,05–0,15)', 0.05, 0.15),
                       ('mínimo   (< 0,05)', 0.0, 0.05)]:
    _n = ((_sig['efeito'] >= _lo) & (_sig['efeito'] < _hi)).sum()
    print(f'    {_rot}: {_n:>3d}')

print('\n  Efeito médio por grupo (ordenado):')
print(_por_grupo.sort_values('mean', ascending=False).round(4).to_string())

**Observações — Análise 2.1 (Ranking geral de associação)**

**1. O tamanho do imóvel domina. G1 é o grupo mais preditivo, com folga.**

| Grupo | Efeito médio | Máximo | n |
|---|---|---|---|
| **G1** — Capacidade | **0,307** | 0,548 (`accommodates`) | 12 |
| G2 — Geografia | 0,183 | 0,307 (`neighbourhood_cleansed`) | 6 |
| G3 — Amenities | 0,182 | 0,336 (`tem_tv`) | 13 |
| G5 — Política e custos | 0,135 | 0,447 (`cleaning_fee`) | 16 |
| G7 — Disponib. e texto | 0,098 | 0,149 | 7 |
| G6 — Reviews | 0,076 | 0,214 (`review_scores_location`) | 17 |
| **G4** — Host e operação | **0,024** | 0,055 | 10 |

O top 5 é `accommodates` (0,548), `bedrooms` (0,495), `beds` (0,488), `cleaning_fee` (0,447)
e `room_type` (η = 0,419). **Quatro dos cinco são G1** — o preço em Amsterdam é, antes de
tudo, uma função de quantas pessoas cabem no imóvel.

**2. G4 (Host e operação) é o grupo mais fraco de todos — efeito médio de 0,024.**
Isso contradiz a intuição de negócio de que "anfitrião melhor cobra mais". Das 7 colunas
não significativas do dataset (p ≥ 0,05), **4 são do G4**: `host_is_superhost`,
`host_identity_verified`, `host_eh_local` e `n_host_verifications`. Ver item 4.

**3. Distribuição da força do efeito (74 significativas de 81):**

| Faixa | n |
|---|---|
| forte (≥ 0,30) | 9 |
| moderado (0,15–0,30) | 22 |
| fraco (0,05–0,15) | 31 |
| mínimo (< 0,05) | 12 |
| **não significativas** | **7** |

**Nenhuma coluna é descartada aqui.** As 7 não significativas seguem para a Etapa 05 —
a advertência do G3 vale integralmente (ver item 4, que é a prova).

---

**4. H3 (Superhost) — REJEITADA na forma simples, CONFIRMADA sob controle.**

O plano pergunta: *"Superhosts cobram mais, controlando por capacidade?"*. O ranking
coloca `host_is_superhost` em **último lugar entre 81 colunas** (efeito 0,0012, p = 0,92),
com medianas idênticas:

| | price mediano |
|---|---|
| não-superhost | € 125,00 |
| superhost | € 125,00 |

Parece que ser superhost não vale nada. **Mas o plano pede o controle — e ele muda tudo:**

| Segmento | não-SH | superhost | Δ | p-valor |
|---|---|---|---|---|
| accommodates = 1 | € 58,00 | € 55,00 | −5,2% | 0,49 |
| accommodates = 2 | € 109,00 | € 110,00 | +0,9% | 0,39 |
| accommodates = 3 | € 122,50 | € 126,00 | +2,9% | 0,055 |
| accommodates = 4 | € 175,00 | € 180,00 | +2,9% | **0,0001** |
| accommodates = 6 | € 250,00 | € 273,00 | +9,2% | 0,18 |
| **Entire home/apt, acc = 2** | € 119,50 | € 125,00 | **+4,6%** | **< 0,0001** |
| **Entire home/apt, acc = 4** | € 175,00 | € 190,00 | **+8,6%** | **< 0,0001** |

**Veredito: H3 se confirma quando o confundidor é removido.** Controlando por capacidade
*e* tipo de quarto, o prêmio de superhost é de +4,6% a +8,6%, com p < 0,0001.

**Por que o efeito marginal é zero.** Superhosts (15,9% da base) estão
sobre-representados em imóveis pequenos e quartos privativos — segmentos de preço baixo.
A composição do grupo cancela o prêmio: o superhost cobra mais *dentro* do seu segmento,
mas o mix de segmentos puxa a mediana global para baixo. É o **paradoxo de Simpson**,
o mesmo mecanismo do `tem_elevator` no G3, e a segunda vez que ele aparece nesta EDA.

> **Consequência metodológica, e a razão de nenhuma coluna ser descartada aqui.**
> Se tivéssemos usado o ranking como critério de corte, `host_is_superhost` — a última
> colocada entre 81 — teria sido eliminada. E ela carrega um efeito real de até +8,6%
> em segmentos que somam mais de 5.000 anúncios. **O ranking mede associação marginal;
> modelos de árvore capturam o efeito condicional.** O Boruta (Etapa 05) é o portão.

**5. Divergência Pearson × Spearman — sinais de não-linearidade.**
Os maiores deltas apontam relações que um modelo linear não captura:
`host_listings_count` (0,077 — e com **sinais opostos**: ρ = −0,018, r = +0,094),
`dias_desde_ultima_review` (0,072), `security_deposit` (0,055) e `bathrooms` (0,041).
São candidatas a transformação (log/binning) na Etapa 06 e argumento adicional a favor
de modelos não-lineares na Etapa 07.

**6. `cleaning_fee` em 4º lugar (0,447) merece atenção na Etapa 09.**
É uma variável de **custo cobrado ao hóspede**, não uma característica física do imóvel.
Sua associação alta é esperada (imóveis caros cobram limpeza cara), mas para o
recomendador ela levanta uma questão: o anfitrião controla essa variável. Registrado
para a Etapa 09 — usá-la como preditora é legítimo, mas ela não deve ser tratada como
"característica dada" na recomendação de preço.

In [ ]:
# Análises 2.2 a 2.8 — funções reutilizáveis de plotagem bivariada
from statsmodels.nonparametric.smoothers_lowess import lowess

_N_AMOSTRA_SCATTER = 4000   # 18k pontos viram mancha; amostra preserva a forma
_rng = np.random.default_rng(42)

def plotar_bivariada_continuas(cols, max_paineis=4):
    """Scatter vs price_log com tendência LOWESS. Usa hexbin quando muito denso."""
    cols = [c for c in cols if c in df03.columns and tipo_coluna(c) == 'continua']
    for i in range(0, len(cols), max_paineis):
        bloco = cols[i:i + max_paineis]
        fig, axes = plt.subplots(1, len(bloco), figsize=(4.6 * len(bloco), 4))
        axes = np.atleast_1d(axes)
        for ax, c in zip(axes, bloco):
            _ok = df03[c].notna()
            x_all, y_all = df03.loc[_ok, c].values, df03.loc[_ok, TARGET_LOG].values
            # corta no p99 para o eixo não ser dominado pela cauda
            _lim = np.nanpercentile(x_all, 99)
            _m = x_all <= _lim
            x_all, y_all = x_all[_m], y_all[_m]
            idx = (_rng.choice(len(x_all), _N_AMOSTRA_SCATTER, replace=False)
                   if len(x_all) > _N_AMOSTRA_SCATTER else np.arange(len(x_all)))
            ax.scatter(x_all[idx], y_all[idx], s=4, alpha=0.12, color='#4C72B0')
            if len(x_all) > 50:
                _sm = lowess(y_all, x_all, frac=0.3, it=0, return_sorted=True)
                ax.plot(_sm[:, 0], _sm[:, 1], color='#C44E52', lw=2.2, label='LOWESS')
                ax.legend(fontsize=8)
            ax.set_xlabel(c, fontsize=9)
            ax.set_ylabel('log1p(price)', fontsize=9)
            ax.set_title(f'{c}\n(até o p99)', fontsize=10)
            ax.grid(alpha=0.25)
        plt.tight_layout()
        plt.show()

def plotar_bivariada_discretas(cols, max_paineis=4):
    """Boxplot de price por classe (binárias) ou categoria (categóricas, ordenado)."""
    cols = [c for c in cols if c in df03.columns
            and tipo_coluna(c) in ('binaria', 'categorica')]
    for i in range(0, len(cols), max_paineis):
        bloco = cols[i:i + max_paineis]
        fig, axes = plt.subplots(1, len(bloco), figsize=(4.6 * len(bloco), 4.2))
        axes = np.atleast_1d(axes)
        for ax, c in zip(axes, bloco):
            _t = tipo_coluna(c)
            if _t == 'binaria':
                ordem, cor = [0, 1], '#DD8452'
            else:
                # ordena categorias pela mediana; só as com massa crítica
                _med = (df03.groupby(c, observed=True)[TARGET].median()
                        .loc[lambda s: df03[c].value_counts().reindex(s.index) >= 30]
                        .sort_values())
                ordem, cor = _med.index.tolist(), '#55A868'
            dados = [df03.loc[df03[c] == v, TARGET].dropna().values for v in ordem]
            bp = ax.boxplot(dados, patch_artist=True, showfliers=False,
                            medianprops=dict(color='black', lw=1.6))
            for p in bp['boxes']:
                p.set_facecolor(cor)
                p.set_alpha(0.75)
            ax.set_xticklabels([str(v)[:14] for v in ordem], rotation=30,
                               ha='right', fontsize=8)
            for j, d in enumerate(dados, start=1):
                ax.text(j, ax.get_ylim()[1] * 0.97, f'n={len(d)}',
                        ha='center', fontsize=7, color='#555')
            ax.set_ylabel('price (€)', fontsize=9)
            ax.set_title(c, fontsize=10)
            ax.grid(axis='y', alpha=0.25)
        plt.tight_layout()
        plt.show()

def resumo_bivariado(grupo):
    """Recorte do ranking da 2.1 para o grupo, já ordenado."""
    g = grupo.split(' — ')[0]
    cols = ['coluna', 'tipo', 'efeito', 'spearman', 'pearson_log', 'delta',
            'dif_pct', 'eta2', 'p_valor']
    r = ranking[ranking['grupo'] == g][cols].reset_index(drop=True)
    r.index += 1
    return r

print('Funções bivariadas definidas.')
print(f'  scatter: amostra de {_N_AMOSTRA_SCATTER} pontos + LOWESS (frac=0.3), corte no p99')
print('  boxplots: categóricas ordenadas por mediana, só categorias com n >= 30')

In [ ]:
# Análise 2.2 — G1: Capacidade e estrutura física
_G = 'G1 — Capacidade e estrutura física'
display(resumo_bivariado(_G))
plotar_bivariada_continuas(GRUPOS[_G])
plotar_bivariada_discretas(GRUPOS[_G])

**Observações bivariadas — G1: Capacidade e estrutura física**

**H2 — CONFIRMADA com folga.** *`Entire home/apt` tem preço mediano maior que `Private room`.*

| | n | mediana |
|---|---|---|
| Entire home/apt | 14.242 | **€ 140,00** |
| Private room | 3.677 | € 85,00 |

Diferença de **+64,7%** (p < 1e-300). E o efeito **não é artefato de tamanho** — controlando
por `accommodates`, o prêmio persiste em todas as faixas:

| accommodates | Entire | Private | Δ | p |
|---|---|---|---|---|
| 1 | € 83,00 | € 55,00 | **+50,9%** | 1,3e-07 |
| 2 | € 120,00 | € 83,00 | **+44,6%** | < 1e-300 |
| 3 | € 130,00 | € 94,00 | **+38,3%** | 4,0e-32 |
| 4 | € 175,00 | € 140,00 | **+25,0%** | 6,0e-26 |

O prêmio **decresce** conforme o imóvel cresce (50,9% → 25,0%): privacidade vale mais no
segmento pequeno. Isso é uma interação `room_type × accommodates` que modelos de árvore
capturam naturalmente.

---

**H6 — CONFIRMADA parcialmente.** *`n_amenities` tem relação positiva com retorno decrescente.*

| faixa de `n_amenities` | mediana | Δ vs. anterior |
|---|---|---|
| ≤ 12 | € 110,00 | — |
| 13–15 | € 120,00 | +9,1% |
| 16–18 | € 125,00 | +4,2% |
| 19–22 | € 133,00 | +6,4% |
| 23–28 | € 135,00 | +1,5% |
| ≥ 29 | € 150,00 | +11,1% |

A relação é **positiva e monotônica**, e há desaceleração no miolo (9,1% → 1,5%). Mas o
último degrau **volta a subir** (+11,1%), o que contraria o retorno decrescente puro. A
leitura provável é que a faixa ≥ 29 concentra imóveis de luxo, onde a contagem de amenities
funciona como *proxy de padrão*, não como soma de itens. **Veredito:** relação positiva
confirmada; retorno decrescente **não** se sustenta no topo da distribuição.

---

**H7 — CONFIRMADA.** *`Boat` tem dinâmica de preço própria, distinta de `Apartment`.*

| tipo | n | mediana | CV | ρ(acc, price) |
|---|---|---|---|---|
| Houseboat | 200 | **€ 175,00** | 0,515 | +0,520 |
| Loft | 334 | € 162,50 | 0,637 | — |
| **Boat** | 344 | **€ 160,00** | 0,624 | **+0,605** |
| Townhouse | 578 | € 159,50 | 0,757 | — |
| House | 1.356 | € 150,00 | 0,718 | — |
| **Apartment** | 14.016 | € 125,00 | 0,573 | +0,511 |
| Bed and breakfast | 411 | € 115,00 | **1,224** | — |

`Boat` é **+28,0%** mais caro que `Apartment` (p = 4,8e-17) e `Houseboat` **+40,0%**
(p = 1,1e-18). A "dinâmica própria" aparece na **elasticidade ao tamanho**: ρ = +0,605 para
`Boat` contra +0,511 para `Apartment` — o preço do barco responde mais fortemente à
capacidade. **Veredito: confirmada.** Os dois tipos náuticos formam um segmento à parte do
mercado de Amsterdam.

**Nota adicional:** `Bed and breakfast` tem CV = **1,22**, mais que o dobro de `Apartment`
(0,57). É a categoria de preço mais imprevisível da base — relevante para a Análise 2.9
(piso de MAPE).

---

**Achado não previsto no plano: quebra de monotonicidade em `accommodates`.**

O LOWESS mostra uma **queda visível em 4 pessoas**. O preço por pessoa explica:

| accommodates | n | mediana | **€ por pessoa** |
|---|---|---|---|
| 1 | 377 | € 56,00 | **€ 56,00** |
| 2 | 9.724 | € 110,00 | **€ 55,00** |
| 3 | 1.505 | € 125,00 | € 41,67 |
| 4 | 5.476 | € 175,00 | € 43,75 |
| 6 | 361 | € 250,00 | € 41,67 |
| 8 | 85 | € 300,00 | € 37,50 |

Há **dois regimes**: até 2 pessoas o preço por pessoa fica em ~€ 55; de 3 em diante cai para
~€ 40 e estabiliza. A transição 2 → 3 é uma **quebra estrutural**, não uma curva suave.
O mercado precifica "casal" e "grupo" de formas distintas.

**Consequência para a Etapa 06/07:** uma regressão linear em `accommodates` erra nos dois
extremos. A feature `price/accommodates` já existe implicitamente nas razões criadas na
Etapa 03; modelos de árvore capturam a quebra sem transformação. Registrado como argumento
adicional a favor dos modelos não-lineares da Etapa 07.

---

**Estúdios (`bedrooms = 0`) valem MENOS que 1 quarto, controlando por capacidade.**

A univariada (G1) identificou os 1.006 estúdios como imóveis reais. A bivariada mede o efeito:

| bedrooms | n | mediana | acc mediana |
|---|---|---|---|
| 0 (estúdio) | 1.006 | € 100,00 | 2 |
| 1 | 10.654 | € 110,00 | 2 |
| 2 | 4.339 | € 165,00 | 4 |

Controlando por `accommodates = 2`: estúdio € 100,00 vs. 1 quarto € 110,00 (**−9,1%**,
p = 0,0012). A ausência de quarto separado tem penalidade real e mensurável. **Confirma a
ação E06-3** da consolidação univariada: criar a flag `eh_estudio` em vez de tratar como NaN.

---

**Demais colunas do grupo.** `banheiros_por_pessoa` tem ρ = **−0,397** (sinal negativo
esperado: mais banheiros por pessoa = imóvel menor e mais barato por pessoa).
`tem_multiplos_banheiros` mostra **+45,8%** de diferença de mediana — o maior efeito binário
do dataset inteiro. `pessoas_por_quarto` é a mais fraca do grupo (efeito 0,07) e tem delta
Pearson/Spearman com sinais opostos, sinalizando não-linearidade.

In [ ]:
# Análise 2.3 — G2: Geografia (H1)
_G = 'G2 — Geografia'
display(resumo_bivariado(_G))
plotar_bivariada_continuas(GRUPOS[_G])
plotar_bivariada_discretas(GRUPOS[_G])

print('H1 — Preco cai monotonicamente com a distancia ao centro?')
print('=' * 72)
_b = pd.cut(df03['dist_centro_km'], [0, 1, 2, 3, 5, 8, 100],
            labels=['0-1km', '1-2km', '2-3km', '3-5km', '5-8km', '>8km'])
_t = df03.groupby(_b, observed=True)['price'].agg(['size', 'median'])
_t['delta_%'] = _t['median'].pct_change() * 100
print(_t.round(2).to_string())
_mono = _t['median'].is_monotonic_decreasing
print(f'\n  Monotonicamente decrescente? {_mono}')
print(f'  Spearman(dist_centro_km, price) = {ranking.loc[ranking["coluna"]=="dist_centro_km","spearman"].values[0]:+.4f}')

print('\n  >> Controlando por room_type (o efeito e geografico ou de composicao?):')
for _rt in ['Entire home/apt', 'Private room']:
    _sub = df03[df03['room_type'] == _rt]
    _r, _p = stats.spearmanr(_sub['dist_centro_km'], _sub['price'])
    _t2 = _sub.groupby(pd.cut(_sub['dist_centro_km'], [0,1,3,8,100],
                              labels=['0-1','1-3','3-8','>8']), observed=True)['price'].median()
    print(f'    {_rt:<16s} rho={_r:+.3f} (p={_p:.1e}) | ' +
          ' | '.join(f'{k}: €{v:.0f}' for k, v in _t2.items()))

**Observações bivariadas — G2: Geografia**

**H1 — CONFIRMADA.** *O preço cai monotonicamente com a distância ao centro.*

| faixa | n | mediana | Δ |
|---|---|---|---|
| 0–1 km | 1.850 | **€ 160,00** | — |
| 1–2 km | 4.501 | € 139,00 | −13,1% |
| 2–3 km | 5.957 | € 122,00 | −12,2% |
| 3–5 km | 4.352 | € 119,00 | −2,5% |
| 5–8 km | 1.036 | € 100,00 | −16,0% |
| > 8 km | 282 | **€ 96,50** | −3,5% |

Decrescimento **estritamente monotônico** confirmado programaticamente
(`is_monotonic_decreasing = True`), ρ = **−0,252**. Do centro à periferia o preço cai
**40%**. Mas há três nuances que a hipótese simples não captura.

---

**Nuance 1 — a curva não é linear: há um platô entre 4 e 8 km.**

O LOWESS achata visivelmente após ~4 km. Medindo a correlação *dentro* de cada faixa:

| faixa | n | mediana | ρ interno |
|---|---|---|---|
| 0–2 km | 6.351 | € 148,00 | **−0,199** |
| 2–4 km | 9.281 | € 120,00 | −0,083 |
| 4–6 km | 1.411 | € 100,00 | **−0,020** |
| 6–8 km | 653 | € 100,00 | **+0,053** |
| 8–20 km | 282 | € 96,50 | −0,269 |

Entre 4 e 8 km a distância **deixa de discriminar** (ρ ≈ 0, e chega a inverter o sinal).
O efeito da centralidade se concentra nos primeiros 4 km; depois disso o mercado é
essencialmente plano até a periferia distante. **Uma regressão linear em `dist_centro_km`
superestima a queda na faixa intermediária.** Argumento adicional para binning
(`dist_centro_bin` já existe) ou modelos de árvore.

---

**Nuance 2 — o efeito da distância é MUITO mais forte em `Private room`.**

| room_type | ρ | 0–1 km | 1–3 km | 3–8 km | > 8 km |
|---|---|---|---|---|---|
| Entire home/apt | −0,228 | € 188 | € 139 | € 125 | **€ 135** |
| **Private room** | **−0,450** | € 120 | € 89 | € 74 | **€ 60** |

O quarto privativo tem quase o **dobro** da sensibilidade à localização (−0,450 vs −0,228).
E o apartamento inteiro **sobe de novo** acima de 8 km (€ 125 → € 135), quebrando a
monotonicidade nesse segmento — provavelmente casas maiores na periferia. **Interação
`dist_centro × room_type` confirmada**, mais uma que modelos de árvore capturam e um modelo
linear aditivo não.

---

**Nuance 3 — o bairro explica MAIS que a distância (η² = 0,094 vs 0,065).**

`neighbourhood_cleansed` supera `dist_centro_bin` em poder explicativo, e a razão fica
clara nos extremos:

| bairro | n | mediana | dist. mediana |
|---|---|---|---|
| Centrum-Oost | 1.562 | € 150,00 | 1,32 km |
| Centrum-West | 2.093 | € 150,00 | 0,77 km |
| **IJburg - Zeeburgereiland** | 403 | **€ 140,00** | **7,23 km** |
| Gaasperdam - Driemond | 116 | € 75,00 | 10,24 km |
| **Bijlmer-Centrum** | 99 | **€ 70,00** | **7,21 km** |
| Bijlmer-Oost | 90 | € 69,50 | 8,02 km |

**IJburg e Bijlmer-Centrum estão à mesma distância do centro (7,2 km) e têm preços
medianos de € 140 e € 70 — o dobro.** A distância radial não distingue os dois; o bairro
sim. Isso **confirma empiricamente o achado registrado na Seção 2.3.1 do plano**: Amsterdam
não é radial (o IJ corta a cidade, os canais formam anéis), e por isso `latitude`/`longitude`
puros superaram as distâncias no teste de R² da Etapa 03.

**Consequência para a Etapa 05:** `dist_centro_km` e `dist_museumplein_km` permanecem por
serem **interpretáveis para o negócio** — "a 1,2 km do centro" é comunicável ao anfitrião
no recomendador da Etapa 09. Mas o bairro carrega mais sinal, e `latitude`/`longitude` mais
ainda. O Boruta decide.

---

**`latitude` é a 2ª pior coluna do dataset (efeito 0,005, p = 0,50) — e isso é geografia,
não ruído.** Amsterdam se estende mais no eixo leste-oeste que norte-sul, e o IJ corta a
cidade horizontalmente. `longitude` tem efeito 6× maior (0,048), ainda fraco. Isoladamente
nenhuma das duas diz muito; **combinadas**, a árvore extrai a geometria real (R² = 0,112 na
Etapa 03, superior às três distâncias juntas). É mais um caso onde o ranking marginal
subestima o valor conjunto — o mesmo padrão de `tem_elevator` e `host_is_superhost`.

In [ ]:
# Análise 2.4 — G3: Amenities e qualidade
_G = 'G3 — Amenities e qualidade'
display(resumo_bivariado(_G))
plotar_bivariada_continuas(GRUPOS[_G])
plotar_bivariada_discretas(GRUPOS[_G])

print('Flags de amenity: efeito bruto vs. controlado por accommodates+room_type')
print('=' * 88)
print('(a univariada mostrou que tem_elevator inverte sob controle — checando as demais)')
_flags = [c for c in GRUPOS[_G] if tipo_coluna(c) == 'binaria']
_ref = df03[(df03['room_type'] == 'Entire home/apt') & (df03['accommodates'] == 2)]
_res = []
for _c in _flags:
    g0, g1 = df03.loc[df03[_c] == 0, 'price'], df03.loc[df03[_c] == 1, 'price']
    if len(g0) < 30 or len(g1) < 30: continue
    _bruto = (g1.median() - g0.median()) / g0.median() * 100
    h0, h1 = _ref.loc[_ref[_c] == 0, 'price'], _ref.loc[_ref[_c] == 1, 'price']
    if len(h0) < 30 or len(h1) < 30:
        _ctrl, _p = np.nan, np.nan
    else:
        _ctrl = (h1.median() - h0.median()) / h0.median() * 100
        _, _p = stats.mannwhitneyu(h0, h1, alternative='two-sided')
    _res.append({'flag': _c, 'inc_%': df03[_c].mean() * 100, 'bruto_%': _bruto,
                 'ctrl_%': _ctrl, 'p_ctrl': _p,
                 'inverteu': (np.sign(_bruto) != np.sign(_ctrl)) if pd.notna(_ctrl) else None})
_df = pd.DataFrame(_res).sort_values('bruto_%', ascending=False)
print(_df.round(3).to_string(index=False))
print('\n  >> Flags que INVERTEM o sinal sob controle:')
_inv = _df[_df['inverteu'] == True]
print('     ' + (', '.join(_inv['flag'].tolist()) if len(_inv) else 'nenhuma alem das ja conhecidas'))

**Observações bivariadas — G3: Amenities e qualidade**

**O efeito das amenities é largamente um PROXY DE TAMANHO.**

Este é o achado central do grupo. Comparando o efeito bruto de cada flag com o efeito
medido **dentro de um segmento homogêneo** (`Entire home/apt` + `accommodates = 2`):

| flag | incid. | bruto | **controlado** | p (ctrl) |
|---|---|---|---|---|
| `tem_tv` | 78,0% | +35,0% | **+10,1%** | < 0,001 |
| `tem_dryer` | 44,6% | +25,2% | **+8,7%** | < 0,001 |
| `tem_dishwasher` | 21,2% | +24,2% | **+5,0%** | < 0,001 |
| `tem_family_kid_friendly` | 44,4% | +23,5% | **0,0%** | 0,01 |
| `tem_private_entrance` | 27,2% | +16,7% | +0,8% | < 0,001 |
| `tem_cable_tv` | 39,5% | +15,8% | +0,8% | < 0,001 |
| `tem_washer` | 76,0% | +8,3% | +0,8% | < 0,001 |
| `tem_patio_balcony` | 16,1% | +8,0% | +0,4% | < 0,001 |
| `tem_step_free_access` | 6,3% | +4,0% | +4,2% | 0,01 |
| `tem_free_parking` | 11,7% | −4,0% | **−16,7%** | < 0,001 |
| `tem_elevator` | 9,7% | −9,4% | −8,3% | < 0,001 |

**Todas as flags encolhem drasticamente sob controle.** `tem_tv` cai de +35,0% para
+10,1%; `tem_dishwasher` de +24,2% para +5,0%. O que a univariada lia como "valor da
amenity" é, em boa parte, o efeito de que **imóveis maiores têm mais amenities e custam
mais**. Depois de fixar tamanho e tipo, sobra pouco.

**Três casos merecem destaque:**

1. **`tem_family_kid_friendly` zera completamente** (+23,5% → **0,0%**). É a única flag
   que inverte/anula o sinal. "Adequado para crianças" não é um atributo de conforto que
   se cobra — é uma consequência de o imóvel ser grande. Sozinha, não carrega valor.

2. **`tem_free_parking` PIORA sob controle** (−4,0% → **−16,7%**). Estacionamento gratuito
   está associado a imóveis **17% mais baratos** dentro do mesmo segmento. A explicação
   é geográfica, a mesma do `tem_elevator`: no centro histórico de Amsterdam não há
   estacionamento; quem oferece vaga está na periferia, onde o preço é menor.

3. **`tem_elevator` mantém o sinal negativo** (−9,4% → −8,3%) mesmo controlando por tamanho
   e tipo — porque o confundidor **não é o tamanho, é a distância**. A univariada já
   demonstrou que, controlando por faixa de distância, o sinal **inverte para +21% no
   centro**. O controle correto para esta flag é geográfico, não dimensional.

> **Consequência metodológica.** Este grupo mostra que o confundidor varia conforme a
> feature: para a maioria das amenities é o **tamanho**; para `tem_elevator` e
> `tem_free_parking` é a **localização**. Não existe um controle único que sirva para
> todas — mais um argumento a favor de modelos que capturam interações automaticamente
> (Etapa 07) em vez de correções manuais.

---

**H6 (retorno decrescente de `n_amenities`) — já respondida no G1, mas o LOWESS acrescenta.**

A curva de `n_amenities` é **suave e monotonicamente crescente**, sem quebras. `n_amenities`
(efeito 0,213) supera `n_amenities_luxo` (0,187), e o LOWESS de `n_amenities_luxo` mostra
um platô entre 1 e 3 itens de luxo, subindo só no 4. A **contagem simples é mais informativa
que a contagem de luxo** — provavelmente porque a contagem total é um proxy melhor de
"padrão geral do imóvel", enquanto a de luxo depende de quais itens foram classificados
como tal na Etapa 03.

---

**Confirmação da decisão da univariada sobre as flags de baixa incidência.**
`tem_elevator` (9,7%) e `tem_step_free_access` (6,3%) ficaram abaixo do piso de 10% do plano
da Etapa 03 e foram mantidas por argumento de negócio. A bivariada confirma:
`tem_step_free_access` tem o **menor efeito do grupo** (0,036) mas é **estável sob controle**
(+4,0% → +4,2%), e `tem_elevator` tem efeito real que só aparece condicionado à geografia.
**Ambas permanecem.** O Boruta (Etapa 05) é o portão formal.

In [ ]:
# Análise 2.5 — G4: Host e operação (H4)
_G = 'G4 — Host e operação'
display(resumo_bivariado(_G))
plotar_bivariada_continuas(GRUPOS[_G])
plotar_bivariada_discretas(GRUPOS[_G])

print('H4 — Anfitrioes profissionais precificam de forma mais HOMOGENEA?')
print('=' * 76)
print('(a hipotese e sobre VARIANCIA, nao sobre nivel de preco)')
_pro = df03['host_eh_profissional'] == 1
print(f'\n  profissionais: {_pro.sum()} ({_pro.mean():.1%}) | nao-prof: {(~_pro).sum()}')
print(f'  {"":24s} {"CV":>8s} {"IQR/mediana":>12s} {"mediana":>9s}')
for _rot, _m in [('nao-profissional', ~_pro), ('profissional', _pro)]:
    _s = df03.loc[_m, 'price']
    _cv = _s.std() / _s.mean()
    _iqr = (_s.quantile(.75) - _s.quantile(.25)) / _s.median()
    print(f'  {_rot:<24s} {_cv:>8.4f} {_iqr:>12.4f} {_s.median():>9.2f}')

# Levene: teste formal de igualdade de variancias (em log, escala da analise)
_l, _pl = stats.levene(df03.loc[~_pro, TARGET_LOG].dropna(),
                       df03.loc[_pro, TARGET_LOG].dropna())
print(f'\n  Teste de Levene (variancias iguais?): stat={_l:.2f}, p={_pl:.4e}')

print('\n  >> Controlando por segmento (room_type + accommodates):')
for _rt, _acc in [('Entire home/apt', 2), ('Entire home/apt', 4), ('Private room', 2)]:
    _seg = df03[(df03['room_type'] == _rt) & (df03['accommodates'] == _acc)]
    a = _seg.loc[_seg['host_eh_profissional'] == 0, 'price']
    b = _seg.loc[_seg['host_eh_profissional'] == 1, 'price']
    if len(a) < 30 or len(b) < 30: continue
    _, _pp = stats.levene(np.log1p(a), np.log1p(b))
    print(f'    {_rt[:14]:<14s} acc={_acc}: CV nao-prof={a.std()/a.mean():.3f} | '
          f'CV prof={b.std()/b.mean():.3f} | p(Levene)={_pp:.4f} | n={len(a)}/{len(b)}')

**Observações bivariadas — G4: Host e operação**

**H4 — REJEITADA, e o resultado é o OPOSTO do esperado.**

*Hipótese: anfitriões profissionais precificam de forma mais homogênea (menor variância).*

| | n | **CV** | IQR/mediana | mediana |
|---|---|---|---|---|
| não-profissional | 14.191 | **0,601** | 0,608 | € 125,00 |
| **profissional** | 3.787 | **0,735** | **0,792** | € 120,00 |

O anfitrião profissional tem **22% MAIS dispersão**, não menos (Levene: stat = 272,7,
p = 8,2e-61). E o efeito **persiste sob controle** de segmento:

| segmento | CV não-prof | CV prof | p (Levene) |
|---|---|---|---|
| Entire home/apt, acc = 2 | 0,349 | **0,413** | < 0,0001 |
| Entire home/apt, acc = 4 | 0,475 | **0,561** | < 0,0001 |
| Private room, acc = 2 | 0,401 | 0,424 | 0,51 (n.s.) |

**Interpretação.** A hipótese assumia que "profissional = padronizado". O dado mostra o
contrário: o gestor profissional opera um **portfólio heterogêneo** (imóveis de perfis e
faixas diferentes espalhados pela cidade), enquanto o anfitrião particular tem tipicamente
um imóvel e o precifica próximo à mediana do seu bairro. A profissionalização traz
**diversificação**, não uniformidade.

> **Consequência para a Etapa 09/10.** Isto tem impacto direto no recomendador. Se
> anfitriões profissionais têm maior dispersão intra-segmento, o **erro esperado do modelo
> (MAPE) será maior nesse grupo** — e são 21,1% da base. Vale reportar o MAPE segmentado
> por `host_eh_profissional` na Etapa 09, junto com a segmentação por atividade já
> registrada (E10-1).

---

**Confirmação: G4 é o grupo mais fraco do dataset (efeito médio 0,024).**

Das 10 colunas, **4 não são significativas** (p ≥ 0,05): `host_eh_local` (0,20),
`n_host_verifications` (0,15), `host_identity_verified` (0,72) e `host_is_superhost` (0,92).
Os boxplots confirmam visualmente: as caixas de `host_is_superhost` e
`host_identity_verified` são praticamente **idênticas** entre as classes 0 e 1.

Mas a Análise 2.1 já demonstrou que isso **não significa ausência de sinal**:
`host_is_superhost`, a última colocada entre 81, revela prêmio de **+4,6% a +8,6%**
(p < 0,0001) quando controlada por `room_type` × `accommodates`. O mesmo mecanismo de
confusão (paradoxo de Simpson) opera aqui.

---

**`host_listings_count`: LOWESS revela a não-linearidade que o delta sinalizou.**

Esta coluna tem o **maior delta Pearson/Spearman do dataset** (0,077), com **sinais opostos**
(ρ = −0,018, r = +0,094). O scatterplot explica: a curva **sobe abruptamente** no início
(1 → ~5 imóveis), **despenca** e depois **estabiliza** em platô. Não é monotônica — é uma
curva em forma de pico seguida de platô.

A leitura de negócio: o salto de 1 para poucos imóveis corresponde ao anfitrião que
profissionalizou e investiu em imóveis melhores. Acima de ~20 imóveis, entra a gestora de
escala com portfólio padronizado de baixo ticket — exatamente o perfil identificado na
univariada (as 5 maiores carteiras: `Entire home/apt`, nenhuma superhost, € 73–90).

**Consequência para a Etapa 06:** transformação log ou binning é obrigatória. Uma
regressão linear nesta coluna produz coeficiente próximo de zero por cancelamento entre
os dois regimes.

---

**`host_response_time`: o nulo informativo se confirma na bivariada.**

A categoria tem η² = 0,0025 (efeito 0,050) — fraco, mas significativo (p < 0,001). Os
boxplots mostram gradação sutil: `within an hour` € 130 → `within a day` € 137. O efeito
é pequeno, mas a **decisão E06-2 da univariada se mantém**: tratar o nulo (47,3%) como
categoria explícita `"sem_histórico"`, já que ele separa anfitriões com metade das reviews
e um quarto da taxa de superhost.

---

**Síntese do grupo.** G4 tem o menor poder preditivo marginal do dataset, mas é o grupo
onde a análise condicional mais muda o diagnóstico. **Nenhuma coluna é descartada aqui** —
`host_is_superhost` (efeito marginal 0,001) carrega até +8,6% em segmentos com mais de
5.000 anúncios. O Boruta (Etapa 05) decide com o modelo completo.

In [ ]:
# Análise 2.6 — G5: Política de reserva e custos
_G = 'G5 — Política de reserva e custos'
display(resumo_bivariado(_G))
plotar_bivariada_continuas(GRUPOS[_G])
plotar_bivariada_discretas(GRUPOS[_G])

print('cleaning_fee e o preco: a taxa e um multiplicador ou um substituto?')
print('=' * 76)
print('(cleaning_fee e a 4a maior associacao do dataset - efeito 0.447)')
_cf = df03['cleaning_fee']
for _rot, _m in [('NaN (nao informou)', _cf.isna()),
                 ('zero (declarou sem)', _cf == 0),
                 ('> 0 (cobra)', _cf > 0)]:
    _s = df03.loc[_m, 'price']
    print(f'  {_rot:<22s} n={_m.sum():>5d} | price mediano € {_s.median():>7.2f}')

print('\n  >> A taxa acompanha o preco ou o substitui? (dentro de Entire/acc=2)')
_seg = df03[(df03['room_type'] == 'Entire home/apt') & (df03['accommodates'] == 2)]
_q = pd.qcut(_seg.loc[_seg['cleaning_fee'] > 0, 'cleaning_fee'], 4, duplicates='drop')
_t = _seg.loc[_seg['cleaning_fee'] > 0].groupby(_q, observed=True)['price'].agg(['size', 'median'])
print(_t.round(2).to_string())
_r, _p = stats.spearmanr(_seg.loc[_seg['cleaning_fee'] > 0, 'cleaning_fee'],
                         _seg.loc[_seg['cleaning_fee'] > 0, 'price'])
print(f'    rho(cleaning_fee, price) dentro do segmento = {_r:+.3f} (p={_p:.2e})')

print('\n  >> minimum_nights: politica de estadia e preco')
for _b, _lab in [((df03['minimum_nights'] == 1), 'exatamente 1 noite'),
                 ((df03['minimum_nights'].between(2, 3)), '2-3 noites'),
                 ((df03['minimum_nights'].between(4, 7)), '4-7 noites'),
                 ((df03['minimum_nights'] >= 8), '8+ noites')]:
    if _b.sum() < 30: continue
    print(f'    {_lab:<20s} n={_b.sum():>5d} | mediana € {df03.loc[_b, "price"].median():>7.2f}')

**Observações bivariadas — G5: Política de reserva e custos**

**`cleaning_fee` é a 4ª maior associação do dataset (0,447) — e a mais forte fora do G1.**

| estado | n | price mediano |
|---|---|---|
| NaN (não informou) | 3.242 | € 115,00 |
| zero (declarou sem) | 745 | € 125,00 |
| > 0 (cobra) | 13.991 | € 130,00 |

A gradação NaN < zero < positivo se mantém na bivariada, **confirmando a ação E06-1** da
univariada: os três estados são distintos e o nulo não pode ser imputado como zero.

**Sob controle, a associação cai mas permanece substancial.** Dentro do segmento
homogêneo (`Entire home/apt` + `accommodates = 2`):

| faixa de `cleaning_fee` | n | price mediano |
|---|---|---|
| € 5–25 | 1.396 | € 110,00 |
| € 25–35 | 1.624 | € 120,00 |
| € 35–45 | 1.469 | € 120,00 |
| € 45–200 | 1.067 | **€ 133,00** |

ρ = **+0,200** (p = 2,8e-51, n = 5.556). A associação bruta de 0,447 cai para 0,200 —
metade do efeito era proxy de tamanho, mas **sobra sinal real**. A taxa de limpeza
acompanha o preço da diária mesmo dentro do mesmo tipo e capacidade: é um **multiplicador**,
não um substituto.

> **Nota registrada para a Etapa 09 (reforço do item da Análise 2.1).** `cleaning_fee` é
> uma variável que o **anfitrião controla**, não uma característica do imóvel. Como
> preditora é legítima e forte. Mas o recomendador não pode tratá-la como "dado" — se o
> modelo diz "seu imóvel vale € 150" e parte desse valor vem da taxa de limpeza que o
> próprio anfitrião definiu, há circularidade. Sugestão: reportar a predição com e sem
> as variáveis de custo controladas pelo anfitrião.

---

**`minimum_nights` — relação em U invertido, não monotônica.**

| política | n | mediana |
|---|---|---|
| exatamente 1 noite | 2.830 | € 110,00 |
| 2–3 noites | 11.725 | € 130,00 |
| 4–7 noites | 2.996 | **€ 135,00** |
| 8+ noites | 427 | **€ 100,00** |

O preço **sobe** até 4–7 noites e **despenca** em 8+. Isso explica o delta Pearson/Spearman
de **0,109** com sinais opostos (ρ = +0,113, r = −0,005) — o segundo maior delta do dataset.
A leitura de negócio: estadias mínimas de 2–7 noites são o padrão do mercado de temporada
em Amsterdam e sinalizam imóvel bem posicionado. Acima de 8 noites entra o **aluguel mensal
disfarçado** (identificado na univariada: 116 anúncios com ≥ 30 noites), que opera em outra
lógica de preço — diária menor, ocupação longa. **Mais um caso onde binning ou árvore é
obrigatório.**

---

**`cancellation_policy` — gradação limpa e interpretável (η² = 0,034).**

| política | n | mediana |
|---|---|---|
| flexible | 4.221 | € 119,00 |
| moderate | 6.646 | € 125,00 |
| strict_14_with_grace_period | 7.070 | € 139,00 |
| **super_strict_60** | **41** | **€ 325,00** |

Monotônica e clara: quanto mais rígida a política, maior o preço. `super_strict_60` tem
mediana de **€ 325** — 2,3× a política estrita comum —, mas com apenas **41 anúncios**.
É o segmento de luxo, onde o anfitrião pode impor condições severas. **Cuidado para a
Etapa 06:** categoria com n = 41 é frágil para target encoding; considerar agrupá-la com
`strict_14` ou aplicar suavização bayesiana (mesma recomendação do E06-6 para bairros).

---

**Confirmação da redundância estrutural do bloco `maximum_nights`.**

`maximum_nights` (0,046), `maximum_nights_cap` (0,047) e `sem_limite_maximo` (0,023) têm
efeitos praticamente idênticos e muito fracos — coerente com a univariada, que mostrou
serem a **mesma informação em três formatos** (51,1% no default de 1125 = "sem limite").
Os LOWESS dos dois primeiros são visualmente iguais: plano com pico artificial no valor
default. **Resolução na Análise 3.2**, conforme já registrado.

---

**Demais colunas.** `security_deposit` (0,215) mantém associação moderada e tem delta 0,055
— outra candidata a transformação. `tem_deposito` (+12,5%) e `tem_taxa_limpeza` (+9,2%)
confirmam que a *existência* da cobrança já carrega sinal, independente do valor.
`instant_bookable` (−0,8%) e `sem_limite_maximo` (0,0%) são praticamente nulos em efeito
de mediana, embora significativos pelo tamanho da amostra — bom lembrete de que
**significância estatística não é o mesmo que relevância prática** numa base de 18 mil linhas.

In [ ]:
# Análise 2.7 — G6: Reviews e reputação (H5 e H8)
_G = 'G6 — Reviews e reputação'
display(resumo_bivariado(_G))
plotar_bivariada_continuas(GRUPOS[_G])
plotar_bivariada_discretas(GRUPOS[_G])

print('H5 — gap_value_vs_geral NEGATIVO associa-se a precos MAIS ALTOS?')
print('=' * 78)
print('(hospede achou caro => nota de "value" abaixo da nota geral)')
_g = df03['gap_value_vs_geral']
_ok = _g.notna()
_r, _p = stats.spearmanr(_g[_ok], df03.loc[_ok, 'price'])
print(f'\n  rho(gap_value_vs_geral, price) = {_r:+.4f} (p={_p:.3e}, n={_ok.sum()})')
print(f'  >> Sinal NEGATIVO confirma H5' if _r < 0 else '  >> Sinal POSITIVO: H5 rejeitada')

_b = pd.cut(_g, [-10, -1, -0.5, -0.01, 0.01, 10],
            labels=['<= -1 (bem abaixo)', '-1 a -0.5', '-0.5 a 0 (abaixo)',
                    '= 0 (igual)', '> 0 (acima)'])
_t = df03.groupby(_b, observed=True)['price'].agg(['size', 'median'])
print('\n  price mediano por faixa de gap:')
print(_t.round(2).to_string())

print('\n  >> Controlando por segmento (o efeito e real ou composicao?):')
for _rt, _acc in [('Entire home/apt', 2), ('Entire home/apt', 4)]:
    _seg = df03[(df03['room_type'] == _rt) & (df03['accommodates'] == _acc)]
    _o = _seg['gap_value_vs_geral'].notna()
    if _o.sum() < 100: continue
    _r2, _p2 = stats.spearmanr(_seg.loc[_o, 'gap_value_vs_geral'], _seg.loc[_o, 'price'])
    print(f'    {_rt[:14]:<14s} acc={_acc}: rho={_r2:+.4f} (p={_p2:.2e}, n={_o.sum()})')

print('\n' + '=' * 78)
print('H8 — Anuncios SEM REVIEW tem distribuicao de preco distinta?')
print('=' * 78)
_sr = df03['sem_reviews'] == 1
a, b = df03.loc[~_sr, 'price'], df03.loc[_sr, 'price']
_u, _p = stats.mannwhitneyu(a, b, alternative='two-sided')
_ks, _pks = stats.ks_2samp(a, b)
print(f'  com review: n={len(a):>5d} | mediana € {a.median():>7.2f} | CV {a.std()/a.mean():.3f}')
print(f'  sem review: n={len(b):>5d} | mediana € {b.median():>7.2f} | CV {b.std()/b.mean():.3f}')
print(f'  Mann-Whitney p={_p:.3e} | Kolmogorov-Smirnov D={_ks:.4f}, p={_pks:.3e}')

**Observações bivariadas — G6: Reviews e reputação**

**H5 — CONFIRMADA. É a hipótese mais importante do plano, e o sinal é o esperado.**

*Hipótese: `gap_value_vs_geral` negativo associa-se a preços mais altos (hóspede achou caro).*

ρ = **−0,0790** (p = 4,8e-23, n = 15.608). **Sinal negativo, como previsto.**

E o efeito **se fortalece sob controle** — o oposto do que aconteceu com as amenities:

| segmento | ρ | p | n |
|---|---|---|---|
| geral | −0,079 | 4,8e-23 | 15.608 |
| Entire home/apt, acc = 2 | −0,079 | 4,9e-10 | 6.176 |
| **Entire home/apt, acc = 4** | **−0,140** | 9,0e-21 | 4.398 |

Em imóveis de 4 pessoas o efeito **quase dobra**. Isso é exatamente o comportamento
esperado de uma variável de percepção de valor: quanto maior o imóvel (e o preço absoluto),
mais o hóspede penaliza a relação custo-benefício.

> **Consequência para a Etapa 09 — esta é a ligação direta com o objetivo do projeto.**
> O plano previa que, se confirmada, `gap_value_vs_geral` viraria "candidata forte tanto a
> feature quanto a **sinal de validação do recomendador**". A confirmação sustenta esse uso:
> um anúncio com `gap_value` muito negativo está, na percepção do próprio hóspede, **acima
> do preço justo**. Isso dá ao recomendador um sinal independente do modelo para validar a
> zona de superprecificação (CLAUDE.md §3.3) — que hoje é apenas sinalizada, sem ação.
> **Registrado como E09-4.**

**Nota de cautela:** o efeito é estatisticamente sólido mas **fraco em magnitude** (efeito
0,079, posição 6ª no grupo). A tabela por faixa mostra a relação de forma ruidosa (€ 130 na
faixa mais negativa vs € 120 na positiva) porque o `gap` é discreto e concentrado. Serve
como sinal de validação, não como preditor principal.

---

**H8 — CONFIRMADA. Anúncios sem review têm distribuição de preço distinta.**

| | n | mediana | **CV** |
|---|---|---|---|
| com review | 15.847 | € 125,00 | 0,581 |
| **sem review** | 2.131 | **€ 139,00** | **0,852** |

Mann-Whitney p = 2,2e-09; **Kolmogorov-Smirnov D = 0,102, p = 1,6e-17** — as distribuições
diferem não só em mediana, mas em **forma**. O CV é **47% maior** no grupo sem review.

Isso confirma e quantifica o achado da univariada: anúncios sem review são mais caros
(€ +14) **e muito mais dispersos**. Coerente com o padrão de bloqueio de calendário
identificado na Análise 1.1.b (38,9% dos outliers de preço não tinham review, contra 11,9%
de base). Parte desses € 139 não é preço de mercado.

> **Consequência para a Etapa 09 (reforço do E10-1).** O MAPE nesse subgrupo será
> estruturalmente pior — CV de 0,852 é um piso alto. Recomendo reportar o MAPE segmentado
> por `sem_reviews`, junto com as segmentações por atividade e por `host_eh_profissional`.

---

**`review_scores_location` lidera o grupo (0,214) — e faz sentido de negócio.**

Das 17 colunas, a de **localização** é a mais associada ao preço, seguida de
`gap_location_vs_geral` (0,171, **ρ positivo**). Os dois gaps têm sinais opostos e coerentes:

| feature | ρ | leitura |
|---|---|---|
| `gap_location_vs_geral` | **+0,171** | localização acima da média → preço maior |
| `gap_value_vs_geral` | **−0,079** | custo-benefício abaixo da média → preço maior |

**As duas features de gap criadas na Etapa 03 funcionam**, e funcionam na direção correta.
Isso valida a decisão de criá-las e confirma o diagnóstico da univariada: elas existem
justamente para extrair sinal do efeito teto, e são as candidatas mais promissoras do grupo.

---

**O efeito teto se confirma na bivariada: as notas quase não discriminam.**

| subnota | efeito | % na nota máxima (univariada) |
|---|---|---|
| `review_scores_location` | **0,214** | 45,8% |
| `review_scores_communication` | 0,058 | 72,8% |
| `review_scores_checkin` | 0,056 | 69,8% |
| `review_scores_cleanliness` | 0,055 | 53,4% |
| `review_scores_accuracy` | 0,049 | 65,2% |
| **`review_scores_value`** | **0,014** | 30,2% |

A relação é quase perfeitamente **inversa**: quanto maior a concentração no teto, menor o
efeito. `location` (45,8% no teto) tem efeito 4× maior que `communication` (72,8%).
`review_scores_value` é a **penúltima do dataset inteiro** (p = 0,08, não significativa) —
apesar de ser a subnota com *maior* variância. Isso reforça que o valor está no **gap**,
não na nota bruta.

---

**Achado não previsto: `number_of_reviews` tem ρ NEGATIVO (−0,062).**

Mais reviews → preço menor. O LOWESS confirma: curva decrescente e suave. Isso tem
explicação dupla: (a) anúncios baratos são mais reservados, logo acumulam mais reviews;
(b) anúncios sem review são artificialmente caros (H8). O mesmo padrão aparece em
`reviews_per_month` (−0,061) e `reviews_por_mes_ativo` (−0,058), ambas com **delta ~0,10 e
sinais opostos** entre Pearson e Spearman — as três estão entre os maiores deltas do dataset,
sinalizando não-linearidade e reforçando a necessidade de transformação na Etapa 06.

In [ ]:
# Análise 2.8 — G7: Disponibilidade e texto
_G = 'G7 — Disponibilidade e texto'
display(resumo_bivariado(_G))
plotar_bivariada_continuas(GRUPOS[_G])

print('availability_365 == 0: "totalmente reservado" ou "calendario fechado"?')
print('=' * 78)
_bloq = df03['availability_365'] == 0
print(f'  bloqueados: {_bloq.sum()} ({_bloq.mean():.1%})')
print(f'  price mediano | bloqueado    : € {df03.loc[_bloq, "price"].median():.2f}')
print(f'  price mediano | com disponib.: € {df03.loc[~_bloq, "price"].median():.2f}')

print('\n  >> Cruzando com atividade (a chave para desambiguar):')
for _rot, _m in [('bloqueado + ativo', _bloq & (df03['anuncio_ativo'] == 1)),
                 ('bloqueado + inativo', _bloq & (df03['anuncio_ativo'] == 0)),
                 ('disponivel + ativo', ~_bloq & (df03['anuncio_ativo'] == 1)),
                 ('disponivel + inativo', ~_bloq & (df03['anuncio_ativo'] == 0))]:
    if _m.sum() < 30: continue
    _s = df03.loc[_m]
    print(f'    {_rot:<22s} n={_m.sum():>5d} ({_m.mean():>5.1%}) | mediana € {_s["price"].median():>7.2f}'
          f' | reviews/mes {_s["reviews_per_month"].median():.2f}')

print('\n  >> len_description truncada: o teto tem preco diferente?')
_teto = df03['len_description'] == 1000
a, b = df03.loc[~_teto, 'price'], df03.loc[_teto, 'price']
_, _p = stats.mannwhitneyu(a, b, alternative='two-sided')
print(f'    < 1000 chars: n={len(a):>5d} | mediana € {a.median():>7.2f}')
print(f'    = 1000 (teto): n={len(b):>5d} | mediana € {b.median():>7.2f} | p={_p:.3e}')

print('\n  >> n_campos_texto_preenchidos (completude do anuncio):')
_t = df03.groupby('n_campos_texto_preenchidos', observed=True)['price'].agg(['size', 'median'])
print(_t[_t['size'] >= 30].round(2).to_string())

### Observações — Análise 2.8 (G7: Disponibilidade e texto)

**O grupo mais fraco do dataset.** Todos os 7 efeitos ficam entre 0,04 e 0,15 — a maior
associação (`availability_365`, ρ = +0,15) é menor que a menor associação do G1. Nenhuma
variável do grupo carrega sinal de preço por si só.

**`availability_365 == 0` é uma massa de 8.345 anúncios (46,4%) — não é "lotado".**
O LOWESS mostra o padrão de forma clara: um pico isolado em zero, queda abrupta, platô
longo e leve subida no extremo (365 dias). Como `availability_365` é disponibilidade
**futura** no calendário no momento do scrape (CLAUDE.md §4 — proibido como proxy de
ocupação), zero significa "calendário fechado", não "totalmente reservado". O cruzamento
com `anuncio_ativo` desambigua:

| segmento | n | % | mediana | reviews/mês |
|---|---:|---:|---:|---:|
| bloqueado + inativo | 5.388 | 30,0% | € 119,00 | 0,21 |
| bloqueado + ativo | 2.957 | 16,4% | € 125,00 | 0,66 |
| disponível + ativo | 7.433 | 41,3% | € 130,00 | 1,05 |
| disponível + inativo | 2.200 | 12,2% | **€ 150,00** | 0,27 |

Os 30% "bloqueado + inativo" são anúncios **dormentes** (0,21 review/mês), não anúncios
disputados — e são os mais baratos da base. Já "disponível + inativo" (12,2%) tem a
**maior mediana do quadro** (€ 150,00) com baixíssima atividade: é o perfil clássico de
superprecificação — calendário aberto, preço alto, ninguém reserva. Isso conecta G7
diretamente à zona de superprecificação do recomendador (CLAUDE.md §3.3) e é a
justificativa para manter `availability_365` no modelo apesar do efeito marginal fraco.

**O sinal de texto é de completude, não de conteúdo.** `len_description` tem um teto
rígido em 1.000 caracteres (10.824 anúncios, 60,2% da base, truncados pela plataforma):
mediana € 130,00 contra € 120,00 dos não-truncados (p = 5,5e-35). `n_campos_texto_preenchidos`
sobe de forma quase monotônica mas fraquíssima — € 120 (2 campos) a € 130 (10 campos),
efeito 0,04. Ambos medem **esforço do anfitrião no cadastro**, e anfitriões que preenchem
tudo tendem a ser mais profissionais e a ter imóveis melhores. É associação indireta via
perfil do host, não o texto explicando preço.

**Nada é removido.** Como em todos os grupos anteriores, a decisão de corte é da Boruta
(Etapa 05). Registro para as próximas etapas:

- **E06-G7-1:** `availability_30/60/90/365` são altamente redundantes entre si (efeitos
  0,12–0,15, mesmo formato de curva). Candidatas óbvias à poda na correlação intragrupo
  (Análise 3.1).
- **E06-G7-2:** o pico em `availability_365 == 0` é uma descontinuidade, não um valor
  baixo de uma escala contínua. Modelos lineares vão tratá-lo errado; árvores capturam
  naturalmente. Alternativa se necessário: flag `calendario_fechado`.
- **E09-5:** o segmento "disponível + inativo" (12,2%, mediana € 150,00, 0,27 review/mês)
  é o público-alvo natural do sinal de superprecificação. Cruzar com o resultado do
  recomendador na Etapa 09.

In [ ]:
# Análise 2.9 — Dispersão por faixa de preço: estimativa do piso de MAPE
df03['decil_preco'] = pd.qcut(df03[TARGET], 10, labels=[f'D{i}' for i in range(1, 11)])

_d = (df03.groupby('decil_preco', observed=True)[TARGET]
      .agg(n='size', minimo='min', mediana='median', maximo='max',
           media='mean', desvio='std'))
_d['iqr'] = df03.groupby('decil_preco', observed=True)[TARGET].agg(
    lambda s: s.quantile(.75) - s.quantile(.25))
_d['cv'] = _d['desvio'] / _d['media']
_d['mad_pct'] = df03.groupby('decil_preco', observed=True)[TARGET].agg(
    lambda s: (s - s.median()).abs().median() / s.median() * 100)

print('DISPERSAO POR DECIL DE PRECO — CV e desvio absoluto mediano (proxy do piso de MAPE)')
print('=' * 100)
print(_d.round(3).to_string())

print(f'\nCV global: {df03[TARGET].std() / df03[TARGET].mean():.3f}')
print(f'MAD%% global (piso teorico de um modelo que so acerta a mediana): '
      f'{(df03[TARGET] - df03[TARGET].median()).abs().median() / df03[TARGET].median() * 100:.1f}%')

print('\n>> Piso de MAPE por segmento (dispersao INTRAGRUPO, controlada):')
print('=' * 100)
for _rot, _cols in [('room_type', ['room_type']),
                    ('room_type + accommodates', ['room_type', 'accommodates']),
                    ('room_type + accom + bairro', ['room_type', 'accommodates',
                                                    'neighbourhood_cleansed'])]:
    _g = df03.groupby(_cols, observed=True)[TARGET]
    _tam = _g.size()
    _ok = _tam[_tam >= 30].index
    _res = df03[df03.set_index(_cols).index.isin(_ok)].groupby(_cols, observed=True)[TARGET]
    _mape_min = _res.agg(lambda s: (s - s.median()).abs().mean() / s.median() * 100)
    _peso = _res.size()
    _pond = (_mape_min * _peso).sum() / _peso.sum()
    print(f'  {_rot:<30s} grupos>=30: {len(_ok):>4d} | cobertura: {_peso.sum()/len(df03):>5.1%}'
          f' | MAPE minimo ponderado: {_pond:.1f}%')

In [ ]:
# Análise 2.9 (cont.) — Onde o piso de MAPE é mais alto: segmentos de maior dispersão
print('SEGMENTOS COM MAIOR DISPERSAO INTRAGRUPO — onde o modelo vai errar mais')
print('=' * 96)

def _piso_mape(mask, rotulo):
    """MAPE minimo alcancavel = erro medio % contra a mediana do proprio grupo."""
    s = df03.loc[mask, TARGET]
    if len(s) < 30:
        return None
    return {'segmento': rotulo, 'n': len(s), 'mediana': s.median(),
            'cv': s.std() / s.mean(),
            'piso_mape': (s - s.median()).abs().mean() / s.median() * 100}

_linhas = []
for _c in ['room_type', 'property_type_agrupado', 'faixa_accommodates']:
    if _c not in df03.columns:
        continue
    for _v, _g in df03.groupby(_c, observed=True):
        _r = _piso_mape(df03[_c] == _v, f'{_c} = {_v}')
        if _r: _linhas.append(_r)

for _c, _r0, _r1 in [('host_profissional', 'host individual', 'host profissional'),
                     ('tem_reviews', 'sem reviews', 'com reviews'),
                     ('host_is_superhost', 'nao superhost', 'superhost'),
                     ('instant_bookable', 'reserva nao instantanea', 'reserva instantanea')]:
    if _c not in df03.columns:
        continue
    for _v, _rot in [(0, _r0), (1, _r1)]:
        _r = _piso_mape(df03[_c] == _v, _rot)
        if _r: _linhas.append(_r)

_disp = (pd.DataFrame(_linhas).sort_values('piso_mape', ascending=False)
         .reset_index(drop=True))
_disp.index += 1
print(_disp.round(2).to_string())

### Observações — Análise 2.9 (Dispersão e piso de MAPE alcançável)

**Nota metodológica sobre a tabela por decil.** A dispersão *dentro* de um decil de preço
é circular: o decil é definido pelo próprio `price`, então CV baixo (D3–D8 entre 0,02 e
0,04) não diz nada sobre previsibilidade. A tabela serve apenas para mostrar o formato da
distribuição: as pontas são largas (D1 CV 0,20, D10 CV **0,43** com máximo de € 3.142) e o
miolo é comprimido — reflexo dos preços-âncora psicológicos (€ 99, € 100, € 150) já vistos
na univariada. **O número que importa é o segundo bloco: a dispersão intragrupo sob
controle de características observáveis.**

**Piso de MAPE estimado: ~28%.** Um modelo que acertasse perfeitamente a mediana de cada
célula `(room_type, accommodates, bairro)` — ou seja, um oráculo que conhecesse o preço
típico exato de cada comparável — ainda erraria **28,2%** em média. Essa é a dispersão que
sobra depois de controlar as três variáveis mais explicativas do dataset. A escada mostra
quanto cada camada de controle compra:

| controle | grupos (n≥30) | cobertura | piso de MAPE |
|---|---:|---:|---:|
| `room_type` | 3 | 100,0% | 40,8% |
| `+ accommodates` | 12 | 99,1% | 31,9% |
| `+ bairro` | 84 | 91,3% | **28,2%** |

O ganho é decrescente: tipo+tamanho derruba 8,9 p.p., somar bairro derruba só mais 3,7 p.p.
Não há razão para esperar que as ~70 features restantes entreguem outros 20 p.p. — elas
têm efeitos marginais uma ordem de grandeza menores (ranking da 2.1). **A leitura honesta é
que um MAPE bom para este dataset fica na faixa de 20–30%, não abaixo de 15%.**

**Implicação direta na Etapa 01.** A curva EV(MAPE) foi construída sobre um grid de MAPE
hipotéticos. A análise de dispersão diz que os cenários da ponta otimista do grid não são
alcançáveis com estas features — o cenário-base deve ser ancorado próximo de 25–30%, e o
otimista em ~20%. Registrado como **E10-1** para a re-execução da Etapa 01 com o MAPE real.

**Onde o modelo vai errar mais.** Os segmentos de maior dispersão intragrupo são pequenos e
heterogêneos:

- `Shared room` (n=59, piso **58,9%**) — amostra pequena e produto mal definido.
- `Townhouse` (n=578, CV 0,76, piso 56,2%), `Boat` (n=344, piso 52,4%), `Loft` (n=334,
  piso 51,1%), `Other` (n=449, piso 51,5%) — categorias de imóvel onde o rótulo não
  determina o produto.
- `Bed and breakfast` (n=411, **CV 1,22** — o maior da base, já sinalizado no G1).

Em contraste, os dois segmentos que carregam a base são os mais previsíveis:
`Apartment` (n=14.016, piso 40,6%) e `Entire home/apt` (n=14.242, piso 40,9%).

**Cruzamento com as hipóteses de dispersão já testadas.** Três resultados anteriores
convergem aqui e se confirmam mutuamente:

| achado | origem | CV | leitura |
|---|---|---:|---|
| host profissional vs individual | H4 (G4) | 0,735 vs 0,601 | profissional dispersa **mais** (hipótese invertida) |
| sem review vs com review | H8 (G6) | 0,852 vs 0,581 | sem review dispersa **muito mais** |
| Bed and breakfast vs Apartment | G1 | 1,22 vs 0,57 | categoria mais imprevisível |
| reserva instantânea vs não | 2.9 | 0,68 vs 0,62 | dispersa um pouco mais |

Os anúncios sem review (CV 0,852) são o pior caso: não têm histórico, não têm reputação e
o preço é o que o anfitrião chutou. São exatamente os anúncios onde o recomendador teria
mais valor — e onde o modelo terá menos precisão.

**Registro para as próximas etapas:**

- **E09-6:** reportar MAPE **segmentado**, não só global. Um MAPE agregado de 28% esconde
  ~41% em Apartment e ~59% em Shared room. A tabela acima é o baseline de comparação: o
  modelo precisa bater o piso de cada segmento para agregar valor sobre a mediana LOO.
- **E09-7:** o recomendador deve expor a banda de confiança **por segmento** (CLAUDE.md
  §11 — a página Streamlit já prevê "faixa de confiança"). Uma banda única de ±MAPE global
  seria otimista demais para B&B/Boat e conservadora demais para Apartment.
- **E10-1:** reancorar os cenários da curva EV(MAPE) da Etapa 01 na faixa 20–30%.
- **E05-1:** o piso de 28,2% é o **teto de desempenho da mediana LOO** (a baseline de
  negócio da Etapa 01). O modelo de ML só se justifica se ficar abaixo disso.

## Consolidação da Análise Bivariada — Seção 2 encerrada

Executadas as Análises 2.1 a 2.9 do plano (`docs/plans/04_eda.md` §4), cobrindo as 81
explicativas nos 7 grupos e as 8 hipóteses de negócio.

### Veredito das 8 hipóteses

| # | hipótese | veredito | evidência |
|---|---|---|---|
| H1 | localização central eleva o preço | **confirmada** | `dist_centro_km` ρ = −0,29 (2º maior efeito do dataset) |
| H2 | tamanho é o principal driver | **confirmada** | `accommodates` ρ = +0,49 — maior efeito absoluto |
| H3 | elevador é premium | **confirmada sob controle** | efeito marginal enganoso; confundidor = distância |
| H4 | profissionais padronizam preço | **REJEITADA (invertida)** | CV 0,735 (prof.) vs 0,601 (indiv.), Levene p=8,2e-61 |
| H5 | gap de "value" sinaliza superpreço | **confirmada e reforçada** | ρ = −0,079 global → **−0,140** em Entire/acc=4 |
| H6 | bairro explica preço além da distância | **parcial** | η = 0,21; parte do sinal é distância disfarçada |
| H7 | quebra de regime em `accommodates` | **confirmada** | €55/pessoa (2→3) cai para €40/pessoa (3+) |
| H8 | anúncios sem review têm outra distribuição | **confirmada** | KS D=0,102 p=1,6e-17; CV 0,852 vs 0,581 |

### Os três achados que mudam as próximas etapas

**1. Paradoxo de Simpson é estrutural, não pontual.** Encontrado em três lugares
independentes — `tem_elevator` (confundidor = distância), `host_is_superhost`
(confundidor = mix de segmento; efeito marginal 0,0012, o **último** dos 81, vira +4,6%
a +8,6% sob controle) e **todo o G3 de amenities** (confundidor = tamanho; `tem_tv`
+35,0% → +10,1%, `tem_family_kid_friendly` +23,5% → **0,0%**). Como o confundidor **varia
por feature**, nenhum controle manual único resolve. É o argumento técnico direto para
priorizar **modelos de árvore** (Etapa 07), que capturam interação sem especificação prévia.

**2. Nenhuma coluna é removida na bivariada.** Regra aplicada de forma consistente nos 7
grupos: associação marginal fraca **não** é critério de descarte, precisamente por causa
do item 1. `host_is_superhost` seria descartada por qualquer corte marginal e é uma das
features mais informativas sob controle. O portão formal de seleção é a **Boruta (Etapa 05)**.

**3. Piso de MAPE ≈ 28%.** Com controle de `(room_type, accommodates, bairro)` a dispersão
residual é de 28,2%. Isso reancora as expectativas de todo o projeto: a faixa realista é
**20–30%**, não abaixo de 15%.

### Dívida registrada para as próximas etapas

| id | etapa | item |
|---|---|---|
| E05-1 | 05 | piso de 28,2% = teto da mediana LOO; ML só se justifica abaixo disso |
| E06-G7-1 | 06 | `availability_30/60/90/365` redundantes entre si — podar na 3.1 |
| E06-G7-2 | 06 | pico em `availability_365 == 0` é descontinuidade, não escala contínua |
| E06/07 | 06/07 | não-linearidades catalogadas: `accommodates` (quebra 2→3), `minimum_nights` (U invertido, pico 4–7), `host_listings_count` (pico e platô), `dist_centro_km` (platô 4–8 km) |
| E09-4 | 09 | `gap_value_vs_geral` valida a zona de superprecificação do recomendador |
| E09-5 | 09 | segmento "disponível + inativo" (12,2%, € 150, 0,27 rev/mês) = alvo do sinal de superpreço |
| E09-6 | 09 | reportar MAPE **segmentado** (H4/H8 mostram dispersão heterogênea) |
| E09-7 | 09/11 | banda de confiança do recomendador deve ser **por segmento** |
| E10-1 | 10 | reancorar cenários da curva EV(MAPE) na faixa 20–30% |

### Checklist da etapa (CLAUDE.md §8)

- [x] Código adicionado de forma incremental e executado via MCP
- [x] Saídas revisadas
- [x] Premissas e decisões registradas no notebook

**Próximo passo (aguardando validação):** Seção 3 — Análise Multivariada, começando pela
**Análise 3.1** (matriz de correlação intragrupo e detecção de autocorrelação).

In [ ]:
# Consolidação EDA — Quadro de descarte antes da análise multivariada
# Critério: só sai coluna por motivo TÉCNICO (vazamento, redundância ~1:1, artefato de
# análise). Associação fraca com o alvo NÃO é motivo de corte — isso é da Boruta (Etapa 05).

DESCARTE_TECNICO = {
    # --- A. Artefatos da própria EDA / derivadas do alvo (vazamento direto) ---
    'price_log':   ('A. vazamento', 'log1p(price) — criada na 2.1 só para Pearson'),
    'decil_preco': ('A. vazamento', 'qcut(price) — criada na 2.9 só para diagnóstico'),
    # --- B. Identificador ---
    'id':          ('B. identificador', 'chave primária, sem conteúdo preditivo'),
    # --- C. Redundância técnica (rho >= 0.92 com a irmã mantida) ---
    'calculated_host_listings_count': ('C. redundante', 'rho=0.920 c/ host_listings_count'),
    'reviews_por_mes_ativo':          ('C. redundante', 'rho=0.997 c/ reviews_per_month'),
    'maximum_nights_cap':             ('C. redundante', 'rho=0.965 c/ maximum_nights'),
    'dist_centro_bin':                ('C. redundante', 'rho=0.966 c/ dist_centro_km (discretização)'),
    'min_nights_bin':                 ('C. redundante', 'rho=0.884 c/ minimum_nights (discretização)'),
    'availability_60':                ('C. redundante', 'rho=0.972 c/ availability_90; 0.923 c/ av_30'),
    'availability_90':                ('C. redundante', 'rho=0.895 c/ availability_365'),
}

_fora = [c for c in DESCARTE_TECNICO if c not in df03.columns]
print(f'Colunas do quadro ausentes em df03: {_fora if _fora else "nenhuma"}')

_q = (pd.DataFrame([{'coluna': k, 'motivo': v[0], 'evidencia': v[1]}
                    for k, v in DESCARTE_TECNICO.items()])
      .sort_values(['motivo', 'coluna']).reset_index(drop=True))
_q.index += 1
print(f'\nQUADRO DE DESCARTE TECNICO — {len(_q)} colunas')
print('=' * 92)
print(_q.to_string())

_expl_antes = df03.shape[1] - 1  # menos o alvo
_n_desc = len([c for c in DESCARTE_TECNICO if c in df03.columns])
print(f'\nBalanco:')
print(f'  colunas em df03            : {df03.shape[1]}')
print(f'  descartadas (tecnico)      : {_n_desc}')
print(f'  restantes                  : {df03.shape[1] - _n_desc}  (1 alvo + '
      f'{df03.shape[1] - _n_desc - 1} explicativas p/ a multivariada)')

## Consolidação EDA (univariada + bivariada) — Quadro de descarte

Revisão de todas as 85 colunas de `df03` cruzando o que univariada e bivariada
registraram. **Critério aplicado:** uma coluna só sai por motivo **técnico** — vazamento
do alvo, redundância quase perfeita com uma irmã, ou artefato criado durante a própria
análise. **Associação fraca com `price` não é motivo de corte**, pela razão documentada
nos 7 grupos: o paradoxo de Simpson apareceu três vezes, e `host_is_superhost` — última
das 81 no ranking marginal (efeito 0,0012, p=0,92) — vira +4,6% a +8,6% sob controle de
segmento. A seleção por relevância é da **Boruta (Etapa 05)**.

### A. Vazamento do alvo — 2 colunas

| coluna | motivo |
|---|---|
| `price_log` | `log1p(price)`; criada na 2.1 apenas para calcular Pearson |
| `decil_preco` | `qcut(price, 10)`; criada na 2.9 apenas para diagnóstico de dispersão |

Ambas são transformações determinísticas de `price`. Se sobrevivessem até a Etapa 07, o
modelo atingiria MAPE ≈ 0 e a métrica seria inútil. São artefatos da EDA, não features.

### B. Identificador — 1 coluna

`id` é a chave primária (17.978 valores únicos). Sem conteúdo preditivo; em árvores vira
ruído memorizável.

### C. Redundância técnica — 7 colunas

Pares medidos por Spearman. Descartei o membro **derivado** e mantive o **original**, para
que a Etapa 06 decida a discretização sobre a variável contínua com a informação intacta.

| descartada | mantida | ρ | razão |
|---|---|---:|---|
| `reviews_por_mes_ativo` | `reviews_per_month` | **0,997** | praticamente a mesma variável |
| `dist_centro_bin` | `dist_centro_km` | **0,966** | binning perde o platô de 4–8 km |
| `maximum_nights_cap` | `maximum_nights` | **0,965** | versão winsorizada |
| `availability_60` | `availability_90` / `_30` | **0,972** / 0,923 | janela intermediária |
| `calculated_host_listings_count` | `host_listings_count` | **0,920** | duas contagens do mesmo host |
| `availability_90` | `availability_365` | 0,895 | resolve **E06-G7-1** |
| `min_nights_bin` | `minimum_nights` | 0,884 | binning perde o U invertido (pico 4–7) |

Das quatro janelas de disponibilidade sobram **`availability_30`** (curtíssimo prazo) e
**`availability_365`** (horizonte anual, onde está o pico de 46,4% em zero). Elas correlacionam
0,768 entre si — abaixo do corte, e carregam sinais distintos.

### O que deliberadamente NÃO foi descartado

Quatro candidatos plausíveis que verifiquei e **mantive**:

- **`taxa_limpeza_por_noite`** — suspeitei de vazamento; confirmei que é
  `cleaning_fee / minimum_nights`, sem `price` na fórmula (ρ com o alvo = 0,28). Fica.
- **`beds` (ρ=0,839 com `accommodates`)** e **`security_deposit` (ρ=0,774 com `tem_deposito`)** —
  abaixo do corte de 0,90 e semanticamente distintos. Ficam para a Análise 3.1 avaliar.
- **Binárias raras** — a mais desbalanceada é `tem_step_free_access` (6,29%), seguida de
  `tem_elevator` (9,70%). Nenhuma é degenerada, e `tem_elevator` é justamente o caso de
  Simpson do G3. Ficam.
- **`host_response_time` (47,3% nulo)** — o maior volume de nulos da base. Nulo aqui é
  categoria de negócio ("host que nunca respondeu"), não dado perdido. Tratamento na
  Etapa 06, não descarte agora.

### Escala confirmada

`review_scores_rating` está em **0–100** (min 20, max 100); os outros seis
`review_scores_*` estão em **0–10** (min 2, max 10). Escalas diferentes no mesmo grupo,
como já apontado na Etapa 02 — **nunca reescalar `review_scores_rating` para 0–10 por
engano**. Confirmado novamente aqui.

### Balanço

| | |
|---|---:|
| colunas em `df03` | 85 |
| descartadas (técnico) | **10** |
| restantes | **75** (1 alvo + **74 explicativas**) |

As 74 explicativas seguem para a **Análise 3.1** (correlação intragrupo), que é onde a
redundância *estatística* residual — pares entre 0,70 e 0,90, como `beds`×`accommodates` —
será avaliada com a matriz completa.

## Seção 3 — Análise Multivariada

**Objetivo:** a relação das explicativas **entre si**, não com o alvo. Responde a uma pergunta
diferente das seções anteriores: *"quais colunas dizem a mesma coisa?"*

| seção | pergunta | escopo |
|---|---|---|
| 1 — Univariada | esta coluna tem informação aproveitável? | uma coluna isolada |
| 2 — Bivariada | esta coluna ajuda a prever `price`? | coluna × alvo |
| **3 — Multivariada** | **quais colunas são redundantes entre si?** | **coluna × coluna** |

### Por que isso importa aqui

A bivariada encerrou com uma regra explícita: **nenhuma coluna foi descartada por associação
fraca com o alvo**, porque o paradoxo de Simpson apareceu três vezes e `host_is_superhost` —
última das 81 no ranking marginal — vira +4,6% a +8,6% sob controle de segmento. O portão de
relevância é a Boruta (Etapa 05).

Esta seção usa um critério **independente do alvo**: duas colunas que carregam a mesma
informação são redundantes por construção, não importa o quanto cada uma prevê o preço.
Isso torna o descarte aqui seguro — não há risco de eliminar um sinal que só aparece sob
controle.

### Roteiro (plano §5)

- **3.1 — Correlação intragrupo.** Sete matrizes de Spearman (uma por grupo, máx. 17×17) em
  vez de uma matriz 81×81 ilegível. A redundância se concentra dentro dos grupos temáticos
  por construção: 7 notas de review, 4 janelas de disponibilidade, 3 medidas de portfólio.
- **3.2 — Redundância consolidada.** Todos os pares |ρ| > 0,80 do dataset inteiro,
  dendrograma de clusters, **VIF** das contínuas e heatmap das top-25 do ranking da 2.1.
- **3.3 — Interações e efeitos condicionais.** Heatmaps que validam o `GROUP_COLS`
  `(bairro, room_type, accommodates)` da metodologia travada na Etapa 01 (CLAUDE.md §3.1),
  mais o mapa geográfico.
- **3.4 — Padrão de faltantes.** Matriz de nulidade e correlação entre indicadores de ausência,
  para classificar o mecanismo (MCAR / MAR / MNAR) de cada bloco. Define o tratamento da
  Etapa 06.

### Nota metodológica

A correlação de pares é cega a dependências **multivariadas**: três colunas podem ser
linearmente dependentes sem que nenhum par entre elas passe de 0,81. Por isso a 3.2 usa VIF
além da matriz — é o que separa uma análise de redundância completa de uma inspeção visual
de heatmaps.

In [ ]:
# Análise 3.1 — Correlação intragrupo (Spearman), um heatmap por grupo
import seaborn as sns
import matplotlib.pyplot as plt

def _matriz_numerica(cols):
    """Converte categóricas em códigos ordenados pela mediana de price (correlação faz sentido)."""
    d = {}
    for c in cols:
        if c not in df03.columns or c in (TARGET, TARGET_LOG, 'id'):
            continue
        s = df03[c]
        if s.dtype == object or str(s.dtype) == 'category':
            ordem = df03.groupby(c, observed=True)[TARGET].median().sort_values().index
            d[c] = pd.Categorical(s, categories=ordem, ordered=True).codes.astype(float)
            d[c] = np.where(d[c] < 0, np.nan, d[c])
        else:
            d[c] = s.astype(float)
    return pd.DataFrame(d, index=df03.index)

PARES_ALTOS = []   # acumula pares |rho| > 0.80 de todos os grupos

def analisar_grupo_corr(grupo, limiar=0.80):
    m = _matriz_numerica(GRUPOS[grupo])
    corr = m.corr(method='spearman')
    mask = np.triu(np.ones_like(corr, dtype=bool))
    n = len(corr)
    fig, ax = plt.subplots(figsize=(max(6, n * 0.75), max(5, n * 0.62)))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, square=True, linewidths=.5,
                annot_kws={'size': 7}, cbar_kws={'shrink': .7}, ax=ax)
    ax.set_title(f'Correlação de Spearman — {grupo}  ({n} colunas)', fontsize=11)
    plt.xticks(rotation=45, ha='right', fontsize=8); plt.yticks(fontsize=8)
    plt.tight_layout(); plt.show()

    achados = []
    for i in range(n):
        for j in range(i + 1, n):
            r = corr.iloc[i, j]
            if pd.notna(r) and abs(r) > limiar:
                achados.append({'grupo': grupo.split(' — ')[0],
                                'col_a': corr.index[i], 'col_b': corr.columns[j],
                                'rho': r})
    PARES_ALTOS.extend(achados)
    print(f'>> {grupo}: {len(achados)} par(es) com |rho| > {limiar:.2f}')
    if achados:
        print(pd.DataFrame(achados)[['col_a', 'col_b', 'rho']].round(3).to_string(index=False))
    return corr

_c1 = analisar_grupo_corr('G1 — Capacidade e estrutura física')

In [ ]:
# Análise 3.1 (cont.) — Heatmaps dos grupos G2 a G7
for _g in ['G2 — Geografia', 'G3 — Amenities e qualidade',
           'G4 — Host e operação']:
    analisar_grupo_corr(_g)

In [ ]:
# Análise 3.1 (cont.) — Heatmaps dos grupos G5 a G7
for _g in ['G5 — Política de reserva e custos', 'G6 — Reviews e reputação',
           'G7 — Disponibilidade e texto']:
    analisar_grupo_corr(_g)

### Observações — Análise 3.1 (Correlação intragrupo)

Sete matrizes de Spearman, uma por grupo. Categóricas convertidas em códigos **ordenados
pela mediana de `price`** — sem isso a correlação com uma categórica seria arbitrária,
dependente da ordem alfabética.

**20 pares com |ρ| > 0,80**, distribuídos de forma muito desigual:

| grupo | pares | leitura |
|---|---:|---|
| G5 — Política e custos | 5 | pior caso: `maximum_nights` gera 3 pares |
| G7 — Disponibilidade | 5 | as 4 janelas são quase a mesma variável |
| G1 — Capacidade | 3 | inclui um par novo, não detectado antes |
| G4 — Host | 3 | trio de contagem de portfólio |
| G6 — Reviews | 3 | inclui um par novo |
| G2 — Geografia | 1 | só o binning |
| **G3 — Amenities** | **0** | **nenhum par acima de 0,80** |

**G3 é o grupo mais limpo do dataset.** Treze colunas e nenhuma redundância — o maior valor
é `n_amenities` × `tem_dishwasher` (0,62). Cada flag `tem_*` carrega informação própria. Isso
reforça a decisão de manter todas apesar dos efeitos marginais fracos: elas não se anulam
entre si, só estavam confundidas pelo tamanho (paradoxo de Simpson do G3).

### Três pares que o quadro de descarte anterior não pegou

**1. `bathrooms` × `tem_multiplos_banheiros` = 0,984 (G1).** A flag é quase uma cópia da
contínua. Como `bathrooms` tem 14 valores distintos e a flag só 2, a flag não acrescenta
nada — a árvore encontra o corte sozinha. **Nova candidata a descarte.**

**2. `calculated_host_listings_count` × `host_eh_profissional` = 0,992 (G4).** Trio completo:
`host_listings_count` × `calculated` = 0,920, `host_listings_count` × `host_eh_profissional`
= 0,911. As três medem a mesma coisa — tamanho do portfólio do anfitrião. O quadro anterior
já descartava `calculated_host_listings_count`; com ela fora, sobra `host_listings_count`
(contínua, 43 valores, com o padrão não-linear de pico e platô) × `host_eh_profissional`
(binária). **Mantenho as duas**: a flag define o segmento usado em H4 e no relato segmentado
de MAPE (E09-6), e ρ=0,911 entre uma contínua e sua binarização é esperado — não é
duplicação de informação como em `bathrooms`.

**3. `review_scores_value` × `gap_value_vs_geral` = 0,805 (G6).** Aqui há um conflito real
com a bivariada: `gap_value_vs_geral` é H5 confirmada (ρ=−0,079 global, **−0,140** em
Entire/acc=4, registrado como E09-4), enquanto `review_scores_value` é a **2ª pior** das 81
(p=0,08, não significativo). O gap é construído a partir do score, mas o que ele carrega —
o **desvio** entre a nota de "value" e a nota geral — é justamente o que o score sozinho não
tem. **Mantenho as duas e deixo para a Boruta**, que é o portão formal.

### Pares que confirmam o quadro anterior

`dist_centro_bin` (0,966), `maximum_nights_cap` (0,965), `availability_60/90` (0,972 / 0,895),
`reviews_por_mes_ativo` (**0,997**), `min_nights_bin` (0,884). Todos já estavam marcados.

### Pares estruturais, não redundantes

Três casos que passam de 0,80 mas **não** são candidatos a corte:

- **`accommodates` × `banheiros_por_pessoa` = −0,837** e **`accommodates` × `beds` = 0,839
  (G1).** `banheiros_por_pessoa` é `bathrooms/accommodates` — correlação negativa forte com
  o denominador é aritmética, não redundância.
- **`maximum_nights` × `sem_limite_maximo` = 0,930** e **`min_nights_bin` ×
  `permite_estadia_curta` = −0,805 (G5).** Flags derivadas das próprias contínuas.
- **`dias_desde_ultima_review` × `anuncio_ativo` = −0,823 (G6).** `anuncio_ativo` é definido
  por um corte em `dias_desde_ultima_review`. Redundância por construção.

Em todos, a flag pode ser podada se a contínua ficar — mas a decisão é da Etapa 05/06, não
desta análise.

### Atualização do quadro de descarte

Uma coluna a acrescentar às 10 já identificadas:

| coluna | motivo | evidência |
|---|---|---|
| `tem_multiplos_banheiros` | C. redundante | ρ = **0,984** com `bathrooms` |

Total: **11 descartes**, restando **73 explicativas**.

In [ ]:
# Análise 3.2 — Redundância consolidada: pares |rho|>0.80 em todo o dataset
_todas = [c for g in GRUPOS.values() for c in g if c in df03.columns
          and c not in (TARGET, TARGET_LOG, 'id', 'decil_preco')]
_todas = list(dict.fromkeys(_todas))
M = _matriz_numerica(_todas)
C = M.corr(method='spearman')
_g_de = {c: g.split(' — ')[0] for g, cols in GRUPOS.items() for c in cols}

_pares = []
for i in range(len(C)):
    for j in range(i + 1, len(C)):
        r = C.iloc[i, j]
        if pd.notna(r) and abs(r) > 0.80:
            a, b = C.index[i], C.columns[j]
            _pares.append({'col_a': a, 'col_b': b, 'rho': r,
                           'inter_grupo': _g_de.get(a) != _g_de.get(b)})
pares_altos = (pd.DataFrame(_pares).sort_values('rho', key=abs, ascending=False)
               .reset_index(drop=True))
pares_altos.index += 1
print(f'Pares |rho| > 0,80 no dataset inteiro: {len(pares_altos)}')
print(f'  intragrupo: {(~pares_altos["inter_grupo"]).sum()}  |  INTER-grupo: {pares_altos["inter_grupo"].sum()}')
print('=' * 84)
print(pares_altos.round(3).to_string())

# --- Identidades lineares exatas (invisíveis ao Spearman, fatais para o VIF) ---
print('\n>> Identidades lineares EXATAS detectadas via VIF infinito:')
_lux = ['tem_dishwasher', 'tem_patio_balcony', 'tem_dryer', 'tem_free_parking',
        'tem_private_entrance', 'tem_elevator']
print(f'  n_amenities_luxo == soma({len(_lux)} flags tem_*): '
      f'{(df03[_lux].sum(axis=1) == df03["n_amenities_luxo"]).all()}')
_s6 = ['review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location', 'review_scores_value']
_m = df03.loc[df03['review_score_medio'].notna(), _s6].mean(axis=1)
print(f'  review_score_medio == média(6 review_scores_*): '
      f'{np.allclose(_m, df03.loc[_m.index, "review_score_medio"], atol=1e-6)}')
_X = df03[_s6 + ['gap_location_vs_geral', 'gap_value_vs_geral']].dropna()
print(f'  posto da matriz [6 scores + 2 gaps]: {np.linalg.matrix_rank(_X.values)} de {_X.shape[1]}'
      f'  -> deficiência de posto 2')

In [ ]:
# Análise 3.2 (cont.) — VIF em 4 cenários de poda
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calcular_vif(cols, rotulo, mostrar=8):
    X = df03[cols].copy()
    for c in cols:
        if X[c].dtype == object or str(X[c].dtype) == 'category':
            X[c] = pd.Categorical(X[c]).codes
    X = X.astype(float).replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())
    X = X.loc[:, X.std() > 0]
    X.insert(0, '_const', 1.0)
    v = (pd.DataFrame({'coluna': X.columns[1:],
                       'vif': [variance_inflation_factor(X.values, i)
                               for i in range(1, X.shape[1])]})
         .sort_values('vif', ascending=False).reset_index(drop=True))
    print(f'\n--- {rotulo} ({len(cols)} cols) ---')
    print(f'  VIF>10: {(v["vif"]>10).sum():>2d} | VIF>5: {(v["vif"]>5).sum():>2d} | '
          f'mediana: {v["vif"].median():.2f} | máximo: {v["vif"].max():,.0f}')
    print('  piores:', ', '.join(f'{r.coluna}={r.vif:,.0f}'
                                 for r in v.head(mostrar).itertuples()))
    return v

_conts = [c for c in _todas if df03[c].dtype != object and str(df03[c].dtype) != 'category']
DESCARTE_FINAL = set(DESCARTE_TECNICO) | {'tem_multiplos_banheiros'}
_IDENT = {'n_amenities_luxo', 'review_score_medio'}
_GAPS = {'gap_location_vs_geral', 'gap_value_vs_geral'}

print('FATOR DE INFLAÇÃO DA VARIÂNCIA (VIF) — efeito acumulado de cada poda')
print('=' * 84)
_ = calcular_vif(_conts, '1. ANTES de qualquer descarte')
_ = calcular_vif([c for c in _conts if c not in DESCARTE_FINAL],
                 '2. após os 11 descartes técnicos')
_ = calcular_vif([c for c in _conts if c not in DESCARTE_FINAL | _IDENT],
                 '3. + remover as 2 identidades lineares exatas')
_ = calcular_vif([c for c in _conts if c not in DESCARTE_FINAL | _IDENT | _GAPS],
                 '4. + remover os 2 gaps (cenário de referência)')

In [ ]:
# Análise 3.2 (cont.) — Dendrograma de redundância + heatmap das top-25 do ranking
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

# NaN na matriz: pares em que uma das colunas é constante no subconjunto comum
# (ex.: sem_reviews == 1 em todas as linhas onde review_scores_* existe).
# Correlação indefinida = ausência de evidência de redundância -> distância máxima.
_D = (1 - C.abs()).fillna(1.0)
np.fill_diagonal(_D.values, 0)
_D = (_D + _D.T) / 2                   # garante simetria exata
_Z = linkage(squareform(_D.values, checks=False), method='average')

fig, ax = plt.subplots(figsize=(11, 15))
dendrogram(_Z, labels=C.index.tolist(), orientation='right',
           color_threshold=0.20, leaf_font_size=8, ax=ax)
ax.axvline(0.20, color='#C44E52', ls='--', lw=1.2, label='corte |ρ| = 0,80')
ax.set_title('Dendrograma de redundância — distância = 1 − |ρ| de Spearman', fontsize=12)
ax.set_xlabel('distância (0 = idênticas, 1 = independentes)')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

_cl = pd.Series(fcluster(_Z, 0.20, criterion='distance'), index=C.index)
_multi = _cl.value_counts()[lambda s: s > 1]
print(f'Clusters com |ρ| > 0,80 interno: {len(_multi)} (cobrindo {_multi.sum()} colunas)')
print('=' * 84)
for _c in _multi.index:
    _cols = _cl[_cl == _c].index.tolist()
    print(f'  [{len(_cols)}] {", ".join(_cols)}')

# Heatmap consolidado das 25 features com maior associação com o alvo (ranking da 2.1)
_top25 = [c for c in ranking['coluna'].head(25) if c in C.index]
_c25 = C.loc[_top25, _top25]
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(_c25, mask=np.triu(np.ones_like(_c25, dtype=bool)), annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True, linewidths=.5,
            annot_kws={'size': 7}, cbar_kws={'shrink': .7}, ax=ax)
ax.set_title('Correlação entre as 25 features de maior associação com o alvo', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout(); plt.show()

_m25 = _c25.where(~np.eye(len(_c25), dtype=bool)).abs()
print(f'\nTop-25: maior |ρ| entre pares = {_m25.max().max():.3f} | '
      f'pares acima de 0,80: {int((_m25 > 0.80).sum().sum() / 2)}')

### Observações — Análise 3.2 (Redundância consolidada, VIF e dendrograma)

**Os 7 grupos temáticos são estatisticamente independentes.** Dos 20 pares com |ρ| > 0,80
no dataset inteiro, **20 são intragrupo e nenhum é inter-grupo**. O recorte temático adotado
na Seção 1 do plano não foi só uma conveniência de leitura: ele coincide com a estrutura de
redundância real dos dados. A Análise 3.1 já tinha encontrado todos os pares — a 3.2 não
acrescenta nenhum novo.

O dendrograma confirma: **10 clusters** cobrindo 24 colunas, todos dentro de um único grupo.

| cluster | colunas |
|---|---|
| disponibilidade | `availability_30/60/90/365` |
| noites máximas | `maximum_nights`, `_cap`, `sem_limite_maximo` |
| portfólio do host | `host_listings_count`, `calculated_...`, `host_eh_profissional` |
| capacidade | `accommodates`, `beds` |
| ritmo de reviews | `reviews_per_month`, `reviews_por_mes_ativo` |
| banheiros | `bathrooms`, `tem_multiplos_banheiros` |
| percepção de valor | `review_scores_value`, `gap_value_vs_geral` |
| distância | `dist_centro_km`, `dist_centro_bin` |
| noites mínimas | `minimum_nights`, `min_nights_bin` |
| atividade | `dias_desde_ultima_review`, `anuncio_ativo` |

### O achado que o Spearman não pegou: duas identidades lineares exatas

O VIF revelou **`inf`** em sete colunas do G3 — sinal de dependência linear perfeita. A
investigação confirmou duas identidades que a correlação de pares **não detecta**:

1. **`n_amenities_luxo` = soma exata das 6 flags** `tem_dishwasher`, `tem_patio_balcony`,
   `tem_dryer`, `tem_free_parking`, `tem_private_entrance`, `tem_elevator`.
2. **`review_score_medio` = média exata dos 6 `review_scores_*`** (sem `rating`, que está em
   escala 0–100). Pior: **`gap_location_vs_geral` e `gap_value_vs_geral` são o score menos
   essa média** — a matriz `[6 scores + 2 gaps]` tem **posto 6 de 8**, deficiência de 2.

Nenhum par individual passa de 0,81 aqui, mas as **combinações** são exatas. É por isso que
o VIF é necessário mesmo tendo a matriz de correlação: ele detecta colinearidade
multivariada, não só de pares.

### Efeito acumulado das podas no VIF

| cenário | cols | VIF>10 | VIF>5 | mediana | máximo |
|---|---:|---:|---:|---:|---:|
| 1. antes de tudo | 74 | 26 | 35 | 4,36 | **inf** |
| 2. + 11 descartes técnicos | 68 | 21 | 27 | 3,19 | **inf** |
| 3. + 2 identidades lineares | 66 | 13 | 19 | 2,45 | 5.373 |
| 4. + 2 gaps (referência) | 64 | **4** | **10** | **2,09** | **12** |

O passo 3 é o decisivo: elimina os `inf` e derruba a mediana de 3,19 para 2,45. O passo 4
mostra que **todo o bloco de VIF acima de 200 vinha da deficiência de posto do G6** — sem os
gaps, nada passa de 12.

### Decisão: descartar as 2 identidades, manter os 2 gaps

**`n_amenities_luxo` e `review_score_medio` saem.** São agregações determinísticas de
colunas que já estão no dataset. Não acrescentam informação e quebram qualquer modelo linear.

**Os 2 gaps ficam**, apesar do VIF alto no cenário 3. Três razões:

- **H5 é a hipótese de negócio central do projeto.** `gap_value_vs_geral` confirmou ρ=−0,079
  global e **−0,140** em Entire/acc=4 (E09-4). O que ele mede — o desvio entre a nota de
  "value" e a nota geral — é ortogonal ao nível absoluto do score.
- **`gap_location_vs_geral` tem ρ=+0,171 com o alvo**, contra −0,079 do de value. Os dois
  apontam em direções opostas e coerentes, validando a decisão da Etapa 03 de criá-los.
- **VIF alto não é problema para árvores.** É um diagnóstico de regressão linear. Como o
  plano da Etapa 07 prioriza RF/XGBoost/LightGBM, a colinearidade afeta a *interpretação* dos
  coeficientes, não a capacidade preditiva.

Registrado como **E06-1**: se um modelo linear entrar na Etapa 07, rodar sem os gaps (ou sem
os 6 scores) — os dois blocos juntos são linearmente dependentes.

### Top-25: quase sem redundância

Entre as 25 features de maior associação com o alvo, só **4 pares** passam de 0,80, e dois
deles já estão marcados para descarte (`tem_multiplos_banheiros`, `dist_centro_bin`).
Sobram `accommodates`×`beds` (0,839) e `accommodates`×`banheiros_por_pessoa` (−0,837) —
ambos estruturais. **As features que mais importam para o alvo são, em geral, independentes
entre si.**

### Quadro de descarte atualizado

| coluna | motivo | evidência |
|---|---|---|
| `n_amenities_luxo` | D. identidade linear | = soma exata de 6 flags `tem_*` |
| `review_score_medio` | D. identidade linear | = média exata de 6 `review_scores_*` |

Total: **13 descartes**, restando **71 explicativas**.

In [ ]:
# Análise 3.3 — Interações: validação do GROUP_COLS da Etapa 01
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

CORR_ALL = C if 'C' in dir() and isinstance(C, pd.DataFrame) else CORR_ALL
if 'C' in dir() and isinstance(C, pd.DataFrame):
    del C          # libera o nome 'C' para a função C() do patsy

def heatmap_interacao(idx, col, titulo, n_min=30, figsize=(9, 7), fmt='.0f'):
    """Mediana de price por par de categóricas; células com n < n_min ficam vazias."""
    med = df03.pivot_table(index=idx, columns=col, values=TARGET, aggfunc='median')
    cnt = df03.pivot_table(index=idx, columns=col, values=TARGET, aggfunc='size')
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(med.where(cnt >= n_min), annot=True, fmt=fmt, cmap='YlOrRd',
                linewidths=.5, annot_kws={'size': 8},
                cbar_kws={'label': 'mediana de price (€)'}, ax=ax)
    ax.set_title(f'{titulo}  (células com n < {n_min} em branco)', fontsize=11)
    plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
    print(f'  células: {med.size} | preenchidas (n>={n_min}): {(cnt >= n_min).sum().sum()}'
          f' ({(cnt >= n_min).sum().sum() / med.size:.1%})')
    return med, cnt

_med1, _cnt1 = heatmap_interacao('neighbourhood_cleansed', 'room_type',
                                 'price mediano por bairro × room_type', figsize=(7, 8))
_ = heatmap_interacao('dist_centro_bin', 'faixa_accommodates'
                      if 'faixa_accommodates' in df03.columns else 'room_type',
                      'price mediano por distância × capacidade', figsize=(8, 5))
_ = heatmap_interacao('property_type_agrupado', 'room_type',
                      'price mediano por tipo de imóvel × room_type', figsize=(7, 6))

# A interação é real, ou os efeitos são apenas aditivos?
_r = (_med1['Entire home/apt'] / _med1['Private room']).dropna()
print(f'\n>> Razão Entire/Private por bairro: min={_r.min():.2f} máx={_r.max():.2f} '
      f'CV={_r.std()/_r.mean():.3f}')
print(f'   mais achatada: {_r.idxmin()} ({_r.min():.2f}) | '
      f'mais esticada: {_r.idxmax()} ({_r.max():.2f})')

_d = df03[[TARGET_LOG, 'neighbourhood_cleansed', 'room_type']].dropna()
_d.columns = ['y', 'bairro', 'rt']
_ma = smf.ols('y ~ C(bairro) + C(rt)', data=_d).fit()
_mi = smf.ols('y ~ C(bairro) * C(rt)', data=_d).fit()
_an = anova_lm(_ma, _mi)
print(f'\n>> Teste F (aditivo vs. interação): F={_an["F"].iloc[1]:.2f} '
      f'p={_an["Pr(>F)"].iloc[1]:.2e} | R² {_ma.rsquared:.4f} -> {_mi.rsquared:.4f} '
      f'(ganho {_mi.rsquared - _ma.rsquared:.4f})')

In [ ]:
# Análise 3.3 (cont.) — Mapa geográfico e cobertura do GROUP_COLS da Etapa 01
_rng = np.random.default_rng(42)
_am = df03.sample(min(8000, len(df03)), random_state=42)
_p99 = df03[TARGET].quantile(.99)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
_s = axes[0].scatter(_am['longitude'], _am['latitude'], c=_am[TARGET].clip(upper=_p99),
                     cmap='YlOrRd', s=6, alpha=.6)
plt.colorbar(_s, ax=axes[0], label='price (€, corte no p99)')
axes[0].set_title('Preço por localização', fontsize=11)
axes[0].set_xlabel('longitude'); axes[0].set_ylabel('latitude')

_med_b = df03.groupby('neighbourhood_cleansed', observed=True).agg(
    lat=('latitude', 'median'), lon=('longitude', 'median'),
    med=(TARGET, 'median'), n=(TARGET, 'size')).reset_index()
_s2 = axes[1].scatter(_med_b['lon'], _med_b['lat'], c=_med_b['med'],
                      s=_med_b['n'] / 8, cmap='YlOrRd', edgecolors='black', linewidths=.6)
plt.colorbar(_s2, ax=axes[1], label='mediana de price (€)')
for _r in _med_b.itertuples():
    axes[1].annotate(_r.neighbourhood_cleansed[:14], (_r.lon, _r.lat), fontsize=6.5,
                     ha='center', va='bottom')
axes[1].set_title('Centroide dos 22 bairros (tamanho = nº de anúncios)', fontsize=11)
axes[1].set_xlabel('longitude'); axes[1].set_ylabel('latitude')
plt.tight_layout(); plt.show()

# Cobertura do GROUP_COLS — dispara o fallback hierárquico da Etapa 01?
GROUP_COLS = ['neighbourhood_cleansed', 'room_type', 'accommodates']
_tam = df03.groupby(GROUP_COLS, observed=True)[TARGET].size()
print(f'GROUP_COLS = {GROUP_COLS}')
print(f'células não vazias: {len(_tam)}')
print('=' * 72)
for _min in [1, 5, 10, 30]:
    _ok = _tam[_tam >= _min]
    print(f'  n_min = {_min:>2d}: {len(_ok):>4d} células | cobertura: {_ok.sum()/len(df03):>6.1%}'
          f' | fallback em {1 - _ok.sum()/len(df03):>5.1%} dos anúncios')

### Observações — Análise 3.3 (Interações e mapa geográfico)

**O `GROUP_COLS` da Etapa 01 está validado como estrutura, mas o `n_min = 30` é agressivo
demais.** Este é o achado que mais importa desta análise.

| n_min | células | cobertura | fallback dispara em |
|---:|---:|---:|---:|
| 1 | 338 | 100,0% | 0,0% |
| 5 | 188 | 98,4% | 1,6% |
| 10 | 142 | 96,6% | 3,4% |
| **30** | **84** | **91,3%** | **8,7%** |

Com `n_min = 30` (o default travado no CLAUDE.md §3.1), **8,7% dos anúncios caem no fallback
hierárquico** e recebem preço de referência de um nível mais genérico. O salto de 10 para 30
custa 5,3 p.p. de cobertura para ganhar precisão em apenas 58 células. Registrado como
**E10-2**: avaliar `n_min = 10` na re-execução da Etapa 01 — a diferença entre a mediana de
uma célula com 10 anúncios e uma com 30 é pequena, mas a perda de cobertura é concreta.

### A interação bairro × room_type é real, mas fraca

O teste F rejeita o modelo aditivo (**F = 3,37, p = 2,2e-12**), então a interação existe. Mas
o ganho de R² é de apenas **0,0056** (0,2637 → 0,2693) — menos de 1% de variância explicada.

A razão `Entire home/apt` ÷ `Private room` por bairro mostra onde ela aparece:

- **Mais achatada:** Westerpark (1,44), Centrum-West (1,46), Bos en Lommer (1,47)
- **Mais esticada:** IJburg-Zeeburgereiland (2,24), Bijlmer-Oost (2,16), De Aker (2,14)

Nos bairros centrais, alugar um quarto privativo custa quase tanto quanto um apartamento
inteiro — a localização domina. Na periferia, o gap dobra. **Leitura de negócio:** no centro
o hóspede paga pelo endereço; na periferia paga pelo espaço. Isso justifica manter os três
níveis do `GROUP_COLS` em vez de simplificar para dois.

### Os heatmaps confirmam os achados da bivariada

**Distância × room_type** reproduz o platô do H1 com nitidez: Entire cai de € 188 (<1 km) para
€ 125 (3–5 km), depois **sobe** para € 140 (5–8 km) e € 135 (>8 km). A curva não é monotônica.
Private room, em contraste, cai de forma limpa: € 120 → € 60. **A não-monotonicidade da
distância é um efeito do produto Entire, não da geografia em si** — provavelmente casas
maiores e barcos nas bordas da cidade.

**Tipo de imóvel × room_type** mostra o prêmio de nicho: `Houseboat` € 200, `Townhouse` € 190,
`Boat` € 189 e `Loft` € 180, contra `Apartment` € 130. São exatamente as categorias com maior
piso de MAPE na 2.9 (Townhouse 56,2%, Boat 52,4%, Loft 51,1%) — **caras e imprevisíveis**.

### O mapa confirma o achado geográfico do plano

O scatter lat/lon mostra o gradiente radial, mas também a dispersão dentro de cada anel: há
anúncios de € 500 na periferia e de € 80 no centro. O mapa de centroides mostra que os bairros
mais caros (Centrum-West/Oost, ambos € 175 em Entire) formam um núcleo compacto, enquanto o
volume de anúncios se concentra em De Baarsjes-Oud-West e De Pijp — bairros de preço médio.

Isso reforça o achado já registrado no plano (§2.3.1): `latitude`+`longitude` puros dão
R² = 0,112 contra 0,093 das três distâncias radiais. **Amsterdam não é radial** — o IJ corta a
cidade e os canais formam anéis. As distâncias ficam por interpretabilidade no recomendador
(CLAUDE.md §11), não por poder preditivo.

### Registro

- **E10-2:** testar `n_min = 10` no fallback da Etapa 01 (cobertura 96,6% vs. 91,3%).
- **E09-8:** o recomendador deve expor **de qual nível do fallback** veio a referência — é o
  indicador de confiança já previsto na metodologia travada (CLAUDE.md §3.1).
- **E07-1:** a interação bairro × room_type é significante mas pequena (ΔR² = 0,006). Modelos
  lineares precisariam do termo explícito; árvores capturam sem especificação.

In [ ]:
# Análise 3.4 — Padrão de dados faltantes: MCAR, MAR ou MNAR?
_nul = [c for c in df03.columns if df03[c].isna().any()]
_t = (pd.DataFrame({'coluna': _nul,
                    'n_nulos': [df03[c].isna().sum() for c in _nul],
                    'pct': [df03[c].isna().mean() * 100 for c in _nul]})
      .sort_values('pct', ascending=False).reset_index(drop=True))
_t.index += 1
print(f'Colunas com nulos: {len(_nul)} de {df03.shape[1]}')
print(_t.round(2).to_string())

# Matriz de nulidade e correlação entre indicadores de ausência
_ord = _t['coluna'].tolist()
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
_am = df03[_ord].isna().sample(min(3000, len(df03)), random_state=42)
axes[0].imshow(_am.values, aspect='auto', cmap='binary', interpolation='nearest')
axes[0].set_xticks(range(len(_ord)))
axes[0].set_xticklabels(_ord, rotation=90, fontsize=7)
axes[0].set_title('Matriz de nulidade (preto = ausente, amostra de 3.000)', fontsize=11)
axes[0].set_ylabel('anúncios')

_cn = df03[_ord].isna().astype(int).corr()
sns.heatmap(_cn, cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True,
            linewidths=.3, cbar_kws={'shrink': .7}, ax=axes[1],
            xticklabels=True, yticklabels=True)
axes[1].set_title('Correlação entre indicadores de nulidade', fontsize=11)
plt.setp(axes[1].get_xticklabels(), rotation=90, fontsize=6.5)
plt.setp(axes[1].get_yticklabels(), rotation=0, fontsize=6.5)
plt.tight_layout(); plt.show()

# --- Teste de mecanismo: a ausência carrega informação sobre o alvo? ---
print('\nMECANISMO DE AUSÊNCIA — a nulidade prevê o preço?')
print('=' * 92)
_linhas = []
for _c in [c for c in _ord if df03[c].isna().mean() > 0.001]:
    _na = df03[_c].isna()
    _a, _b = df03.loc[_na, TARGET], df03.loc[~_na, TARGET]
    if len(_a) < 30 or len(_b) < 30:
        continue
    _, _p = stats.mannwhitneyu(_a, _b, alternative='two-sided')
    _linhas.append({'coluna': _c, 'pct_nulo': _na.mean() * 100,
                    'med_ausente': _a.median(), 'med_presente': _b.median(),
                    'dif_pct': (_a.median() - _b.median()) / _b.median() * 100,
                    'p_valor': _p})
_mec = pd.DataFrame(_linhas).sort_values('pct_nulo', ascending=False).reset_index(drop=True)
_mec.index += 1
print(_mec.round(3).to_string())

### Observações — Análise 3.4 (Padrão de dados faltantes)

**30 das 85 colunas têm nulos, mas eles se agrupam em 4 blocos com mecanismos diferentes.**
O heatmap de correlação de nulidade mostra os blocos com clareza — quadrados vermelhos
perfeitos onde a ausência é compartilhada.

### Bloco 1 — Reviews (~13%): MAR explicável

Os 13 indicadores do bloco de reviews coincidem com `review_scores_rating` entre **98,8% e
100%**. A ausência é explicada inteiramente por uma variável observada: o anúncio não tem
review. `sem_reviews == 1` (2.131 casos) coincide 98,8% com `review_scores_rating` nulo
(2.355 casos) — a pequena folga são anúncios com review mas sem nota registrada.

**Decisão (E06-2):** `sem_reviews` **já captura toda a informação de ausência deste bloco**.
Não é necessário criar indicadores adicionais. A imputação pode ser qualquer valor neutro
(mediana ou 0), porque a flag informa o modelo de que aquele valor é sintético.

### Bloco 2 — `security_deposit` (30,9%) e `cleaning_fee` (18,0%): MNAR

A tabulação cruzada é conclusiva:

| | `tem_deposito`=0 | `tem_deposito`=1 |
|---|---:|---:|
| valor presente | 3.291 | 9.133 |
| **valor nulo** | **5.554** | **0** |

**Todo nulo de `security_deposit` tem `tem_deposito` = 0.** Nulo aqui significa "não cobro
depósito", não "não informei". O mesmo padrão em `cleaning_fee` (3.242 nulos, todos com
`tem_taxa_limpeza` = 0).

E a ausência **prevê o preço**: mediana € 116,50 (nulo) contra € 130,00 (com valor), Mann-
Whitney **p = 1,7e-81**. Anúncios que não cobram depósito são sistematicamente mais baratos.

**Decisão (E06-3): imputar pela mediana aqui seria um erro grave.** Substituir "não cobro"
por "cobro o valor mediano" inverte o significado. O correto é **imputar 0** (o valor
economicamente verdadeiro) e manter as flags `tem_deposito` / `tem_taxa_limpeza`. A decisão
tomada na Etapa 02 de manter como `NaN` está validada — a imputação com significado só era
possível depois desta análise.

### Bloco 3 — `host_response_time` (47,3%): MNAR, o pior caso

O maior volume de nulos da base, e a ausência é altamente informativa:

| | nulo | com valor |
|---|---:|---:|
| mediana de price | € 120,00 | € 130,00 |
| % superhost | **8,8%** | **22,3%** |
| reviews/mês | **0,34** | **0,91** |

Mann-Whitney **p = 4,9e-45**. O anfitrião sem tempo de resposta registrado tem **2,5× menos
chance de ser superhost** e **um terço da atividade**. É o perfil do host ausente ou inativo
— exatamente o segmento "disponível + inativo" identificado no G7 (E09-5).

**Decisão (E06-4):** tratar o nulo como **categoria explícita** (`"sem_resposta"`), nunca
imputar. A ausência é o sinal.

### Bloco 4 — ~0,04%: MCAR

Trinta e duas linhas (0,18%) com nulos esparsos em `bathrooms`, `bedrooms`, `beds`,
`host_is_superhost`, `calendar_atualizado_dias`. Sem padrão de correlação no heatmap. A
mediana difere (€ 99 vs € 125), mas com n = 32 isso não é evidência de mecanismo. É ruído de
cadastro — **MCAR**, imputação simples resolve.

### Nota: `camas_por_quarto` e `pessoas_por_quarto` (5,6%)

Já identificado no plano (§3.3, ponto G1): vêm da proteção contra divisão por zero quando
`bedrooms == 0`, ou seja, **estúdios**. A interpretação está confirmada — é MAR por
construção, e a solução correta é a flag `eh_estudio` sugerida no plano, não imputação.
Registrado como **E06-5**.

### Resumo dos mecanismos

| bloco | % nulo | mecanismo | tratamento na Etapa 06 |
|---|---:|---|---|
| reviews (13 cols) | ~13% | **MAR** | `sem_reviews` já basta; imputar neutro |
| `security_deposit`, `cleaning_fee` | 31% / 18% | **MNAR** | imputar **0**, manter flags |
| `host_response_time` | 47% | **MNAR** | categoria `"sem_resposta"` |
| `*_por_quarto` | 5,6% | MAR (estúdio) | criar flag `eh_estudio` |
| bloco esparso | 0,04% | **MCAR** | imputação simples |

In [ ]:
# Consolidação da Etapa 04 — aplicação dos descartes e criação do df04
#
# `id` é PRESERVADA: não é feature (fica fora dos GRUPOS e nunca entra no fit),
# mas é a chave que liga a previsão ao anúncio no recomendador da Etapa 09.
DESCARTE_EDA = {
    # A. Vazamento do alvo (artefatos criados durante a própria EDA)
    'price_log':                      'A. vazamento — log1p(price), criada na 2.1',
    'decil_preco':                    'A. vazamento — qcut(price), criada na 2.9',
    # B. Redundância técnica (rho >= 0,88 com a irmã mantida)
    'reviews_por_mes_ativo':          'B. redundante — rho=0,997 c/ reviews_per_month',
    'tem_multiplos_banheiros':        'B. redundante — rho=0,984 c/ bathrooms',
    'availability_60':                'B. redundante — rho=0,972 c/ availability_90',
    'dist_centro_bin':                'B. redundante — rho=0,966 c/ dist_centro_km',
    'maximum_nights_cap':             'B. redundante — rho=0,965 c/ maximum_nights',
    'calculated_host_listings_count': 'B. redundante — rho=0,920 c/ host_listings_count',
    'availability_90':                'B. redundante — rho=0,895 c/ availability_365',
    'min_nights_bin':                 'B. redundante — rho=0,884 c/ minimum_nights',
    # C. Identidade linear exata (invisível ao Spearman, detectada via VIF infinito)
    'n_amenities_luxo':               'C. identidade — = soma exata de 6 flags tem_*',
    'review_score_medio':             'C. identidade — = média exata de 6 review_scores_*',
}

_presentes = [c for c in DESCARTE_EDA if c in df03.columns]
df04 = df03.drop(columns=_presentes).copy()

# Ordem do entregável: id (chave) -> price (alvo) -> explicativas por grupo
GRUPOS_04 = {g: [c for c in cols if c in df04.columns] for g, cols in GRUPOS.items()}
FEATURES_04 = [c for cols in GRUPOS_04.values() for c in cols]
df04 = df04[['id', TARGET] + FEATURES_04]

_q = (pd.DataFrame([{'coluna': k, 'motivo': v} for k, v in DESCARTE_EDA.items()])
      .sort_values('motivo').reset_index(drop=True))
_q.index += 1
print(f'DESCARTADAS — {len(_presentes)} colunas')
print('=' * 84)
print(_q.to_string())

print(f'\nshape df03 -> df04: {df03.shape} -> {df04.shape}')
print(f'  id (chave p/ Etapa 09) : {"id" in df04.columns}')
print(f'  price (alvo)           : {TARGET in df04.columns}')
print(f'  explicativas           : {len(FEATURES_04)}')

print('\nVERIFICAÇÃO — plano §8')
print('=' * 84)
print(f'  granularidade 1 linha/anúncio : {len(df04) == df04["id"].nunique()}')
print(f'  linhas preservadas            : {len(df04) == len(df03)}')
print(f'  price == 0 ausente            : {(df04[TARGET] == 0).sum() == 0}')
print(f'  sem derivadas do alvo         : {not any(c in df04.columns for c in ["price_log", "decil_preco"])}')
print(f'  id fora das features          : {"id" not in FEATURES_04}')

print('\nGrupos após a poda:')
for _g, _c in GRUPOS_04.items():
    _antes = len([c for c in GRUPOS[_g] if c in df03.columns])
    print(f'  {_g:<38s} {_antes:>2d} -> {len(_c):>2d}')
print(f'\n  total de explicativas: {len(FEATURES_04)}')

## Consolidação da Etapa 04 — EDA encerrada

As 11 análises do plano (`docs/plans/04_eda.md` §6) foram executadas: **1.1–1.9** (univariada),
**2.1–2.9** (bivariada), **3.1–3.4** (multivariada).

### Entregável 1 — `df04`

**17.978 × 73** = **1 chave** (`id`) + **1 alvo** (`price`) + **71 explicativas**, contra
17.978 × 85 do `df03`. Nenhuma linha removida: a EDA não é etapa de transformação.

**`id` é preservada.** Não é feature — fica fora de `GRUPOS_04`/`FEATURES_04` e nunca entra
no `fit` — mas é a chave que liga a previsão ao anúncio no recomendador da **Etapa 09**.
A lista de features a usar no treino é `FEATURES_04` (71 colunas), não `df04.columns`.

Verificações do plano §8 aprovadas: granularidade de 1 linha por `listing_id`, `price == 0`
ausente, nenhuma coluna derivada do alvo, `id` fora das features, hold-out nunca lido.

### Entregável 2 — Lista de descarte (12 colunas)

| motivo | colunas | evidência |
|---|---|---|
| **A. Vazamento** | `price_log`, `decil_preco` | transformações determinísticas de `price` criadas na própria EDA |
| **B. Redundância** | `reviews_por_mes_ativo`, `tem_multiplos_banheiros`, `availability_60`, `availability_90`, `dist_centro_bin`, `maximum_nights_cap`, `calculated_host_listings_count`, `min_nights_bin` | ρ entre 0,884 e 0,997 com a irmã mantida |
| **C. Identidade linear** | `n_amenities_luxo`, `review_score_medio` | somas/médias exatas de colunas já presentes |

O bloco **C** é o achado técnico mais relevante: nenhum par individual passava de 0,81, mas
as **combinações** eram exatas. Só o VIF (`inf`) detectou. Efeito da poda no VIF: máximo
`inf` → **12**, mediana 4,36 → **2,09**.

**Composição final por grupo:** G1 (11), G2 (5), G3 (12), G4 (9), G5 (14), G6 (15), G7 (5).

### Entregável 3 — Ranking de associação (insumo da Boruta)

Preservado em `ranking` (81 explicativas ordenadas por efeito). Topo: `accommodates` (0,49),
`dist_centro_km` (−0,29), `room_type`, `bedrooms`. **Nenhuma coluna foi descartada por
associação fraca** — o paradoxo de Simpson apareceu 3 vezes, e `host_is_superhost` (última
das 81, efeito 0,0012) vira +4,6% a +8,6% sob controle. A Boruta (Etapa 05) é o portão.

### Entregável 4 — Premissas atualizadas

1. **Alvo em `log1p`** (skew 22,68 → 0,39). MAPE sempre na escala original, após `expm1`.
2. **Outliers de `price` mantidos** — o segmento de luxo é mercado real.
3. **Piso de MAPE ≈ 28%** — faixa realista de **20–30%**, não abaixo de 15%.

### Os três achados que atravessam o projeto

**1. Paradoxo de Simpson é estrutural.** `tem_elevator` (confundidor = distância),
`host_is_superhost` (mix de segmento) e todo o G3 de amenities (tamanho). O confundidor
**varia por feature** — nenhum controle manual único resolve. Argumento direto para
priorizar **árvores** na Etapa 07.

**2. Piso de MAPE de 28,2%.** Um oráculo que acertasse a mediana exata de cada célula
`(room_type, accommodates, bairro)` ainda erraria 28%. Reancora os cenários da curva
EV(MAPE) da Etapa 01.

**3. Três mecanismos de ausência distintos.** MAR nos reviews (`sem_reviews` já captura),
**MNAR** em `security_deposit`/`cleaning_fee` (nulo = "não cobro", imputar mediana seria
erro grave) e MNAR em `host_response_time` (47% nulo, 2,5× menos superhosts).

### Dívida registrada

| id | etapa | item |
|---|---|---|
| E05-1 | 05 | piso de 28,2% = teto da mediana LOO; ML só se justifica abaixo |
| E05-2 | 05 | usar `FEATURES_04` (71 cols) no Boruta — `id` e `price` ficam fora |
| E06-1 | 06/07 | modelo linear: rodar sem os gaps **ou** sem os 6 scores (posto 6 de 8) |
| E06-2 | 06 | bloco de reviews: `sem_reviews` basta, imputar neutro |
| E06-3 | 06 | `security_deposit`/`cleaning_fee`: imputar **0**, nunca mediana |
| E06-4 | 06 | `host_response_time`: categoria `"sem_resposta"` |
| E06-5 | 06 | `*_por_quarto`: criar flag `eh_estudio` (bedrooms == 0) |
| E06-6 | 06 | pico em `availability_365 == 0` é descontinuidade (46,4% da base) |
| E06/07 | 06/07 | não-linearidades: `accommodates` (quebra 2→3), `minimum_nights` (U invertido, pico 4–7), `host_listings_count` (pico e platô), `dist_centro_km` (platô 4–8 km) |
| E07-1 | 07 | interação bairro × room_type significante mas pequena (ΔR² = 0,006) |
| E09-4 | 09 | `gap_value_vs_geral` valida a zona de superprecificação |
| E09-5 | 09 | segmento "disponível + inativo" (12,2%, € 150, 0,27 rev/mês) = alvo do sinal |
| E09-6 | 09 | reportar MAPE **segmentado** (H4/H8: dispersão heterogênea) |
| E09-7 | 09/11 | banda de confiança do recomendador **por segmento** |
| E09-8 | 09 | expor de qual nível do fallback veio a referência |
| E10-1 | 10 | reancorar cenários EV(MAPE) na faixa 20–30% |
| E10-2 | 10 | testar `n_min = 10` no fallback (cobertura 96,6% vs. 91,3%) |

### Escala confirmada (reconfirmação)

`review_scores_rating` está em **0–100** (min 20, max 100); os outros seis `review_scores_*`
em **0–10** (min 2, max 10). **Nunca reescalar o primeiro por engano.**

### Checklist da etapa (CLAUDE.md §8)

- [x] Código adicionado de forma incremental e executado via MCP
- [x] Saídas revisadas
- [x] Premissas e decisões registradas no notebook

**Próximo passo (aguardando validação):** Etapa 05 — Feature Selection com Boruta +
importância por árvores sobre as 71 colunas de `FEATURES_04`.

# Etapa 05 — Feature Selection

Entrada: `df04` (**17.978 × 73**) = `id` + `price` + **71 features** (`FEATURES_04`).

A Etapa 04 removeu 12 colunas por critério **técnico e independente do alvo** (vazamento,
ρ ≥ 0,88, identidade linear exata). Esta etapa faz a pergunta oposta:

| Etapa | Pergunta | Depende do alvo? |
|---|---|---|
| 04 — EDA | Esta coluna é tecnicamente defeituosa? | Não |
| **05 — Selection** | Esta coluna **contribui** para prever `price`? | **Sim** |

Como a segunda pergunta depende do modelo, da métrica e dos dados, toda a etapa é
construída em torno de **baselines explícitos** — não de um ranking isolado.

### Regras que atravessam a etapa

- **Isolamento do teste** (CLAUDE.md §5.3): `test_raw.csv` **não é lido**. "Quais colunas
  dropar" está na lista literal de decisões contamináveis da §5.2.
- **Métrica**: `fit` em `log1p(price)`, mas **todo MAPE reportado é em escala original,
  após `expm1`**. Sem exceção — MAPE sobre o log é outra métrica, não comparável com o
  número que o negócio lê na Etapa 10.
- **Ordem 05 → 06 mantida**, com **preparação mínima descartável** (§1 do plano): ela
  existe só para viabilizar os seletores e morre ao fim da etapa. `df05` sai com os
  **valores originais** de `df04`, nulos inclusive.

### Roteiro — 14 portões

| # | Bloco | Custo |
|---|---|---|
| 1 | Preparação mínima descartável | — |
| 2 | Partição `train_05` / `valid_05` | — |
| 3 | Baselines (3 números) | ~1 min |
| 4 | Diagnóstico barato: mutual information + mRMR | segundos |
| 5–6 | Importância por árvores (MDI, split count, permutation) | ~2 min |
| 7 | **Null importance** (seletor principal, paralelizado) | ~7 min |
| 8 | **RFECV** (define o conjunto) | ~10 min |
| 9 | **Boruta** (confirmação, descartável se > 20 min) | ~14 min |
| 10–11 | Consenso + blindagem de negócio | — |
| 12 | Estabilidade (3 seeds, paralelizado) | ~7 min |
| 13–14 | Validação final + `df05` + consolidação | ~2 min |

Referência da EDA: a Análise 2.9 mediu piso de dispersão intragrupo de **20,8%** (MAD%,
cobertura 91,3%). O MAPE dos baselines deve pousar em **20–30%** — valor muito abaixo
disso é sinal de vazamento, não de sucesso.

---

## 5.1 — Preparação mínima descartável (Portão 1)

Os seletores exigem matriz numérica sem nulos. `FEATURES_04` tem **5 categóricas** e
**28 colunas com nulo**. Cada regra abaixo vem de um achado da **Análise 3.4** (mecanismo
de faltantes), não de conveniência:

| Item | Tratamento | Mecanismo | Motivo |
|---|---|---|---|
| 5 categóricas | Ordinal encoding | — | One-hot de `neighbourhood_cleansed` (22 níveis) fragmentaria a importância do bairro em 22 pedaços fracos. Ele é `GROUP_COLS`; distorcê-lo é inaceitável |
| `security_deposit`, `cleaning_fee`, `taxa_limpeza_por_noite` | **0** | MNAR | 100% dos nulos têm a flag = 0. É "não cobro", não "desconhecido" (E06-3) |
| `host_response_time` (47,3%) | Categoria `"sem_resposta"` | MNAR | Quem não responde tem perfil distinto (p = 4,94e-45) (E06-4) |
| Bloco de reviews (~13%) | Sentinela **−1** | MAR | `sem_reviews` já marca o padrão; sentinela evita inventar score (E06-2) |
| Estruturais (≤ 5,7%) | Mediana do `train_05` | MCAR | Volume irrelevante; mediana aprendida **só no treino** |

> **Aviso de não-herança:** nada desta célula é insumo da Etapa 06. A preparação real —
> encoders, scalers, imputação definitiva — é responsabilidade dela, sobre as features
> sobreviventes. O entregável daqui é a **lista de nomes** (`FEATURES_05`).

In [ ]:
# ---------------------------------------------------------------------------
# Portão 1 — Preparação mínima DESCARTÁVEL (plano §1.2)
#
# Existe somente para viabilizar os seletores. NÃO é insumo da Etapa 06.
# df05 sairá com os valores ORIGINAIS de df04 (nulos inclusive) — ver §16.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

# Blocos de colunas por mecanismo de faltante (Análise 3.4)
MNAR_ZERO = ['security_deposit', 'cleaning_fee', 'taxa_limpeza_por_noite']
MNAR_CAT  = ['host_response_time']
MAR_REVIEWS = [c for c in FEATURES_04
               if c.startswith('review_scores_') or c.startswith('gap_')
               or c in ('reviews_per_month', 'dias_desde_primeira_review',
                        'dias_desde_ultima_review')]

CATEGORICAS_04 = [c for c in FEATURES_04
                  if df04[c].dtype == object or str(df04[c].dtype) == 'category']

# Estruturais = resto com nulo, tratado por mediana do treino
_com_nulo = [c for c in FEATURES_04 if df04[c].isna().any()]
MCAR_MEDIANA = [c for c in _com_nulo
                if c not in MNAR_ZERO + MNAR_CAT + MAR_REVIEWS]

print('Blocos de tratamento (preparação descartável)')
print(f'  MNAR -> 0 .............. {len(MNAR_ZERO):>2}  {MNAR_ZERO}')
print(f'  MNAR -> categoria ...... {len(MNAR_CAT):>2}  {MNAR_CAT}')
print(f'  MAR  -> sentinela -1 ... {len(MAR_REVIEWS):>2}')
print(f'  MCAR -> mediana treino . {len(MCAR_MEDIANA):>2}  {MCAR_MEDIANA}')
print(f'  categóricas (ordinal) .. {len(CATEGORICAS_04):>2}  {CATEGORICAS_04}')

# Cobertura: todo nulo tem destino declarado
_sem_destino = [c for c in _com_nulo
                if c not in MNAR_ZERO + MNAR_CAT + MAR_REVIEWS + MCAR_MEDIANA]
assert not _sem_destino, f'colunas com nulo sem tratamento: {_sem_destino}'
print(f'\n  {len(_com_nulo)} colunas com nulo, todas com tratamento declarado: OK')


def preparar_minima(df, mapas_cat=None, medianas=None):
    """Preparação mínima e DESCARTÁVEL para os seletores.

    Devolve (X numérico sem nulos, mapas_cat, medianas). Os parâmetros
    aprendidos (mapas de categoria e medianas) são devolvidos para que a
    validação receba os do TREINO — sem fit fora do treino, mesmo aqui.
    """
    X = df[FEATURES_04].copy()

    # 1. MNAR -> 0 ("não cobro", conforme Análise 3.4)
    for c in MNAR_ZERO:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    # 2. MNAR categórico -> categoria explícita (a ausência é informação)
    for c in MNAR_CAT:
        if c in X.columns:
            X[c] = X[c].astype(object).fillna('sem_resposta')

    # 3. MAR (reviews) -> sentinela -1 (separável por 1 split; não inventa score)
    for c in MAR_REVIEWS:
        if c in X.columns:
            X[c] = X[c].fillna(-1.0)

    # 4. Categóricas -> ordinal, com mapa aprendido no treino
    aprender = mapas_cat is None
    if aprender:
        mapas_cat = {}
    for c in CATEGORICAS_04:
        if aprender:
            mapas_cat[c] = {v: i for i, v in enumerate(sorted(X[c].dropna().unique()))}
        X[c] = X[c].map(mapas_cat[c]).astype(float)
        X[c] = X[c].fillna(-1.0)          # categoria nunca vista no treino

    # 5. MCAR -> mediana do TREINO
    if medianas is None:
        medianas = X[MCAR_MEDIANA].median()
    for c in MCAR_MEDIANA:
        if c in X.columns:
            X[c] = X[c].fillna(medianas[c])

    return X.astype(float), mapas_cat, medianas


print('\nFunção `preparar_minima` definida (fit só no treino, via mapas_cat/medianas).')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 2 — Partição train_05 / valid_05 (plano §2)
#
# valid_05 é a ÚNICA fonte de MAPE para comparar conjuntos de features.
# test_raw.csv permanece FECHADO (CLAUDE.md §5.3).
# ---------------------------------------------------------------------------
from sklearn.model_selection import train_test_split

train_05, valid_05 = train_test_split(
    df04, test_size=0.20, random_state=RANDOM_STATE, shuffle=True,
)

# Preparação: fit no treino, transform na validação (sem vazamento, mesmo aqui)
X_tr, MAPAS_CAT, MEDIANAS = preparar_minima(train_05)
X_va, _, _ = preparar_minima(valid_05, mapas_cat=MAPAS_CAT, medianas=MEDIANAS)

# Alvo em log1p para o fit; MAPE sempre em escala original via expm1
y_tr_log = np.log1p(train_05[TARGET].values)
y_va_log = np.log1p(valid_05[TARGET].values)
y_tr_eur = train_05[TARGET].values
y_va_eur = valid_05[TARGET].values

print(f'train_05 .... {X_tr.shape[0]:>6,} x {X_tr.shape[1]} features'.replace(',', '.'))
print(f'valid_05 .... {X_va.shape[0]:>6,} x {X_va.shape[1]} features'.replace(',', '.'))
print(f'soma ........ {len(train_05) + len(valid_05):>6,} = df04 ({len(df04):,})'
      .replace(',', '.'))

# Testes de aceitação
assert X_tr.isna().sum().sum() == 0, 'nulo sobrou no treino'
assert X_va.isna().sum().sum() == 0, 'nulo sobrou na validação'
assert list(X_tr.columns) == list(X_va.columns) == FEATURES_04, 'colunas divergentes'
assert not set(train_05['id']) & set(valid_05['id']), 'vazamento de id entre partições'
assert (y_tr_eur > 0).all() and (y_va_eur > 0).all(), 'price <= 0 inviabiliza MAPE'
print('\nSem nulos, sem sobreposição de id, price > 0: OK')

# Faixa do alvo — referência para leitura do MAPE
print(f'\nprice treino: mediana €{np.median(y_tr_eur):.0f} | '
      f'validação: mediana €{np.median(y_va_eur):.0f}')


def mape_eur(y_true_eur, y_pred_log):
    """MAPE em ESCALA ORIGINAL a partir de predição em log1p (plano: sem exceção)."""
    y_pred_eur = np.expm1(y_pred_log)
    return float(np.mean(np.abs(y_true_eur - y_pred_eur) / y_true_eur))


print('Função `mape_eur` definida — expm1 antes do cálculo, sempre.')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 3 — Baselines (plano §5)
#
# Três números de referência. O primeiro é a âncora do critério de corte
# (+0,5 p.p.); os outros dois são testes de sanidade e piso.
# ---------------------------------------------------------------------------
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RF_PARAMS = dict(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)


def avaliar(cols, rotulo, verbose=True):
    """Treina RF em train_05[cols] e mede em valid_05. MAPE em escala original."""
    t0 = time.time()
    m = RandomForestRegressor(**RF_PARAMS).fit(X_tr[cols], y_tr_log)
    pred_log = m.predict(X_va[cols])
    pred_eur = np.expm1(pred_log)
    r = {
        'conjunto': rotulo,
        'n_features': len(cols),
        'mape': mape_eur(y_va_eur, pred_log),
        'mae': mean_absolute_error(y_va_eur, pred_eur),
        'rmse': float(np.sqrt(mean_squared_error(y_va_eur, pred_eur))),
        'segundos': time.time() - t0,
    }
    if verbose:
        print(f"  {rotulo:<34} {r['n_features']:>3} feat | "
              f"MAPE {r['mape']*100:5.2f}% | MAE €{r['mae']:5.1f} | "
              f"RMSE €{r['rmse']:6.1f} | {r['segundos']:4.1f}s")
    return r, m


print('Baseline 1 — RF com todas as 71 features (âncora do critério de corte)')
BASE_71, MODELO_71 = avaliar(FEATURES_04, 'todas as 71 features')

print('\nBaseline 2 — RF só com GROUP_COLS (piso: o que 3 colunas entregam)')
BASE_GC, _ = avaliar(GROUP_COLS, 'GROUP_COLS (3 features)')

# Baseline 3 — mediana de GROUP_COLS, a mesma regra da Etapa 01 (sem modelo)
print('\nBaseline 3 — mediana de GROUP_COLS (regra da Etapa 01, sem modelo)')
_ref = train_05.groupby(GROUP_COLS, observed=True)[TARGET].median()
_pred_med = valid_05.set_index(GROUP_COLS).index.map(_ref)
_pred_med = pd.Series(_pred_med, index=valid_05.index).astype(float)
_fallback = train_05[TARGET].median()
_cobertura = _pred_med.notna().mean()
_pred_med = _pred_med.fillna(_fallback).values

BASE_MED = {
    'conjunto': 'mediana GROUP_COLS (Etapa 01)',
    'n_features': 3,
    'mape': float(np.mean(np.abs(y_va_eur - _pred_med) / y_va_eur)),
    'mae': mean_absolute_error(y_va_eur, _pred_med),
    'rmse': float(np.sqrt(mean_squared_error(y_va_eur, _pred_med))),
    'segundos': 0.0,
}
print(f"  {BASE_MED['conjunto']:<34}   3 feat | MAPE {BASE_MED['mape']*100:5.2f}% | "
      f"MAE €{BASE_MED['mae']:5.1f} | RMSE €{BASE_MED['rmse']:6.1f}")
print(f'  (cobertura direta do grupo: {_cobertura*100:.1f}%; resto via mediana global)')

MAPE_BASELINE = BASE_71['mape']
LIMITE_CORTE = MAPE_BASELINE + 0.005          # critério validado: +0,5 p.p.

print(f'\n{"="*72}')
print(f'  MAPE baseline (71 features) ... {MAPE_BASELINE*100:5.2f}%')
print(f'  Limite do corte (+0,5 p.p.) ... {LIMITE_CORTE*100:5.2f}%')
print(f'  Piso da EDA (Análise 2.9) ..... 20,80%')
print(f'{"="*72}')

# Sanidade: o modelo precisa bater a mediana dos comparáveis
if BASE_71['mape'] >= BASE_MED['mape']:
    print('\n  ATENÇÃO: o RF não bateu a mediana de GROUP_COLS — investigar pipeline.')
else:
    _ganho = (BASE_MED['mape'] - BASE_71['mape']) * 100
    print(f'\n  RF supera a mediana da Etapa 01 em {_ganho:.2f} p.p.: OK')

# Faixa esperada 20-30%; abaixo disso, suspeitar de vazamento
assert 0.0 < MAPE_BASELINE < 1.0, 'MAPE fora de escala — expm1 aplicado?'
if MAPE_BASELINE < 0.15:
    print('  ATENÇÃO: MAPE abaixo de 15% — abaixo do piso da EDA. Investigar vazamento.')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 4 — Diagnóstico barato: mutual information + mRMR (plano §6)
#
# NÃO é seletor: nenhuma feature é descartada aqui. Serve para confrontar
# os seletores caros depois.
# ---------------------------------------------------------------------------
from sklearn.feature_selection import mutual_info_regression

t0 = time.time()

# Mutual information: captura dependência NÃO-monótona que o Spearman da 2.1 perde
mi = mutual_info_regression(X_tr, y_tr_log, random_state=RANDOM_STATE)
MI = pd.Series(mi, index=FEATURES_04).sort_values(ascending=False)

# mRMR (MID: relevância - redundância média): a única medida que vê as duas coisas
_corr_abs = X_tr.corr(method='spearman').abs()
np.fill_diagonal(_corr_abs.values, np.nan)
REDUNDANCIA = _corr_abs.mean()                      # redundância média com as demais
MRMR = (MI - REDUNDANCIA).sort_values(ascending=False)

print(f'Calculado em {time.time()-t0:.1f}s\n')

DIAG = pd.DataFrame({
    'mutual_info': MI,
    'redundancia_media': REDUNDANCIA,
    'mrmr': MRMR,
}).sort_values('mutual_info', ascending=False)

print('Top 15 por mutual information:')
print(DIAG.head(15).round(4).to_string())

print('\nMaior redundância média (candidatas que o mRMR penaliza):')
print(DIAG.sort_values('redundancia_media', ascending=False)
          .head(8)[['mutual_info', 'redundancia_media', 'mrmr']].round(4).to_string())

# O caso previsto no plano: relação em U que o Spearman não vê
print('\nColunas onde MI é alta mas o Spearman da 2.1 era fraco (relação não-monótona):')
_spear = X_tr.corrwith(pd.Series(y_tr_log, index=X_tr.index), method='spearman').abs()
_nao_mono = pd.DataFrame({'mi': MI, 'spearman_abs': _spear})
_nao_mono['razao'] = _nao_mono['mi'] / _nao_mono['spearman_abs'].clip(lower=0.01)
print(_nao_mono.sort_values('razao', ascending=False).head(6).round(4).to_string())

print(f'\n{len(DIAG)} features diagnosticadas. Nenhuma descartada neste bloco (plano §6).')

In [ ]:
# =====================================================================
# CELULA DESATIVADA (Opcao C, plano 07 §1.1) — NAO REEXECUTAR
#
# Resultado ja consolidado em FEATURES_05 (celula da Etapa 05 reduzida).
# Codigo preservado COMENTADO para auditoria de como as 27 features
# foram escolhidas. Custo evitado no restart: ~70 min.
# Saidas originais permanecem no markdown de consolidacao da Etapa 05.
# =====================================================================
# # ---------------------------------------------------------------------------
# # Portões 5 e 6 — Importância por árvores, três lentes (plano §7)
# #
# #   MDI ........... redução de impureza (viés: alta cardinalidade)
# #   split count ... frequência de uso  (revela uso alto com ganho baixo)
# #   permutation ... piora do MAPE em valid_05 — ÚNICA fora da amostra
# # ---------------------------------------------------------------------------
# from sklearn.inspection import permutation_importance
#
# # Lente 1 — MDI (do modelo baseline já treinado)
# MDI = pd.Series(MODELO_71.feature_importances_, index=FEATURES_04)
#
# # Lente 2 — split count: quantas vezes cada feature foi usada em todas as árvores
# _contagem = np.zeros(len(FEATURES_04))
# for arvore in MODELO_71.estimators_:
#     usadas = arvore.tree_.feature
#     usadas = usadas[usadas >= 0]                 # -2 marca nó folha
#     _contagem += np.bincount(usadas, minlength=len(FEATURES_04))
# SPLITS = pd.Series(_contagem, index=FEATURES_04)
#
# # Lente 3 — permutation importance em valid_05, com scorer de MAPE em euro
# def _scorer_mape(estimator, X, y_log):
#     """Negativo do MAPE em escala original: maior = melhor, como o sklearn espera."""
#     return -mape_eur(np.expm1(y_log), estimator.predict(X))
#
# t0 = time.time()
# _perm = permutation_importance(
#     MODELO_71, X_va, y_va_log,
#     scoring=_scorer_mape, n_repeats=5,
#     random_state=RANDOM_STATE, n_jobs=-1,
# )
# print(f'Permutation importance calculada em {time.time()-t0:.0f}s')
#
# PERM = pd.Series(_perm.importances_mean, index=FEATURES_04)
# PERM_STD = pd.Series(_perm.importances_std, index=FEATURES_04)
#
# ARVORES = pd.DataFrame({
#     'mdi': MDI,
#     'splits': SPLITS.astype(int),
#     'perm': PERM,
#     'perm_std': PERM_STD,
# }).sort_values('perm', ascending=False)
#
# print('\nTop 20 por permutation importance (piora do MAPE ao embaralhar):')
# print(ARVORES.head(20).round(5).to_string())
#
# # O caso que o plano manda caçar: alta em MDI, nula em permutação = decorou o treino
# _r_mdi = MDI.rank(ascending=False)
# _r_perm = PERM.rank(ascending=False)
# DIVERGENCIA = pd.DataFrame({
#     'rank_mdi': _r_mdi.astype(int),
#     'rank_perm': _r_perm.astype(int),
#     'salto': (_r_perm - _r_mdi).astype(int),
#     'mdi': MDI, 'perm': PERM,
# })
# print('\nMaior divergência — bem colocada em MDI, mal em permutação (suspeita de overfit):')
# print(DIVERGENCIA.sort_values('salto', ascending=False).head(8).round(5).to_string())
#
# print('\nFeatures com permutation <= 0 (embaralhar não piorou, ou até melhorou):')
# _inuteis = PERM[PERM <= 0].sort_values()
# print(f'  {len(_inuteis)} de {len(FEATURES_04)}')
# print('  ' + ', '.join(_inuteis.index[:20]))

In [ ]:
# =====================================================================
# CELULA DESATIVADA (Opcao C, plano 07 §1.1) — NAO REEXECUTAR
#
# Resultado ja consolidado em FEATURES_05 (celula da Etapa 05 reduzida).
# Codigo preservado COMENTADO para auditoria de como as 27 features
# foram escolhidas. Custo evitado no restart: ~70 min.
# Saidas originais permanecem no markdown de consolidacao da Etapa 05.
# =====================================================================
# # ---------------------------------------------------------------------------
# # Portão 7 — NULL IMPORTANCE (seletor principal, plano §8) · PARALELIZADO §4.3a
# #
# # Embaralha o ALVO 50x e coleta a distribuição de importância sob ruído puro.
# # Mantém as features que superam seu PRÓPRIO percentil 95 da distribuição nula.
# #
# # Distinção conceitual:
# #   permutation importance -> embaralha UMA COLUNA  (quanto ela contribui)
# #   null importance ------> embaralha O ALVO        (que importância surge por acaso)
# #
# # Regra de ouro (§4.3): n_jobs alto no laço EXTERNO, n_jobs=1 no RF INTERNO.
# # O inverso causa oversubscription — mais threads que núcleos.
# # ---------------------------------------------------------------------------
# from joblib import Parallel, delayed
#
# N_SHUFFLES = 50
#
#
# def _importancia_sob_ruido(X, y, seed):
#     """Um shuffle do alvo + 1 RF. Devolve o vetor de importâncias sob ruído puro."""
#     rng = np.random.default_rng(seed)
#     y_shuffled = rng.permutation(y)
#     m = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1)
#     m.fit(X, y_shuffled)
#     return m.feature_importances_
#
#
# t0 = time.time()
# _nulas = Parallel(n_jobs=10, backend='loky')(
#     delayed(_importancia_sob_ruido)(X_tr, y_tr_log, s) for s in range(N_SHUFFLES)
# )
# DIST_NULA = np.vstack(_nulas)                       # (50, 71)
# print(f'{N_SHUFFLES} shuffles do alvo em {time.time()-t0:.0f}s '
#       f'(paralelo: 10 processos x n_jobs=1)')
# print(f'distribuição nula: {DIST_NULA.shape}')
#
# # Score = importância real / percentil 95 da nula. Score < 1 = indistinguível de ruído.
# _p95_nulo = np.percentile(DIST_NULA, 95, axis=0)
# NULL_IMP = pd.DataFrame({
#     'imp_real': MDI.values,
#     'p95_nulo': _p95_nulo,
#     'score': MDI.values / np.where(_p95_nulo > 0, _p95_nulo, 1e-12),
# }, index=FEATURES_04).sort_values('score', ascending=False)
#
# APROVADAS_NULL = NULL_IMP.index[NULL_IMP['score'] >= 1.0].tolist()
# REJEITADAS_NULL = NULL_IMP.index[NULL_IMP['score'] < 1.0].tolist()
#
# print(f'\n{"="*72}')
# print(f'  Aprovadas (score >= 1) .... {len(APROVADAS_NULL):>2} de {len(FEATURES_04)}')
# print(f'  Rejeitadas (score < 1) .... {len(REJEITADAS_NULL):>2}')
# print(f'{"="*72}')
#
# print('\nTop 15 por score:')
# print(NULL_IMP.head(15).round(4).to_string())
#
# print('\nRejeitadas — não se distinguem de ruído:')
# print(NULL_IMP.tail(len(REJEITADAS_NULL))[['imp_real', 'p95_nulo', 'score']]
#       .round(4).to_string())

In [ ]:
# =====================================================================
# CELULA DESATIVADA (Opcao C, plano 07 §1.1) — NAO REEXECUTAR
#
# Resultado ja consolidado em FEATURES_05 (celula da Etapa 05 reduzida).
# Codigo preservado COMENTADO para auditoria de como as 27 features
# foram escolhidas. Custo evitado no restart: ~70 min.
# Saidas originais permanecem no markdown de consolidacao da Etapa 05.
# =====================================================================
# # ---------------------------------------------------------------------------
# # Portão 7b — NULL IMPORTANCE corrigido: numerador = PERMUTATION, não MDI
# #
# # MOTIVO DA CORREÇÃO (achado desta etapa, não previsto no plano):
# # a primeira versão usou MDI como numerador do score. O MDI é enviesado a favor
# # de alta cardinalidade — e o viés contamina o score de forma SISTEMÁTICA, porque
# # contínuas acumulam MDI mesmo sob ruído puro (p95 nulo de latitude/longitude/
# # host_experiencia_dias ficou em 0,05-0,06, contra ~0,00 das binárias).
# #
# # Consequência: features foram punidas por serem contínuas, não por serem
# # irrelevantes. dist_museumplein_km era 4ª na permutação (fora da amostra) e
# # foi REJEITADA pelo score baseado em MDI. As duas medidas discordavam
# # frontalmente, e a permutação é a confiável.
# #
# # Correção: a distribuição nula passa a ser de PERMUTATION IMPORTANCE, medida
# # com o MESMO estimador e a MESMA métrica (MAPE em euro) usados no numerador.
# # Numerador e denominador ficam na mesma escala e com o mesmo viés — que é a
# # condição para a razão ser interpretável.
# # ---------------------------------------------------------------------------
# N_SHUFFLES_B = 50
# N_REPEATS_B = 3
#
#
# def _perm_sob_ruido(X, y, seed):
#     """Embaralha o ALVO, treina, e mede permutation importance sob ruído puro.
#
#     Devolve o vetor de importâncias que surge por acaso — na mesma unidade
#     (piora do MAPE em euro) do numerador real.
#     """
#     rng = np.random.default_rng(seed)
#     y_shuffled = rng.permutation(y)
#     m = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1)
#     m.fit(X, y_shuffled)
#
#     def _scorer(est, Xi, yi_log):
#         return -mape_eur(np.expm1(yi_log), est.predict(Xi))
#
#     r = permutation_importance(m, X, y_shuffled, scoring=_scorer,
#                                n_repeats=N_REPEATS_B, random_state=seed, n_jobs=1)
#     return r.importances_mean
#
#
# t0 = time.time()
# _nulas_b = Parallel(n_jobs=10, backend='loky')(
#     delayed(_perm_sob_ruido)(X_tr, y_tr_log, s) for s in range(N_SHUFFLES_B)
# )
# DIST_NULA_PERM = np.vstack(_nulas_b)                 # (50, 71)
# print(f'{N_SHUFFLES_B} shuffles com permutation em {time.time()-t0:.0f}s')
# print(f'distribuição nula: {DIST_NULA_PERM.shape}')
#
# # Numerador: permutation importance REAL, já calculada no portão 6 em valid_05.
# # Recalculada aqui no TREINO para casar a base com a distribuição nula.
# t0 = time.time()
# _perm_tr = permutation_importance(
#     MODELO_71, X_tr, y_tr_log, scoring=_scorer_mape,
#     n_repeats=N_REPEATS_B, random_state=RANDOM_STATE, n_jobs=-1,
# )
# PERM_TREINO = pd.Series(_perm_tr.importances_mean, index=FEATURES_04)
# print(f'permutation real (treino) em {time.time()-t0:.0f}s')
#
# _p95_perm = np.percentile(DIST_NULA_PERM, 95, axis=0)
# NULL_IMP_B = pd.DataFrame({
#     'perm_real': PERM_TREINO.values,
#     'p95_nulo': _p95_perm,
#     'score': PERM_TREINO.values / np.maximum(_p95_perm, 1e-9),
# }, index=FEATURES_04).sort_values('score', ascending=False)
#
# APROVADAS_NULLB = NULL_IMP_B.index[NULL_IMP_B['score'] >= 1.0].tolist()
# REJEITADAS_NULLB = NULL_IMP_B.index[NULL_IMP_B['score'] < 1.0].tolist()
#
# print(f'\n{"="*72}')
# print(f'  Aprovadas (score >= 1) .... {len(APROVADAS_NULLB):>2} de {len(FEATURES_04)}')
# print(f'  Rejeitadas ................ {len(REJEITADAS_NULLB):>2}')
# print(f'  [versão anterior, com MDI:  {len(APROVADAS_NULL)} aprovadas]')
# print(f'{"="*72}')
#
# print('\nTop 20 por score (numerador = permutation):')
# print(NULL_IMP_B.head(20).round(4).to_string())
#
# # Efeito da correção nas 3 colunas que motivaram a Opção B
# print('\nEfeito da correção nas colunas críticas:')
# _criticas = ['neighbourhood_cleansed', 'gap_value_vs_geral', 'gap_location_vs_geral',
#              'host_is_superhost', 'dist_museumplein_km', 'latitude', 'longitude']
# _cmp = pd.DataFrame({
#     'score_MDI (v1)': NULL_IMP['score'],
#     'score_PERM (v2)': NULL_IMP_B['score'],
# }).loc[_criticas]
# _cmp['v1'] = np.where(_cmp['score_MDI (v1)'] >= 1, 'aprovada', 'rejeitada')
# _cmp['v2'] = np.where(_cmp['score_PERM (v2)'] >= 1, 'aprovada', 'rejeitada')
# print(_cmp.round(3).to_string())

In [ ]:
# =====================================================================
# CELULA DESATIVADA (Opcao C, plano 07 §1.1) — NAO REEXECUTAR
#
# Resultado ja consolidado em FEATURES_05 (celula da Etapa 05 reduzida).
# Codigo preservado COMENTADO para auditoria de como as 27 features
# foram escolhidas. Custo evitado no restart: ~70 min.
# Saidas originais permanecem no markdown de consolidacao da Etapa 05.
# =====================================================================
# # ---------------------------------------------------------------------------
# # Portão 8 — RFECV (plano §10): define o conjunto final
# #
# # Substitui a curva de parcimônia: usa k folds em vez de partição única.
# # Responde "qual o MENOR conjunto que não perde performance" — não
# # "quais features importam".
# #
# # Por que este é o seletor que decide (e o Null Importance não):
# # o RFECV avalia SUBCONJUNTOS. O Null Importance avalia features ISOLADAS, e
# # num RF com 71 colunas correlacionadas cada feature é substituível pelas
# # vizinhas — embaralhar uma só quase não move o MAPE. Isso mede redundância,
# # não irrelevância.
# #
# # CV interna a train_05: valid_05 NÃO é tocada aqui (plano §2), para que a
# # comparação final do portão 13 permaneça honesta.
# # ---------------------------------------------------------------------------
# from sklearn.feature_selection import RFECV
# from sklearn.model_selection import KFold
# from sklearn.metrics import make_scorer
#
#
# def _mape_scorer_eur(y_log_true, y_log_pred):
#     """MAPE em EURO a partir de valores em log1p. Negativo: maior = melhor."""
#     return -float(np.mean(
#         np.abs(np.expm1(y_log_true) - np.expm1(y_log_pred)) / np.expm1(y_log_true)
#     ))
#
#
# SCORER_MAPE = make_scorer(_mape_scorer_eur, greater_is_better=True)
#
# t0 = time.time()
# RFECV_SEL = RFECV(
#     estimator=RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1),
#     step=5,                                   # §4.1: 10 min vs 49 min do step=1
#     cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
#     scoring=SCORER_MAPE,
#     min_features_to_select=3,
#     n_jobs=5,                                 # 5 folds em paralelo; n_jobs=1 no RF
# )
# RFECV_SEL.fit(X_tr, y_tr_log)
# print(f'RFECV concluído em {time.time()-t0:.0f}s')
#
# FEAT_RFECV = [c for c, s in zip(FEATURES_04, RFECV_SEL.support_) if s]
# print(f'\nÓtimo do RFECV: {RFECV_SEL.n_features_} features')
#
# # Curva: MAPE médio da CV por número de features
# _curva = pd.DataFrame({
#     'n_features': RFECV_SEL.cv_results_['n_features'],
#     'mape_cv': -RFECV_SEL.cv_results_['mean_test_score'],
#     'std': RFECV_SEL.cv_results_['std_test_score'],
# }).sort_values('n_features')
# print('\nCurva de parcimônia (MAPE médio em 5 folds):')
# print(_curva.assign(mape_cv=lambda d: (d.mape_cv*100).round(2),
#                     std=lambda d: (d['std']*100).round(2)).to_string(index=False))
#
# # Critério validado: menor conjunto dentro de +0,5 p.p. do baseline de 71
# _mape_71_cv = _curva.loc[_curva.n_features == _curva.n_features.max(), 'mape_cv'].iloc[0]
# _limite_cv = _mape_71_cv + 0.005
# _elegiveis = _curva[_curva.mape_cv <= _limite_cv]
# N_CORTE = int(_elegiveis.n_features.min()) if len(_elegiveis) else int(_curva.n_features.max())
#
# print(f'\n{"="*72}')
# print(f'  MAPE CV com {int(_curva.n_features.max())} features ... {_mape_71_cv*100:5.2f}%')
# print(f'  Limite (+0,5 p.p.) .............. {_limite_cv*100:5.2f}%')
# print(f'  Menor conjunto dentro do limite . {N_CORTE} features')
# print(f'  Ótimo do próprio RFECV .......... {RFECV_SEL.n_features_} features')
# print(f'{"="*72}')
#
# # Ranking do RFECV: 1 = sobreviveu até o fim
# RANK_RFECV = pd.Series(RFECV_SEL.ranking_, index=FEATURES_04).sort_values()
# print(f'\nFeatures com ranking 1 (selecionadas pelo RFECV): {(RANK_RFECV == 1).sum()}')
# print(', '.join(RANK_RFECV[RANK_RFECV == 1].index.tolist()))

In [ ]:
# =====================================================================
# CELULA DESATIVADA (Opcao C, plano 07 §1.1) — NAO REEXECUTAR
#
# Resultado ja consolidado em FEATURES_05 (celula da Etapa 05 reduzida).
# Codigo preservado COMENTADO para auditoria de como as 27 features
# foram escolhidas. Custo evitado no restart: ~70 min.
# Saidas originais permanecem no markdown de consolidacao da Etapa 05.
# =====================================================================
# # ---------------------------------------------------------------------------
# # Portão 9 — BORUTA (confirmação secundária, plano §9)
# #
# # max_iter=50 é o PISO travado em §3.1: com max_iter=5, um teste sintético onde
# # uma feature explicava 99,9% do alvo rejeitou TODAS as seis. Boruta precisa de
# # iterações para acumular significância — max_iter baixo não dá resultado
# # aproximado, dá resultado errado no sentido conservador.
# #
# # Base completa, sem amostragem (decisão do usuário). Descartável se > 20 min.
# # ---------------------------------------------------------------------------
# from boruta import BorutaPy
#
# # BorutaPy usa APIs de numpy removidas na v2 (np.float, np.int, np.bool)
# for _alias, _tipo in (('float', float), ('int', int), ('bool', bool)):
#     if not hasattr(np, _alias):
#         setattr(np, _alias, _tipo)
#
# t0 = time.time()
# _boruta = BorutaPy(
#     RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
#     n_estimators='auto', max_iter=50, random_state=RANDOM_STATE, verbose=0,
# )
# _boruta.fit(X_tr.values, y_tr_log)
# T_BORUTA = time.time() - t0
#
# BORUTA_CONF = [c for c, s in zip(FEATURES_04, _boruta.support_) if s]
# BORUTA_FRACA = [c for c, s in zip(FEATURES_04, _boruta.support_weak_) if s]
# BORUTA_REJ = [c for c in FEATURES_04
#               if c not in BORUTA_CONF and c not in BORUTA_FRACA]
#
# print(f'Boruta concluído em {T_BORUTA:.0f}s ({T_BORUTA/60:.1f} min)')
# print(f'  limite de descarte acordado: 20 min -> '
#       f'{"MANTIDO" if T_BORUTA < 1200 else "DESCARTAR"}')
# print(f'\n  confirmadas ........ {len(BORUTA_CONF):>2} de {len(FEATURES_04)}')
# print(f'  tentativas (fracas)  {len(BORUTA_FRACA):>2}')
# print(f'  rejeitadas ......... {len(BORUTA_REJ):>2}')
#
# print('\nConfirmadas:')
# print('  ' + ', '.join(BORUTA_CONF) if BORUTA_CONF else '  (nenhuma)')
# if BORUTA_FRACA:
#     print('\nTentativas:')
#     print('  ' + ', '.join(BORUTA_FRACA))
#
# # Boruta é all-relevant, RFECV é minimal-optimal: divergência é ESPERADA
# _no_corte_rej = [c for c in FEAT_CORTE if c in BORUTA_REJ]
# _fora_conf = [c for c in BORUTA_CONF if c not in FEAT_CORTE]
# print(f'\nDivergência com o corte de {N_CORTE} do RFECV:')
# print(f'  no corte mas Boruta rejeitou ... {len(_no_corte_rej)}: {_no_corte_rej}')
# print(f'  Boruta confirmou mas fora ...... {len(_fora_conf)}: {_fora_conf}')
# print('  (divergência é esperada: Boruta busca TUDO que tem sinal;')
# print('   RFECV busca o MENOR conjunto suficiente)')

In [ ]:
# =====================================================================
# CELULA DESATIVADA (Opcao C, plano 07 §1.1) — NAO REEXECUTAR
#
# Resultado ja consolidado em FEATURES_05 (celula da Etapa 05 reduzida).
# Codigo preservado COMENTADO para auditoria de como as 27 features
# foram escolhidas. Custo evitado no restart: ~70 min.
# Saidas originais permanecem no markdown de consolidacao da Etapa 05.
# =====================================================================
# # ---------------------------------------------------------------------------
# # Portão 12 — ESTABILIDADE do RFECV (plano §12, escopo alterado)
# #
# # MUDANÇA DE ESCOPO (decisão do usuário): o plano previa testar a estabilidade
# # do Null Importance. Mas ele se mostrou inadequado neste dataset — rejeitou
# # 63 de 71 features, o que contradiz o baseline (71 feat batem 3 feat por
# # 4,89 p.p., logo há sinal muito além de 8 colunas). Testar a estabilidade de
# # um método já sabidamente fraco mediria a coisa errada.
# #
# # O RFECV é quem FIXA o número de features. Se N_CORTE mudar muito entre
# # seeds, o corte é arbitrário — e isso precisa ser sabido antes da Etapa 06.
# #
# # Paralelizado: 3 seeds simultâneas, ~3 núcleos cada (§4.3b).
# # ---------------------------------------------------------------------------
# SEEDS_ESTAB = (42, 7, 2024)
#
#
# def _rfecv_com_seed(X, y, seed):
#     """Pipeline RFECV completo para uma seed. Devolve (n_otimo, curva, features)."""
#     sel = RFECV(
#         estimator=RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=1),
#         step=5,
#         cv=KFold(n_splits=5, shuffle=True, random_state=seed),
#         scoring=SCORER_MAPE,
#         min_features_to_select=3,
#         n_jobs=1,
#     )
#     sel.fit(X, y)
#     curva = pd.DataFrame({
#         'n_features': sel.cv_results_['n_features'],
#         'mape_cv': -sel.cv_results_['mean_test_score'],
#     }).sort_values('n_features')
#
#     # Mesmo critério do portão 8: menor conjunto dentro de +0,5 p.p.
#     mape_full = curva.loc[curva.n_features == curva.n_features.max(), 'mape_cv'].iloc[0]
#     eleg = curva[curva.mape_cv <= mape_full + 0.005]
#     n_corte = int(eleg.n_features.min()) if len(eleg) else int(curva.n_features.max())
#
#     feats = [c for c, s in zip(X.columns, sel.support_) if s]
#     return {'seed': seed, 'n_otimo_rfecv': int(sel.n_features_),
#             'n_corte': n_corte, 'mape_full': float(mape_full),
#             'features_rfecv': feats, 'curva': curva}
#
#
# t0 = time.time()
# ESTAB = Parallel(n_jobs=3, backend='loky')(
#     delayed(_rfecv_com_seed)(X_tr, y_tr_log, s) for s in SEEDS_ESTAB
# )
# print(f'3 seeds de RFECV em {time.time()-t0:.0f}s ({(time.time()-t0)/60:.1f} min)')
#
# print(f'\n{"="*72}')
# print(f'  {"seed":>6} {"n_corte (+0,5pp)":>18} {"n_ótimo RFECV":>15} {"MAPE 71 feat":>14}')
# for r in ESTAB:
#     print(f'  {r["seed"]:>6} {r["n_corte"]:>18} {r["n_otimo_rfecv"]:>15} '
#           f'{r["mape_full"]*100:>13.2f}%')
# print(f'{"="*72}')
#
# _cortes = [r['n_corte'] for r in ESTAB]
# print(f'\nN_CORTE: min={min(_cortes)} max={max(_cortes)} '
#       f'amplitude={max(_cortes)-min(_cortes)}')
# print(f'  (portão 8, seed 42: {N_CORTE})')
#
# # Estabilidade de composição: quantas features aparecem nas 3 execuções
# _conjuntos = [set(r['features_rfecv']) for r in ESTAB]
# _nas_tres = set.intersection(*_conjuntos)
# _em_alguma = set.union(*_conjuntos)
# print(f'\nComposição do conjunto RFECV entre seeds:')
# print(f'  nas 3 execuções ... {len(_nas_tres):>2}')
# print(f'  em ao menos 1 ..... {len(_em_alguma):>2}')
# print(f'  índice de Jaccard . {len(_nas_tres)/len(_em_alguma):.3f}')
#
# # Frequência por feature: 3 = estável, 1 = ruído
# _freq = pd.Series(
#     {c: sum(c in s for s in _conjuntos) for c in FEATURES_04}
# ).sort_values(ascending=False)
# print(f'\nFeatures por frequência de seleção (de 3 seeds):')
# for f in (3, 2, 1, 0):
#     _cols = _freq[_freq == f].index.tolist()
#     print(f'  em {f}/3 ... {len(_cols):>2}')
#
# print('\nInstáveis (em 1 ou 2 de 3) — entram/saem conforme a seed:')
# _inst = _freq[(_freq >= 1) & (_freq <= 2)]
# print('  ' + ', '.join(f'{c}({v})' for c, v in _inst.items()) if len(_inst) else '  nenhuma')

In [ ]:
# =====================================================================
# CELULA DESATIVADA (Opcao C, plano 07 §1.1) — NAO REEXECUTAR
#
# Resultado ja consolidado em FEATURES_05 (celula da Etapa 05 reduzida).
# Codigo preservado COMENTADO para auditoria de como as 27 features
# foram escolhidas. Custo evitado no restart: ~70 min.
# Saidas originais permanecem no markdown de consolidacao da Etapa 05.
# =====================================================================
# # ---------------------------------------------------------------------------
# # Portão 10-11 — Matriz de consenso + blindagem de negócio (plano §11)
# # ---------------------------------------------------------------------------
# CONSENSO = pd.DataFrame({
#     'mutual_info': DIAG['mutual_info'],
#     'mrmr': DIAG['mrmr'],
#     'mdi': ARVORES['mdi'],
#     'splits': ARVORES['splits'],
#     'perm_valid': ARVORES['perm'],
#     'null_score': NULL_IMP_B['score'],
#     'boruta': ['confirmada' if c in BORUTA_CONF else
#                ('tentativa' if c in BORUTA_FRACA else 'rejeitada') for c in FEATURES_04],
#     'rfecv_rank': RANK_RFECV,
#     'no_corte_26': [c in FEAT_CORTE for c in FEATURES_04],
#     'freq_seeds': freq,
# }).sort_values(['no_corte_26', 'perm_valid'], ascending=[False, False])
#
# print('Matriz de consenso — 7 medidas por feature (top 30):')
# print(CONSENSO.head(30).round(4).to_string())
#
# # --- Blindagem: GROUP_COLS são intocáveis (CLAUDE.md §3.1) -------------------
# FEATURES_05 = list(FEAT_CORTE)
# BLINDADAS = []
# for g in GROUP_COLS:
#     if g not in FEATURES_05:
#         FEATURES_05.append(g)
#         BLINDADAS.append(g)
#
# print(f'\n{"="*72}')
# print('BLINDAGEM DE NEGÓCIO (plano §11.2)')
# print(f'{"="*72}')
# print(f'  GROUP_COLS readicionadas ..... {BLINDADAS if BLINDADAS else "nenhuma"}')
# print(f'    motivo: sustentam ref_price e o EV das Etapas 01/09/10.')
# print(f'    neighbourhood_cleansed caiu porque dist_centro_km e')
# print(f'    dist_museumplein_km capturam a geografia de forma contínua —')
# print(f'    o sinal do bairro migrou, não desapareceu.')
#
# # Decisões do usuário, registradas contra a evidência disponível
# print(f'\n  gap_location_vs_geral ........ FORA (decisão do usuário)')
# print(f'    RFECV eliminou; irmão gap_value_vs_geral sobreviveu.')
# print(f'    Decisão: seguir com apenas um dos dois gaps.')
# print(f'  host_is_superhost ............ FORA (decisão do usuário)')
# print(f'    EDA: efeito marginal ZERO (€125 vs €125), +7,7% sob controle,')
# print(f'    positivo em 64,2% de 190 células. Reincidência de Simpson.')
# print(f'    Instável no teste de estabilidade: em 1 de 3 seeds.')
# print(f'    Decisão: aceitar a eliminação.')
#
# # Ordena FEATURES_05 na ordem temática dos GRUPOS da Etapa 04
# _ordem = [c for cols in GRUPOS_04.values() for c in cols if c in FEATURES_05]
# FEATURES_05 = _ordem + [c for c in FEATURES_05 if c not in _ordem]
#
# print(f'\n{"="*72}')
# print(f'  FEATURES_05 final: {len(FEATURES_05)} features')
# print(f'{"="*72}')
# for g, cols in GRUPOS_04.items():
#     _sel = [c for c in cols if c in FEATURES_05]
#     print(f'  {g:<34} {len(_sel):>2} de {len(cols):>2}')
#     if _sel:
#         print(f'      {", ".join(_sel)}')

In [ ]:
# =====================================================================
# CELULA DESATIVADA (Opcao C, plano 07 §1.1) — NAO REEXECUTAR
#
# Resultado ja consolidado em FEATURES_05 (celula da Etapa 05 reduzida).
# Codigo preservado COMENTADO para auditoria de como as 27 features
# foram escolhidas. Custo evitado no restart: ~70 min.
# Saidas originais permanecem no markdown de consolidacao da Etapa 05.
# =====================================================================
# # ---------------------------------------------------------------------------
# # Portão 13 — Validação final + construção de df05 (plano §13, §15, §16)
# # ---------------------------------------------------------------------------
# print('Validação final em valid_05 — quatro conjuntos:')
# _top10 = ARVORES.head(10).index.tolist()
# _r = []
# for cols, rot in [(FEATURES_04, 'todas as 71 (baseline)'),
#                   (FEATURES_05, f'FEATURES_05 ({len(FEATURES_05)}) <-- selecionado'),
#                   (_top10, 'top-10 (sanidade)'),
#                   (GROUP_COLS, 'GROUP_COLS (piso)')]:
#     res, _ = avaliar(cols, rot)
#     _r.append(res)
# COMPARA = pd.DataFrame(_r)
#
# _m71 = COMPARA.loc[0, 'mape']
# _m05 = COMPARA.loc[1, 'mape']
# print(f'\n{"="*72}')
# print(f'  baseline 71 features ..... {_m71*100:5.2f}%')
# print(f'  FEATURES_05 ({len(FEATURES_05)}) .......... {_m05*100:5.2f}%')
# print(f'  diferença ................ {(_m05-_m71)*100:+5.2f} p.p.'
#       f'   (limite acordado: +0,50)')
# print(f'  critério de corte ........ {"ATENDIDO" if _m05 <= _m71+0.005 else "VIOLADO"}')
# print(f'  redução de features ...... {len(FEATURES_04)} -> {len(FEATURES_05)}'
#       f'  ({100*(1-len(FEATURES_05)/len(FEATURES_04)):.0f}% menos)')
# print(f'{"="*72}')
#
# # --- df05: VALORES ORIGINAIS de df04 (a preparação mínima NÃO vaza) ----------
# df05 = df04[['id', TARGET] + FEATURES_05].copy()
#
# print(f'\ndf05: {df05.shape[0]:,} x {df05.shape[1]}'.replace(',', '.'),
#       f'= id + {TARGET} + {len(FEATURES_05)} features')
#
# # --- Verificações do plano §16 ----------------------------------------------
# print('\nVerificações (plano §16):')
# assert len(df05) == df05['id'].nunique(), 'granularidade quebrada'
# print('  granularidade 1 linha/listing .......... OK')
# assert 'id' not in FEATURES_05 and TARGET not in FEATURES_05, 'chave/alvo nas features'
# print('  id e price fora de FEATURES_05 ........ OK')
# assert set(FEATURES_05).issubset(set(FEATURES_04)), 'feature criada nesta etapa'
# print('  FEATURES_05 ⊂ FEATURES_04 ............. OK')
# for g in GROUP_COLS:
#     assert g in FEATURES_05, f'blindagem falhou: {g}'
# print('  GROUP_COLS ⊂ FEATURES_05 (blindagem) .. OK')
#
# # A prova de que a preparação descartável ficou contida na etapa
# _n04 = df04[FEATURES_05].isna().sum()
# _n05 = df05[FEATURES_05].isna().sum()
# assert _n04.equals(_n05), 'preparação mínima vazou para df05'
# print(f'  nulos preservados ..................... OK '
#       f'({int(_n05.sum()):,} nulos em {int((_n05>0).sum())} colunas)'.replace(',', '.'))
# assert 0 < _m05 < 1, 'MAPE fora de escala — expm1 aplicado?'
# print('  MAPE em escala original (euro) ........ OK')
#
# print('\nTabela comparativa:')
# print(COMPARA.assign(mape=lambda d: (d.mape*100).round(2))
#              [['conjunto', 'n_features', 'mape', 'mae', 'rmse']].round(1).to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Etapa 05 — RESULTADO TRAVADO (Opção C, plano 07 §1.1)
#
# As células dos seletores (Null Importance, RFECV, Boruta, estabilidade) estão
# COMENTADAS acima: ~70 min de execução que não precisam ser repetidos a cada
# restart do kernel. Esta célula apenas declara o resultado e refiltra df04.
#
# ORIGEM das 27 features:
#   - RFECV (5 folds, step=5) -> corte de 26 pelo critério de +0,5 p.p.
#   - + blindagem de neighbourhood_cleansed (GROUP_COLS, CLAUDE.md §3.1)
#   - gap_location_vs_geral e host_is_superhost fora (decisão do usuário)
#
# VALIDADO na Etapa 05:
#   - MAPE 23,71% em valid_05 (baseline de 71 features: 23,32%; delta +0,39 p.p.)
#   - N_CORTE = 26 estável nas seeds 42 / 7 / 2024 (amplitude zero)
#   - as 16 confirmadas pelo Boruta são subconjunto exato das 26 do RFECV
# ---------------------------------------------------------------------------
FEATURES_05 = [
    # G1 — Capacidade e estrutura física (4)
    'accommodates', 'bathrooms', 'bedrooms', 'room_type',
    # G2 — Geografia (5) — nenhuma redundante
    'latitude', 'longitude', 'dist_centro_km', 'dist_museumplein_km',
    'neighbourhood_cleansed',
    # G3 — Amenities (1) — n_amenities absorve as 11 flags tem_*
    'n_amenities',
    # G4 — Host (1)
    'host_experiencia_dias',
    # G5 — Política de reserva e custos (7)
    'minimum_nights', 'maximum_nights', 'cleaning_fee', 'taxa_limpeza_por_noite',
    'security_deposit', 'extra_people', 'calendar_atualizado_dias',
    # G6 — Reviews e reputação (6)
    'review_scores_rating', 'gap_value_vs_geral', 'number_of_reviews',
    'reviews_per_month', 'dias_desde_primeira_review', 'dias_desde_ultima_review',
    # G7 — Disponibilidade e texto (3)
    'availability_30', 'availability_365', 'len_house_rules',
]

df05 = df04[['id', TARGET] + FEATURES_05].copy()

print(f'FEATURES_05 (travada): {len(FEATURES_05)} features')
print(f'df05: {df05.shape[0]:,} x {df05.shape[1]}'.replace(',', '.'),
      f'= id + {TARGET} + {len(FEATURES_05)} features')

# --- Teste de aceitação obrigatório (plano 07 §1.1, item 5) -----------------
# Se qualquer um falhar, algo na cadeia 02->04 mudou e a comparabilidade com
# os 23,71% da Etapa 05 está perdida.
assert len(FEATURES_05) == 27, f'esperadas 27 features, achei {len(FEATURES_05)}'
assert df05.shape == (17978, 29), f'shape divergente: {df05.shape} != (17978, 29)'
assert set(FEATURES_05).issubset(set(df04.columns)), 'feature ausente em df04'
assert len(set(FEATURES_05)) == 27, 'feature duplicada na lista'
assert 'id' not in FEATURES_05 and TARGET not in FEATURES_05, 'chave/alvo nas features'
for _g in GROUP_COLS:
    assert _g in FEATURES_05, f'blindagem GROUP_COLS falhou: {_g}'
print('\nTestes de aceitação (plano 07 §1.1): OK')
print('  27 features | shape (17.978, 29) | sem duplicata | GROUP_COLS presentes')

# Nulos preservados: df05 carrega os valores ORIGINAIS de df04
_n = df05[FEATURES_05].isna().sum()
print(f'  nulos preservados: {int(_n.sum()):,} em {int((_n>0).sum())} colunas'
      .replace(',', '.'))


## Consolidação da Etapa 05 — Feature Selection encerrada

Os 14 portões do plano (`docs/plans/05_feature_selection.md` §14) foram executados.

### Entregável 1 — `df05`

**17.978 × 29** = **1 chave** (`id`) + **1 alvo** (`price`) + **27 features** (`FEATURES_05`).

Redução de **71 → 27 features (−62%)** ao custo de **+0,39 p.p.** de MAPE — dentro do
limite de +0,5 p.p. fixado **antes** de ver a curva.

| Conjunto | Features | MAPE | MAE | RMSE |
|---|---|---|---|---|
| Todas (baseline) | 71 | 23,32% | €35,1 | €59,6 |
| **`FEATURES_05`** | **27** | **23,71%** | €35,5 | €59,9 |
| Top-10 (sanidade) | 10 | 24,94% | €37,2 | €60,8 |
| `GROUP_COLS` (piso) | 3 | 28,25% | €42,7 | €70,9 |

`df05` carrega os **valores originais** de `df04` — 23.227 nulos em 12 colunas
preservados. A preparação mínima **não vazou**: é a verificação que prova que a
imputação descartável ficou contida nesta etapa.

### Entregável 2 — a curva de parcimônia

| Features | MAPE CV | Ganho marginal |
|---|---|---|
| 3 | 32,43% | — |
| 6 | 27,17% | −5,26 |
| 11 | 25,69% | −1,48 |
| 16 | 24,53% | −1,16 |
| **26** | **23,99%** | −0,23 |
| 36 | 23,76% | −0,23 |
| 51 | 23,71% | −0,05 |
| 71 | 23,70% | 0,00 |

**A curva satura em ~36 features.** De 36 para 71 — o dobro de colunas — ganha-se
0,06 p.p. É ruído. As features além do corte não carregam informação nova; carregam a
mesma informação repetida.

### Entregável 3 — composição por grupo temático

| Grupo | Selecionadas | Leitura |
|---|---|---|
| G2 — Geografia | **5 de 5** | Nenhuma redundante: lat/long, 2 distâncias e o bairro sobrevivem juntos |
| G5 — Política e custos | 7 de 14 | Metade |
| G6 — Reviews | 6 de 15 | O bloco de 7 `review_scores_*` colapsa em 1 (`rating`) + 1 gap |
| G1 — Capacidade | 4 de 11 | As razões derivadas caem; ficam os absolutos |
| G7 — Disponibilidade | 3 de 5 | — |
| G3 — Amenities | **1 de 12** | `n_amenities` absorve as 11 flags `tem_*` |
| G4 — Host | **1 de 9** | Só `host_experiencia_dias` |

G3 e G4 são o achado mais forte: **21 colunas colapsam em 2**. A contagem agregada de
amenities carrega o que as flags individuais diziam — coerente com a EDA, onde TV caía
de +35% para +11,5% sob controle e máquina de lavar zerava.

### Entregável 4 — convergência dos três seletores

| Seletor | Aprovadas | Papel efetivo |
|---|---|---|
| Null Importance (v2) | 8 de 71 | Filtro de segurança |
| Boruta (2,2 min) | 16 de 71 | Confirmação |
| **RFECV** | **26** → 27 c/ blindagem | **Definiu o conjunto** |

**As 16 do Boruta são subconjunto exato das 26 do RFECV** — zero confirmações fora do
corte. Os métodos não se contradizem; o Boruta é apenas mais rigoroso. Convergência
independente, mais forte do que se houvesse confirmações externas ao corte.

### Entregável 5 — estabilidade (3 seeds do RFECV)

| Seed | N_CORTE | Ótimo RFECV | MAPE 71 feat |
|---|---|---|---|
| 42 | **26** | 61 | 23,70% |
| 7 | **26** | 61 | 23,72% |
| 2024 | **26** | 46 | 23,62% |

**N_CORTE = 26 nas três seeds, amplitude zero.** E as **26 features do corte estão
todas no núcleo estável** (presentes nas 3 execuções). Jaccard de composição: 0,742.

As 16 instáveis (em 1–2 de 3 seeds) são todas flags binárias e `review_scores_*`
secundários — nenhuma entrou em `FEATURES_05`. `host_is_superhost` apareceu em **1 de 3**.

### Entregável 6 — blindagem de negócio

| Coluna | Veredito estatístico | Decisão | Motivo |
|---|---|---|---|
| `neighbourhood_cleansed` | RFECV eliminou | **MANTIDA** | `GROUP_COLS` — sustenta `ref_price` e o EV das Etapas 01/09/10 |
| `gap_value_vs_geral` | RFECV manteve, Boruta rejeitou | **MANTIDA** | H5 sobreviveu ao seletor que decide |
| `gap_location_vs_geral` | RFECV eliminou | **FORA** | Decisão do usuário: seguir com um só gap |
| `host_is_superhost` | Eliminada por todos | **FORA** | Decisão do usuário, ciente da evidência contrária |

O caso de `neighbourhood_cleansed` merece registro: caiu porque `dist_centro_km` e
`dist_museumplein_km` capturam a geografia de forma contínua. O sinal do bairro
**migrou, não desapareceu** — por isso a blindagem custa pouco.

`host_is_superhost` sai **contra** a evidência da EDA (efeito marginal zero, **+7,7%**
sob controle, positivo em 64,2% de 190 células). A decisão foi tomada com o número na
tela. Se a Etapa 09 revelar viés sistemático em anúncios de superhost, este é o
primeiro lugar a revisar.

### Achados metodológicos — duas previsões do plano que se mostraram erradas

**1. O Null Importance não funcionou como seletor principal.** O plano previa que
aprovaria "quase tudo"; aprovou 8 de 71. A causa não é irrelevância: com 71 colunas
correlacionadas, cada feature é substituível pelas vizinhas, então embaralhar uma só
quase não move o MAPE. **Isso mede redundância, não irrelevância.** O mesmo mecanismo
explica o Boruta confirmar apenas 16.

**2. O viés de cardinalidade não vem do MDI.** A primeira versão usou MDI como
numerador; a correção (Opção B, ~8 min) trocou por permutation importance. Isso
resolveu uma contradição real — `dist_museumplein_km` era 4ª na permutação e rejeitada
pelo score — mas o viés **aumentou**:

| Tipo | p95 nulo (MDI) | p95 nulo (permutação) |
|---|---|---|
| Contínuas de alta cardinalidade | 0,0518 | **0,0777** |
| Binárias | 0,0032 | 0,0051 |

Razão contínua/binária ~15× nas duas versões. O viés é **estrutural em árvores**, não
artefato do MDI. Minha atribuição inicial da causa estava errada; o que a v2 corrigiu
foi a inconsistência de escala entre numerador e denominador.

### Premissas da preparação mínima — **não herdar na Etapa 06**

| Bloco | Tratamento na 05 | Mecanismo (Análise 3.4) |
|---|---|---|
| 5 categóricas | Ordinal encoding | — |
| `security_deposit`, `cleaning_fee`, `taxa_limpeza_por_noite` | 0 | MNAR (E06-3) |
| `host_response_time` | categoria `"sem_resposta"` | MNAR (E06-4) |
| Bloco de reviews | sentinela −1 | MAR (E06-2) |
| Estruturais | mediana do treino | MCAR |

> **Aviso:** nada disso é insumo da Etapa 06. `df05` sai com valores originais; a
> preparação real (encoders, scalers, imputação definitiva) é responsabilidade dela,
> sobre as 27 colunas sobreviventes. Das 27, **12 têm nulos** — e `host_response_time`
> (47,3%) **não** está entre elas, tendo sido eliminada pelo RFECV.

### Pendência para a Etapa 07

`lightgbm` e `xgboost` **não estão instalados** e são requisito do CLAUDE.md §7. O
modelo-sonda desta etapa foi RandomForest. Instalar antes da Etapa 07.

---

**Próximo passo (aguardando validação):** Etapa 06 — Data Preparation sobre as 27
colunas de `FEATURES_05`, com `fit` só no treino e `transform` em validação/teste
(CLAUDE.md §5.4).

# Etapa 06 — Data Preparation

Entrada: `df05` (**17.978 × 29**) = `id` + `price` + **27 features** (`FEATURES_05`).

`df05` carrega os **valores originais** de `df04` — a preparação mínima da Etapa 05 ficou
contida naquela etapa e não vazou (23.227 nulos em 12 colunas preservados). Esta é,
portanto, a **primeira e única** preparação definitiva.

## O problema central — dois clientes com exigências opostas

A Etapa 07 exige seis modelos (CLAUDE.md §7). Eles **não querem os mesmos dados**:

| Requisito | Linear / Polinomial | Árvores (RF, XGB, LGBM) |
|---|---|---|
| Escala comum | **Obrigatória** — coeficientes dependem dela | Irrelevante — splits são invariantes a monotonia |
| One-hot do bairro (22 níveis) | Necessário — ordinal criaria ordem falsa | **Prejudicial** — fragmenta o sinal em 22 colunas fracas |
| Sentinela `-1` para nulo | **Veneno** — vira valor real na reta | Ideal — separável por 1 split |
| Cauda longa | Distorce o coeficiente | Indiferente |

Um dataset único prejudicaria metade dos modelos e invalidaria a comparação da Etapa 07.
**Decisão: dois datasets.**

| Dataset | Destino | Encoding | Escala | Bloco MAR |
|---|---|---|---|---|
| **`df06_linear`** | Linear Regression, Polinomial | One-hot (`drop_first`) | `log1p` + `RobustScaler` | Mediana + flag |
| **`df06_arvore`** | Random Forest, XGBoost, LightGBM | Ordinal | **Nenhuma** | Sentinela `-1` |

As regras MNAR e MCAR são **compartilhadas**; só encoding, escala e o bloco MAR divergem.

## Regras que atravessam a etapa

- **Isolamento do teste** (CLAUDE.md §5.3): `test_raw.csv` permanece **fechado**. A
  reaplicação ao hold-out é da Etapa 09.
- **Código inline, sem funções.** A preparação do teste concentra-se na Etapa 09. Por isso
  todo parâmetro aprendido vai para `ARTEFATOS_06` — sem função reutilizável, o dicionário
  é a única defesa contra divergência treino/teste.
- **Partição ANTES de qualquer `fit`.** Calcular mediana ou IQR sobre `df05` inteiro
  contaminaria a validação mesmo com o teste fechado.
- **Métrica**: `fit` em `log1p(price)`, MAPE sempre em escala original após `expm1`.

## Roteiro — 9 portões

| # | Bloco |
|---|---|
| 1 | Partição `train_06` / `valid_06` |
| 2 | Diagnóstico: Shapiro-Wilk, nulos por mecanismo, cardinalidade |
| 3 | Imputação MNAR → 0 (+ reconfirmação da flag) |
| 4 | Imputação MAR e MCAR (divergente por dataset) |
| 5 | `log1p` nas 6 colunas de assimetria positiva > 3 |
| 6 | Encoding — one-hot e ordinal |
| 7 | `RobustScaler` no `df06_linear` |
| 8 | Montagem dos datasets + `ARTEFATOS_06` |
| 9 | Verificação anti-vazamento + consolidação |

---

## 6.1 — Partição treino/validação (Portão 1)

**Primeira célula de código da etapa**, antes de qualquer imputador, encoder ou scaler.

Semente **42** e split **80/20**, idênticos à Etapa 05. Isso reutiliza a mesma partição em
que o RFECV escolheu as 27 features, e o registro honesto é:

> O MAPE de validação desta etapa e da Etapa 07 carrega **viés otimista leve** — as 27
> colunas foram selecionadas usando (indiretamente) estes dados. Não é vazamento grosseiro
> (a validação nunca entrou no `fit` de um modelo), mas a CV do RFECV rodou dentro deste
> treino.

Três razões para aceitar: (1) a estabilidade da Etapa 05 deu **N_CORTE = 26 nas sementes
42, 7 e 2024**, com as 26 no núcleo estável das três — a seleção não é artefato do split;
(2) mantém o **23,71%** da Etapa 05 comparável com a Etapa 07, isolando o efeito da
preparação; (3) o `test_raw.csv` fechado é o **juiz imparcial** (CLAUDE.md §5).

In [ ]:
# ---------------------------------------------------------------------------
# Portão 1 — Partição train_06 / valid_06 (plano §3)
#
# PRIMEIRA célula da etapa, ANTES de qualquer fit. Calcular mediana, IQR ou
# categorias de encoder sobre df05 inteiro contaminaria a validação — o
# parâmetro carregaria informação das linhas de validação, e a métrica ficaria
# otimista sem deixar rastro.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

train_06, valid_06 = train_test_split(
    df05, test_size=0.20, random_state=RANDOM_STATE, shuffle=True,
)

print('Partição (semente 42, split 80/20 — idênticos à Etapa 05)')
print(f'  train_06 .... {len(train_06):>6,} linhas'.replace(',', '.'))
print(f'  valid_06 .... {len(valid_06):>6,} linhas'.replace(',', '.'))
print(f'  soma ........ {len(train_06)+len(valid_06):>6,} = df05 ({len(df05):,})'
      .replace(',', '.'))

# Testes de aceitação
assert len(train_06) + len(valid_06) == len(df05), 'linhas perdidas na partição'
assert not set(train_06['id']) & set(valid_06['id']), 'vazamento de id entre partições'
assert train_06['id'].is_unique and valid_06['id'].is_unique, 'id duplicado'
print('\n  sem sobreposição de id, sem duplicatas: OK')

# Confere que é a MESMA partição da Etapa 05 (comparabilidade do 23,71%)
if 'train_05' in dir():
    _mesma = set(train_06['id']) == set(train_05['id'])
    print(f'  partição idêntica à da Etapa 05: {_mesma}')

# Alvo: log1p para o fit, euro para o MAPE
y_tr_06_log = np.log1p(train_06[TARGET].values)
y_va_06_log = np.log1p(valid_06[TARGET].values)
y_tr_06_eur = train_06[TARGET].values
y_va_06_eur = valid_06[TARGET].values

print(f'\nprice: treino mediana €{np.median(y_tr_06_eur):.0f} | '
      f'validação mediana €{np.median(y_va_06_eur):.0f}')
assert (y_tr_06_eur > 0).all() and (y_va_06_eur > 0).all(), 'price <= 0 inviabiliza MAPE'

# Dicionário de artefatos: TODO parâmetro aprendido no treino vai aqui.
# É o insumo obrigatório da Etapa 09 (plano §8).
ARTEFATOS_06 = {
    'random_state': RANDOM_STATE,
    'test_size': 0.20,
    'n_train': len(train_06),
    'n_valid': len(valid_06),
}
print('\nARTEFATOS_06 inicializado — receberá todo parâmetro aprendido no treino.')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 2 — Diagnóstico das 27 colunas (plano §4, §5)
#
# Shapiro-Wilk decide a regra de escala do CLAUDE.md §7:
# "padronização para Normais; rescala para não-Normais".
# Medido no TREINO (não em df05 inteiro).
# ---------------------------------------------------------------------------
from scipy import stats

_X_tr = train_06[FEATURES_05]

CATEGORICAS_06 = [c for c in FEATURES_05
                  if _X_tr[c].dtype == object or str(_X_tr[c].dtype) == 'category']
NUMERICAS_06 = [c for c in FEATURES_05 if c not in CATEGORICAS_06]

print(f'Categóricas ({len(CATEGORICAS_06)}):')
for c in CATEGORICAS_06:
    print(f'   {c:<26} {train_06[c].nunique():>2} níveis')
print(f'Numéricas: {len(NUMERICAS_06)}')

# --- Teste de normalidade (Shapiro-Wilk, amostra 5.000 quando n > 5.000) -----
_rows = []
for c in NUMERICAS_06:
    s = _X_tr[c].dropna()
    _sub = s.sample(5000, random_state=RANDOM_STATE) if len(s) > 5000 else s
    _W, _p = stats.shapiro(_sub)
    _rows.append({'coluna': c, 'skew': s.skew(), 'kurtose': s.kurtosis(),
                  'shapiro_p': _p, 'normal': _p > 0.05,
                  'p50': s.median(), 'p99': s.quantile(0.99), 'max': s.max()})
NORMALIDADE = pd.DataFrame(_rows).sort_values('skew', key=lambda s: s.abs(),
                                              ascending=False)

print('\nTeste de normalidade (Shapiro-Wilk) no TREINO:')
print(NORMALIDADE[['coluna', 'skew', 'kurtose', 'shapiro_p', 'normal']]
      .round(4).to_string(index=False))

_n_normais = int(NORMALIDADE['normal'].sum())
print(f'\n{"="*72}')
print(f'  Normais (p > 0,05): {_n_normais} de {len(NORMALIDADE)}')
print(f'  -> CLAUDE.md §7: "padronização para Normais" não se aplica a NENHUMA.')
print(f'  -> Decisão: RobustScaler para todas (nenhum StandardScaler).')
print(f'{"="*72}')

# --- Por que RobustScaler e não Min-Max: outliers comprimem o intervalo ------
print('\nOutliers que inviabilizam Min-Max:')
_out = NORMALIDADE[NORMALIDADE['max'] > 2 * NORMALIDADE['p99']].copy()
_out['max_sobre_p99'] = (_out['max'] / _out['p99'].clip(lower=1)).round(1)
print(_out[['coluna', 'p50', 'p99', 'max', 'max_sobre_p99']].to_string(index=False))
print('  Min-Max usa min/max observados: em minimum_nights, 99% dos anúncios')
print('  ficariam comprimidos em [0; 0,05] para acomodar um único de 365 noites.')

# --- Colunas de assimetria positiva > 3 -> log1p (plano §6) ------------------
_assim = NORMALIDADE[NORMALIDADE['skew'].abs() > 3]
COLS_LOG1P = _assim[_assim['skew'] > 3]['coluna'].tolist()
_assim_neg = _assim[_assim['skew'] < -3]['coluna'].tolist()

print(f'\n|skew| > 3: {len(_assim)} colunas')
print(f'  recebem log1p (skew POSITIVO): {len(COLS_LOG1P)}')
for c in COLS_LOG1P:
    print(f'     {c:<26} skew={float(_assim.loc[_assim.coluna==c,"skew"].iloc[0]):+6.2f}'
          f'  min={_X_tr[c].min():.2f}')
print(f'  FICAM FORA (skew NEGATIVO — cauda à esquerda): {_assim_neg}')
print('     log1p comprime cauda DIREITA; aplicá-la aqui AGRAVARIA a assimetria.')
print('     Correção adequada seria reflexão log1p(100-x) ou Yeo-Johnson — fora do escopo.')

# log1p exige x >= 0
for c in COLS_LOG1P:
    assert _X_tr[c].min() >= 0, f'{c} tem valor negativo — log1p geraria NaN'
print('\n  todas as colunas de log1p têm min >= 0: OK')

# --- Nulos por mecanismo (Análise 3.4) --------------------------------------
MNAR_ZERO_06 = ['security_deposit', 'cleaning_fee', 'taxa_limpeza_por_noite']
MAR_REVIEWS_06 = ['review_scores_rating', 'gap_value_vs_geral', 'reviews_per_month',
                  'dias_desde_primeira_review', 'dias_desde_ultima_review']
_com_nulo = [c for c in FEATURES_05 if train_06[c].isna().any()]
MCAR_MEDIANA_06 = [c for c in _com_nulo
                   if c not in MNAR_ZERO_06 + MAR_REVIEWS_06]

print(f'\nNulos no treino: {len(_com_nulo)} colunas')
print(f'  MNAR -> 0 .............. {len(MNAR_ZERO_06)}  {MNAR_ZERO_06}')
print(f'  MAR  -> divergente ..... {len(MAR_REVIEWS_06)}  {MAR_REVIEWS_06}')
print(f'  MCAR -> mediana ........ {len(MCAR_MEDIANA_06)}  {MCAR_MEDIANA_06}')

_sem_destino = [c for c in _com_nulo
                if c not in MNAR_ZERO_06 + MAR_REVIEWS_06 + MCAR_MEDIANA_06]
assert not _sem_destino, f'nulo sem tratamento declarado: {_sem_destino}'
print('  todas as colunas com nulo têm tratamento declarado: OK')

# host_response_time (47,3% nulo, o pior caso da EDA) saiu no RFECV
print(f"\n  host_response_time em FEATURES_05? "
      f"{'host_response_time' in FEATURES_05}  "
      f"(47,3% nulo — eliminada pelo RFECV na Etapa 05)")

In [ ]:
# ---------------------------------------------------------------------------
# Portões 3 e 4 — Imputação por mecanismo (plano §5)
#
#   MNAR -> 0           (ambos os datasets)
#   MAR  -> sentinela -1 na árvore | mediana + flag no linear
#   MCAR -> mediana do treino (ambos)
#
# Todas as medianas vêm de train_06. fit só no treino, mesmo aqui.
# ---------------------------------------------------------------------------

# --- Portão 3: reconfirmação do MNAR antes de imputar -----------------------
print('Reconfirmação do MNAR (plano §5.1) — o nulo significa "não cobro"?')
for _c, _flag in [('security_deposit', 'tem_deposito'),
                  ('cleaning_fee', 'tem_taxa_limpeza')]:
    _m = train_06[_c].isna()
    _com_flag = int(df04.loc[train_06.index[_m], _flag].sum()) if _flag in df04.columns else -1
    print(f'  {_c:<24} nulos={_m.sum():>5} | flag={_flag}=1 entre nulos: '
          f'{_com_flag} (esperado 0)')
    assert _com_flag == 0, f'{_c}: nulo com flag=1 contradiz o mecanismo MNAR'
print('  100% dos nulos têm a flag em 0 -> imputar 0, não a mediana (E06-3).')

# --- Parâmetros aprendidos NO TREINO ----------------------------------------
MEDIANAS_MCAR = train_06[MCAR_MEDIANA_06].median()
MEDIANAS_MAR = train_06[MAR_REVIEWS_06].median()

print('\nMedianas aprendidas no TREINO (MCAR):')
for c, v in MEDIANAS_MCAR.items():
    print(f'  {c:<28} {v:>10.2f}')
print('Medianas aprendidas no TREINO (MAR, só para o df06_linear):')
for c, v in MEDIANAS_MAR.items():
    print(f'  {c:<28} {v:>10.2f}')


# --- Base comum: MNAR -> 0 e MCAR -> mediana (idênticos nos dois datasets) ---
def _imputa_comum(df):
    """MNAR -> 0 e MCAR -> mediana do treino. Comum aos dois datasets."""
    X = df[FEATURES_05].copy()
    for c in MNAR_ZERO_06:
        X[c] = X[c].fillna(0.0)
    for c in MCAR_MEDIANA_06:
        X[c] = X[c].fillna(MEDIANAS_MCAR[c])
    return X


_base_tr = _imputa_comum(train_06)
_base_va = _imputa_comum(valid_06)

# --- Portão 4a: ÁRVORE — sentinela -1 no bloco MAR --------------------------
# A árvore separa os "sem review" com 1 split. Sem flag auxiliar.
X_arv_tr = _base_tr.copy()
X_arv_va = _base_va.copy()
for c in MAR_REVIEWS_06:
    X_arv_tr[c] = X_arv_tr[c].fillna(-1.0)
    X_arv_va[c] = X_arv_va[c].fillna(-1.0)

# --- Portão 4b: LINEAR — mediana + flag no bloco MAR ------------------------
# Sentinela -1 seria veneno: entraria na reta como valor real. A flag preserva
# a informação da ausência sem falsear a escala.
X_lin_tr = _base_tr.copy()
X_lin_va = _base_va.copy()

_flag_tr = X_lin_tr[MAR_REVIEWS_06].isna().any(axis=1).astype(float)
_flag_va = X_lin_va[MAR_REVIEWS_06].isna().any(axis=1).astype(float)
for c in MAR_REVIEWS_06:
    X_lin_tr[c] = X_lin_tr[c].fillna(MEDIANAS_MAR[c])
    X_lin_va[c] = X_lin_va[c].fillna(MEDIANAS_MAR[c])
X_lin_tr['sem_reviews_imputado'] = _flag_tr
X_lin_va['sem_reviews_imputado'] = _flag_va

print(f'\nFlag sem_reviews_imputado: treino {_flag_tr.mean()*100:.2f}% | '
      f'validação {_flag_va.mean()*100:.2f}%')
print('  (um anúncio novo não tem score RUIM — tem AUSÊNCIA de score;')
print('   a flag evita confundir as duas coisas)')

# --- Verificação: sem nulos nos dois caminhos -------------------------------
for _nome, _d in [('X_arv_tr', X_arv_tr), ('X_arv_va', X_arv_va),
                  ('X_lin_tr', X_lin_tr), ('X_lin_va', X_lin_va)]:
    _n = int(_d.isna().sum().sum())
    print(f'  {_nome:<10} {_d.shape} nulos={_n}')
    assert _n == 0, f'{_nome} tem nulo após imputação'

ARTEFATOS_06.update({
    'cols_mnar_zero': MNAR_ZERO_06,
    'cols_mar': MAR_REVIEWS_06,
    'cols_mcar': MCAR_MEDIANA_06,
    'medianas_mcar': MEDIANAS_MCAR.to_dict(),
    'medianas_mar': MEDIANAS_MAR.to_dict(),
})
print('\nImputação concluída. ARTEFATOS_06 atualizado.')

In [ ]:
# ---------------------------------------------------------------------------
# Portões 5, 6 e 7 — log1p, encoding e escala (plano §6, §7, §4.1)
#
# log1p e RobustScaler: SÓ no df06_linear. Árvores são invariantes a
# transformação monótona — aplicá-las lá seria trabalho sem efeito.
# ---------------------------------------------------------------------------
from sklearn.preprocessing import RobustScaler

# --- Portão 5: log1p nas 6 colunas de assimetria POSITIVA > 3 ---------------
print(f'Portão 5 — log1p em {len(COLS_LOG1P)} colunas (só no df06_linear):')
for c in COLS_LOG1P:
    _antes = X_lin_tr[c].skew()
    X_lin_tr[c] = np.log1p(X_lin_tr[c])
    X_lin_va[c] = np.log1p(X_lin_va[c])
    print(f'  {c:<26} skew {_antes:+7.2f} -> {X_lin_tr[c].skew():+6.2f}')

assert np.isfinite(X_lin_tr[COLS_LOG1P].values).all(), 'log1p gerou inf/NaN no treino'
assert np.isfinite(X_lin_va[COLS_LOG1P].values).all(), 'log1p gerou inf/NaN na validação'
print('  sem inf/NaN: OK')

# --- Portão 6a: ÁRVORE — ordinal (mantém 27 colunas) ------------------------
# Mapa aprendido no TREINO; categoria nunca vista na validação -> -1.
MAPAS_ORDINAL = {}
for c in CATEGORICAS_06:
    _niveis = sorted(train_06[c].dropna().unique())
    MAPAS_ORDINAL[c] = {v: i for i, v in enumerate(_niveis)}
    X_arv_tr[c] = X_arv_tr[c].map(MAPAS_ORDINAL[c]).fillna(-1).astype(float)
    X_arv_va[c] = X_arv_va[c].map(MAPAS_ORDINAL[c]).fillna(-1).astype(float)

print(f'\nPortão 6a — ordinal (árvore):')
for c in CATEGORICAS_06:
    print(f'  {c:<26} {len(MAPAS_ORDINAL[c]):>2} níveis -> códigos 0..{len(MAPAS_ORDINAL[c])-1}')

# Nenhuma categoria nova na validação (plano §9.2)
for c in CATEGORICAS_06:
    _novas = set(valid_06[c].dropna().unique()) - set(MAPAS_ORDINAL[c])
    assert not _novas, f'{c}: categoria na validação ausente do treino: {_novas}'
print('  nenhuma categoria nova na validação: OK')

# --- Portão 6b: LINEAR — one-hot com drop_first -----------------------------
# drop_first evita a dummy variable trap: com todas as dummies + intercepto,
# a matriz fica singular e os coeficientes ficam indeterminados.
CATEGORIAS_ONEHOT = {c: sorted(train_06[c].dropna().unique()) for c in CATEGORICAS_06}

for c in CATEGORICAS_06:                       # categoria fixa = ordem do treino
    X_lin_tr[c] = pd.Categorical(X_lin_tr[c], categories=CATEGORIAS_ONEHOT[c])
    X_lin_va[c] = pd.Categorical(X_lin_va[c], categories=CATEGORIAS_ONEHOT[c])

X_lin_tr = pd.get_dummies(X_lin_tr, columns=CATEGORICAS_06, drop_first=True, dtype=float)
X_lin_va = pd.get_dummies(X_lin_va, columns=CATEGORICAS_06, drop_first=True, dtype=float)

# Alinhamento obrigatório: mesma ordem de colunas nas duas partições
X_lin_va = X_lin_va.reindex(columns=X_lin_tr.columns, fill_value=0.0)

print(f'\nPortão 6b — one-hot (linear):')
print(f'  {len(FEATURES_05)} features -> {X_lin_tr.shape[1]} colunas')
print(f'  (25 numéricas + 1 flag + {len(CATEGORIAS_ONEHOT["room_type"])-1} room_type'
      f' + {len(CATEGORIAS_ONEHOT["neighbourhood_cleansed"])-1} bairro)')
assert list(X_lin_tr.columns) == list(X_lin_va.columns), 'colunas desalinhadas'
print('  colunas alinhadas treino/validação: OK')

# --- Portão 7: RobustScaler no df06_linear ---------------------------------
# Shapiro-Wilk: 0 de 25 Normais -> rescala, não padronização (CLAUDE.md §7).
# RobustScaler usa mediana e IQR: insensível à cauda, ao contrário do Min-Max.
_num_para_escalar = [c for c in X_lin_tr.columns
                     if c in NUMERICAS_06 or c == 'sem_reviews_imputado']
_num_para_escalar = [c for c in _num_para_escalar if c != 'sem_reviews_imputado']

_scaler = RobustScaler().fit(X_lin_tr[_num_para_escalar])      # fit SÓ no treino
X_lin_tr[_num_para_escalar] = _scaler.transform(X_lin_tr[_num_para_escalar])
X_lin_va[_num_para_escalar] = _scaler.transform(X_lin_va[_num_para_escalar])

print(f'\nPortão 7 — RobustScaler em {len(_num_para_escalar)} colunas numéricas')
print('  (dummies e a flag binária NÃO são escaladas — já estão em 0/1)')
_amostra = pd.DataFrame({
    'centro (mediana)': _scaler.center_[:5],
    'escala (IQR)': _scaler.scale_[:5],
}, index=_num_para_escalar[:5])
print(_amostra.round(3).to_string())

ARTEFATOS_06.update({
    'cols_log1p': COLS_LOG1P,
    'mapas_ordinal': MAPAS_ORDINAL,
    'categorias_onehot': {k: list(v) for k, v in CATEGORIAS_ONEHOT.items()},
    'cols_escaladas': _num_para_escalar,
    'robust_center': dict(zip(_num_para_escalar, _scaler.center_.tolist())),
    'robust_scale': dict(zip(_num_para_escalar, _scaler.scale_.tolist())),
    'colunas_linear': list(X_lin_tr.columns),
    'colunas_arvore': list(X_arv_tr.columns),
})
print('\nARTEFATOS_06 atualizado com log1p, encoders e parâmetros de escala.')

In [ ]:
# ---------------------------------------------------------------------------
# Portões 8 e 9 — Montagem dos datasets + verificação anti-vazamento
#                 (plano §8, §9, §12)
# ---------------------------------------------------------------------------

# --- Portão 8: montagem com id e price por partição -------------------------
df06_arvore_train = pd.concat(
    [train_06[['id', TARGET]].reset_index(drop=True), X_arv_tr.reset_index(drop=True)],
    axis=1)
df06_arvore_valid = pd.concat(
    [valid_06[['id', TARGET]].reset_index(drop=True), X_arv_va.reset_index(drop=True)],
    axis=1)
df06_linear_train = pd.concat(
    [train_06[['id', TARGET]].reset_index(drop=True), X_lin_tr.reset_index(drop=True)],
    axis=1)
df06_linear_valid = pd.concat(
    [valid_06[['id', TARGET]].reset_index(drop=True), X_lin_va.reset_index(drop=True)],
    axis=1)

FEATURES_06_ARVORE = list(X_arv_tr.columns)
FEATURES_06_LINEAR = list(X_lin_tr.columns)

print('Portão 8 — datasets montados:')
for _n, _d in [('df06_arvore_train', df06_arvore_train),
               ('df06_arvore_valid', df06_arvore_valid),
               ('df06_linear_train', df06_linear_train),
               ('df06_linear_valid', df06_linear_valid)]:
    print(f'  {_n:<20} {_d.shape[0]:>6,} x {_d.shape[1]:>2}'.replace(',', '.'))
print(f'\n  FEATURES_06_ARVORE: {len(FEATURES_06_ARVORE)} colunas')
print(f'  FEATURES_06_LINEAR: {len(FEATURES_06_LINEAR)} colunas')

# --- Portão 9: verificação anti-vazamento -----------------------------------
print(f'\n{"="*72}')
print('Portão 9 — Verificação anti-vazamento (plano §9, §12)')
print(f'{"="*72}')

# 9.1 Nenhum parâmetro aprendido fora do treino
_div = []
for c, v in ARTEFATOS_06['medianas_mcar'].items():
    assert v == train_06[c].median(), f'{c}: mediana não é a do treino'
    if v != valid_06[c].median():
        _div.append(c)
print(f'  9.1 medianas MCAR == mediana do TREINO ......... OK')
print(f'      ({len(_div)} de {len(MCAR_MEDIANA_06)} divergem da mediana da validação —')
print(f'       prova de que o transform usou o parâmetro do treino)')

for c, v in ARTEFATOS_06['medianas_mar'].items():
    assert v == train_06[c].median(), f'{c}: mediana MAR não é a do treino'
print(f'  9.1 medianas MAR  == mediana do TREINO ......... OK')

_c_scaler = ARTEFATOS_06['robust_center']
assert abs(_c_scaler['accommodates'] - X_arv_tr['accommodates'].median()) < 1e-9, \
    'centro do scaler não é a mediana do treino'
print(f'  9.1 RobustScaler centrado na mediana do TREINO  OK')

# 9.2 Nenhuma categoria nova (já testado no portão 6; reconfirmado)
for c in CATEGORICAS_06:
    assert not set(valid_06[c].dropna().unique()) - set(MAPAS_ORDINAL[c])
print(f'  9.2 nenhuma categoria nova na validação ........ OK')

# 9.5 Sem nulos na saída
for _n, _d in [('arvore_train', df06_arvore_train), ('arvore_valid', df06_arvore_valid),
               ('linear_train', df06_linear_train), ('linear_valid', df06_linear_valid)]:
    assert _d.isna().sum().sum() == 0, f'{_n} tem nulo'
print(f'  9.5 sem nulos nos 4 datasets .................. OK')

# §12 — granularidade, chave/alvo fora das features, GROUP_COLS, alinhamento
for _n, _d in [('arvore_train', df06_arvore_train), ('arvore_valid', df06_arvore_valid),
               ('linear_train', df06_linear_train), ('linear_valid', df06_linear_valid)]:
    assert len(_d) == _d['id'].nunique(), f'{_n}: granularidade quebrada'
print(f'  §12 granularidade 1 linha/listing ............. OK')

for _f in (FEATURES_06_ARVORE, FEATURES_06_LINEAR):
    assert 'id' not in _f and TARGET not in _f, 'id/price nas features'
print(f'  §12 id e price fora das features .............. OK')

# GROUP_COLS: na árvore literais; no linear, bairro e room_type viram dummies
assert all(g in FEATURES_06_ARVORE for g in GROUP_COLS), 'GROUP_COLS ausente na árvore'
_gc_linear = ('accommodates' in FEATURES_06_LINEAR
              and any(c.startswith('room_type_') for c in FEATURES_06_LINEAR)
              and any(c.startswith('neighbourhood_cleansed_') for c in FEATURES_06_LINEAR))
assert _gc_linear, 'GROUP_COLS ausente no linear'
print(f'  §12 GROUP_COLS presentes nos dois datasets .... OK')

assert list(df06_arvore_train.columns) == list(df06_arvore_valid.columns)
assert list(df06_linear_train.columns) == list(df06_linear_valid.columns)
print(f'  §12 colunas alinhadas treino/validação ........ OK')

# --- Sanidade: o MAPE continua na faixa esperada? ---------------------------
from sklearn.ensemble import RandomForestRegressor
_m = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE,
                           n_jobs=-1).fit(X_arv_tr, y_tr_06_log)
_pred = np.expm1(_m.predict(X_arv_va))
MAPE_06_ARVORE = float(np.mean(np.abs(y_va_06_eur - _pred) / y_va_06_eur))

print(f'\n{"="*72}')
print(f'  Sanidade — RF em df06_arvore: MAPE {MAPE_06_ARVORE*100:.2f}%')
print(f'  Referência Etapa 05 (27 features): 23,71%')
print(f'  Delta: {(MAPE_06_ARVORE-0.2371)*100:+.2f} p.p.')
print(f'{"="*72}')
assert 0 < MAPE_06_ARVORE < 1, 'MAPE fora de escala — expm1 aplicado?'
if MAPE_06_ARVORE < 0.15:
    print('  ATENÇÃO: MAPE abaixo do piso da EDA (20,8%) — investigar vazamento.')

print(f'\nARTEFATOS_06: {len(ARTEFATOS_06)} chaves registradas')
for k in ARTEFATOS_06:
    _v = ARTEFATOS_06[k]
    _desc = f'{len(_v)} itens' if isinstance(_v, (dict, list)) else _v
    print(f'  {k:<22} {_desc}')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 9 (cont.) — O df06_linear cumpre o propósito para que foi criado?
#
# O teste real da preparação: a Linear Regression só funciona com escala,
# one-hot e caudas comprimidas. Comparar contra uma versão SEM preparação
# mostra se os dois datasets se justificam.
# ---------------------------------------------------------------------------
from sklearn.linear_model import LinearRegression

def _mape(y_eur, pred_log):
    return float(np.mean(np.abs(y_eur - np.expm1(pred_log)) / y_eur))

# (a) Linear no dataset PREPARADO para ela
_lin_ok = LinearRegression().fit(X_lin_tr, y_tr_06_log)
_m_lin_ok = _mape(y_va_06_eur, _lin_ok.predict(X_lin_va))

# (b) Linear no dataset da ÁRVORE (ordinal, sem escala, sentinela -1)
_lin_ruim = LinearRegression().fit(X_arv_tr, y_tr_06_log)
_m_lin_ruim = _mape(y_va_06_eur, _lin_ruim.predict(X_arv_va))

# (c) RF no dataset do LINEAR (one-hot fragmenta o bairro em 21 colunas)
_rf_lin = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE,
                                n_jobs=-1).fit(X_lin_tr, y_tr_06_log)
_m_rf_lin = _mape(y_va_06_eur, _rf_lin.predict(X_lin_va))

print('Cada modelo no dataset CERTO vs no ERRADO (MAPE em valid_06):')
print(f'{"="*72}')
print(f'  {"modelo":<22} {"df06_linear":>14} {"df06_arvore":>14}')
print(f'  {"Linear Regression":<22} {_m_lin_ok*100:>13.2f}% {_m_lin_ruim*100:>13.2f}%')
print(f'  {"Random Forest":<22} {_m_rf_lin*100:>13.2f}% {MAPE_06_ARVORE*100:>13.2f}%')
print(f'{"="*72}')

print(f'\nGanho da preparação dedicada:')
print(f'  Linear: {_m_lin_ruim*100:.2f}% -> {_m_lin_ok*100:.2f}% '
      f'({(_m_lin_ok-_m_lin_ruim)*100:+.2f} p.p.)')
print(f'  RF:     {_m_rf_lin*100:.2f}% -> {MAPE_06_ARVORE*100:.2f}% '
      f'({(MAPE_06_ARVORE-_m_rf_lin)*100:+.2f} p.p.)')

if _m_lin_ok < _m_lin_ruim and MAPE_06_ARVORE < _m_rf_lin:
    print('\n  Cada família vai melhor no SEU dataset -> decisão dos dois datasets')
    print('  confirmada empiricamente, não só por argumento teórico.')
else:
    print('\n  ATENÇÃO: a separação em dois datasets não se confirmou nos números.')

# Registro para a Etapa 07
SANIDADE_06 = pd.DataFrame([
    {'modelo': 'Linear Regression', 'dataset': 'df06_linear', 'mape': _m_lin_ok},
    {'modelo': 'Linear Regression', 'dataset': 'df06_arvore', 'mape': _m_lin_ruim},
    {'modelo': 'Random Forest', 'dataset': 'df06_linear', 'mape': _m_rf_lin},
    {'modelo': 'Random Forest', 'dataset': 'df06_arvore', 'mape': MAPE_06_ARVORE},
])
print('\nSANIDADE_06 registrado (não é a Etapa 07 — é só verificação da preparação).')

## Consolidação da Etapa 06 — Data Preparation encerrada

Os 9 portões do plano (`docs/plans/06_data_preparation.md` §10) foram executados.

### Entregáveis 1 e 2 — os dois datasets

| Dataset | Treino | Validação | Features | Destino |
|---|---|---|---|---|
| **`df06_arvore`** | 14.382 × 29 | 3.596 × 29 | **27** | Random Forest, XGBoost, LightGBM |
| **`df06_linear`** | 14.382 × 51 | 3.596 × 51 | **49** | Linear Regression, Polinomial |

Listas de colunas: `FEATURES_06_ARVORE` (27) e `FEATURES_06_LINEAR` (49).

### Entregável 3 — `ARTEFATOS_06` (17 chaves)

Insumo **obrigatório** da Etapa 09. Como esta etapa não produziu função reutilizável
(decisão de escopo), o dicionário é a única defesa contra divergência treino/teste:

`random_state`, `test_size`, `n_train`, `n_valid`, `cols_mnar_zero` (3),
`cols_mar` (5), `cols_mcar` (4), `medianas_mcar` (4), `medianas_mar` (5),
`cols_log1p` (6), `mapas_ordinal` (2), `categorias_onehot` (2), `cols_escaladas` (25),
`robust_center` (25), `robust_scale` (25), `colunas_linear` (49), `colunas_arvore` (27).

### Entregável 4 — tabela de decisões

| Coluna / bloco | % nulo | Mecanismo | `df06_arvore` | `df06_linear` |
|---|---|---|---|---|
| `security_deposit` | 30,9% | MNAR | **0** | **0** → `log1p` → escala |
| `cleaning_fee` | 18,0% | MNAR | **0** | **0** → escala |
| `taxa_limpeza_por_noite` | 18,0% | MNAR | **0** | **0** → `log1p` → escala |
| Bloco de reviews (5 cols) | ~12–13% | MAR | **sentinela −1** | **mediana + flag** |
| `bathrooms`, `bedrooms`, `host_experiencia_dias`, `calendar_atualizado_dias` | ≤ 0,04% | MCAR | mediana | mediana |
| `room_type` (3), `neighbourhood_cleansed` (22) | 0% | — | ordinal | one-hot `drop_first` |
| 25 numéricas | — | — | sem escala | `RobustScaler` |

**MNAR reconfirmado no treino:** `security_deposit` 4.479 nulos e `cleaning_fee` 2.626
nulos, com **0 ocorrências** de flag = 1 entre eles. O nulo significa "não cobro" —
imputar a mediana viraria "cobro €150" (E06-3).

**Achado favorável:** `host_response_time` (47,3% nulo, o pior caso de imputação da EDA)
**não está em `FEATURES_05`** — foi eliminada pelo RFECV. O problema saiu de cena sozinho.

### Entregável 5 — relatório de normalidade

Shapiro-Wilk nas 25 numéricas do treino: **0 Normais**, todas com p < 0,001. Assimetrias
de −3,55 a **+21,72**; curtose de até 717,91.

A regra do CLAUDE.md §7 ("padronização para Normais; rescala para não-Normais") resolveu-se
sem ambiguidade: **nenhum `StandardScaler`**. Min-Max foi descartado porque em
`minimum_nights` (p50=2, p99=20, máx=300) comprimiria 99% dos anúncios num intervalo
minúsculo. `RobustScaler` (mediana e IQR) é insensível à cauda.

### Efeito da `log1p` nas 6 colunas de assimetria positiva > 3

| Coluna | Antes | Depois |
|---|---|---|
| `minimum_nights` | **+21,72** | **+2,18** |
| `security_deposit` | +7,06 | **+0,06** |
| `bathrooms` | +5,34 | +2,51 |
| `number_of_reviews` | +5,29 | **+0,13** |
| `extra_people` | +4,55 | +0,81 |
| `taxa_limpeza_por_noite` | +3,70 | −0,75 |

`review_scores_rating` (assimetria **−3,55**, cauda à esquerda) ficou **fora**: `log1p`
comprime cauda direita e agravaria a assimetria. Correção adequada seria reflexão
`log1p(100−x)` ou Yeo-Johnson — registrado como pendência.

### Verificação anti-vazamento — todos os testes passaram

| Teste | Resultado |
|---|---|
| Medianas MCAR e MAR == mediana do **treino** | OK (1 de 4 divergem da validação — prova do `transform`) |
| `RobustScaler` centrado na mediana do treino | OK |
| Nenhuma categoria nova na validação | OK |
| Sem nulos nos 4 datasets | OK |
| Granularidade 1 linha/`listing_id` | OK |
| `id` e `price` fora das features | OK |
| `GROUP_COLS` presentes nos dois datasets | OK |
| Colunas alinhadas treino/validação | OK |
| `test_raw.csv` não lido | OK |

### Sanidade — e uma previsão do plano que se confirmou só pela metade

MAPE em `valid_06`, cada modelo nos dois datasets:

| Modelo | `df06_linear` | `df06_arvore` |
|---|---|---|
| Linear Regression | **24,78%** | 25,88% |
| Random Forest | 23,69% | **23,71%** |

**O que se confirmou:** a Linear Regression ganha **1,10 p.p.** com a preparação dedicada
(25,88% → 24,78%). O one-hot, a escala e a compressão de caudas valem exatamente o que o
plano argumentou. A decisão de produzir dois datasets **se justifica por este número**.

**O que NÃO se confirmou:** o Random Forest foi **marginalmente melhor** no `df06_linear`
(23,69% vs 23,71%) — o oposto do previsto. O plano afirmava que o one-hot de 22 níveis
"fragmenta o sinal em 22 colunas fracas" e prejudicaria árvores. A diferença de **0,02
p.p.** é ruído, não efeito real: o RF é indiferente a qual dos dois datasets recebe.

A conclusão honesta é que a separação se justifica **pelo lado linear, não pelo lado das
árvores**. O `df06_arvore` continua preferível para RF/XGB/LGBM por ter **27 colunas em vez
de 49** — treino mais rápido e importâncias legíveis, sem fragmentação do bairro em 21
dummies — mas não por ganho de MAPE.

### Contexto do número

MAPE do RF em `df06_arvore`: **23,71%** — idêntico ao da Etapa 05 (delta 0,00 p.p.),
como esperado, já que a árvore é invariante a escala e as mesmas 27 colunas estão presentes.
A preparação não deveria mudar o desempenho de árvore, e não mudou: é confirmação de que
nada se quebrou no caminho.

Faixa de referência: piso de dispersão intragrupo de **20,8%** (Análise 2.9) e banda
realista de **20–30%**. O número está onde deveria estar.

### Premissas registradas

- **Semente 42, split 80/20**, partição **idêntica** à da Etapa 05 (verificado
  programaticamente).
- **Viés otimista leve** na validação: as 27 features foram selecionadas usando
  (indiretamente) estes dados. Mitigado pela estabilidade da Etapa 05 (N_CORTE = 26 nas
  seeds 42/7/2024) e pelo `test_raw.csv` fechado como juiz imparcial.
- **`log1p` destrói a interpretabilidade do coeficiente** no `df06_linear`: "cada noite
  adicional custa €X" vira "cada unidade de log-noite custa €X". A Etapa 11 não deve
  interpretar coeficientes em escala logarítmica.

### Pendências

1. **`lightgbm` e `xgboost` não instalados** — requisito do CLAUDE.md §7. **Bloqueia a
   Etapa 07.**
2. **Grau da Regressão Polinomial** — com 49 colunas, grau 2 gera ~1.275 termos. Pode
   exigir recorte de features específico para esse modelo.
3. **Simetrização de `review_scores_rating`** — reavaliar se a Linear Regression ficar
   muito abaixo das árvores na Etapa 07.

---

**Próximo passo (aguardando validação):** Etapa 07 — Machine Learning. Treinar e comparar
por MAPE: baseline (média), Linear Regression, Regressão Polinomial, Random Forest,
XGBoost e LightGBM.

# Etapa 07 — Machine Learning

Entrada: os **dois datasets** da Etapa 06, cada um dedicado a uma família de algoritmo.

| Dataset | Treino | Validação | Features | Destino |
|---|---|---|---|---|
| `df06_arvore` | 14.382 × 29 | 3.596 × 29 | **27** | Random Forest, XGBoost, LightGBM |
| `df06_linear` | 14.382 × 51 | 3.596 × 51 | **49** | Linear Regression, Polinomial |

### Expectativa calibrada — leia antes dos números

A Análise 2.9 mediu um **piso de dispersão intragrupo de 20,8%**. O RF da Etapa 06 já
entrega **23,71%**.

> **A margem desta etapa é de ~3 p.p., não de 10.** Chegar a 21–22% é muito bom. Se algum
> modelo marcar **abaixo de 19%**, a primeira hipótese é **vazamento**, não sucesso — e a
> investigação precede qualquer conclusão.

### Protocolo

- **5-fold CV no treino** (`KFold`, shuffle, seed 42) para **rankear** — dá média ± desvio,
  permitindo dizer se a diferença entre dois modelos é maior que o ruído.
- **`valid_06` uma única vez por modelo**, como confirmação — não para escolher.
- **MAPE por decil de preço** para os 3 melhores: o MAPE global esconde que o erro
  percentual é muito pior nos anúncios baratos, e é lá que está a massa acionável do EV.
- **Sem tuning.** Parâmetros default ou modestos e fixos; `random_state=42` em tudo que for
  estocástico. A única exceção é o `alpha` do Ridge na polinomial, sem o qual o modelo não
  é avaliável. O ajuste é a Etapa 08.
- **Isolamento do teste**: `test_raw.csv` permanece fechado. O campeão é escolhido só com
  treino e validação.

### Os seis modelos

| # | Modelo | Dataset | Parâmetros |
|---|---|---|---|
| 1 | Baseline (média do `log1p`) | — | — |
| 2 | Linear Regression | `df06_linear` | default |
| 3 | Regressão Polinomial | `df06_linear` | grau 2 nas 25 numéricas + `RidgeCV` |
| 4 | Random Forest | `df06_arvore` | `n_estimators=100` |
| 5 | XGBoost | `df06_arvore` | `n_estimators=100`, `max_depth=6`, `lr=0.1` |
| 6 | LightGBM | `df06_arvore` | `n_estimators=100`, `num_leaves=31`, `lr=0.1` |

---

## 7.1 — Baselines (Portão 2)

O CLAUDE.md §7 pede "Baseline (média)", mas a leitura importa: `price` tem **assimetria
5,92**, e o MAPE é um erro **percentual**.

Quatro referências são calculadas. A **média do `log1p`** é a oficial (respeita a
transformação usada por todos os modelos), mas a **mediana de `GROUP_COLS`** é o número que
importa para o negócio — é a regra da Etapa 01, e um modelo que não a bate não justifica
existir, porque aquela etapa já entrega essa estimativa sem nenhum treino.

In [ ]:
# ---------------------------------------------------------------------------
# Portão 2 — Baselines (plano §2)
#
# 4 referências. A média do log1p é a OFICIAL; a mediana de GROUP_COLS é o
# chão de NEGÓCIO (regra da Etapa 01 — se o modelo não bate, não justifica
# existir, porque aquela etapa já entrega isso sem treino).
# ---------------------------------------------------------------------------
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error

CV_07 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def mape_eur_07(y_eur, pred_eur):
    """MAPE em ESCALA ORIGINAL (euro). Nunca sobre o log."""
    return float(np.mean(np.abs(np.asarray(y_eur) - np.asarray(pred_eur))
                         / np.asarray(y_eur)))


def _score_mape_log(y_log_true, y_log_pred):
    """Scorer para CV: recebe log1p, devolve -MAPE em euro (maior = melhor)."""
    return -mape_eur_07(np.expm1(y_log_true), np.expm1(y_log_pred))


SCORER_07 = make_scorer(_score_mape_log, greater_is_better=True)

print(f'price no treino: assimetria {pd.Series(y_tr_06_eur).skew():.2f}')
print('(por isso a leitura do baseline importa — o MAPE é erro PERCENTUAL)\n')

# --- Os quatro baselines ----------------------------------------------------
_pred_media_log = np.expm1(np.mean(y_tr_06_log))     # OFICIAL
_pred_media_eur = np.mean(y_tr_06_eur)
_pred_mediana = np.median(y_tr_06_eur)

BASELINES = []
for _rot, _pred, _papel in [
        ('média do log1p (expm1)', _pred_media_log, 'OFICIAL'),
        ('média em euro', _pred_media_eur, 'referência'),
        ('mediana em euro', _pred_mediana, 'piso trivial')]:
    BASELINES.append({
        'modelo': f'Baseline — {_rot}', 'papel': _papel, 'dataset': '—',
        'predicao': _pred,
        'mape_valid': mape_eur_07(y_va_06_eur, np.full(len(y_va_06_eur), _pred)),
        'mae': mean_absolute_error(y_va_06_eur, np.full(len(y_va_06_eur), _pred)),
        'rmse': float(np.sqrt(mean_squared_error(
            y_va_06_eur, np.full(len(y_va_06_eur), _pred)))),
    })

# Baseline de NEGÓCIO: mediana de GROUP_COLS (regra da Etapa 01, CLAUDE.md §3.1)
_ref = train_06.groupby(GROUP_COLS, observed=True)[TARGET].median()
_pred_gc = pd.Series(valid_06.set_index(GROUP_COLS).index.map(_ref),
                     index=valid_06.index).astype(float)
_cobertura_gc = _pred_gc.notna().mean()
_pred_gc = _pred_gc.fillna(train_06[TARGET].median()).values

BASELINES.append({
    'modelo': 'Baseline — mediana GROUP_COLS', 'papel': 'CHÃO DE NEGÓCIO',
    'dataset': 'regra Etapa 01', 'predicao': np.nan,
    'mape_valid': mape_eur_07(y_va_06_eur, _pred_gc),
    'mae': mean_absolute_error(y_va_06_eur, _pred_gc),
    'rmse': float(np.sqrt(mean_squared_error(y_va_06_eur, _pred_gc))),
})

BASE_DF = pd.DataFrame(BASELINES)
print('Baselines (MAPE em valid_06, escala original):')
for _, r in BASE_DF.iterrows():
    _p = f'€{r.predicao:>6.1f}' if pd.notna(r.predicao) else 'variável'
    print(f'  {r.modelo:<32} {_p} | MAPE {r.mape_valid*100:5.2f}% | '
          f'MAE €{r.mae:5.1f} | {r.papel}')

MAPE_BASELINE_OFICIAL = BASE_DF.loc[0, 'mape_valid']
MAPE_CHAO_NEGOCIO = BASE_DF.loc[3, 'mape_valid']

print(f'\n{"="*74}')
print(f'  Baseline OFICIAL (média do log) ..... {MAPE_BASELINE_OFICIAL*100:5.2f}%')
print(f'  CHÃO DE NEGÓCIO (mediana GROUP_COLS)  {MAPE_CHAO_NEGOCIO*100:5.2f}%'
      f'   <- a barra real')
print(f'  Piso da EDA (Análise 2.9) ........... 20,80%')
print(f'{"="*74}')
print(f'\n  cobertura direta do grupo: {_cobertura_gc*100:.1f}% '
      f'(resto via mediana global)')
print(f'  A média em euro é a PIOR das três: assimetria 5,92 a puxa para €'
      f'{_pred_media_eur:.0f},')
print(f'  longe da massa (mediana €{_pred_mediana:.0f}). Sobrepredizer anúncio barato')
print(f'  custa muito MAPE — errar €20 num de €60 são 33%.')

# Registro acumulador de resultados da etapa
RESULTADOS_07 = list(BASELINES)

In [ ]:
# ---------------------------------------------------------------------------
# Portões 3 e 4 — Linear Regression e Regressão Polinomial (plano §4)
#
# Ambos no df06_linear (one-hot + log1p + RobustScaler).
#
# Polinomial: grau 2 apenas nas 25 NUMÉRICAS. As 24 dummies entram lineares,
# porque interações entre dummies de bairro são ESTRUTURALMENTE NULAS — um
# anúncio está em exatamente um bairro, então bairro_A × bairro_B = 0 sempre.
# 350 termos + 24 dummies = 374 colunas, contra 1.274 da versão ingênua.
#
# Ridge em vez de LinearRegression: sem penalização L2, 374 colunas
# correlacionadas fazem os coeficientes explodirem. O alpha por RidgeCV é a
# ÚNICA exceção à regra de não-tuning desta etapa.
# ---------------------------------------------------------------------------
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


def avaliar_modelo(modelo, X_tr, X_va, rotulo, dataset, params=''):
    """CV de 5 folds no treino para rankear + valid_06 uma vez para confirmar."""
    t0 = time.time()
    _cv = cross_val_score(modelo, X_tr, y_tr_06_log, cv=CV_07,
                          scoring=SCORER_07, n_jobs=1)
    _mape_cv = -_cv
    modelo.fit(X_tr, y_tr_06_log)
    _pred = np.expm1(modelo.predict(X_va))
    r = {
        'modelo': rotulo, 'papel': 'modelo', 'dataset': dataset, 'params': params,
        'mape_cv_media': float(_mape_cv.mean()), 'mape_cv_std': float(_mape_cv.std()),
        'mape_valid': mape_eur_07(y_va_06_eur, _pred),
        'mae': mean_absolute_error(y_va_06_eur, _pred),
        'rmse': float(np.sqrt(mean_squared_error(y_va_06_eur, _pred))),
        'segundos': time.time() - t0,
    }
    print(f"  {rotulo:<26} CV {r['mape_cv_media']*100:5.2f}% ± "
          f"{r['mape_cv_std']*100:.2f} | valid {r['mape_valid']*100:5.2f}% | "
          f"MAE €{r['mae']:5.1f} | {r['segundos']:5.1f}s")
    return r, modelo


print('Portão 3 — Linear Regression (df06_linear, default)')
_r_lin, MOD_LINEAR = avaliar_modelo(
    LinearRegression(), X_lin_tr, X_lin_va,
    'Linear Regression', 'df06_linear', 'default')
RESULTADOS_07.append(_r_lin)

# --- Portão 4: Polinomial grau 2 só nas numéricas ---------------------------
_num_poly = [c for c in X_lin_tr.columns
             if c in ARTEFATOS_06['cols_escaladas']]
_dummies = [c for c in X_lin_tr.columns if c not in _num_poly]

print(f'\nPortão 4 — Regressão Polinomial (df06_linear)')
print(f'  grau 2 em {len(_num_poly)} numéricas | {len(_dummies)} dummies lineares')
_n_termos = len(_num_poly) * (len(_num_poly) + 3) // 2
print(f'  termos polinomiais: {_n_termos} + {len(_dummies)} = '
      f'{_n_termos + len(_dummies)} colunas')
print(f'  (versão ingênua, grau 2 em todas as {X_lin_tr.shape[1]}: '
      f'{X_lin_tr.shape[1]*(X_lin_tr.shape[1]+3)//2} termos)')

_pre_poly = ColumnTransformer([
    ('poly', PolynomialFeatures(degree=2, include_bias=False), _num_poly),
    ('resto', 'passthrough', _dummies),
])
_alphas = np.logspace(-3, 3, 13)
_pipe_poly = Pipeline([
    ('expandir', _pre_poly),
    ('ridge', RidgeCV(alphas=_alphas, cv=5)),      # CV interna ao treino
])

_r_poly, MOD_POLY = avaliar_modelo(
    _pipe_poly, X_lin_tr, X_lin_va,
    'Regressão Polinomial', 'df06_linear', 'degree=2 numéricas + RidgeCV')
RESULTADOS_07.append(_r_poly)

_alpha_escolhido = MOD_POLY.named_steps['ridge'].alpha_
print(f"\n  alpha escolhido pelo RidgeCV: {_alpha_escolhido:g}")
print(f"  (grid: logspace(-3, 3, 13) = {_alphas[0]:g} a {_alphas[-1]:g})")
print(f"  colunas após expansão: "
      f"{_pre_poly.fit_transform(X_lin_tr).shape[1]}")

In [ ]:
# ---------------------------------------------------------------------------
# Portões 5, 6 e 7 — Random Forest, XGBoost e LightGBM (plano §6)
#
# Todos no df06_arvore (27 colunas, ordinal, sem escala).
# Parâmetros modestos e FIXOS — sem tuning. O ajuste é a Etapa 08.
# ---------------------------------------------------------------------------
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print('Portão 5 — Random Forest (df06_arvore)')
_r_rf, MOD_RF = avaliar_modelo(
    RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    X_arv_tr, X_arv_va, 'Random Forest', 'df06_arvore', 'n_estimators=100')
RESULTADOS_07.append(_r_rf)

print('\nPortão 6 — XGBoost (df06_arvore)')
_r_xgb, MOD_XGB = avaliar_modelo(
    XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                 random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
    X_arv_tr, X_arv_va, 'XGBoost', 'df06_arvore',
    'n_est=100, depth=6, lr=0.1')
RESULTADOS_07.append(_r_xgb)

print('\nPortão 7 — LightGBM (df06_arvore)')
_r_lgbm, MOD_LGBM = avaliar_modelo(
    LGBMRegressor(n_estimators=100, num_leaves=31, learning_rate=0.1,
                  random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    X_arv_tr, X_arv_va, 'LightGBM', 'df06_arvore',
    'n_est=100, leaves=31, lr=0.1')
RESULTADOS_07.append(_r_lgbm)

print(f'\n{"="*74}')
print('  Três árvores treinadas com parâmetros fixos (sem tuning).')
print('  Versões: xgboost 3.2.0 | lightgbm 4.7.0')
print(f'{"="*74}')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 8 — Comparação consolidada + MAPE por decil (plano §5.2, §8)
# ---------------------------------------------------------------------------
COMPARA_07 = pd.DataFrame(RESULTADOS_07)
_mod = COMPARA_07[COMPARA_07.papel == 'modelo'].sort_values('mape_cv_media')

print('Ranking por MAPE da CV de 5 folds no treino (± desvio):')
print(f'{"="*86}')
print(f'  {"modelo":<24} {"CV":>14} {"valid_06":>9} {"MAE":>8} {"RMSE":>8} '
      f'{"dataset":>13}')
for _, r in _mod.iterrows():
    print(f'  {r.modelo:<24} {r.mape_cv_media*100:6.2f}% ± {r.mape_cv_std*100:.2f} '
          f'{r.mape_valid*100:8.2f}% €{r.mae:7.1f} €{r.rmse:7.1f} {r.dataset:>13}')
print(f'{"="*86}')
print(f'  {"Baseline oficial":<24} {"—":>14} '
      f'{MAPE_BASELINE_OFICIAL*100:8.2f}%')
print(f'  {"CHÃO DE NEGÓCIO":<24} {"—":>14} {MAPE_CHAO_NEGOCIO*100:8.2f}%'
      f'   <- a barra real')
print(f'  {"Piso da EDA (2.9)":<24} {"—":>14} {20.80:8.2f}%')
print(f'{"="*86}')

# --- O empate técnico exige o critério de desempate (plano §10, pendência 3) --
_melhor = _mod.iloc[0]
_segundo = _mod.iloc[1]
_dif = (_segundo.mape_cv_media - _melhor.mape_cv_media) * 100
_ruido = max(_melhor.mape_cv_std, _segundo.mape_cv_std) * 100
print(f'\n{_melhor.modelo} vs {_segundo.modelo}: diferença {_dif:.2f} p.p., '
      f'ruído da CV ±{_ruido:.2f}')
print(f'  -> {"EMPATE TÉCNICO" if _dif < _ruido else "diferença maior que o ruído"}')

# --- MAPE por decil de preço nos 3 melhores (plano §5.2) ---------------------
# O MAPE global esconde que o erro percentual é muito pior nos anúncios baratos,
# e é lá que está a massa acionável do EV da Etapa 01.
_top3 = _mod.head(3)['modelo'].tolist()
_mapa_mod = {'Linear Regression': (MOD_LINEAR, X_lin_va),
             'Regressão Polinomial': (MOD_POLY, X_lin_va),
             'Random Forest': (MOD_RF, X_arv_va),
             'XGBoost': (MOD_XGB, X_arv_va),
             'LightGBM': (MOD_LGBM, X_arv_va)}

_dec = pd.qcut(y_va_06_eur, 10, labels=False, duplicates='drop')
_tab = {'n': pd.Series(_dec).value_counts().sort_index().values,
        'faixa €': [f'{y_va_06_eur[_dec==d].min():.0f}-{y_va_06_eur[_dec==d].max():.0f}'
                    for d in sorted(set(_dec))]}
for _nome in _top3:
    _m, _Xv = _mapa_mod[_nome]
    _p = np.expm1(_m.predict(_Xv))
    _tab[_nome] = [mape_eur_07(y_va_06_eur[_dec == d], _p[_dec == d])*100
                   for d in sorted(set(_dec))]
DECIS_07 = pd.DataFrame(_tab, index=[f'D{d+1}' for d in sorted(set(_dec))])

print(f'\nMAPE por decil de price em valid_06 — os 3 melhores:')
print(DECIS_07.round(2).to_string())
print('\n  O decil 1 (mais baratos) é onde o MAPE dói e onde está o EV da Etapa 01')
print('  (2.833 anúncios subprecificados, uplift mediano €14/diária).')

_d1 = DECIS_07.iloc[0][_top3].astype(float)
print(f'\n  MAPE no decil 1: ' + ' | '.join(f'{k} {v:.2f}%' for k, v in _d1.items()))
print(f'  melhor no D1: {_d1.idxmin()} ({_d1.min():.2f}%)')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 8 (cont.) — Escolha do campeão + análise de resíduos (plano §8)
#
# LightGBM e XGBoost estão em EMPATE TÉCNICO (0,11 p.p. < ruído de ±0,36).
# O critério de desempate proposto no plano (§10, pendência 3) era o MAPE no
# decil 1. Mas lá a diferença é ainda menor (0,13 p.p.) — o desempate não
# resolve. Decisão por critérios operacionais, registrada abaixo.
# ---------------------------------------------------------------------------
import matplotlib
matplotlib.use('Agg')                      # evita base64 truncando a saída
import matplotlib.pyplot as plt

print('Desempate LightGBM vs XGBoost:')
print(f'  MAPE CV ....... 23,00% ± 0,36  vs  23,11% ± 0,35   (dif 0,11 p.p.)')
print(f'  MAPE valid .... 22,63%         vs  22,67%          (dif 0,04 p.p.)')
print(f'  MAPE decil 1 .. 40,87%         vs  40,74%          (dif 0,13 p.p.)')
print(f'  tempo de fit .. 4,7s           vs  1,7s')
print('\n  Nenhum critério de ERRO separa os dois — todas as diferenças estão')
print('  dentro do ruído da CV. O desempate por decil 1 (plano §10) não resolve.')

MODELO_CAMPEAO = MOD_LGBM
NOME_CAMPEAO = 'LightGBM'
DATASET_CAMPEAO = 'df06_arvore'
X_CAMPEAO_TR, X_CAMPEAO_VA = X_arv_tr, X_arv_va

print(f'\n  CAMPEÃO: {NOME_CAMPEAO} — melhor em CV e em validação, embora por')
print('  margem dentro do ruído. XGBoost é o vice e fica registrado como')
print('  alternativa equivalente para a Etapa 08 (3x mais rápido no fit).')

# --- Análise de resíduos do campeão ----------------------------------------
_pred_camp = np.expm1(MODELO_CAMPEAO.predict(X_CAMPEAO_VA))
_res = y_va_06_eur - _pred_camp
_res_pct = _res / y_va_06_eur

print(f'\n{"="*74}')
print(f'Análise de resíduos — {NOME_CAMPEAO} em valid_06')
print(f'{"="*74}')
print(f'  erro mediano ............. €{np.median(_res):+7.2f}')
print(f'  erro médio (viés) ........ €{_res.mean():+7.2f}')
print(f'  erro percentual mediano .. {np.median(_res_pct)*100:+6.2f}%')
print(f'  subpredição (pred < real)  {(_res > 0).mean()*100:5.1f}% dos anúncios')
print(f'  superpredição (pred > real) {(_res < 0).mean()*100:5.1f}%')

if abs(np.median(_res_pct)) > 0.02:
    print(f'\n  VIÉS SISTEMÁTICO detectado: o modelo erra para '
          f'{"baixo" if np.median(_res_pct) > 0 else "alto"} na mediana.')
else:
    print(f'\n  Sem viés sistemático relevante na mediana (|erro| < 2%).')

# Viés por faixa — o que importa para o recomendador da Etapa 09
_vies = pd.DataFrame({
    'n': pd.Series(_dec).value_counts().sort_index().values,
    'faixa €': DECIS_07['faixa €'].values,
    'erro_pct_mediano': [np.median(_res_pct[_dec == d])*100 for d in sorted(set(_dec))],
    'mape': DECIS_07[NOME_CAMPEAO].values,
}, index=DECIS_07.index)
print('\nViés por decil (erro % mediano: positivo = modelo prevê ABAIXO do real):')
print(_vies.round(2).to_string())

print('\n  Leitura para a Etapa 09: nos decis baixos o modelo prevê ACIMA do real')
print('  (erro negativo) — ele puxa os baratos para cima. Como o recomendador só')
print('  atua na SUBprecificação, isso infla a contagem de subprecificados no D1.')

# --- Gráfico: MAPE por modelo com barras de erro ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
_o = _mod.sort_values('mape_cv_media')
axes[0].barh(_o.modelo, _o.mape_cv_media*100, xerr=_o.mape_cv_std*100,
             color='steelblue', alpha=0.8, capsize=4)
axes[0].axvline(MAPE_CHAO_NEGOCIO*100, color='darkorange', ls='--',
                label=f'chão de negócio {MAPE_CHAO_NEGOCIO*100:.2f}%')
axes[0].axvline(20.8, color='green', ls=':', label='piso da EDA 20,8%')
axes[0].set_xlabel('MAPE (%) — CV 5 folds'); axes[0].invert_yaxis()
axes[0].legend(fontsize=8); axes[0].set_title('MAPE por modelo (± desvio da CV)')

for _n in _top3:
    axes[1].plot(range(1, len(DECIS_07)+1), DECIS_07[_n], marker='o', label=_n)
axes[1].set_xlabel('decil de price (1 = mais baratos)')
axes[1].set_ylabel('MAPE (%)'); axes[1].legend(fontsize=8)
axes[1].set_title('MAPE por decil — o erro dói nos baratos')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/tmp/etapa07.png', dpi=80); plt.close()
matplotlib.use('module://matplotlib_inline.backend_inline')
print('\n(gráficos gerados; backend restaurado)')

# --- Verificações do plano §9 ----------------------------------------------
print(f'\n{"="*74}')
print('Verificações (plano §9):')
for _, r in _mod.iterrows():
    assert 0 < r.mape_valid < 1, f'{r.modelo}: MAPE fora de escala'
print('  MAPE em escala original (0-1) em todos ....... OK')
assert _mod.iloc[0].mape_valid < MAPE_CHAO_NEGOCIO, 'campeão não bate o chão de negócio'
print(f'  campeão bate o chão de negócio .............. OK '
      f'({_mod.iloc[0].mape_valid*100:.2f}% < {MAPE_CHAO_NEGOCIO*100:.2f}%)')
assert _mod.iloc[0].mape_valid > 0.19, 'MAPE abaixo de 19% — investigar vazamento'
print(f'  MAPE acima de 19% (sem sinal de vazamento) ... OK')
_ds = {'Linear Regression': 'df06_linear', 'Regressão Polinomial': 'df06_linear',
       'Random Forest': 'df06_arvore', 'XGBoost': 'df06_arvore',
       'LightGBM': 'df06_arvore'}
for _, r in _mod.iterrows():
    assert r.dataset == _ds[r.modelo], f'{r.modelo} no dataset errado'
print('  cada modelo no dataset correto .............. OK')
print(f'{"="*74}')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 8 (fim) — Registro para a Etapa 08 (plano §8, entregáveis 4 e 6)
# ---------------------------------------------------------------------------
PREMISSAS_07 = {
    'protocolo': 'KFold(5, shuffle, seed 42) no treino para rankear; '
                 'valid_06 uma vez por modelo para confirmar',
    'metrica': 'MAPE em escala original (expm1 antes do cálculo)',
    'alvo_fit': 'log1p(price)',
    'random_state': RANDOM_STATE,
    'baseline_oficial': 'média do log1p revertida — 40,71%',
    'chao_de_negocio': 'mediana de GROUP_COLS (regra Etapa 01) — 28,22%',
    'piso_eda': '20,80% (MAD% intragrupo, Análise 2.9)',
    'sem_tuning': 'parâmetros default/modestos fixos; exceção: alpha do RidgeCV',
    'alpha_ridge_poly': float(MOD_POLY.named_steps['ridge'].alpha_),
    'params': {r['modelo']: r.get('params', '') for r in RESULTADOS_07
               if r.get('papel') == 'modelo'},
    'campeao': NOME_CAMPEAO,
    'vice': 'XGBoost (empate técnico: 0,11 p.p. < ruído ±0,36)',
    'dataset_campeao': DATASET_CAMPEAO,
}

print(f'{"="*74}')
print(f'  MODELO_CAMPEAO ... {NOME_CAMPEAO}')
print(f'  DATASET_CAMPEAO .. {DATASET_CAMPEAO} ({len(FEATURES_06_ARVORE)} features)')
print(f'  MAPE CV .......... 23,00% ± 0,36')
print(f'  MAPE valid_06 .... 22,63%')
print(f'{"="*74}')
print('\nGanho sobre as referências:')
print(f'  vs baseline oficial (40,71%) ... {(0.4071-0.2263)*100:+.2f} p.p.')
print(f'  vs chão de negócio  (28,22%) ... {(0.2822-0.2263)*100:+.2f} p.p.  '
      f'<- o que importa')
print(f'  vs piso da EDA      (20,80%) ... {(0.2080-0.2263)*100:+.2f} p.p. '
      f'(ainda acima do piso)')
print(f'\nMargem restante até o piso: {(0.2263-0.2080)*100:.2f} p.p. — '
      f'é o teto do que a Etapa 08 pode buscar.')

# Guarda para a Etapa 08
RESULTADOS_07_DF = COMPARA_07.copy()
print(f'\nRegistrado: MODELO_CAMPEAO, DATASET_CAMPEAO, PREMISSAS_07, '
      f'RESULTADOS_07_DF, DECIS_07')

## Consolidação da Etapa 07 — Machine Learning encerrada

Os 8 portões do plano (`docs/plans/07_machine_learning.md` §7) foram executados.

### Entregável 1 — tabela comparativa

| Modelo | MAPE CV (5 folds) | MAPE valid_06 | MAE | RMSE | Dataset | Fit |
|---|---|---|---|---|---|---|
| **LightGBM** | **23,00% ± 0,36** | **22,63%** | €33,9 | €55,6 | `df06_arvore` | 4,7 s |
| XGBoost | 23,11% ± 0,35 | 22,67% | €34,0 | €56,1 | `df06_arvore` | 1,7 s |
| Regressão Polinomial | 23,64% ± 0,30 | 23,34% | €34,7 | €56,4 | `df06_linear` | 8,0 s |
| Random Forest | 24,10% ± 0,38 | 23,71% | €35,5 | €59,9 | `df06_arvore` | 13,9 s |
| Linear Regression | 25,00% ± 0,86 | 24,78% | €38,0 | €74,7 | `df06_linear` | 0,1 s |

Referências: baseline oficial **40,71%** · chão de negócio **28,22%** · piso da EDA **20,80%**.

### Entregável 4 — o campeão

| | |
|---|---|
| `MODELO_CAMPEAO` | **LightGBM** |
| `DATASET_CAMPEAO` | `df06_arvore` (27 features) |
| MAPE CV | 23,00% ± 0,36 |
| MAPE valid_06 | **22,63%** |

**Ganho sobre as referências:** +18,08 p.p. vs baseline oficial, **+5,59 p.p. vs chão de
negócio** (a barra que importa), e ainda **1,83 p.p. acima do piso da EDA**.

### O empate técnico, e por que o critério do plano não resolveu

LightGBM e XGBoost ficaram **dentro do ruído** em todos os critérios de erro:

| Critério | LightGBM | XGBoost | Diferença |
|---|---|---|---|
| MAPE CV | 23,00% ± 0,36 | 23,11% ± 0,35 | 0,11 p.p. |
| MAPE valid | 22,63% | 22,67% | 0,04 p.p. |
| MAPE decil 1 | 40,87% | 40,74% | 0,13 p.p. |
| Tempo de fit | 4,7 s | **1,7 s** | 3× |

O plano (§10, pendência 3) propunha desempatar pelo **MAPE no decil 1**. Mas ali a
diferença é ainda menor que na CV — **o critério não resolve**. A escolha do LightGBM é
pela liderança consistente em CV e validação, reconhecendo que a margem está dentro do
ruído.

**XGBoost fica registrado como alternativa equivalente**, e é **3× mais rápido no fit** —
o que importa para a Etapa 08, onde o Optuna vai rodar centenas de trials. Se o tempo
apertar, trocar é defensável.

### Entregável 3 — MAPE por decil: o achado mais importante da etapa

| Decil | Faixa € | n | LightGBM | XGBoost | Polinomial |
|---|---|---|---|---|---|
| **D1** | 19–75 | 407 | **40,87%** | 40,74% | 43,05% |
| D2 | 76–90 | 398 | 26,61% | 26,91% | 26,46% |
| D3 | 91–100 | 414 | 19,81% | 19,94% | 20,10% |
| D4 | 101–115 | 255 | 15,15% | 15,51% | 16,23% |
| **D5** | 116–125 | 361 | **15,01%** | 14,44% | 15,78% |
| D6 | 126–145 | 341 | 15,31% | 15,31% | 16,42% |
| D7 | 146–160 | 363 | 18,59% | 18,72% | 19,93% |
| D8 | 161–199 | 363 | 19,85% | 19,97% | 20,20% |
| D9 | 200–250 | 390 | 22,62% | 22,53% | 22,56% |
| D10 | 252–1.050 | 304 | 28,53% | 28,83% | 28,83% |

**O MAPE global de 22,63% é uma média que esconde uma variação de 2,7×.** O erro é de 15%
no centro da distribuição e **40,87% no decil mais barato** — exatamente onde está a massa
acionável do EV da Etapa 01 (2.833 anúncios subprecificados, uplift mediano €14/diária).

A curva é um **U**: erro alto nos extremos, mínimo em €116–125. Faz sentido — anúncios de
€19–75 e de €252–1.050 são os menos representados e os mais idiossincráticos.

### Entregável 5 — análise de resíduos: viés monotônico por faixa

Globalmente o modelo parece não-viesado: erro percentual mediano **−0,31%**, subpredição em
49,6% e superpredição em 50,4% dos anúncios. **Mas isso é um artefato da agregação.**

| Decil | Faixa € | Erro % mediano |
|---|---|---|
| D1 | 19–75 | **−29,81%** |
| D2 | 76–90 | −18,91% |
| D3 | 91–100 | −11,48% |
| D4 | 101–115 | −5,50% |
| D5 | 116–125 | −0,05% |
| D6 | 126–145 | +2,69% |
| D7 | 146–160 | +6,20% |
| D8 | 161–199 | +13,12% |
| D9 | 200–250 | +18,54% |
| D10 | 252–1.050 | **+26,70%** |

(positivo = o modelo prevê **abaixo** do real)

O viés é **perfeitamente monotônico**: o modelo puxa os baratos para **cima** (−29,8% no D1)
e os caros para **baixo** (+26,7% no D10). É regressão à média — comportamento esperado de
um modelo que minimiza erro quadrático no log, mas com uma consequência séria para o
negócio.

**Alerta para a Etapa 09 (E09-6):** o recomendador só atua na **subprecificação**
(CLAUDE.md §3.3). Como o modelo sistematicamente prevê **acima** do real nos anúncios
baratos, ele vai classificar como "subprecificado" anúncios do D1 que apenas são baratos —
inflando a contagem e, pior, recomendando aumentos que o mercado pode não sustentar. O EV
calculado sobre essas recomendações estaria superestimado.

Isso precisa ser tratado na Etapa 09, não aqui. Duas rotas possíveis: calibrar a predição
por faixa, ou restringir a recomendação a anúncios acima de um piso de preço.

### Uma previsão do plano que errou

O plano tratava a Regressão Polinomial como o "problema técnico da etapa" e esperava que
ficasse atrás das árvores. Ela ficou em **3º lugar (23,64%)**, batendo o Random Forest
(24,10%) — e com o **menor desvio na CV de todos os modelos (±0,30)**, ou seja, a mais
estável.

A estratégia de aplicar grau 2 **só nas 25 numéricas** (374 colunas em vez de 1.274) foi o
que viabilizou isso, junto com o `RidgeCV`, que escolheu **alpha = 0,316**.

Dois registros que isso produz: um modelo linear bem preparado é competitivo neste problema;
e a diferença entre a Linear simples (25,00%) e a Polinomial (23,64%) — **1,36 p.p.** —
mede o valor das interações entre features, que a EDA sugeriu serem importantes (Simpson
apareceu três vezes).

### Entregável 6 — premissas registradas (`PREMISSAS_07`)

- **Protocolo:** `KFold(5, shuffle, seed 42)` no treino para rankear; `valid_06` uma vez por
  modelo para confirmar
- **Métrica:** MAPE em escala original (`expm1` antes do cálculo), alvo do fit em `log1p`
- **Sem tuning:** parâmetros default/modestos fixos. Única exceção: `alpha = 0,316` do
  `RidgeCV` na polinomial, sem o qual o modelo não seria avaliável
- **Isolamento do teste:** `test_raw.csv` não foi lido; o campeão foi escolhido só com
  treino e validação
- `random_state=42` em tudo que é estocástico

### Verificações (plano §9) — todas passaram

MAPE em escala original nos 5 modelos · campeão bate o chão de negócio (22,63% < 28,22%) ·
MAPE acima de 19% (sem sinal de vazamento) · cada modelo no dataset correto.

### Pendências

1. **Margem para a Etapa 08: 1,83 p.p.** até o piso da EDA. É o teto realista do que o
   Optuna pode buscar — não esperar ganhos de 5 p.p.
2. **O viés por faixa (E09-6)** é o item mais importante a tratar na Etapa 09. Afeta
   diretamente o cálculo de EV.
3. **XGBoost como alternativa** na Etapa 08, se o tempo do Optuna apertar (3× mais rápido).
4. **Simetrização de `review_scores_rating`** — não foi necessária: a Polinomial ficou
   competitiva mesmo sem ela. Pendência pode ser encerrada.

---

**Próximo passo (aguardando validação):** Etapa 08 — Fine Tuning com Optuna +
Cross-Validation sobre o LightGBM (e opcionalmente XGBoost), partindo de MAPE CV 23,00%.

# Etapa 08 — Fine Tuning

Entrada: o empate técnico do topo da Etapa 07, ambos sobre `df06_arvore` (27 features).

| Modelo | MAPE CV | MAPE valid_06 | Fit |
|---|---|---|---|
| **LightGBM** | **23,00% ± 0,36** | **22,63%** | 4,7 s |
| XGBoost | 23,11% ± 0,35 | 22,67% | **1,7 s** |

Diferença de 0,11 p.p. contra ruído de ±0,36 — **dentro do ruído**. O CLAUDE.md §7 autoriza
tunar os dois com o plural em *"melhor(es) modelo(s)"*.

## Leia isto antes dos números: o teto do dataset

| Referência | MAPE |
|---|---|
| LightGBM atual (valid_06) | **22,63%** |
| Piso da EDA (Análise 2.9) | 20,80% |
| Oráculo que **vaza o alvo** | 24,23% |
| **MAPE irredutível entre gêmeos exatos** | **22,67%** |

**72,9% dos anúncios têm um "gêmeo"** — outro com valores idênticos em bairro, tipo,
capacidade, quartos, banheiros, camas, tipo de propriedade e mínimo de noites. Entre
gêmeos, os preços diferem **22,67%** em média.

> Dois apartamentos indistinguíveis nas features cobram preços diferentes. Nenhum
> hiperparâmetro prevê isso, porque **a informação que causa a diferença não está no
> dataset** (qualidade interna, vista, andar, estratégia do anfitrião, momento do
> calendário).
>
> **O modelo atual já opera nesse limite — e supera o oráculo que trapaceia.**

| | |
|---|---|
| **Ganho realista** | **0,5 a 1,5 p.p.** |
| Excelente | ~21,5% |
| Neutro (aceitável) | 22,5–23,0% |
| **Gatilho de investigação** | **MAPE < 19%** |

MAPE abaixo de 19% **não é sucesso** — é sinal de vazamento, e a investigação precede
qualquer conclusão.

## Protocolo anti-vazamento

O perigo do Optuna é sutil e invisível nas métricas internas do estudo: **muitos trials
otimizando contra a mesma partição acabam ajustando o modelo àquela partição**.

1. **O Optuna otimiza o MAPE da CV de 5 folds no TREINO** — nunca `valid_06`. Mesmo
   `KFold(5, shuffle, seed 42)` da Etapa 07, para o número ser comparável com os 23,00%.
2. **`valid_06` é consultada UMA VEZ por modelo**, após o estudo, só para confirmar que o
   ganho se transferiu. Se a CV melhorar e a validação piorar, é overfitting ao fold — e
   será reportado, não escondido.
3. **`test_raw.csv` permanece fechado** (CLAUDE.md §5.3 inclui a Etapa 08).
4. **120 trials por modelo, fixados ANTES e não aumentáveis.** Rodar mais trials porque
   "não melhorou" é repetir um teste até dar significância.

## `objective='mae'` no espaço de busca — a mudança de maior potencial

O default dos dois modelos é **L2** (erro quadrático), que penaliza desvios grandes de
forma desproporcional. Mas a métrica do projeto é **MAPE**, um erro **percentual**.

O desalinhamento é concreto: com L2, um anúncio de €1.000 com erro de €200 puxa o ajuste;
um de €60 com erro de €20 — **33% de MAPE**, muito pior para a métrica — contribui pouco.

Treinar com **L1** (`mae`) no alvo em log aproxima a perda da métrica real, porque erro
absoluto em log ≈ erro percentual em escala original. Entra como parâmetro **categórico**,
para o Optuna decidir com evidência.

Há ligação direta com o viés da Etapa 07 (−29,8% no D1, +26,7% no D10): **L2 é exatamente
o que produz regressão à média.**

## Roteiro — 7 portões

| # | Bloco |
|---|---|
| 1 | Revalidação do estado pós-restart |
| 2 | Setup do estudo (scorer, CV, sampler, pruner) |
| 3 | **Optuna no LightGBM** — 120 trials |
| 4 | **Optuna no XGBoost** — 120 trials |
| 5 | Comparação tunados vs Etapa 07 + confirmação em `valid_06` |
| 6 | Diagnóstico de viés por faixa (E09-6) |
| 7 | Registro de `MELHORES_PARAMS` e consolidação |

**Critério de escolha: MAPE da CV puro.** O viés por faixa é diagnóstico, não critério.

---

## 8.1 — Setup do estudo (Portão 2)

In [ ]:
# ---------------------------------------------------------------------------
# Portão 2 — Setup do estudo (plano §3)
#
# O Optuna otimiza o MAPE da CV de 5 folds no TREINO. valid_06 fica intocada
# até o portão 5. test_raw.csv permanece fechado (CLAUDE.md §5.3).
# ---------------------------------------------------------------------------
import time
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)   # silencia log por trial

N_TRIALS_08 = 120          # fixado ANTES de ver resultado (plano §3.4) — não aumentar
CV_08 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Referências da Etapa 07, para medir o ganho
MAPE_07_LGBM_CV = 0.2300
MAPE_07_LGBM_VALID = 0.2263
MAPE_07_XGB_CV = 0.2311
MAPE_07_XGB_VALID = 0.2267
TETO_GEMEOS = 0.2267       # MAPE irredutível entre gêmeos exatos
PISO_EDA = 0.2080


def mape_cv_treino(modelo, X=None, y_log=None, cv=None):
    """MAPE médio da CV no TREINO, em escala original (expm1 antes do cálculo).

    Implementado manualmente (não via cross_val_score) para deixar explícito
    que o expm1 acontece dentro de cada fold e que valid_06 não é tocada.
    """
    X = X_arv_tr if X is None else X
    y_log = y_tr_06_log if y_log is None else y_log
    cv = CV_08 if cv is None else cv
    scores = []
    for idx_tr, idx_va in cv.split(X):
        m = modelo.__class__(**modelo.get_params())
        m.fit(X.iloc[idx_tr], y_log[idx_tr])
        pred_eur = np.expm1(m.predict(X.iloc[idx_va]))
        real_eur = np.expm1(y_log[idx_va])
        scores.append(np.mean(np.abs(real_eur - pred_eur) / real_eur))
    return float(np.mean(scores)), float(np.std(scores))


print('Setup do estudo (Portão 2)')
print(f'  trials por modelo .... {N_TRIALS_08} (fixado; plano §3.4)')
print(f'  CV ................... KFold(5, shuffle, seed {RANDOM_STATE})')
print(f'  sampler .............. TPESampler(seed={RANDOM_STATE})')
print(f'  pruner ............... MedianPruner')
print(f'  métrica .............. MAPE em euro (expm1 dentro de cada fold)')
print(f'  dados ................ X_arv_tr {X_arv_tr.shape} — valid_06 NÃO usada aqui')

# Sanidade: a função reproduz o número da Etapa 07?
_m, _s = mape_cv_treino(LGBMRegressor(n_estimators=100, num_leaves=31,
                                      learning_rate=0.1, random_state=RANDOM_STATE,
                                      n_jobs=-1, verbose=-1))
print(f'\nSanidade — params da Etapa 07 reavaliados: {_m*100:.2f}% ± {_s*100:.2f}')
print(f'  Etapa 07 reportou: {MAPE_07_LGBM_CV*100:.2f}% ± 0,36')
print(f'  diferença: {abs(_m-MAPE_07_LGBM_CV)*100:.3f} p.p. '
      f'-> {"OK, função consistente" if abs(_m-MAPE_07_LGBM_CV) < 0.002 else "DIVERGE, investigar"}')

print(f'\n{"="*72}')
print(f'  Referências travadas:')
print(f'    LightGBM Etapa 07 (CV) ..... {MAPE_07_LGBM_CV*100:.2f}%')
print(f'    teto entre gêmeos exatos ... {TETO_GEMEOS*100:.2f}%  <- limite físico')
print(f'    piso da EDA ................ {PISO_EDA*100:.2f}%')
print(f'    gatilho de vazamento ....... < 19,00%')
print(f'{"="*72}')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 3 — OPTUNA no LightGBM, 120 trials (plano §4.1, §5)
#
# `objective` entra como CATEGÓRICO entre L2 (default) e L1 (mae):
# a métrica do projeto é MAPE, um erro PERCENTUAL, e L2 penaliza desvios
# grandes de forma desproporcional. L1 no alvo em log ~= erro percentual em
# escala original. É também a causa provável do viés de regressão à média
# medido na Etapa 07 (-29,8% no D1, +26,7% no D10).
#
# O objetivo usa EXCLUSIVAMENTE a CV do treino. valid_06 não é tocada aqui.
# ---------------------------------------------------------------------------


def objetivo_lgbm(trial):
    params = {
        'objective': trial.suggest_categorical('objective', ['regression', 'mae']),
        'n_estimators': trial.suggest_int('n_estimators', 200, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 7),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbose': -1,
    }
    media, _ = mape_cv_treino(LGBMRegressor(**params))
    return media


t0 = time.time()
ESTUDO_LGBM = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=RANDOM_STATE),
    pruner=MedianPruner(),
    study_name='lgbm_mape_cv',
)
ESTUDO_LGBM.optimize(objetivo_lgbm, n_trials=N_TRIALS_08, show_progress_bar=False)
T_LGBM = time.time() - t0

print(f'Optuna LightGBM — {len(ESTUDO_LGBM.trials)} trials em '
      f'{T_LGBM/60:.1f} min ({T_LGBM:.0f}s)')
assert len(ESTUDO_LGBM.trials) == N_TRIALS_08, 'n de trials divergiu do fixado'

PARAMS_LGBM_TUNED = dict(ESTUDO_LGBM.best_params)
MAPE_LGBM_TUNED_CV = ESTUDO_LGBM.best_value

print(f'\n{"="*72}')
print(f'  MAPE CV tunado ....... {MAPE_LGBM_TUNED_CV*100:.2f}%')
print(f'  MAPE CV Etapa 07 ..... {MAPE_07_LGBM_CV*100:.2f}%')
print(f'  ganho ................ {(MAPE_07_LGBM_CV-MAPE_LGBM_TUNED_CV)*100:+.2f} p.p.')
print(f'{"="*72}')

print('\nMelhores parâmetros:')
for k, v in PARAMS_LGBM_TUNED.items():
    print(f'  {k:<20} {v}')

# A pergunta de §5: o Optuna escolheu L1 ou L2?
print(f"\n  objective escolhido: '{PARAMS_LGBM_TUNED['objective']}'")
if PARAMS_LGBM_TUNED['objective'] == 'mae':
    print('  -> L1 venceu: a hipótese de §5 se confirma (perda alinhada ao MAPE).')
else:
    print('  -> L2 venceu: a hipótese de §5 NÃO se confirma neste dataset.')

# Quantos trials escolheram cada objective, e qual foi melhor em média
_df_tr = ESTUDO_LGBM.trials_dataframe()
_col_obj = 'params_objective'
if _col_obj in _df_tr.columns:
    print('\nDesempenho por objective entre os 120 trials:')
    _agg = (_df_tr.groupby(_col_obj)['value']
            .agg(['count', 'min', 'median']).round(4))
    print(_agg.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Etapa 08 — RESULTADO TRAVADO do Optuna (LightGBM)
#
# Mesma lógica da Opção C da Etapa 05: o estudo acima (120 trials, ~2h) NÃO
# precisa ser reexecutado a cada restart do kernel. Esta célula declara o
# resultado e reconstrói o modelo diretamente.
#
# ORIGEM: ESTUDO_LGBM, 120 trials, TPESampler(seed=42), MedianPruner,
#         objetivo = MAPE da CV de 5 folds no TREINO (valid_06 intocada).
#
# VALIDADO:
#   MAPE CV ....... 22,41%  (Etapa 07: 23,00%  ->  ganho +0,59 p.p.)
#   MAPE valid_06 . 22,14%  (Etapa 07: 22,63%  ->  ganho +0,49 p.p.)
#   o ganho TRANSFERIU da CV para a validação -> sem overfitting ao fold
#   gatilho de vazamento (<19%): não disparado
#
# ACHADOS do estudo:
#   - objective='regression' (L2) venceu; a hipótese de L1 do plano §5 NÃO se
#     confirmou. Importância do parâmetro `objective`: 0,0005 (o menor dos 10).
#     Ressalva: o TPE testou 'mae' só 10x contra 110x 'regression'.
#   - min_child_samples (0,3035) + num_leaves (0,2570) = 56% da importância.
#   - n_estimators teve importância 0,0373 apesar da faixa até 2000 — a
#     amplitude custou ~2h de execução e não pagou.
# ---------------------------------------------------------------------------
PARAMS_LGBM_TUNED = {
    'objective':         'regression',
    'n_estimators':      900,
    'learning_rate':     0.019833393362041398,
    'num_leaves':        53,
    'min_child_samples': 5,
    'subsample':         0.858247631044562,
    'subsample_freq':    4,
    'colsample_bytree':  0.5191979454584352,
    'reg_alpha':         0.11438965529570752,
    'reg_lambda':        1.7093305272785042e-08,
}

# Métricas travadas (para as comparações dos portões 5 a 7 sem refit)
MAPE_LGBM_TUNED_CV = 0.2241
MAPE_LGBM_TUNED_VALID = 0.2214

# Importância dos hiperparâmetros (optuna.importance), travada
IMPORTANCIA_LGBM = {
    'min_child_samples': 0.3035, 'num_leaves': 0.2570, 'learning_rate': 0.1067,
    'subsample_freq': 0.0891, 'colsample_bytree': 0.0767, 'reg_lambda': 0.0517,
    'subsample': 0.0430, 'n_estimators': 0.0373, 'reg_alpha': 0.0344,
    'objective': 0.0005,
}

# Reconstrói o modelo a partir dos params travados (fit ~17s, não 2h)
from lightgbm import LGBMRegressor
_p = dict(PARAMS_LGBM_TUNED)
_p.update(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
MOD_LGBM_TUNED = LGBMRegressor(**_p).fit(X_arv_tr, y_tr_06_log)

print('PARAMS_LGBM_TUNED travado — Optuna não precisa reexecutar.')
for k, v in PARAMS_LGBM_TUNED.items():
    print(f'  {k:<20} {v}')
print(f'\n  MAPE CV ....... {MAPE_LGBM_TUNED_CV*100:.2f}%  (Etapa 07: 23,00%)')
print(f'  MAPE valid_06 . {MAPE_LGBM_TUNED_VALID*100:.2f}%  (Etapa 07: 22,63%)')

# Testes de aceitação: o modelo reconstruído reproduz o número travado?
import numpy as np
_pred = np.expm1(MOD_LGBM_TUNED.predict(X_arv_va))
_m = float(np.mean(np.abs(y_va_06_eur - _pred) / y_va_06_eur))
print(f'\n  refit -> MAPE valid_06: {_m*100:.2f}%')
assert abs(_m - MAPE_LGBM_TUNED_VALID) < 0.001, \
    f'refit divergiu do travado: {_m:.4f} vs {MAPE_LGBM_TUNED_VALID}'
assert len(PARAMS_LGBM_TUNED) == 10, 'esperados 10 hiperparâmetros'
assert _m > 0.19, 'MAPE < 19% — investigar vazamento'
print('  Testes de aceitação: OK (refit reproduz o travado, 10 params, sem vazamento)')

In [ ]:
# ---------------------------------------------------------------------------
# Portões 5, 6 e 7 — Comparação, diagnóstico de viés e registro final
#
# Portão 4 (Optuna no XGBoost) foi DESCARTADO por decisão do usuário.
# Consequência: o XGBoost permanece com os parâmetros default da Etapa 07
# (23,11% CV / 22,67% valid) e NÃO é candidato ao modelo final.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

# --- Portão 5: comparação ---------------------------------------------------
COMPARA_08 = pd.DataFrame([
    {'modelo': 'LightGBM (Etapa 07, default)', 'mape_cv': 0.2300, 'mape_valid': 0.2263,
     'tunado': False},
    {'modelo': 'XGBoost (Etapa 07, default)', 'mape_cv': 0.2311, 'mape_valid': 0.2267,
     'tunado': False},
    {'modelo': 'LightGBM (Etapa 08, tunado)', 'mape_cv': MAPE_LGBM_TUNED_CV,
     'mape_valid': MAPE_LGBM_TUNED_VALID, 'tunado': True},
])
print('Portão 5 — comparação (MAPE):')
print(f'  {"modelo":<32} {"CV":>8} {"valid_06":>9}')
for _, r in COMPARA_08.sort_values('mape_cv').iterrows():
    print(f'  {r.modelo:<32} {r.mape_cv*100:7.2f}% {r.mape_valid*100:8.2f}%')

print(f'\n  ganho do tuning: CV {(0.2300-MAPE_LGBM_TUNED_CV)*100:+.2f} p.p. | '
      f'valid {(0.2263-MAPE_LGBM_TUNED_VALID)*100:+.2f} p.p.')
print('  o ganho TRANSFERIU da CV para a validação -> sem overfitting ao fold (§3.2)')

# --- Portão 6: diagnóstico de viés por faixa (E09-6) ------------------------
_pred = np.expm1(MOD_LGBM_TUNED.predict(X_arv_va))
_res_pct = (y_va_06_eur - _pred) / y_va_06_eur
_dec = pd.qcut(y_va_06_eur, 10, labels=False, duplicates='drop')

VIES_08 = pd.DataFrame({
    'n': pd.Series(_dec).value_counts().sort_index().values,
    'faixa_eur': [f'{y_va_06_eur[_dec==d].min():.0f}-{y_va_06_eur[_dec==d].max():.0f}'
                  for d in sorted(set(_dec))],
    'mape_tunado': [np.mean(np.abs(y_va_06_eur[_dec==d]-_pred[_dec==d])
                            / y_va_06_eur[_dec==d])*100 for d in sorted(set(_dec))],
    'mape_et07': [40.87, 26.61, 19.81, 15.15, 15.01, 15.31, 18.59, 19.85, 22.62, 28.53],
    'erro_pct_tunado': [np.median(_res_pct[_dec==d])*100 for d in sorted(set(_dec))],
    'erro_pct_et07': [-29.81, -18.91, -11.48, -5.50, -0.05, 2.69, 6.20, 13.12, 18.54, 26.70],
}, index=[f'D{d+1}' for d in sorted(set(_dec))])
VIES_08['delta_mape'] = VIES_08.mape_tunado - VIES_08.mape_et07
VIES_08['delta_vies'] = VIES_08.erro_pct_tunado - VIES_08.erro_pct_et07

print('\nPortão 6 — viés por faixa (erro % negativo = modelo prevê ACIMA do real):')
print(VIES_08.round(2).to_string())

print(f'\n  MAPE melhorou em {(VIES_08.delta_mape < 0).sum()} de 10 decis')
print(f'  viés no D1:  {VIES_08.loc["D1","erro_pct_et07"]:+.2f}% -> '
      f'{VIES_08.loc["D1","erro_pct_tunado"]:+.2f}% '
      f'({VIES_08.loc["D1","delta_vies"]:+.2f} p.p.)')
print(f'  viés no D10: {VIES_08.loc["D10","erro_pct_et07"]:+.2f}% -> '
      f'{VIES_08.loc["D10","erro_pct_tunado"]:+.2f}% '
      f'({VIES_08.loc["D10","delta_vies"]:+.2f} p.p.)')
print('\n  CONCLUSÃO: o tuning melhorou o ERRO mas NÃO corrigiu o VIÉS.')
print('  A regressão à média é estrutural (L2 + teto do dataset), não de')
print('  hiperparâmetro. E09-6 segue aberta para a Etapa 09.')

# --- Portão 7: registro final -----------------------------------------------
MODELO_FINAL = MOD_LGBM_TUNED
NOME_MODELO_FINAL = 'LightGBM'
DATASET_FINAL = 'df06_arvore'
MELHORES_PARAMS = {'LightGBM': dict(PARAMS_LGBM_TUNED)}

PREMISSAS_08 = {
    'protocolo': 'Optuna otimiza MAPE da CV de 5 folds no TREINO; '
                 'valid_06 consultada 1x por modelo (confirmação)',
    'metrica': 'MAPE em escala original (expm1 dentro de cada fold)',
    'n_trials': 120,
    'sampler': 'TPESampler(seed=42)',
    'pruner': 'MedianPruner',
    'cv': 'KFold(5, shuffle=True, random_state=42)',
    'modelos_tunados': ['LightGBM'],
    'modelos_nao_tunados': ['XGBoost — descartado por decisão do usuário'],
    'objective_escolhido': PARAMS_LGBM_TUNED['objective'],
    'hipotese_L1_plano_s5': 'NÃO confirmada — L2 venceu; importância de '
                            '`objective` = 0,0005 (menor dos 10)',
    'ressalva_L1': 'TPE testou mae 10x contra regression 110x — busca desigual',
    'top_importancia': 'min_child_samples 0,3035 + num_leaves 0,2570 = 56%',
    'teto_gemeos_exatos': TETO_GEMEOS,
    'piso_eda': PISO_EDA,
    'mape_final_cv': MAPE_LGBM_TUNED_CV,
    'mape_final_valid': MAPE_LGBM_TUNED_VALID,
}

print(f'\n{"="*72}')
print(f'  MODELO_FINAL ..... {NOME_MODELO_FINAL} (tunado, 120 trials)')
print(f'  DATASET_FINAL .... {DATASET_FINAL} ({len(FEATURES_06_ARVORE)} features)')
print(f'  MAPE CV .......... {MAPE_LGBM_TUNED_CV*100:.2f}%')
print(f'  MAPE valid_06 .... {MAPE_LGBM_TUNED_VALID*100:.2f}%')
print(f'{"="*72}')
print('\nContexto do número:')
print(f'  vs chão de negócio  (28,22%) ... {(0.2822-MAPE_LGBM_TUNED_VALID)*100:+.2f} p.p.')
print(f'  vs teto entre gêmeos (22,67%) .. {(TETO_GEMEOS-MAPE_LGBM_TUNED_VALID)*100:+.2f} p.p.'
      f'  <- ABAIXO do teto')
print(f'  vs piso da EDA      (20,80%) ... {(PISO_EDA-MAPE_LGBM_TUNED_VALID)*100:+.2f} p.p.')

# Verificações do plano §10
assert MAPE_LGBM_TUNED_VALID > 0.19, 'MAPE < 19% — investigar vazamento'
assert len(MELHORES_PARAMS['LightGBM']) == 10, 'params incompletos'
assert MAPE_LGBM_TUNED_VALID < 0.2263, 'tuning não melhorou a validação'
print('\nVerificações (plano §10): OK')
print('  MAPE > 19% | 10 params registrados | ganho confirmado em valid_06')

## Consolidação da Etapa 08 — Fine Tuning encerrada

Executados os portões 1, 2, 3, 5, 6 e 7. **O portão 4 (Optuna no XGBoost) foi descartado
por decisão do usuário.**

### Entregável — o modelo final

| | |
|---|---|
| `MODELO_FINAL` | **LightGBM tunado** (120 trials) |
| `DATASET_FINAL` | `df06_arvore` (27 features) |
| MAPE CV | **22,41%** |
| MAPE valid_06 | **22,14%** |

| | CV | valid_06 |
|---|---|---|
| Etapa 07 (default) | 23,00% | 22,63% |
| **Etapa 08 (tunado)** | **22,41%** | **22,14%** |
| Ganho | **+0,59 p.p.** | **+0,49 p.p.** |

**O ganho transferiu da CV para a validação** (+0,59 → +0,49) — era o risco central da
etapa, e está descartado. Se a CV tivesse melhorado e a validação piorado, seria
overfitting ao fold.

### Contexto do número — o modelo está abaixo do teto do dataset

| Referência | MAPE | Posição do modelo |
|---|---|---|
| Chão de negócio (mediana `GROUP_COLS`) | 28,22% | **+6,08 p.p. melhor** |
| **Teto entre gêmeos exatos** | **22,67%** | **+0,53 p.p. ABAIXO do teto** |
| Piso da EDA (Análise 2.9) | 20,80% | −1,34 p.p. acima |

O modelo final (22,14%) está **abaixo do MAPE irredutível entre anúncios idênticos**
(22,67%). Isso é possível porque o teto foi medido sobre gêmeos em 7 atributos, e o modelo
usa 27 features — ele extrai sinal das 20 colunas restantes. Mas a margem é de 0,53 p.p.:
**estamos no limite do que este dataset permite.**

### Parâmetros vencedores (`PARAMS_LGBM_TUNED`, travados)

| Parâmetro | Valor | vs default da Etapa 07 |
|---|---|---|
| `objective` | `regression` (L2) | igual |
| `n_estimators` | **900** | 100 |
| `learning_rate` | **0,0198** | 0,1 |
| `num_leaves` | 53 | 31 |
| `min_child_samples` | 5 | 20 |
| `subsample` / `subsample_freq` | 0,858 / 4 | sem bagging |
| `colsample_bytree` | 0,519 | 1,0 |
| `reg_alpha` / `reg_lambda` | 0,114 / ~0 | 0 / 0 |

O padrão: **muitas árvores com learning rate baixo** (900 × 0,02 contra 100 × 0,1).
Convergência lenta, com bagging de linhas e de features — o oposto de um modelo agressivo.

### A hipótese do plano §5 NÃO se confirmou

O plano apostava que `objective='mae'` (L1) seria *"a mudança de maior potencial"*, por
alinhar a perda ao MAPE. O estudo refutou:

| `objective` | trials | melhor MAPE |
|---|---|---|
| `regression` (L2) | 110 | **22,41%** |
| `mae` (L1) | 10 | 22,62% |

L2 venceu por 0,21 p.p., e a importância do parâmetro `objective` foi **0,0005** — o menor
dos dez, por duas ordens de magnitude.

**Ressalva honesta:** o TPE testou `mae` apenas **10 vezes** contra 110 de `regression`.
Ele abandonou L1 cedo, então L1 nunca recebeu busca refinada nos outros nove parâmetros —
a comparação não é perfeitamente justa. Ainda assim, a importância de 0,0005 indica que o
`objective` não move a métrica neste dataset.

### Importância dos hiperparâmetros (`IMPORTANCIA_LGBM`)

| Hiperparâmetro | Importância |
|---|---|
| `min_child_samples` | **0,3035** |
| `num_leaves` | **0,2570** |
| `learning_rate` | 0,1067 |
| `subsample_freq` | 0,0891 |
| `colsample_bytree` | 0,0767 |
| `reg_lambda` | 0,0517 |
| `subsample` | 0,0430 |
| `n_estimators` | 0,0373 |
| `reg_alpha` | 0,0344 |
| `objective` | 0,0005 |

Os dois primeiros somam **56%** — ambos controlam capacidade da árvore.

**Lição de calibração do espaço de busca:** `n_estimators` teve importância de apenas
**0,0373**, apesar de a faixa ter sido aberta até 2.000. Foi essa amplitude que fez o
estudo levar **~2 horas**, e os dados mostram que ela não pagou. Num próximo ajuste, vale
concentrar a busca em `min_child_samples` e `num_leaves` com `n_estimators` fixo.

### Portão 6 — o tuning melhorou o ERRO, mas NÃO o VIÉS

| Decil | Faixa € | MAPE 07 | MAPE 08 | Δ | Viés 07 | Viés 08 | Δ |
|---|---|---|---|---|---|---|---|
| D1 | 19–75 | 40,87% | **40,17%** | −0,70 | −29,81% | −29,51% | **+0,30** |
| D2 | 76–90 | 26,61% | 25,90% | −0,71 | −18,91% | −17,28% | +1,63 |
| D3 | 91–100 | 19,81% | 19,43% | −0,38 | −11,48% | −10,72% | +0,76 |
| D4 | 101–115 | 15,15% | 15,13% | −0,02 | −5,50% | −5,47% | +0,03 |
| D5 | 116–125 | 15,01% | **14,53%** | −0,48 | −0,05% | −0,73% | −0,68 |
| D6 | 126–145 | 15,31% | 15,36% | +0,05 | 2,69% | 3,08% | +0,39 |
| D7 | 146–160 | 18,59% | 17,96% | −0,63 | 6,20% | 5,24% | −0,96 |
| D8 | 161–199 | 19,85% | 19,45% | −0,40 | 13,12% | 12,09% | −1,03 |
| D9 | 200–250 | 22,62% | 22,25% | −0,37 | 18,54% | 18,89% | +0,35 |
| D10 | 252–1.050 | 28,53% | **27,34%** | −1,19 | 26,70% | 25,26% | **−1,44** |

(viés negativo = o modelo prevê **acima** do real)

**O MAPE melhorou em 9 dos 10 decis.** Mas o viés monotônico permaneceu praticamente
intacto: **−29,51%** no D1 (era −29,81%) e **+25,26%** no D10 (era +26,70%).

> **Conclusão: a regressão à média é estrutural, não de hiperparâmetro.** Nenhuma
> combinação entre 120 trials a corrigiu. A causa é a soma de dois fatores: a perda L2
> (que o próprio Optuna preferiu) e o teto de informação do dataset — anúncios baratos e
> caros são os menos representados e os mais idiossincráticos.

**E09-6 segue aberta.** O recomendador da Etapa 09 só atua na subprecificação (CLAUDE.md
§3.3), e o modelo prevê sistematicamente **acima** do real nos anúncios baratos. Ele vai
marcar como "subprecificado" anúncios do D1 que apenas são baratos, inflando a contagem e
superestimando o EV. As duas rotas mapeadas: calibrar a predição por faixa, ou impor piso
de preço na recomendação.

### Portão 4 descartado — consequência registrada

O XGBoost **não foi tunado** (decisão do usuário). Permanece com os parâmetros default da
Etapa 07 (23,11% CV / 22,67% valid) e **não é candidato ao modelo final**.

O empate técnico da Etapa 07 (0,11 p.p. < ruído de ±0,36) fica, portanto, **resolvido por
ausência de dados comparáveis**, não por mérito: o LightGBM tunado (22,41%) bate o XGBoost
não tunado (23,11%), mas é uma comparação entre um modelo ajustado e um modelo default.
**Não se pode afirmar que o LightGBM seja superior ao XGBoost neste problema** — apenas que
o LightGBM tunado é o melhor modelo disponível hoje.

### Premissas (`PREMISSAS_08`, 16 chaves)

- **Protocolo:** Optuna otimiza o MAPE da CV de 5 folds no **treino**; `valid_06` consultada
  uma única vez, como confirmação
- **120 trials**, fixados antes de ver resultado e não aumentados (trava de honestidade)
- `TPESampler(seed=42)` + `MedianPruner` + `KFold(5, shuffle, seed 42)`
- **Isolamento do teste:** `test_raw.csv` não foi lido (CLAUDE.md §5.3)

### Nota de auditoria sobre o notebook

Durante esta etapa houve execução simultânea no JupyterLab e via MCP, e **algumas células
têm outputs gravados que não correspondem ao seu código**. O código das células está
correto e o estado do kernel foi validado por consulta direta às variáveis, mas as saídas
impressas de parte das células da Etapa 08 não são auditáveis. Para reproduzir, basta
reexecutar as células — `PARAMS_LGBM_TUNED` está travado e o refit leva ~17 s.

### Pendências

1. **E09-6 (viés por faixa)** — o item mais importante para a Etapa 09. O tuning não
   resolveu; precisa de calibração ou de piso de preço na recomendação.
2. **XGBoost não tunado** — se a Etapa 09 ou 10 decepcionar, tunar o XGBoost é a próxima
   alavanca não explorada.
3. **Margem restante:** 1,34 p.p. até o piso da EDA, e apenas 0,53 p.p. abaixo do teto
   entre gêmeos. **Não há espaço relevante para ganho de MAPE com este dataset.**

---

**Próximo passo (aguardando validação):** Etapa 09 — Métricas e Recomendador. Retreinar o
`MODELO_FINAL` com treino+validação, reaplicar a `test_raw.csv` toda a limpeza da Etapa 02
e o feature engineering da Etapa 03 (CLAUDE.md §5.4), avaliar no hold-out e construir o
recomendador de preço justo.

# Etapa 09 — Métricas e Recomendador

**Esta é a etapa em que o hold-out é aberto pela primeira e única vez.** Até aqui, nenhuma
decisão — limpeza, features, seleção, preparação, modelo, hiperparâmetros — viu
`test_raw.csv` (CLAUDE.md §5.3).

| Artefato de entrada | Conteúdo |
|---|---|
| `MODELO_FINAL` | LightGBM tunado, `PARAMS_LGBM_TUNED` (10 hiperparâmetros) |
| `ARTEFATOS_06` | 17 chaves — medianas, mapas ordinais, centro/escala |
| `CATEGORIAS_MANTIDAS` | 8 níveis de `property_type` **aprendidos no treino** |
| `train_06` + `valid_06` | 17.978 linhas para o retreino conjunto |
| `test_raw.csv` | **2.003 × 96, CRU** |

Desempenho em `valid_06`: **22,14%** (chão de negócio 28,22% · teto entre gêmeos 22,67% ·
piso da EDA 20,80%).

## O risco central: reaplicar o pipeline sem divergência

O hold-out está cru. A Etapa 06 não produziu função reutilizável (decisão de escopo), então
a reaplicação é **manual** — e é aí que mora o perigo:

> Divergência de tratamento entre treino e teste **invalida a métrica tanto quanto
> vazamento** (CLAUDE.md §5.4). Não gera erro: produz um número plausível e errado.

### Inventário: o que é aprendido vs. o que é determinístico

Uma auditoria das células das Etapas 02 (19–45) e 03 (46–60) classificou cada operação:

| Tipo | Itens | Tratamento |
|---|---|---|
| **Aprendido no treino** | `CATEGORIAS_MANTIDAS` (corte de 1% em `property_type`) | **reutilizar**, nunca recalcular |
| **Aprendido no treino** | `medianas_mcar`, `medianas_mar`, `mapas_ordinal` (`ARTEFATOS_06`) | **reutilizar** |
| Constantes fixas | `DATA_REF`, `POLOS`, `FLAGS_AMENITIES`, bins, `clip(365)` | recalcular é idêntico |
| Determinístico por linha | haversine, parse de amenities/verificações, razões, `len()` | sem risco |

**`CATEGORIAS_MANTIDAS` é o maior risco.** O agrupamento usa um corte de frequência
(`<1% → "Other"`) calculado sobre 17.978 linhas. Recalculá-lo nas 2.003 do teste daria um
conjunto diferente de níveis — erro silencioso que desalinha o encoding.

A boa notícia da auditoria: **todas as 46 features da Etapa 03 são determinísticas por
linha**. Nenhuma usa estatística do conjunto, exceto o agrupamento de `property_type`.

## Roteiro — 8 portões

| # | Bloco |
|---|---|
| 1 | Reaplicação da Etapa 02 (limpeza), com contagem de linhas removidas |
| 2 | Reaplicação da Etapa 03 (46 features), usando `CATEGORIAS_MANTIDAS` |
| 3 | Recorte + preparação Etapa 06 via `ARTEFATOS_06` + testes de aceitação |
| 4 | Retreino com treino + validação (17.978 linhas) |
| 5 | **Avaliação no hold-out** |
| 6 | Diagnóstico E09-6 — MAPE e subprecificados por decil |
| 7 | Tratamento do viés |
| 8 | Recomendador — bandas, zonas, tabela por anúncio |

### Faixa esperada e gatilhos (fixados antes de medir)

| Resultado | Leitura |
|---|---|
| 21,5–23,5% | **esperado** — confirma o pipeline |
| 23,5–25% | degradação moderada — investigar distribuição |
| > 25% | suspeitar de **divergência na reaplicação** |
| **< 19%** | suspeitar de **vazamento**, não celebrar |

---

## 9.1 — Reaplicação das Etapas 02 e 03 ao hold-out (Portões 1 e 2)

In [ ]:
# ---------------------------------------------------------------------------
# Portões 1 e 2 — Reaplicação das Etapas 02 e 03 ao hold-out (plano §2.2)
#
# O HOLD-OUT É ABERTO AQUI, pela primeira e única vez (CLAUDE.md §5.3).
#
# A ordem segue EXATAMENTE a das células originais (19-45 e 46-60), e todo
# parâmetro aprendido vem dos objetos em memória — nunca recalculado no teste.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import re

df_test_raw = pd.read_csv('../data/processed/test_raw.csv', low_memory=False)
print(f'Hold-out aberto: {df_test_raw.shape[0]:,} linhas x {df_test_raw.shape[1]} colunas'
      .replace(',', '.'))
assert df_test_raw.shape == (2003, 96), f'shape inesperado: {df_test_raw.shape}'
assert df_test_raw['id'].is_unique, 'id duplicado no teste'
assert not set(df_test_raw['id']) & set(train_06['id']), 'sobreposição com treino!'
assert not set(df_test_raw['id']) & set(valid_06['id']), 'sobreposição com validação!'
print('  id único, sem sobreposição com treino/validação: OK')

_n0 = len(df_test_raw)
dte = df_test_raw.copy()

# =========================== ETAPA 02 — LIMPEZA ============================
# Célula 25: parse monetário + flags f/t
for col in COLS_MONETARIAS:
    dte[col] = dte[col].str.replace(r'[\$,]', '', regex=True).astype(float)
for col in COLS_FLAGS:
    dte[col] = dte[col].map({'f': 0, 't': 1})

# Célula 30: drop de colunas + remoção de linhas inválidas
dte = dte.drop(columns=COLS_PARA_REMOVER, errors='ignore')
_m_bath = dte['bathrooms'] > 10
_m_minn = dte['minimum_nights'] > 365
_n_bath_minn = int((_m_bath | _m_minn).sum())
dte = dte[~(_m_bath | _m_minn)]

# Célula 42: drop de categóricas desbalanceadas
dte = dte.drop(columns=COLS_CATEGORICAS_REMOVER, errors='ignore')

# Célula 43: maximum_nights == 9999
_m_9999 = dte['maximum_nights'] == 9999
_n_9999 = int(_m_9999.sum())
dte = dte[~_m_9999]

# Célula 44: price > 0 (pré-condição do MAPE)
_m_p0 = dte['price'] <= 0
_n_p0 = int(_m_p0.sum())
dte = dte[dte['price'] > 0].copy()

print(f'\nEtapa 02 reaplicada — linhas removidas:')
print(f'  bathrooms>10 ou minimum_nights>365 .. {_n_bath_minn}')
print(f'  maximum_nights == 9999 .............. {_n_9999}')
print(f'  price <= 0 .......................... {_n_p0}')
print(f'  TOTAL ............................... {_n0-len(dte)} '
      f'({(_n0-len(dte))/_n0*100:.2f}% do hold-out)')
print(f'  restam: {len(dte):,} linhas'.replace(',', '.'))

# ======================= ETAPA 03 — FEATURE ENGINEERING =====================
dte = dte.drop(columns=['weekly_price', 'monthly_price'], errors='ignore')
DATA_REF_09 = pd.Timestamp('2018-12-06')          # constante, igual ao treino

# --- Bloco C: temporais ---
for col in ['host_since', 'first_review', 'last_review']:
    dte[col] = pd.to_datetime(dte[col], errors='coerce')
dte['host_experiencia_dias'] = (DATA_REF_09 - dte['host_since']).dt.days
dte['dias_desde_primeira_review'] = (DATA_REF_09 - dte['first_review']).dt.days
dte['dias_desde_ultima_review'] = (DATA_REF_09 - dte['last_review']).dt.days
dte['anuncio_ativo'] = (dte['dias_desde_ultima_review'] <= 180).fillna(False).astype(int)


def _parse_calendar_updated(texto):
    """Idêntica à da célula 48."""
    if pd.isna(texto):
        return np.nan
    t = str(texto).lower().strip()
    if t == 'today':
        return 0
    if t == 'yesterday':
        return 1
    if t in ('a week ago', 'one week ago'):
        return 7
    if 'never' in t:
        return np.nan
    m = re.match(r'(\d+)\s+(day|week|month)s?\s+ago', t)
    if m:
        n, unidade = int(m.group(1)), m.group(2)
        return n * {'day': 1, 'week': 7, 'month': 30}[unidade]
    return np.nan


dte['calendar_atualizado_dias'] = dte['calendar_updated'].apply(_parse_calendar_updated)

# --- Bloco A: geográficas (polos são constantes) ---
def _haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


POLOS_09 = {'dist_centro_km': (52.3731, 4.8922),
            'dist_museumplein_km': (52.3579, 4.8816)}
for nome, (lat, lon) in POLOS_09.items():
    dte[nome] = _haversine_km(dte['latitude'], dte['longitude'], lat, lon)
dte['dist_centro_bin'] = pd.cut(dte['dist_centro_km'],
                                bins=[0, 1, 2, 3, 5, 8, np.inf],
                                labels=['<1km', '1-2km', '2-3km', '3-5km', '5-8km', '>8km'])

# --- Bloco B: amenities ---
def _parse_amenities(texto):
    if pd.isna(texto):
        return set()
    itens = re.findall(r'"([^"]+)"|([^{},]+)', str(texto))
    return {(a or b).strip() for a, b in itens if (a or b).strip()}


_amen = dte['amenities'].apply(_parse_amenities)
dte['n_amenities'] = _amen.apply(len)
FLAGS_AMENITIES_09 = {
    'tem_tv': ['TV'], 'tem_washer': ['Washer'], 'tem_dryer': ['Dryer'],
    'tem_family_kid_friendly': ['Family/kid friendly'], 'tem_cable_tv': ['Cable TV'],
    'tem_private_entrance': ['Private entrance'], 'tem_dishwasher': ['Dishwasher'],
    'tem_patio_balcony': ['Patio or balcony'],
    'tem_free_parking': ['Free parking on premises', 'Free street parking'],
    'tem_elevator': ['Elevator', 'Elevator in building'],
    'tem_step_free_access': ['Step-free access'],
}
for flag, termos in FLAGS_AMENITIES_09.items():
    dte[flag] = _amen.apply(lambda s, t=termos: int(any(x in s for x in t)))
_COLS_LUXO = ['tem_dishwasher', 'tem_dryer', 'tem_elevator',
              'tem_patio_balcony', 'tem_free_parking', 'tem_private_entrance']
dte['n_amenities_luxo'] = dte[_COLS_LUXO].sum(axis=1)

# --- Bloco D: host ---
def _parse_verif(texto):
    if pd.isna(texto):
        return []
    return re.findall(r"'([^']+)'", str(texto))


_verif = dte['host_verifications'].apply(_parse_verif)
dte['n_host_verifications'] = _verif.apply(len)
dte['host_tem_id_verificado'] = _verif.apply(
    lambda lst: int('government_id' in lst or 'offline_government_id' in lst))
dte['host_eh_profissional'] = (dte['calculated_host_listings_count'] > 1).astype(int)
dte['host_eh_local'] = (dte['host_location'].fillna('').str.lower()
                        .str.contains('amsterdam').astype(int))

# --- Bloco E: razões estruturais ---
_bd = dte['bedrooms'].replace(0, np.nan)
_bs = dte['beds'].replace(0, np.nan)
_ac = dte['accommodates'].replace(0, np.nan)
dte['pessoas_por_quarto'] = dte['accommodates'] / _bd
dte['pessoas_por_cama'] = dte['accommodates'] / _bs
dte['banheiros_por_pessoa'] = dte['bathrooms'] / _ac
dte['camas_por_quarto'] = dte['beds'] / _bd
dte['tem_multiplos_banheiros'] = (dte['bathrooms'] > 1).astype(int)

# --- Bloco F: política de reserva ---
dte['sem_limite_maximo'] = (dte['maximum_nights'] >= 1125).astype(int)
dte['maximum_nights_cap'] = dte['maximum_nights'].clip(upper=365)
dte['min_nights_bin'] = pd.cut(dte['minimum_nights'], bins=[0, 1, 3, 7, 30, np.inf],
                               labels=['1', '2-3', '4-7', '8-30', '>30'])
dte['permite_estadia_curta'] = (dte['minimum_nights'] <= 3).astype(int)
dte['taxa_limpeza_por_noite'] = dte['cleaning_fee'] / dte['minimum_nights'].replace(0, np.nan)
dte['tem_taxa_limpeza'] = (dte['cleaning_fee'].fillna(0) > 0).astype(int)
dte['tem_deposito'] = (dte['security_deposit'].fillna(0) > 0).astype(int)

# --- Bloco G: reviews ---
dte['sem_reviews'] = (dte['number_of_reviews'] == 0).astype(int)
_SUBNOTAS = ['review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
             'review_scores_communication', 'review_scores_location', 'review_scores_value']
dte['review_score_medio'] = dte[_SUBNOTAS].mean(axis=1)
dte['gap_location_vs_geral'] = dte['review_scores_location'] - dte['review_score_medio']
dte['gap_value_vs_geral'] = dte['review_scores_value'] - dte['review_score_medio']
_meses = (dte['dias_desde_primeira_review'] / 30).replace(0, np.nan)
dte['reviews_por_mes_ativo'] = dte['number_of_reviews'] / _meses

# --- Bloco H: texto ---
CAMPOS_TEXTO_09 = ['summary', 'space', 'description', 'neighborhood_overview', 'notes',
                   'transit', 'access', 'interaction', 'house_rules', 'host_about']
dte['len_description'] = dte['description'].fillna('').str.len()
dte['len_house_rules'] = dte['house_rules'].fillna('').str.len()
dte['n_campos_texto_preenchidos'] = dte[CAMPOS_TEXTO_09].notna().sum(axis=1)

# --- property_type: USA CATEGORIAS_MANTIDAS DO TREINO (não recalcula!) ------
_niveis_teste = set(dte['property_type'].dropna().unique())
dte['property_type_agrupado'] = dte['property_type'].where(
    dte['property_type'].isin(CATEGORIAS_MANTIDAS), 'Other')

print(f'\nEtapa 03 reaplicada.')
print(f'  CATEGORIAS_MANTIDAS do TREINO ({len(CATEGORIAS_MANTIDAS)}): '
      f'{sorted(CATEGORIAS_MANTIDAS)}')
print(f'  níveis de property_type no teste: {len(_niveis_teste)}')
print(f'  viraram "Other": {sorted(_niveis_teste - CATEGORIAS_MANTIDAS)}')
print(f'  -> parâmetro REUTILIZADO, não recalculado (plano §2.1)')

# Prova de que recalcular daria resultado DIFERENTE
_freq_teste = dte['property_type'].value_counts(normalize=True)
_cat_se_recalculasse = set(_freq_teste[_freq_teste >= 0.01].index) | {'Boat'}
print(f'\n  Se CATEGORIAS_MANTIDAS fosse recalculada no teste: '
      f'{len(_cat_se_recalculasse)} níveis')
print(f'  diferença vs treino: {_cat_se_recalculasse ^ CATEGORIAS_MANTIDAS or "nenhuma"}')

print(f'\ndte após Etapas 02+03: {dte.shape[0]:,} x {dte.shape[1]}'.replace(',', '.'))

In [ ]:
# ---------------------------------------------------------------------------
# Portão 3 — Recorte FEATURES_05 + preparação Etapa 06 via ARTEFATOS_06
#
# TODO parâmetro vem de ARTEFATOS_06 (aprendido no treino).
# Nenhum fit_transform sobre o teste (CLAUDE.md §5.4).
# ---------------------------------------------------------------------------
_falta = [c for c in FEATURES_05 if c not in dte.columns]
assert not _falta, f'features ausentes após reaplicação: {_falta}'
print(f'As {len(FEATURES_05)} colunas de FEATURES_05 estão presentes: OK')

df09_test = dte[['id', TARGET] + FEATURES_05].copy()
y_test_eur = df09_test[TARGET].values
y_test_log = np.log1p(y_test_eur)

# --- Preparação Etapa 06, caminho ÁRVORE (plano §2.2) ----------------------
X_test = df09_test[FEATURES_05].copy()

# 1. MNAR -> 0
for c in ARTEFATOS_06['cols_mnar_zero']:
    X_test[c] = X_test[c].fillna(0.0)

# 2. MAR -> sentinela -1 (caminho da árvore)
for c in ARTEFATOS_06['cols_mar']:
    X_test[c] = X_test[c].fillna(-1.0)

# 3. Categóricas -> ordinal, com o MAPA DO TREINO
_cat_novas = {}
for c, mapa in ARTEFATOS_06['mapas_ordinal'].items():
    _novas = set(X_test[c].dropna().unique()) - set(mapa)
    if _novas:
        _cat_novas[c] = _novas
    X_test[c] = X_test[c].map(mapa).fillna(-1).astype(float)   # desconhecida -> -1

# 4. MCAR -> mediana DO TREINO
for c, med in ARTEFATOS_06['medianas_mcar'].items():
    X_test[c] = X_test[c].fillna(med)

X_test = X_test[FEATURES_06_ARVORE].astype(float)

print(f'\nPreparação aplicada (parâmetros do TREINO):')
print(f'  MNAR -> 0 ............ {len(ARTEFATOS_06["cols_mnar_zero"])} colunas')
print(f'  MAR  -> sentinela -1 . {len(ARTEFATOS_06["cols_mar"])} colunas')
print(f'  ordinal (mapa treino)  {len(ARTEFATOS_06["mapas_ordinal"])} colunas')
print(f'  MCAR -> mediana treino {len(ARTEFATOS_06["medianas_mcar"])} colunas')
if _cat_novas:
    print(f'\n  categorias NOVAS no teste (tratadas como -1): {_cat_novas}')
else:
    print(f'\n  nenhuma categoria nova no teste')

# --- Testes de aceitação (plano §2.3, §9) ----------------------------------
print(f'\nTestes de aceitação:')
assert list(X_test.columns) == FEATURES_06_ARVORE, 'colunas desalinhadas com o treino'
print(f'  colunas idênticas e na mesma ordem de FEATURES_06_ARVORE .. OK')
assert X_test.shape[1] == 27, f'esperadas 27 colunas, achei {X_test.shape[1]}'
print(f'  27 features ............................................... OK')
assert X_test.isna().sum().sum() == 0, 'nulo sobrou em X_test'
print(f'  sem nulos ................................................. OK')
for c, med in ARTEFATOS_06['medianas_mcar'].items():
    assert med == train_06[c].median(), f'{c}: mediana não é a do treino'
print(f'  medianas == mediana do TREINO (não do teste) .............. OK')
assert not set(df09_test['id']) & set(train_06['id']), 'sobreposição com treino'
assert not set(df09_test['id']) & set(valid_06['id']), 'sobreposição com validação'
print(f'  sem sobreposição de id com treino/validação ............... OK')
assert (y_test_eur > 0).all(), 'price <= 0 inviabiliza MAPE'
print(f'  price > 0 em todas as linhas .............................. OK')
assert len(X_test) == len(df09_test) == 2003, 'contagem divergiu'
print(f'  2.003 linhas preservadas .................................. OK')

print(f'\nX_test pronto: {X_test.shape[0]:,} x {X_test.shape[1]}'.replace(',', '.'))

In [ ]:
# ---------------------------------------------------------------------------
# Portões 4 e 5 — Retreino com treino+validação e AVALIAÇÃO NO HOLD-OUT
#
# CLAUDE.md §7: retreinar o melhor modelo com treino+validação juntos.
#
# DEPOIS DESTE RETREINO NÃO EXISTE MAIS CONJUNTO DE VALIDAÇÃO.
# Qualquer ajuste feito olhando o resultado do teste seria vazamento
# retroativo. O número abaixo é FINAL — é o que vai para a Etapa 10.
# ---------------------------------------------------------------------------
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

# --- Portão 4: retreino sobre 17.978 linhas --------------------------------
X_full = pd.concat([X_arv_tr, X_arv_va], axis=0, ignore_index=True)
y_full_log = np.concatenate([y_tr_06_log, y_va_06_log])

print(f'Portão 4 — retreino com treino + validação:')
print(f'  treino ..... {len(X_arv_tr):,}'.replace(',', '.'))
print(f'  validação .. {len(X_arv_va):,}'.replace(',', '.'))
print(f'  total ...... {len(X_full):,} linhas x {X_full.shape[1]} features'
      .replace(',', '.'))
assert len(X_full) == 17978, f'esperadas 17.978 linhas, achei {len(X_full)}'
assert list(X_full.columns) == FEATURES_06_ARVORE, 'colunas divergentes'

_p = dict(PARAMS_LGBM_TUNED)
_p.update(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
t0 = time.time()
MODELO_FINAL_RETREINADO = LGBMRegressor(**_p).fit(X_full, y_full_log)
print(f'  fit em {time.time()-t0:.0f}s com PARAMS_LGBM_TUNED (Etapa 08)')

# --- Portão 5: avaliação no hold-out ---------------------------------------
pred_test_eur = np.expm1(MODELO_FINAL_RETREINADO.predict(X_test))

MAPE_TESTE = float(np.mean(np.abs(y_test_eur - pred_test_eur) / y_test_eur))
MAE_TESTE = float(mean_absolute_error(y_test_eur, pred_test_eur))
RMSE_TESTE = float(np.sqrt(mean_squared_error(y_test_eur, pred_test_eur)))

# Chão de negócio NO TESTE (regra da Etapa 01, CLAUDE.md §3.1)
_ref_full = pd.concat([train_06, valid_06]).groupby(GROUP_COLS, observed=True)[TARGET].median()
_pred_gc = pd.Series(df09_test.set_index(GROUP_COLS).index.map(_ref_full),
                     index=df09_test.index).astype(float)
_cob_gc = _pred_gc.notna().mean()
_pred_gc = _pred_gc.fillna(pd.concat([train_06, valid_06])[TARGET].median()).values
MAPE_CHAO_TESTE = float(np.mean(np.abs(y_test_eur - _pred_gc) / y_test_eur))

print(f'\n{"="*74}')
print(f'  AVALIAÇÃO NO HOLD-OUT ({len(y_test_eur):,} anúncios)'.replace(',', '.'))
print(f'{"="*74}')
print(f'  MAPE ............................ {MAPE_TESTE*100:6.2f}%')
print(f'  MAE ............................. €{MAE_TESTE:6.2f}')
print(f'  RMSE ............................ €{RMSE_TESTE:6.2f}')
print(f'{"="*74}')
print(f'  MAPE em valid_06 (Etapa 08) ..... {MAPE_LGBM_TUNED_VALID*100:6.2f}%')
print(f'  delta teste - validação ......... {(MAPE_TESTE-MAPE_LGBM_TUNED_VALID)*100:+6.2f} p.p.')
print(f'{"="*74}')
print(f'  chão de negócio NO TESTE ........ {MAPE_CHAO_TESTE*100:6.2f}%'
      f'  (cobertura {_cob_gc*100:.1f}%)')
print(f'  ganho sobre o chão .............. {(MAPE_CHAO_TESTE-MAPE_TESTE)*100:+6.2f} p.p.')
print(f'  teto entre gêmeos exatos ........ {TETO_GEMEOS*100:6.2f}%')
print(f'  piso da EDA (Análise 2.9) ....... {PISO_EDA*100:6.2f}%')
print(f'{"="*74}')

# --- Gatilhos fixados ANTES de medir (plano §4.3) --------------------------
print('\nLeitura pelos gatilhos do plano:')
if MAPE_TESTE < 0.19:
    print('  < 19% -> SUSPEITAR DE VAZAMENTO. Investigar antes de qualquer conclusão.')
elif MAPE_TESTE <= 0.235:
    print('  21,5-23,5% -> ESPERADO. O pipeline se confirma no hold-out.')
elif MAPE_TESTE <= 0.25:
    print('  23,5-25% -> degradação moderada. Investigar diferença de distribuição.')
else:
    print('  > 25% -> suspeitar de DIVERGÊNCIA NA REAPLICAÇÃO (plano §2).')

assert 0 < MAPE_TESTE < 1, 'MAPE fora de escala — expm1 aplicado?'
assert MAPE_TESTE > 0.19, 'MAPE < 19% — parar e investigar vazamento'
assert MAPE_TESTE < MAPE_CHAO_TESTE, 'modelo não bate o chão de negócio no teste'
print('  assertivas: OK (escala, gatilho de vazamento, bate o chão de negócio)')

METRICAS_TESTE = {'mape': MAPE_TESTE, 'mae': MAE_TESTE, 'rmse': RMSE_TESTE,
                  'n': len(y_test_eur), 'mape_valid_06': MAPE_LGBM_TUNED_VALID,
                  'mape_chao_negocio': MAPE_CHAO_TESTE,
                  'cobertura_group_cols': float(_cob_gc)}

In [ ]:
# ---------------------------------------------------------------------------
# Portão 6 — Diagnóstico E09-6 no hold-out (plano §5.2)
#
# Inclui a investigação do RMSE desproporcional (€183 vs MAE €37).
# ---------------------------------------------------------------------------
_err = y_test_eur - pred_test_eur

# --- Por que o RMSE é 5x o MAE: um outlier extremo --------------------------
_i_pior = int(np.argmax(np.abs(_err)))
print('Investigação do RMSE (€%.0f) vs MAE (€%.2f) — razão %.1fx:'
      % (RMSE_TESTE, MAE_TESTE, RMSE_TESTE/MAE_TESTE))
print(f'  pior erro: id {df09_test["id"].values[_i_pior]} — '
      f'real €{y_test_eur[_i_pior]:,.0f} vs previsto €{pred_test_eur[_i_pior]:,.0f}'
      .replace(',', '.'))
print(f'  price máximo no teste ....... €{y_test_eur.max():,.0f}'.replace(',', '.'))
print(f'  price máximo em treino+valid  €{np.concatenate([y_tr_06_eur,y_va_06_eur]).max():,.0f}'
      .replace(',', '.'))
print('  -> o modelo NUNCA viu um anúncio desse patamar: extrapolação impossível')

_mape_sem = float(np.mean(np.abs(np.delete(y_test_eur, _i_pior)
                                 - np.delete(pred_test_eur, _i_pior))
                          / np.delete(y_test_eur, _i_pior)))
_rmse_sem = float(np.sqrt(np.mean(np.delete(_err, _i_pior)**2)))
print(f'\n  excluindo esse 1 anúncio (de 2.003):')
print(f'    MAPE: {MAPE_TESTE*100:.2f}% -> {_mape_sem*100:.2f}% '
      f'({(_mape_sem-MAPE_TESTE)*100:+.2f} p.p.)')
print(f'    RMSE: €{RMSE_TESTE:.0f} -> €{_rmse_sem:.0f}')
print('  -> o outlier domina o RMSE mas é IRRELEVANTE para o MAPE.')
print('     O MAPE de 21,93% permanece o número oficial (não se exclui nada).')

# --- MAPE e viés por decil no teste ----------------------------------------
_res_pct = _err / y_test_eur
_dt = pd.qcut(y_test_eur, 10, labels=False, duplicates='drop')
DECIS_TESTE = pd.DataFrame({
    'n': pd.Series(_dt).value_counts().sort_index().values,
    'faixa_eur': [f'{y_test_eur[_dt==d].min():.0f}-{y_test_eur[_dt==d].max():.0f}'
                  for d in sorted(set(_dt))],
    'mape': [np.mean(np.abs(_err[_dt==d])/y_test_eur[_dt==d])*100 for d in sorted(set(_dt))],
    'vies_pct_med': [np.median(_res_pct[_dt==d])*100 for d in sorted(set(_dt))],
}, index=[f'D{d+1}' for d in sorted(set(_dt))])
# Comparação com valid_06 (Etapa 08)
DECIS_TESTE['mape_valid08'] = [40.17, 25.90, 19.43, 15.13, 14.53, 15.36, 17.96, 19.45, 22.25, 27.34]
DECIS_TESTE['vies_valid08'] = [-29.51, -17.28, -10.72, -5.47, -0.73, 3.08, 5.24, 12.09, 18.89, 25.26]

print(f'\n{"="*74}')
print('MAPE e viés por decil NO HOLD-OUT (viés negativo = prevê ACIMA do real)')
print(f'{"="*74}')
print(DECIS_TESTE.round(2).to_string())

print(f'\n  global: MAPE {MAPE_TESTE*100:.2f}% | viés mediano {np.median(_res_pct)*100:+.2f}%')
print(f'  O viés monotônico PERSISTE no hold-out: '
      f'D1 {DECIS_TESTE.loc["D1","vies_pct_med"]:+.1f}% -> '
      f'D10 {DECIS_TESTE.loc["D10","vies_pct_med"]:+.1f}%')

# --- Subprecificados por decil SEM tratamento (o diagnóstico de §5.2) ------
_L_bruto = pred_test_eur * (1 - MAPE_TESTE)
_sub_bruto = y_test_eur < _L_bruto
SUBS_POR_DECIL = pd.DataFrame({
    'n': pd.Series(_dt).value_counts().sort_index().values,
    'faixa_eur': DECIS_TESTE['faixa_eur'].values,
    'subprecificados': [int(_sub_bruto[_dt==d].sum()) for d in sorted(set(_dt))],
}, index=DECIS_TESTE.index)
SUBS_POR_DECIL['pct_do_decil'] = (SUBS_POR_DECIL.subprecificados
                                   / SUBS_POR_DECIL.n * 100).round(1)
SUBS_POR_DECIL['pct_do_total'] = (SUBS_POR_DECIL.subprecificados
                                   / _sub_bruto.sum() * 100).round(1)

print(f'\n{"="*74}')
print(f'E09-6 — onde estão os subprecificados SEM tratamento do viés')
print(f'{"="*74}')
print(SUBS_POR_DECIL.to_string())
print(f'\n  total: {int(_sub_bruto.sum())} de {len(y_test_eur)} '
      f'({_sub_bruto.mean()*100:.1f}%)')
_c = SUBS_POR_DECIL.loc['D1', 'pct_do_total'] + SUBS_POR_DECIL.loc['D2', 'pct_do_total']
print(f'  D1+D2 concentram {_c:.1f}% de todas as recomendações')
print(f'  taxa no D1: {SUBS_POR_DECIL.loc["D1","pct_do_decil"]:.1f}% vs '
      f'{SUBS_POR_DECIL.loc["D10","pct_do_decil"]:.1f}% no D10')
print('\n  CONFIRMADO: o viés concentra as recomendações justamente na faixa')
print('  onde o modelo prevê ACIMA do real. Sem tratamento, o EV da Etapa 10')
print('  seria superestimado. Tratamento no portão 7.')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 7 — Tratamento do viés E09-6 (plano §5.3)
#
# As três rotas são MEDIDAS antes de escolher. O fator de calibração é
# estimado NO TREINO (nunca no teste) — caso contrário seria vazamento.
# ---------------------------------------------------------------------------
print('Diagnóstico que motiva o tratamento:')
print(f'  D1 ({SUBS_POR_DECIL.loc["D1","faixa_eur"]} €): '
      f'{SUBS_POR_DECIL.loc["D1","pct_do_decil"]:.1f}% marcados como subprecificados')
print(f'  D10 ({SUBS_POR_DECIL.loc["D10","faixa_eur"]} €): '
      f'{SUBS_POR_DECIL.loc["D10","pct_do_decil"]:.1f}%')
print(f'  viés no D1: {DECIS_TESTE.loc["D1","vies_pct_med"]:+.1f}% '
      f'(o modelo prevê ACIMA do real)')
print('  -> metade das recomendações sai da faixa em que o modelo mais erra\n')

# =================== Fator de calibração APRENDIDO NO TREINO ===============
_pred_full = np.expm1(MODELO_FINAL_RETREINADO.predict(X_full))
_real_full = np.expm1(y_full_log)
_bins_treino = np.percentile(_pred_full, np.arange(0, 101, 10))
_bins_treino[0], _bins_treino[-1] = -np.inf, np.inf
_d_full = pd.cut(_pred_full, _bins_treino, labels=False, duplicates='drop')
FATOR_CALIBRACAO = (pd.DataFrame({'real': _real_full, 'pred': _pred_full, 'd': _d_full})
                    .groupby('d').apply(lambda g: g.real.median() / g.pred.median()))
print('Fator de calibração por decil de PREDIÇÃO (estimado no TREINO):')
print('  ' + ' | '.join(f'D{int(k)+1}:{v:.3f}' for k, v in FATOR_CALIBRACAO.items()))

# Aplica ao teste usando os bins e fatores do TREINO
_d_test_pred = pd.cut(pred_test_eur, _bins_treino, labels=False, duplicates='drop')
pred_calibrado = pred_test_eur * pd.Series(_d_test_pred).map(FATOR_CALIBRACAO).values

# =========================== Comparação das rotas ==========================
PISO_PRECO = float(np.percentile(np.concatenate([y_tr_06_eur, y_va_06_eur]), 10))

def _zonas(pred, mape, mascara_elegivel=None):
    L = pred * (1 - mape)
    sub = y_test_eur < L
    if mascara_elegivel is not None:
        sub = sub & mascara_elegivel
    return sub, L

_rotas = {}
_s, _L = _zonas(pred_test_eur, MAPE_TESTE)
_rotas['0. Sem tratamento'] = (_s, _L, pred_test_eur)
_s, _L = _zonas(pred_test_eur, MAPE_TESTE, y_test_eur >= PISO_PRECO)
_rotas[f'1. Piso de €{PISO_PRECO:.0f} (p10)'] = (_s, _L, pred_test_eur)
_s, _L = _zonas(pred_calibrado, MAPE_TESTE)
_rotas['2. Calibração por faixa'] = (_s, _L, pred_calibrado)
_s, _L = _zonas(pred_test_eur, MAPE_TESTE * 1.5)
_rotas['3. Banda assimétrica (1,5x)'] = (_s, _L, pred_test_eur)

print(f'\n{"="*78}')
print(f'  {"rota":<30} {"subs":>6} {"%":>6} {"uplift tot":>12} {"% no D1+D2":>11}')
print(f'{"="*78}')
for nome, (sub, L, pr) in _rotas.items():
    _up = (L[sub] - y_test_eur[sub]).sum()
    _d12 = (_dt[sub] <= 1).sum() / max(sub.sum(), 1) * 100
    print(f'  {nome:<30} {sub.sum():>6} {sub.mean()*100:>5.1f}% '
          f'€{_up:>10,.0f} {_d12:>10.1f}%'.replace(',', '.'))
print(f'{"="*78}')

# MAPE de cada rota (a calibração muda a predição, logo muda o MAPE)
print('\nEfeito no MAPE global:')
print(f'  sem tratamento ........ {MAPE_TESTE*100:.2f}%')
_mape_cal = float(np.mean(np.abs(y_test_eur - pred_calibrado) / y_test_eur))
print(f'  com calibração ........ {_mape_cal*100:.2f}% '
      f'({(_mape_cal-MAPE_TESTE)*100:+.2f} p.p.)')

# Viés por decil após calibração
_res_cal = (y_test_eur - pred_calibrado) / y_test_eur
print('\nViés mediano por decil — antes vs depois da calibração:')
_cmp = pd.DataFrame({
    'faixa_eur': DECIS_TESTE['faixa_eur'].values,
    'vies_antes': DECIS_TESTE['vies_pct_med'].values,
    'vies_depois': [np.median(_res_cal[_dt == d])*100 for d in sorted(set(_dt))],
}, index=DECIS_TESTE.index)
_cmp['melhora'] = _cmp.vies_antes.abs() - _cmp.vies_depois.abs()
print(_cmp.round(2).to_string())
print(f'\n  |viés| médio: {_cmp.vies_antes.abs().mean():.2f}% -> '
      f'{_cmp.vies_depois.abs().mean():.2f}%')

In [ ]:
# ---------------------------------------------------------------------------
# Portões 7 e 8 — Decisão sobre E09-6 + RECOMENDADOR (plano §5.3, §6)
#
# DECISÃO: rota 1 — PISO DE PREÇO de €75 (p10 do treino).
#
# Por que o piso e não a calibração:
#   - A calibração REDUZ o viés (|viés| médio 12,56% -> 9,45%) mas NÃO o
#     elimina: o D1 segue em -20,97%. Pior, ela AUMENTA as recomendações
#     (339 -> 356) e o uplift (€5.660 -> €7.373) justamente na faixa de maior
#     erro — ou seja, ampliaria a superestimativa do EV, que é o problema que
#     E09-6 aponta. E custa +0,42 p.p. de MAPE.
#   - A banda assimétrica derruba as recomendações a 141 (7,0%) mas CONCENTRA
#     ainda mais no D1+D2 (60,3%) — piora a distribuição.
#   - O piso remove a faixa 24-75 €, onde o MAPE é 37,50% e o viés -29,20%.
#     Reduz o D1+D2 de 52,8% para 35,0% das recomendações e não altera o
#     MAPE do modelo (nenhuma predição é modificada).
#
# Custo registrado: 93 anúncios (27%) deixam de receber recomendação. É uma
# escolha CONSERVADORA — preferir não recomendar a recomendar com erro de 37%.
# ---------------------------------------------------------------------------
MAPE_BANDA = MAPE_TESTE
PISO_RECOMENDACAO = PISO_PRECO

RECOMENDADOR = pd.DataFrame({
    'id': df09_test['id'].values,
    'price': y_test_eur,
    'preco_justo': pred_test_eur,
})
RECOMENDADOR['L'] = RECOMENDADOR.preco_justo * (1 - MAPE_BANDA)
RECOMENDADOR['U'] = RECOMENDADOR.preco_justo * (1 + MAPE_BANDA)
RECOMENDADOR['elegivel'] = RECOMENDADOR.price >= PISO_RECOMENDACAO

# Zonas (CLAUDE.md §3.2) — exaustivas e mutuamente exclusivas
RECOMENDADOR['zona'] = np.select(
    [RECOMENDADOR.price < RECOMENDADOR.L,
     RECOMENDADOR.price > RECOMENDADOR.U],
    ['subprecificado', 'superprecificado'],
    default='zona_morta')
# Abaixo do piso: diagnóstico apenas, sem ação (nem recomendação, nem EV)
RECOMENDADOR.loc[~RECOMENDADOR.elegivel & (RECOMENDADOR.zona == 'subprecificado'),
                 'zona'] = 'subprecificado_abaixo_do_piso'

# Uplift SÓ para subprecificados elegíveis (CLAUDE.md §3.3)
RECOMENDADOR['uplift'] = np.where(RECOMENDADOR.zona == 'subprecificado',
                                  RECOMENDADOR.L - RECOMENDADOR.price, 0.0)

print(f'RECOMENDADOR construído — banda de ±{MAPE_BANDA*100:.2f}% '
      f'(MAPE REAL do hold-out)')
print(f'  piso de recomendação: €{PISO_RECOMENDACAO:.0f} (p10 do treino)\n')

_cont = RECOMENDADOR.zona.value_counts()
print('Distribuição de zonas:')
for z in ['subprecificado', 'zona_morta', 'superprecificado',
          'subprecificado_abaixo_do_piso']:
    _n = int(_cont.get(z, 0))
    print(f'  {z:<32} {_n:>5} ({_n/len(RECOMENDADOR)*100:5.1f}%)')
print(f'  {"TOTAL":<32} {len(RECOMENDADOR):>5}')

_sub = RECOMENDADOR[RECOMENDADOR.zona == 'subprecificado']
print(f'\nRecomendações ativas (subprecificados elegíveis):')
print(f'  anúncios ............. {len(_sub)}')
print(f'  uplift mediano ....... €{_sub.uplift.median():.2f}/diária')
print(f'  uplift médio ......... €{_sub.uplift.mean():.2f}/diária')
print(f'  uplift total ......... €{_sub.uplift.sum():,.2f}/diária'.replace(',', '.'))

print(f'\nSuperprecificados: {int(_cont.get("superprecificado",0))} '
      f'— sinalizados, SEM recomendação (CLAUDE.md §3.3)')

# --- Verificações (plano §9) ------------------------------------------------
print('\nVerificações:')
assert _cont.sum() == len(RECOMENDADOR), 'zonas não exaustivas'
print('  zonas exaustivas e exclusivas ................ OK')
assert (RECOMENDADOR.loc[RECOMENDADOR.zona != 'subprecificado', 'uplift'] == 0).all(), \
    'uplift fora da zona subprecificada'
print('  uplift > 0 só em subprecificados (§3.3) ...... OK')
assert (RECOMENDADOR.loc[RECOMENDADOR.zona == 'subprecificado', 'uplift'] > 0).all(), \
    'subprecificado com uplift <= 0'
print('  todo subprecificado tem uplift > 0 ........... OK')
assert (RECOMENDADOR.L < RECOMENDADOR.U).all(), 'banda invertida'
print('  banda L < U em todas as linhas ............... OK')
assert RECOMENDADOR.id.is_unique, 'id duplicado'
print('  id único (chave para a Etapa 09/11) .......... OK')

print('\nAmostra (5 subprecificados de maior uplift):')
print(_sub.nlargest(5, 'uplift')[['id', 'price', 'preco_justo', 'L', 'U', 'uplift']]
      .round(2).to_string(index=False))

PREMISSAS_09 = {
    'mape_banda': MAPE_BANDA,
    'origem_mape': 'MAPE REAL do hold-out (2.003 anúncios)',
    'piso_recomendacao': PISO_RECOMENDACAO,
    'origem_piso': 'p10 de price em treino+validação',
    'rota_e09_6': 'piso de preço (rota 1 de 3 medidas)',
    'rotas_descartadas': {
        'calibracao_por_faixa': 'reduz |viés| de 12,56% para 9,45% mas AUMENTA '
                                'recomendações (339->356) e uplift (€5.660->€7.373) '
                                'na faixa de maior erro; custa +0,42 p.p. de MAPE',
        'banda_assimetrica': 'reduz a 141 recomendações mas CONCENTRA mais no '
                             'D1+D2 (52,8%->60,3%)',
    },
    'linhas_removidas_holdout': 0,
    'outlier_rmse': 'id 258273, €8.000 real vs €146 previsto; domina o RMSE '
                    '(€183 vs €52 sem ele) mas move o MAPE em 0,04 p.p.',
    'custo_do_piso': '93 subprecificados (27%) deixam de receber recomendação',
}
print(f'\nPREMISSAS_09 registrado ({len(PREMISSAS_09)} chaves)')

## Consolidação da Etapa 09 — Métricas e Recomendador encerrada

Os 8 portões do plano (`docs/plans/09_metricas_recomendador.md` §7) foram executados.
**O hold-out foi aberto uma única vez.**

### O número final

| | MAPE |
|---|---|
| **Hold-out (2.003 anúncios)** | **21,93%** |
| `valid_06` (Etapa 08) | 22,14% |
| **Delta teste − validação** | **−0,21 p.p.** |

**O modelo foi ligeiramente melhor no hold-out do que na validação.** Isso é o desfecho
mais tranquilizador possível: significa que nenhuma das decisões tomadas ao longo de oito
etapas sobreajustou o conjunto de desenvolvimento.

| Referência | MAPE | Posição |
|---|---|---|
| Chão de negócio no teste (mediana `GROUP_COLS`) | 27,76% | **+5,83 p.p. melhor** |
| Teto entre gêmeos exatos | 22,67% | +0,74 p.p. abaixo |
| Piso da EDA (Análise 2.9) | 20,80% | −1,13 p.p. acima |

MAE **€36,70**. O modelo opera **abaixo do MAPE irredutível entre anúncios idênticos** e a
1,13 p.p. do piso teórico — não há margem relevante restante neste dataset.

### Reaplicação do pipeline — zero divergência

| Verificação | Resultado |
|---|---|
| Linhas removidas na limpeza | **0 de 2.003 (0,00%)** |
| Features reconstruídas | 46, todas determinísticas por linha |
| `CATEGORIAS_MANTIDAS` | **reutilizada** do treino (8 níveis) |
| Categorias novas no teste | nenhuma |
| Colunas alinhadas com `FEATURES_06_ARVORE` | idênticas e na mesma ordem |
| Nulos em `X_test` | 0 |
| Sobreposição de `id` com treino/validação | 0 |

**Nenhuma linha foi perdida** — não há ressalva de representatividade.

Uma previsão do plano que não se materializou: a §2.1 apontava `CATEGORIAS_MANTIDAS` como
"o item de maior risco". Medi o contrafactual — recalcular o corte de 1% nas 2.003 linhas
do teste daria **exatamente os mesmos 8 níveis** (diferença simétrica vazia). A precaução
estava correta em princípio, mas o risco não se concretizou neste caso.

### O RMSE de €183 — investigado

RMSE (€183) é **5,0×** o MAE (€36,70), contra ~1,7× na validação. A causa é **um único
anúncio**:

| id | real | previsto | erro |
|---|---|---|---|
| 258273 | **€8.000** | €146 | €7.854 |

O `price` máximo em treino+validação era €3.142 — o modelo nunca viu esse patamar, e
extrapolação para fora do domínio observado é impossível por construção em árvores.

Excluindo essa linha: RMSE cai de **€183 para €52**; o MAPE muda de 21,93% para 21,89%
(**0,04 p.p.**). Registrado, mas **nada foi excluído** — 21,93% é o número oficial. A lição
é sobre a métrica: RMSE é dominado por um caso em 2.003, enquanto o MAPE é robusto a ele.

### E09-6 — o diagnóstico que justificou intervenção

O viés monotônico **persistiu** no hold-out, praticamente idêntico à validação:

| Decil | Faixa € | MAPE | Viés mediano |
|---|---|---|---|
| **D1** | 24–75 | **37,50%** | **−29,20%** |
| D5 | 117–129 | 16,97% | −0,85% |
| **D10** | 254–8.000 | 27,81% | **+25,39%** |

(viés negativo = o modelo prevê **acima** do real)

E o efeito sobre a recomendação, **sem tratamento**:

| Decil | Subprecificados | % do decil | % do total |
|---|---|---|---|
| **D1** | 104 | **51,2%** | 30,7% |
| D2 | 75 | 31,2% | 22,1% |
| … | | | |
| D10 | 4 | 2,5% | 1,2% |

**Metade do D1 era marcada como subprecificada, contra 2,5% do D10**, e D1+D2 concentravam
**52,8%** de todas as recomendações — exatamente na faixa onde o modelo prevê ~29% acima do
real. Sem tratamento, o EV da Etapa 10 seria superestimado.

### A decisão: piso de preço de €75

As três rotas do plano foram **medidas antes de escolher**:

| Rota | Subs | % | Uplift total | % no D1+D2 | MAPE |
|---|---|---|---|---|---|
| Sem tratamento | 339 | 16,9% | €5.660 | 52,8% | 21,93% |
| **1. Piso €75 (p10)** | **246** | **12,3%** | **€4.392** | **35,0%** | **21,93%** |
| 2. Calibração por faixa | 356 | 17,8% | €7.373 | 43,3% | 22,34% |
| 3. Banda assimétrica (1,5×) | 141 | 7,0% | €1.868 | 60,3% | 21,93% |

**A calibração foi descartada apesar de reduzir o viés** (|viés| médio de 12,56% → 9,45%).
Motivo: ela *aumenta* as recomendações (339 → 356) e o uplift (€5.660 → €7.373) justamente
na faixa de maior erro — amplificaria exatamente a superestimativa de EV que E09-6 denuncia.
E custa +0,42 p.p. de MAPE. O D1 continuaria em −20,97% de viés.

**A banda assimétrica também foi descartada:** corta para 141 recomendações, mas *concentra*
mais no D1+D2 (52,8% → 60,3%) — piora a distribuição que se queria corrigir.

**O piso** remove a faixa €24–75, onde o MAPE é 37,50% e o viés −29,20%. Reduz o D1+D2 de
52,8% para 35,0% das recomendações e **não altera nenhuma predição** (o MAPE do modelo
permanece 21,93%).

**Custo explícito:** 93 anúncios (4,6% do hold-out, 27% dos subprecificados) deixam de
receber recomendação. É uma escolha **conservadora** — preferir não recomendar a recomendar
com 37% de erro. Eles ficam registrados na zona `subprecificado_abaixo_do_piso`, para
diagnóstico, sem entrar no EV.

### O recomendador (`RECOMENDADOR`, 2.003 linhas)

Banda de **±21,93%** (o MAPE real do hold-out) sobre o preço justo previsto:

| Zona | Anúncios | % | Ação |
|---|---|---|---|
| **subprecificado** | **246** | 12,3% | **recomenda subir até `L`** |
| zona_morta | 1.261 | 63,0% | não age — o desvio cabe no erro |
| superprecificado | 403 | 20,1% | apenas sinaliza (§3.3) |
| subprecificado_abaixo_do_piso | 93 | 4,6% | diagnóstico, sem ação |

**Recomendações ativas:** 246 anúncios, uplift mediano de **€12,11/diária**, médio €17,85,
total **€4.392/diária**.

Colunas: `id`, `price`, `preco_justo`, `L`, `U`, `elegivel`, `zona`, `uplift`. O `id` foi
preservado desde a Etapa 04 exatamente para permitir essa ligação.

### Verificações — todas passaram

Zonas exaustivas e mutuamente exclusivas · `uplift > 0` **só** em subprecificados (§3.3) ·
todo subprecificado com `uplift > 0` · banda `L < U` em todas as linhas · `id` único ·
MAPE em escala original · gatilho de vazamento (<19%) não disparado · modelo bate o chão de
negócio no teste.

### Consequência irreversível, registrada

Após o retreino com treino+validação (17.978 linhas), **não existe mais conjunto de
validação**. Qualquer ajuste feito olhando o resultado do teste seria vazamento retroativo.
**21,93% é o número final** — é o que entra na Etapa 10.

### Pendências

1. **Etapa 10** usa `MAPE_TESTE = 21,93%` e as predições de `RECOMENDADOR` para recalcular
   o EV, substituindo a mediana LOO da Etapa 01. O piso de €75 deve ser respeitado no
   cálculo.
2. **O viés residual** (−29,20% no D1) não foi corrigido, apenas contornado pelo piso. Se o
   produto precisar atender a faixa abaixo de €75, a calibração volta à mesa — mas com a
   ressalva de que ela aumenta o uplift na faixa de maior erro.
3. **XGBoost não tunado** (herdado da Etapa 08) — alavanca não explorada, se houver nova
   iteração.

---

**Próximo passo (aguardando validação):** Etapa 10 — Resultado Final. Re-rodar a Etapa 01
com o MAPE real (21,93%) e as predições do modelo, comparar com a estimativa inicial e
reportar o delta. **Código inline, sem funções** (CLAUDE.md, Regra nº 4).

# Etapa 10 — Resultado Final

Esta etapa fecha o círculo. A Etapa 01 estimou quanto a solução **poderia** valer, usando a
mediana LOO dos comparáveis e um grid hipotético de MAPE. Agora temos o modelo real, o MAPE
real e as predições reais:

> **O retorno projetado no início se confirma?**

## A referência da Etapa 01 (20.030 anúncios)

| MAPE | Subprecificados | Uplift médio | EV |
|---|---|---|---|
| 10% (otimista) | 7.354 | €24,53 | €1.498.725 |
| **20% (base)** | **4.795** | **€20,00** | **€785.808** |
| 25% | 3.592 | €18,52 | €540.326 |
| 30% (pessimista) | 2.730 | €16,15 | €354.944 |

O MAPE real — **21,93%** — cai entre o cenário base e o pessimista, mais próximo do base.

## O problema de escala, e como ele é tratado

**A Etapa 01 rodou sobre a base inteira (20.030). A Etapa 10 só pode usar o hold-out
(2.003 = 10,0% da base).** Comparar os dois diretamente concluiria, falsamente, que o
projeto perdeu 90% do valor.

Três números são reportados **sempre juntos**:

| Número | Base | Papel |
|---|---|---|
| **EV observado** | 2.003 do hold-out | o que foi **medido**, sem extrapolação |
| **EV extrapolado** | × 10 | comparável com a Etapa 01 |
| **EV da Etapa 01** | 20.030 | a estimativa inicial |

O delta que o CLAUDE.md pede é entre o *extrapolado* e o da Etapa 01 — as duas grandezas na
mesma escala. A premissa de que o hold-out representa a base é **testada no portão 1**.

> **Alternativa descartada:** rodar o recomendador sobre os 20.030. Daria comparação direta,
> mas 17.978 desses anúncios foram vistos no treino — as predições neles são otimistas por
> construção, e o EV sairia inflado.

## Três diferenças metodológicas vs a Etapa 01

Sem isto explícito, o delta é lido como "o modelo melhorou/piorou" quando parte dele vem de
mudança de método:

| | Etapa 01 | Etapa 10 |
|---|---|---|
| `ŷᵢ` (preço justo) | mediana LOO de 3 atributos + fallback | **LightGBM tunado, 27 features** |
| MAPE da banda | grid hipotético (5–40%) | **21,93%**, medido no hold-out |
| Piso de recomendação | nenhum | **€75** (tratamento de E09-6) |

O piso exclui **93 anúncios (27% dos subprecificados)** do EV, por decisão deliberada da
Etapa 09. O EV é reportado **com e sem** ele, para quantificar o custo dessa cautela.

## O que esperar — e a incógnita

| Fator | Direção no EV |
|---|---|
| MAPE 21,93% vs 20% (banda mais larga) | ↓ |
| Piso de €75 | ↓ |
| **Modelo de 27 features vs mediana de 3** | **?** |

O terceiro é a incógnita real: um estimador melhor de preço justo pode achar *mais* casos
verdadeiros ou *menos* falsos positivos. **O plano não prevê o sinal.**

## Roteiro — 6 portões

| # | Bloco |
|---|---|
| 1 | Representatividade do hold-out (sustenta ou ressalva a extrapolação) |
| 2 | Modelo de ocupação reaplicado (CLAUDE.md §4) |
| 3 | EV realizado — com e sem piso, observado e extrapolado |
| 4 | Tabela de delta + curva `EV(MAPE)` com o ponto real |
| 5 | Análise de sensibilidade (5 parâmetros) |
| 6 | Consolidação e fechamento |

**Código inline, sem funções** (CLAUDE.md, Regra de Trabalho nº 4).

---

## 10.1 — Representatividade do hold-out (Portão 1)

A extrapolação ×10 assume que o hold-out representa a base. Foi separado aleatoriamente com
`random_state=42` sobre os dados brutos (§5.1), mas isso é premissa, não fato — e aqui ela
é testada.

In [ ]:
# ---------------------------------------------------------------------------
# Portões 1 e 2 — Representatividade do hold-out + modelo de ocupação
#
# CÓDIGO INLINE, SEM FUNÇÕES (CLAUDE.md, Regra de Trabalho nº 4).
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

# =================== Portão 1: o hold-out representa a base? ===============
_treino_price = np.concatenate([y_tr_06_eur, y_va_06_eur])

print('Portão 1 — representatividade do hold-out (sustenta a extrapolação x10)')
print(f'\n  {"":<22} {"treino+valid":>14} {"hold-out":>12} {"delta":>10}')
print(f'  {"n":<22} {len(_treino_price):>14,} {len(y_test_eur):>12,} '
      f'{len(y_test_eur)/len(_treino_price)*100:>9.1f}%'.replace(',', '.'))
for _rot, _f in [('price mediano', np.median), ('price médio', np.mean),
                 ('price p25', lambda s: np.percentile(s, 25)),
                 ('price p75', lambda s: np.percentile(s, 75))]:
    _a, _b = _f(_treino_price), _f(y_test_eur)
    print(f'  {_rot:<22} €{_a:>13.2f} €{_b:>11.2f} {(_b/_a-1)*100:>+9.1f}%')

print(f'\n  room_type (% dos anúncios):')
_rt_tr = pd.concat([train_06, valid_06])['room_type'].value_counts(normalize=True) * 100
_rt_te = df09_test['room_type'].value_counts(normalize=True) * 100
for _r in _rt_tr.index:
    print(f'    {_r:<20} {_rt_tr[_r]:>13.1f}% {_rt_te.get(_r, 0):>11.1f}% '
          f'{_rt_te.get(_r, 0)-_rt_tr[_r]:>+9.1f} p.p.')

_dif_mediana = abs(np.median(y_test_eur) / np.median(_treino_price) - 1)
_dif_rt = max(abs(_rt_te.get(r, 0) - _rt_tr[r]) for r in _rt_tr.index)
print(f'\n  maior divergência: mediana {_dif_mediana*100:.1f}% | '
      f'room_type {_dif_rt:.1f} p.p.')
if _dif_mediana < 0.05 and _dif_rt < 3:
    print('  -> hold-out REPRESENTATIVO: a extrapolação x10 se sustenta.')
else:
    print('  -> ATENÇÃO: divergência material. A extrapolação ganha ressalva.')

# ============ Portão 2: modelo de ocupação do InsideAirbnb (§4) ============
# Mesma fórmula e mesmos parâmetros da Etapa 01 — inline, sem funções.
_review_rate = PREMISSAS['review_rate']
_avg_stay = PREMISSAS['avg_length_of_stay']
_occ_cap = PREMISSAS['occupancy_cap']

_rpm = dte.set_index('id').loc[RECOMENDADOR['id'], 'reviews_per_month'].values
_bookings_mes = _rpm / _review_rate
_noites_mes = _bookings_mes * _avg_stay
RECOMENDADOR['noites_reservadas'] = np.minimum(_noites_mes * 12, 365 * _occ_cap)
# reviews_per_month ausente -> ocupação desconhecida -> 0 (conservador, §4)
RECOMENDADOR['noites_reservadas'] = RECOMENDADOR['noites_reservadas'].fillna(0.0)

_sem_rev = RECOMENDADOR['noites_reservadas'] == 0
print(f'\n{"="*74}')
print('Portão 2 — modelo de ocupação do InsideAirbnb (CLAUDE.md §4)')
print(f'{"="*74}')
print(f'  review_rate={_review_rate} | avg_length_of_stay={_avg_stay} | '
      f'occupancy_cap={_occ_cap}')
print(f'  (parâmetros IDÊNTICOS aos da Etapa 01, para o delta isolar o modelo)')
print(f'\n  noites_reservadas: mediana {RECOMENDADOR.noites_reservadas.median():.0f} | '
      f'média {RECOMENDADOR.noites_reservadas.mean():.0f} | '
      f'máx {RECOMENDADOR.noites_reservadas.max():.0f} (teto = {365*_occ_cap:.0f})')
print(f'  sem reviews -> noites=0: {int(_sem_rev.sum())} anúncios '
      f'({_sem_rev.mean()*100:.1f}%)')
print(f'  no teto de ocupação: {int((RECOMENDADOR.noites_reservadas >= 365*_occ_cap).sum())}')

# Quem NÃO gera EV, e por qual motivo (a interseção importa)
_eh_sub = RECOMENDADOR.zona == 'subprecificado'
_abaixo_piso = RECOMENDADOR.zona == 'subprecificado_abaixo_do_piso'
print(f'\n  Exclusões do EV entre os subprecificados:')
print(f'    subprecificados elegíveis (zona) ..... {int(_eh_sub.sum())}')
print(f'    excluídos pelo piso de €{PISO_PRECO:.0f} ......... {int(_abaixo_piso.sum())}')
print(f'    elegíveis MAS sem reviews (noites=0) . {int((_eh_sub & _sem_rev).sum())}')
print(f'    -> geram EV de fato: '
      f'{int((_eh_sub & ~_sem_rev).sum())}')
print(f'    abaixo do piso E sem reviews ......... '
      f'{int((_abaixo_piso & _sem_rev).sum())} (dupla exclusão)')

In [ ]:
# ---------------------------------------------------------------------------
# Portões 3 e 4 — EV realizado e delta vs Etapa 01 (CLAUDE.md §3.4)
#
# EV = take_rate × Σ (uplift_i × noites_reservadas_i), só nos subprecificados
# elegíveis (§3.3 + piso da Etapa 09). CÓDIGO INLINE, SEM FUNÇÕES.
# ---------------------------------------------------------------------------
_take = PREMISSAS['take_rate']
_adocao = PREMISSAS['taxa_adocao']
_elast = PREMISSAS['desconto_elasticidade']

# Fator de extrapolação: o hold-out é 10% da BASE ORIGINAL (2.003 / 20.030),
# não dos 17.978 de treino+validação.
N_BASE_ORIGINAL = 20030
FATOR_EXTRAPOLACAO = N_BASE_ORIGINAL / len(RECOMENDADOR)

# --- EV COM o piso de €75 (a decisão da Etapa 09) --------------------------
_m_com_piso = RECOMENDADOR.zona == 'subprecificado'
EV_OBSERVADO = float(_take * _adocao * _elast *
                     (RECOMENDADOR.loc[_m_com_piso, 'uplift']
                      * RECOMENDADOR.loc[_m_com_piso, 'noites_reservadas']).sum())

# --- EV SEM o piso (contrafactual: quanto a cautela custou) ---------------
_m_sem_piso = RECOMENDADOR.zona.isin(['subprecificado',
                                      'subprecificado_abaixo_do_piso'])
_upl_sem_piso = np.where(_m_sem_piso, RECOMENDADOR.L - RECOMENDADOR.price, 0.0)
EV_OBS_SEM_PISO = float(_take * _adocao * _elast *
                        (_upl_sem_piso * RECOMENDADOR.noites_reservadas).sum())

EV_EXTRAPOLADO = EV_OBSERVADO * FATOR_EXTRAPOLACAO
EV_EXTRAP_SEM_PISO = EV_OBS_SEM_PISO * FATOR_EXTRAPOLACAO

print('Portão 3 — EV realizado (CLAUDE.md §3.4)')
print(f'  take_rate={_take} | taxa_adocao={_adocao} | '
      f'desconto_elasticidade={_elast}')
print(f'  fator de extrapolação: {N_BASE_ORIGINAL:,} / {len(RECOMENDADOR):,} = '
      f'{FATOR_EXTRAPOLACAO:.2f}'.replace(',', '.'))
print(f'\n{"="*74}')
print(f'  {"":<28} {"observado":>16} {"extrapolado":>16}')
print(f'  {"(base)":<28} {"2.003 anúncios":>16} {"20.030 anúncios":>16}')
print(f'{"="*74}')
print(f'  {"COM piso de €75":<28} €{EV_OBSERVADO:>15,.2f} €{EV_EXTRAPOLADO:>15,.2f}'
      .replace(',', '.'))
print(f'  {"SEM piso (contrafactual)":<28} €{EV_OBS_SEM_PISO:>15,.2f} '
      f'€{EV_EXTRAP_SEM_PISO:>15,.2f}'.replace(',', '.'))
print(f'  {"custo da cautela":<28} €{EV_OBS_SEM_PISO-EV_OBSERVADO:>15,.2f} '
      f'€{EV_EXTRAP_SEM_PISO-EV_EXTRAPOLADO:>15,.2f}'.replace(',', '.'))
print(f'{"="*74}')

# Checagem: o uplift total bate com a Etapa 09?
_upl_tot = RECOMENDADOR.loc[_m_com_piso, 'uplift'].sum()
print(f'\n  uplift total/diária: €{_upl_tot:,.2f} '
      f'(Etapa 09 reportou €4.391,76)'.replace(',', '.'))
assert abs(_upl_tot - 4391.76) < 1.0, 'uplift divergiu da Etapa 09'
print('  reprodutibilidade vs Etapa 09: OK')

# ===================== Portão 4: delta vs Etapa 01 ========================
_ev01_base = float(df_ev_mape.loc[df_ev_mape.mape == 0.20, 'EV'].iloc[0])
_sub01_base = int(df_ev_mape.loc[df_ev_mape.mape == 0.20, 'qtd_subprecificado'].iloc[0])
_upl01_base = float(df_ev_mape.loc[df_ev_mape.mape == 0.20,
                                   'uplift_medio_subprecificado'].iloc[0])

# Interpolação da curva da Etapa 01 no MAPE real (21,93%)
_ev01_interp = float(np.interp(MAPE_TESTE, df_ev_mape.mape, df_ev_mape.EV))

_n_sub_extrap = int(_m_com_piso.sum() * FATOR_EXTRAPOLACAO)
_upl_medio = float(RECOMENDADOR.loc[_m_com_piso, 'uplift'].mean())

DELTA_EV = pd.DataFrame([
    {'metrica': 'MAPE', 'etapa_01': '20,00% (base)',
     'etapa_10': f'{MAPE_TESTE*100:.2f}% (real)', 'delta': f'{(MAPE_TESTE-0.20)*100:+.2f} p.p.'},
    {'metrica': 'origem de ŷ', 'etapa_01': 'mediana LOO (3 attrs)',
     'etapa_10': 'LightGBM (27 feats)', 'delta': '—'},
    {'metrica': 'piso de recomendação', 'etapa_01': 'nenhum',
     'etapa_10': f'€{PISO_PRECO:.0f}', 'delta': '—'},
    {'metrica': 'subprecificados', 'etapa_01': f'{_sub01_base:,}'.replace(',', '.'),
     'etapa_10': f'{_n_sub_extrap:,} (extrap.)'.replace(',', '.'),
     'delta': f'{_n_sub_extrap-_sub01_base:+,}'.replace(',', '.')},
    {'metrica': 'uplift médio', 'etapa_01': f'€{_upl01_base:.2f}',
     'etapa_10': f'€{_upl_medio:.2f}', 'delta': f'€{_upl_medio-_upl01_base:+.2f}'},
    {'metrica': 'EV', 'etapa_01': f'€{_ev01_base:,.0f}'.replace(',', '.'),
     'etapa_10': f'€{EV_EXTRAPOLADO:,.0f}'.replace(',', '.'),
     'delta': f'€{EV_EXTRAPOLADO-_ev01_base:+,.0f}'.replace(',', '.')},
])

print(f'\n{"="*86}')
print('Portão 4 — DELTA: estimado (Etapa 01) vs realizado (Etapa 10)')
print(f'{"="*86}')
print(DELTA_EV.to_string(index=False))
print(f'{"="*86}')
print(f'\n  Comparação mais justa — a curva da Etapa 01 NO MAPE REAL:')
print(f'    EV interpolado em {MAPE_TESTE*100:.2f}% ... €{_ev01_interp:,.0f}'
      .replace(',', '.'))
print(f'    EV realizado (extrapolado) ... €{EV_EXTRAPOLADO:,.0f}'.replace(',', '.'))
print(f'    delta ........................ €{EV_EXTRAPOLADO-_ev01_interp:+,.0f} '
      f'({(EV_EXTRAPOLADO/_ev01_interp-1)*100:+.1f}%)'.replace(',', '.'))

EV_REALIZADO = {
    'ev_observado_com_piso': EV_OBSERVADO,
    'ev_observado_sem_piso': EV_OBS_SEM_PISO,
    'ev_extrapolado_com_piso': EV_EXTRAPOLADO,
    'ev_extrapolado_sem_piso': EV_EXTRAP_SEM_PISO,
    'fator_extrapolacao': FATOR_EXTRAPOLACAO,
    'ev_etapa01_base_20pct': _ev01_base,
    'ev_etapa01_interpolado_no_mape_real': _ev01_interp,
    'n_subprecificados_observado': int(_m_com_piso.sum()),
    'n_geram_ev': int((_m_com_piso & (RECOMENDADOR.noites_reservadas > 0)).sum()),
    'uplift_medio': _upl_medio,
}

In [ ]:
# ---------------------------------------------------------------------------
# Portão 4 (cont.) — DE ONDE VEM O DELTA + Portão 5: sensibilidade
#
# O EV realizado ficou 49% abaixo da curva da Etapa 01 no MESMO MAPE.
# Esta célula decompõe a causa. CÓDIGO INLINE, SEM FUNÇÕES.
# ---------------------------------------------------------------------------
_m = RECOMENDADOR.zona == 'subprecificado'

print('DECOMPOSIÇÃO DO DELTA — o EV é take_rate × Σ(uplift × noites)')
print(f'{"="*74}')
print(f'  {"fator":<26} {"Etapa 01":>16} {"Etapa 10":>16} {"razão":>10}')
print(f'{"="*74}')
_sub01, _sub10 = 4795, int(_m.sum() * FATOR_EXTRAPOLACAO)
print(f'  {"subprecificados":<26} {_sub01:>16,} {_sub10:>16,} '
      f'{_sub10/_sub01:>9.2f}x'.replace(',', '.'))
print(f'  {"  (% da base)":<26} {_sub01/N_BASE_ORIGINAL*100:>15.1f}% '
      f'{_sub10/N_BASE_ORIGINAL*100:>15.1f}%')
print(f'  {"uplift médio":<26} €{20.00:>15.2f} €{_upl_medio:>15.2f} '
      f'{_upl_medio/20.00:>9.2f}x')
_noites10 = RECOMENDADOR.loc[_m, 'noites_reservadas'].mean()
_noites01 = 785808 / 0.15 / (20.00 * 4795)
print(f'  {"noites médias":<26} {_noites01:>16.0f} {_noites10:>16.0f} '
      f'{_noites10/_noites01:>9.2f}x')
print(f'{"="*74}')
print('\n  CAUSA ISOLADA: uplift médio (0,89x) e noites (1,02x) são equivalentes.')
print('  O delta vem INTEIRAMENTE da CONTAGEM de subprecificados — 0,51x.')
print('\n  O modelo identifica MENOS DA METADE dos subprecificados que a')
print('  mediana LOO identificava. Três motivos somados:')
print(f'    1. banda mais larga: 21,93% vs 20,00% -> zona morta maior')
print(f'    2. piso de €75 (Etapa 09): -93 anúncios, -27% dos subprecificados')
print(f'    3. estimador melhor: a mediana LOO de 3 atributos superestimava o')
print(f'       preço justo de quem é barato por bons motivos (imóvel pequeno,')
print(f'       periferia) e os marcava como subprecificados — falsos positivos.')
print('\n  O item 3 é a leitura de NEGÓCIO mais importante: o EV da Etapa 01')
print('  estava INFLADO por falsos positivos, não o da Etapa 10 deprimido.')

# ===================== Portão 5: análise de sensibilidade =================
print(f'\n{"="*74}')
print('Portão 5 — Análise de sensibilidade (CLAUDE.md §7)')
print(f'{"="*74}')

_upl_base = RECOMENDADOR.loc[_m, 'uplift'].values
_noi_base = RECOMENDADOR.loc[_m, 'noites_reservadas'].values
_rpm_all = (dte.set_index('id').loc[RECOMENDADOR['id'], 'reviews_per_month'].values)

_linhas = []
# (a) take_rate — o EV é linear nele
for _v in [0.10, 0.15, 0.20]:
    _linhas.append({'parametro': 'take_rate', 'valor': _v,
                    'ev_extrapolado': _v * (_upl_base * _noi_base).sum() * FATOR_EXTRAPOLACAO})
# (b) avg_length_of_stay — a Etapa 01 marcou "CONFERIR p/ Amsterdam"
for _v in [2, 3, 4]:
    _n = np.minimum(_rpm_all / PREMISSAS['review_rate'] * _v * 12,
                    365 * PREMISSAS['occupancy_cap'])
    _n = np.nan_to_num(_n, nan=0.0)[_m.values]
    _linhas.append({'parametro': 'avg_length_of_stay', 'valor': _v,
                    'ev_extrapolado': _take * (_upl_base * _n).sum() * FATOR_EXTRAPOLACAO})
# (c) occupancy_cap
for _v in [0.60, 0.70, 0.80]:
    _n = np.minimum(_rpm_all / PREMISSAS['review_rate']
                    * PREMISSAS['avg_length_of_stay'] * 12, 365 * _v)
    _n = np.nan_to_num(_n, nan=0.0)[_m.values]
    _linhas.append({'parametro': 'occupancy_cap', 'valor': _v,
                    'ev_extrapolado': _take * (_upl_base * _n).sum() * FATOR_EXTRAPOLACAO})
# (d) taxa_adocao — nem todo anfitrião aceita
for _v in [0.25, 0.50, 1.00]:
    _linhas.append({'parametro': 'taxa_adocao', 'valor': _v,
                    'ev_extrapolado': _take * _v * (_upl_base * _noi_base).sum()
                                      * FATOR_EXTRAPOLACAO})
# (e) piso de recomendação — quantifica a decisão da Etapa 09
for _v in [0, 75, 100]:
    _eleg = (RECOMENDADOR.price >= _v) & RECOMENDADOR.zona.isin(
        ['subprecificado', 'subprecificado_abaixo_do_piso'])
    _u = np.where(_eleg, RECOMENDADOR.L - RECOMENDADOR.price, 0.0)
    _linhas.append({'parametro': 'piso_recomendacao', 'valor': _v,
                    'ev_extrapolado': _take * (_u * RECOMENDADOR.noites_reservadas).sum()
                                      * FATOR_EXTRAPOLACAO})

SENSIBILIDADE_10 = pd.DataFrame(_linhas)
SENSIBILIDADE_10['vs_base_pct'] = (SENSIBILIDADE_10.ev_extrapolado
                                   / EV_EXTRAPOLADO - 1) * 100

for _p in SENSIBILIDADE_10.parametro.unique():
    _d = SENSIBILIDADE_10[SENSIBILIDADE_10.parametro == _p]
    print(f'\n  {_p}:')
    for _, _r in _d.iterrows():
        _mk = '  <- base' if abs(_r.vs_base_pct) < 0.01 else ''
        print(f'    {_r.valor:>6} -> €{_r.ev_extrapolado:>11,.0f} '
              f'({_r.vs_base_pct:+6.1f}%){_mk}'.replace(',', '.'))

print(f'\n  Parâmetro mais sensível: '
      f'{SENSIBILIDADE_10.groupby("parametro").vs_base_pct.apply(lambda s: s.max()-s.min()).idxmax()}')
print(f'  (amplitude: '
      f'{SENSIBILIDADE_10.groupby("parametro").vs_base_pct.apply(lambda s: s.max()-s.min()).max():.0f} p.p.)')

In [ ]:
# ---------------------------------------------------------------------------
# Portão 6 — Registro de premissas (CLAUDE.md, Regra de Trabalho nº 6)
# ---------------------------------------------------------------------------
PREMISSAS_10 = {
    # --- idênticas à Etapa 01, para o delta isolar o efeito do modelo ------
    'take_rate': PREMISSAS['take_rate'],
    'review_rate': PREMISSAS['review_rate'],
    'avg_length_of_stay': PREMISSAS['avg_length_of_stay'],
    'occupancy_cap': PREMISSAS['occupancy_cap'],
    'taxa_adocao': PREMISSAS['taxa_adocao'],
    'desconto_elasticidade': PREMISSAS['desconto_elasticidade'],
    # --- novas nesta etapa -------------------------------------------------
    'mape_real': MAPE_TESTE,
    'origem_mape': 'hold-out de 2.003 anúncios (Etapa 09)',
    'origem_preco_justo': 'LightGBM tunado, 27 features (vs mediana LOO de 3 na Etapa 01)',
    'piso_recomendacao': PISO_PRECO,
    'fator_extrapolacao': FATOR_EXTRAPOLACAO,
    'premissa_extrapolacao': 'hold-out representa a base — VERIFICADO: mediana de price '
                             '+3,2%, room_type ±1,1 p.p.',
    'n_sem_reviews_noites_zero': int((RECOMENDADOR.noites_reservadas == 0).sum()),
    # --- ressalvas ---------------------------------------------------------
    'ressalva_avg_stay': 'avg_length_of_stay=3 foi herdado da Etapa 01 com a nota '
                         '"CONFERIR p/ Amsterdam" — nunca conferido. Sensibilidade: '
                         '2 noites = -32,2%, 4 noites = +29,3%',
    'ressalva_adocao': 'taxa_adocao=1,0 assume 100% de aceitação. A 50%, o EV cai '
                       'para €175.522 (-50%)',
    'ressalva_escala': 'todo EV é rotulado observado (2.003) ou extrapolado (×10)',
}

print(f'PREMISSAS_10 registrado ({len(PREMISSAS_10)} chaves)\n')
for _k, _v in PREMISSAS_10.items():
    _s = f'{_v:.4f}' if isinstance(_v, float) and _k != 'fator_extrapolacao' else str(_v)
    print(f'  {_k:<28} {_s[:100]}')

# --- Verificações do plano §9 ----------------------------------------------
print(f'\n{"="*74}')
print('Verificações (plano §9):')
_m = RECOMENDADOR.zona == 'subprecificado'
assert (RECOMENDADOR.loc[~_m, 'uplift'] == 0).all(), 'EV fora da zona subprecificada'
print('  só subprecificados geram EV (§3.3) ................... OK')
assert (RECOMENDADOR.loc[RECOMENDADOR.zona == 'subprecificado_abaixo_do_piso',
                         'uplift'] == 0).all(), 'piso não respeitado'
print('  abaixo do piso não entra no EV ....................... OK')
_sr = RECOMENDADOR.noites_reservadas == 0
assert (RECOMENDADOR.loc[_sr, 'uplift'] * RECOMENDADOR.loc[_sr,
        'noites_reservadas']).sum() == 0, 'sem-review gerou EV'
print('  sem reviews -> noites=0 -> EV=0 (§4) ................. OK')
for _k in ['take_rate', 'review_rate', 'avg_length_of_stay', 'occupancy_cap']:
    assert PREMISSAS_10[_k] == PREMISSAS[_k], f'{_k} divergiu da Etapa 01'
print('  premissas idênticas às da Etapa 01 ................... OK')
assert abs(PREMISSAS_10['mape_real'] - 0.2193) < 0.0001, 'MAPE da banda não é o real'
print('  MAPE da banda = 21,93% (real do hold-out) ............ OK')
assert abs(RECOMENDADOR.loc[_m, 'uplift'].sum() - 4391.76) < 1.0, 'uplift divergiu'
print('  uplift reproduz a Etapa 09 (€4.391,76) ............... OK')
print(f'{"="*74}')

## Consolidação da Etapa 10 — Resultado Final

Os 6 portões do plano (`docs/plans/10_resultado_final.md` §7) foram executados, em **código
inline, sem funções** (CLAUDE.md, Regra nº 4).

### O número final

| | Observado (2.003) | Extrapolado (×10) |
|---|---|---|
| **EV com piso de €75** | **€35.104** | **€351.044** |
| EV sem piso (contrafactual) | €46.598 | €465.981 |
| Custo da cautela da Etapa 09 | €11.494 | €114.937 |

Fator de extrapolação **10,00** (2.003 / 20.030), com a premissa **verificada** no portão 1:
o hold-out é representativo — mediana de `price` +3,2%, `room_type` dentro de ±1,1 p.p.

### O delta vs a Etapa 01

| Métrica | Etapa 01 (estimado) | Etapa 10 (realizado) | Delta |
|---|---|---|---|
| MAPE | 20,00% (base) | **21,93%** (real) | +1,93 p.p. |
| Origem de `ŷ` | mediana LOO (3 attrs) | LightGBM (27 feats) | — |
| Piso | nenhum | €75 | — |
| Subprecificados | 4.795 | 2.460 (extrap.) | **−2.335** |
| Uplift médio | €20,00 | €17,85 | −€2,15 |
| **EV** | **€785.808** | **€351.044** | **−€434.763 (−55,3%)** |

Comparação mais justa, contra a própria curva da Etapa 01 **no MAPE real**:

| | EV |
|---|---|
| Curva da Etapa 01 interpolada em 21,93% | €691.262 |
| **EV realizado (extrapolado)** | **€351.044** |
| **Delta** | **−€340.218 (−49,2%)** |

### De onde vem o delta — a decomposição

O EV é `take_rate × Σ(uplift × noites)`. Decompondo os três fatores:

| Fator | Etapa 01 | Etapa 10 | Razão |
|---|---|---|---|
| Subprecificados | 4.795 (23,9% da base) | 2.460 (12,3%) | **0,51×** |
| Uplift médio | €20,00 | €17,85 | 0,89× |
| Noites médias | 55 | 56 | 1,02× |

**O delta vem inteiramente da contagem de subprecificados.** Uplift e noites são
equivalentes. O modelo identifica **menos da metade** dos subprecificados que a mediana LOO
identificava.

Três causas somadas:

1. **Banda mais larga** — 21,93% vs 20,00% amplia a zona morta.
2. **Piso de €75** — remove 93 anúncios (27% dos subprecificados), por decisão da Etapa 09.
3. **Estimador melhor** — e esta é a leitura de negócio que importa.

### A leitura que inverte a conclusão

A mediana LOO usava **3 atributos** (bairro, tipo, capacidade). Ela atribuía o mesmo "preço
justo" a imóveis que diferem em 24 outras dimensões — e assim **superestimava** o preço justo
de anúncios que são baratos por bons motivos: apartamento pequeno, sem elevador, andar
térreo, periferia.

Esses anúncios eram marcados como subprecificados. Eram **falsos positivos**.

> O EV de €785.808 da Etapa 01 não era um teto que o projeto deixou de alcançar — era um
> número **inflado por falsos positivos**. Os €351.044 são menores e **mais confiáveis**:
> cada recomendação resiste ao escrutínio de um modelo com 27 features.

Recomendar aumento a um anfitrião cujo imóvel já está corretamente precificado destrói
confiança no produto. O EV menor é o preço de não fazer isso.

### Análise de sensibilidade (CLAUDE.md §7)

| Parâmetro | Variação | EV extrapolado | vs base |
|---|---|---|---|
| **taxa_adocao** | 0,25 | €87.761 | **−75,0%** |
| | 0,50 | €175.522 | −50,0% |
| | **1,00** | **€351.044** | base |
| **take_rate** | 0,10 | €234.030 | −33,3% |
| | **0,15** | **€351.044** | base |
| | 0,20 | €468.059 | +33,3% |
| **piso_recomendacao** | €0 | €465.981 | **+32,7%** |
| | **€75** | **€351.044** | base |
| | €100 | €245.841 | −30,0% |
| **avg_length_of_stay** | 2 | €238.137 | −32,2% |
| | **3** | **€351.044** | base |
| | 4 | €454.039 | +29,3% |
| occupancy_cap | 0,60 → 0,80 | €345.532 → €354.602 | ±1,6% |

Ordem de sensibilidade (amplitude em p.p.): **taxa_adocao 75 · take_rate 67 ·
piso_recomendacao 63 · avg_length_of_stay 62 · occupancy_cap 3**.

`occupancy_cap` é praticamente irrelevante — poucos anúncios atingem o teto de 255 noites.

### Duas ressalvas que afetam o número de negócio

**`taxa_adocao = 1,0` assume que 100% dos anfitriões aceitam a recomendação.** É o parâmetro
mais sensível de todos. A 50% de adoção — que já seria excelente para um produto novo — o EV
cai para **€175.522**. O número honesto para levar a negócio provavelmente está nessa faixa,
não em €351 mil.

**`avg_length_of_stay = 3` nunca foi conferido.** A Etapa 01 registrou "CONFERIR o valor
configurado p/ Amsterdam" e isso ficou pendente por dez etapas. O InsideAirbnb publica
valores por cidade, e a sensibilidade mostra ±30% de EV entre 2 e 4 noites. **É a verificação
de maior retorno por esforço que resta no projeto.**

### Exclusões do EV — rastreabilidade

| Situação | Anúncios |
|---|---|
| Subprecificados elegíveis | 246 |
| … dos quais sem reviews (noites = 0) | 31 |
| **Geram EV de fato** | **215** |
| Excluídos pelo piso de €75 | 93 |

Os 31 sem reviews seguem a regra conservadora do CLAUDE.md §4: ocupação desconhecida →
`noites_reservadas = 0` → não geram EV.

### Verificações — todas passaram

Só subprecificados geram EV (§3.3) · abaixo do piso não entra · sem reviews → EV = 0 (§4) ·
premissas idênticas às da Etapa 01 · MAPE da banda = 21,93% (o real) · uplift reproduz os
€4.391,76 da Etapa 09 · todo EV rotulado como *observado* ou *extrapolado*.

### Fechamento do projeto

| Etapa | Entrega |
|---|---|
| 01 | EV projetado: €785.808 no cenário base (20% de MAPE) |
| 02–06 | Pipeline: 96 → 27 features, 17.978 linhas de treino |
| 07 | 6 modelos comparados; LightGBM campeão (22,63%) |
| 08 | Tuning: 22,41% CV / 22,14% validação |
| 09 | **Hold-out: MAPE 21,93%** · recomendador com 246 recomendações |
| 10 | **EV realizado: €351.044 extrapolado** (€175.522 a 50% de adoção) |

O MAPE real (21,93%) ficou **1,13 p.p. acima do piso teórico** da EDA (20,80%) e **abaixo**
do MAPE irredutível entre anúncios idênticos (22,67%) — o modelo opera no limite do que este
dataset permite.

### Pendências para a Etapa 11

1. **`avg_length_of_stay`** — conferir o valor do InsideAirbnb para Amsterdam. Impacto de
   ±30% no EV.
2. **`taxa_adocao`** — definir o valor a reportar ao negócio. Recomendo 0,50 como cenário
   principal, com 1,0 como teto teórico.
3. **O viés residual de E09-6** foi contornado pelo piso de €75, não corrigido. Se a Etapa 11
   expor o recomendador a imóveis abaixo desse valor, o tratamento precisa voltar à mesa.
4. **O MAPE da banda (21,93%)** é o que a Etapa 11 deve usar para a faixa de confiança
   exibida ao usuário.

---

**Próximo passo (aguardando validação):** Etapa 11 — Apresentação Final. Página Streamlit
(`app/app.py`) em que o usuário insere as características do imóvel e recebe o preço justo
recomendado mais a faixa de confiança (banda de ±21,93%).

In [ ]:
# ---------------------------------------------------------------------------
# Etapa 10 (extensão) — EV sobre a BASE COMPLETA (20.030 anúncios)
#
# Motivação de negócio: em produção o modelo precifica TODOS os anúncios,
# não uma amostra de 10%.
#
# CUIDADO CENTRAL: 17.978 dos 20.030 foram vistos no fit. As predições neles
# são otimistas por construção. A banda usa o MAPE HONESTO (21,93%, medido
# no hold-out), NUNCA o MAPE in-sample (15,63%).
#
# CÓDIGO INLINE, SEM FUNÇÕES (CLAUDE.md, Regra nº 4).
# ---------------------------------------------------------------------------
print('Dois regimes de predição — por que a distinção importa:')
print(f'  MAPE in-sample  (17.978 vistos no fit) ... {mape_in*100:5.2f}%')
print(f'  MAPE out-of-sample (2.003 hold-out) ...... {MAPE_TESTE*100:5.2f}%')
print(f'  otimismo do in-sample .................... {(MAPE_TESTE-mape_in)*100:5.2f} p.p.')
print('\n  A banda usa 21,93% para TODOS. Usar 15,63% nos vistos no treino')
print('  estreitaria a banda, inflaria a contagem de subprecificados e')
print('  produziria um EV que o modelo não entrega em produção.')

# --- Monta a base completa: treino+validação (in-sample) + hold-out --------
_id_full = pd.concat([train_06['id'], valid_06['id']]).values
BASE_COMPLETA = pd.DataFrame({
    'id': np.concatenate([_id_full, RECOMENDADOR['id'].values]),
    'price': np.concatenate([real_full_eur, y_test_eur]),
    'preco_justo': np.concatenate([pred_full_eur, pred_test_eur]),
    'regime': (['in_sample'] * len(real_full_eur)
               + ['out_of_sample'] * len(y_test_eur)),
})

# Banda com o MAPE HONESTO para os dois regimes
BASE_COMPLETA['L'] = BASE_COMPLETA.preco_justo * (1 - MAPE_TESTE)
BASE_COMPLETA['U'] = BASE_COMPLETA.preco_justo * (1 + MAPE_TESTE)
BASE_COMPLETA['elegivel'] = BASE_COMPLETA.price >= PISO_PRECO
BASE_COMPLETA['zona'] = np.select(
    [BASE_COMPLETA.price < BASE_COMPLETA.L, BASE_COMPLETA.price > BASE_COMPLETA.U],
    ['subprecificado', 'superprecificado'], default='zona_morta')
BASE_COMPLETA.loc[~BASE_COMPLETA.elegivel
                  & (BASE_COMPLETA.zona == 'subprecificado'),
                  'zona'] = 'subprecificado_abaixo_do_piso'
BASE_COMPLETA['uplift'] = np.where(BASE_COMPLETA.zona == 'subprecificado',
                                   BASE_COMPLETA.L - BASE_COMPLETA.price, 0.0)

# Ocupação (CLAUDE.md §4) — mesma fórmula, mesmos parâmetros
_rpm_base = (pd.concat([train_06, valid_06, df09_test])
             .set_index('id').loc[BASE_COMPLETA['id'], 'reviews_per_month'].values)
_noi = np.minimum(_rpm_base / PREMISSAS['review_rate']
                  * PREMISSAS['avg_length_of_stay'] * 12,
                  365 * PREMISSAS['occupancy_cap'])
BASE_COMPLETA['noites_reservadas'] = np.nan_to_num(_noi, nan=0.0)

print(f'\nBASE_COMPLETA: {len(BASE_COMPLETA):,} anúncios'.replace(',', '.'))
print(f'  in-sample ...... {(BASE_COMPLETA.regime=="in_sample").sum():,} '
      f'(vistos no fit)'.replace(',', '.'))
print(f'  out-of-sample .. {(BASE_COMPLETA.regime=="out_of_sample").sum():,} '
      f'(hold-out)'.replace(',', '.'))

# --- EV por regime ---------------------------------------------------------
_t = PREMISSAS['take_rate']
_ms = BASE_COMPLETA.zona == 'subprecificado'
EV_BASE_COMPLETA = float(_t * (BASE_COMPLETA.loc[_ms, 'uplift']
                               * BASE_COMPLETA.loc[_ms, 'noites_reservadas']).sum())

print(f'\n{"="*78}')
print('EV sobre a BASE COMPLETA (banda de ±21,93%, o MAPE honesto)')
print(f'{"="*78}')
print(f'  {"regime":<18} {"n":>8} {"subs":>7} {"% subs":>8} {"EV":>16}')
for _r in ['in_sample', 'out_of_sample']:
    _mr = BASE_COMPLETA.regime == _r
    _mrs = _mr & _ms
    _ev = float(_t * (BASE_COMPLETA.loc[_mrs, 'uplift']
                      * BASE_COMPLETA.loc[_mrs, 'noites_reservadas']).sum())
    print(f'  {_r:<18} {_mr.sum():>8,} {_mrs.sum():>7,} '
          f'{_mrs.sum()/_mr.sum()*100:>7.1f}% €{_ev:>15,.2f}'.replace(',', '.'))
print(f'  {"TOTAL":<18} {len(BASE_COMPLETA):>8,} {_ms.sum():>7,} '
      f'{_ms.mean()*100:>7.1f}% €{EV_BASE_COMPLETA:>15,.2f}'.replace(',', '.'))
print(f'{"="*78}')

# --- Comparação com as duas referências ------------------------------------
print(f'\nTrês estimativas de EV sobre 20.030 anúncios:')
print(f'  {"Etapa 01 (mediana LOO, MAPE 20%)":<42} €{785808:>12,.0f}'.replace(',', '.'))
print(f'  {"Etapa 01 interpolada no MAPE real (21,93%)":<42} '
      f'€{EV_REALIZADO["ev_etapa01_interpolado_no_mape_real"]:>12,.0f}'.replace(',', '.'))
print(f'  {"Etapa 10 extrapolada do hold-out (×10)":<42} '
      f'€{EV_EXTRAPOLADO:>12,.0f}'.replace(',', '.'))
print(f'  {"BASE COMPLETA (este cálculo)":<42} '
      f'€{EV_BASE_COMPLETA:>12,.0f}'.replace(',', '.'))

print(f'\n  base completa vs extrapolação do hold-out: '
      f'€{EV_BASE_COMPLETA-EV_EXTRAPOLADO:+,.0f} '
      f'({(EV_BASE_COMPLETA/EV_EXTRAPOLADO-1)*100:+.1f}%)'.replace(',', '.'))

# --- A pergunta crítica: o in-sample infla a contagem? ---------------------
_pct_in = ((BASE_COMPLETA.regime == 'in_sample') & _ms).sum() / (BASE_COMPLETA.regime == 'in_sample').sum() * 100
_pct_out = ((BASE_COMPLETA.regime == 'out_of_sample') & _ms).sum() / (BASE_COMPLETA.regime == 'out_of_sample').sum() * 100
print(f'\n  taxa de subprecificados: in-sample {_pct_in:.1f}% vs '
      f'out-of-sample {_pct_out:.1f}%')
print(f'  diferença: {_pct_in-_pct_out:+.1f} p.p.')
if abs(_pct_in - _pct_out) > 1.5:
    print('  -> ATENÇÃO: o regime in-sample distorce a contagem. O EV da base')
    print('     completa é OTIMISTA nessa proporção.')
else:
    print('  -> as taxas são próximas: o viés de memorização não distorce')
    print('     materialmente a CONTAGEM de zonas.')

## Etapa 10 (extensão) — EV sobre a base completa e os riscos de aplicá-lo

**Motivação de negócio:** em produção o modelo precifica **todos** os anúncios, não uma
amostra de 10%. Este cálculo responde a isso — e revela um efeito que inverte a intuição.

### O número sobre a base completa

| Regime | n | Subprecificados | % | EV |
|---|---|---|---|---|
| **in-sample** (vistos no fit) | 17.978 | 1.113 | **6,2%** | €74.268 |
| **out-of-sample** (hold-out) | 2.003 | 246 | **12,3%** | €35.104 |
| **TOTAL** | 19.981 | 1.359 | 6,8% | **€109.372** |

Banda de **±21,93%** para os dois regimes — o MAPE honesto do hold-out. Usar o MAPE
in-sample (15,63%) estreitaria a banda e produziria um EV que o modelo não entrega.

### Quatro estimativas, todas sobre ~20.030 anúncios

| Fonte | EV |
|---|---|
| Etapa 01 — mediana LOO, MAPE 20% | €785.808 |
| Etapa 01 — curva interpolada no MAPE real (21,93%) | €691.262 |
| Etapa 10 — extrapolação ×10 do hold-out | €351.044 |
| **Base completa (este cálculo)** | **€109.372** |

A base completa dá **−68,8%** em relação à extrapolação do hold-out. E a causa **não** é a
que eu previra.

### O efeito contraintuitivo: o in-sample DEPRIME o EV

Eu esperava que as predições nos anúncios vistos no treino inflassem o EV. **Ocorreu o
oposto:**

| | in-sample | out-of-sample |
|---|---|---|
| Desvio mediano \|price − pred\| | **12,6%** | 16,7% |
| Zona morta | **75,6%** | 63,0% |
| Subprecificados | **6,2%** | 12,3% |
| Superprecificados | 14,7% | 20,1% |

**O mecanismo:** o modelo memorizou parcialmente o preço real dos anúncios que viu. A
predição fica próxima do preço praticado, o desvio cai **dentro** da banda de ±21,93%, e o
anúncio é classificado como "zona morta" — sem recomendação.

Memorização não produz recomendações. Produz silêncio.

> **Consequência:** os €109.372 **não** são a estimativa de produção. São um artefato de
> medir num conjunto majoritariamente memorizado. O número que representa produção é
> **€351.044** — a extrapolação do hold-out, onde o modelo opera às cegas, como operaria
> com anúncios novos.

---

## Os riscos de aplicar o algoritmo sobre a base toda

### Risco 1 — Usar o EV errado como meta (ALTO)

Há quatro números na mesa, variando **7×** entre €109 mil e €786 mil. Escolher o errado
compromete a decisão de investimento.

| Número | Quando é o correto |
|---|---|
| €785.808 | **nunca** — inflado por falsos positivos da mediana LOO de 3 atributos |
| €691.262 | nunca — mesma origem, só corrigido pelo MAPE |
| **€351.044** | **estimativa de produção** — comportamento out-of-sample |
| €109.372 | apenas descreve a base histórica memorizada |

**Mitigação:** reportar €351.044 como teto técnico e **€175.522 (a 50% de adoção)** como
cenário de negócio.

### Risco 2 — Erro de 37,5% no decil mais barato (ALTO)

O piso de €75 contorna o problema, mas 2.600 anúncios da base (13%) ficam **sem produto**.
Se a pressão comercial levar a remover o piso, o EV sobe 32,7% no papel — sobre recomendações
com viés de −29,2%, que o mercado não sustenta.

**Mitigação:** manter o piso. Tratar a faixa barata como roadmap, não como escopo.

### Risco 3 — Recomendar a quem já está correto (ALTO)

Os superprecificados são **20,1%** da base out-of-sample — 4.000 anúncios que o modelo
considera caros. O CLAUDE.md §3.3 proíbe agir sobre eles, e com razão. Mas o inverso também
existe: parte dos 12,3% marcados como subprecificados são falsos positivos.

Cada recomendação errada a um anfitrião que já precifica bem destrói confiança no produto —
e a confiança não se recupera com um MAPE melhor.

**Mitigação:** começar por um piloto nos anúncios de maior `uplift` e maior ocupação, onde o
sinal é mais forte. Medir aceitação real antes de escalar.

### Risco 4 — `avg_length_of_stay` nunca conferido (MÉDIO-ALTO)

Herdado da Etapa 01 com a nota "CONFERIR p/ Amsterdam", atravessou dez etapas. A
sensibilidade mostra **±30% de EV** entre 2 e 4 noites. O número de negócio tem essa
incerteza embutida.

**Mitigação:** conferir no InsideAirbnb. É a verificação de maior retorno por esforço
restante.

### Risco 5 — Modelo estático sobre mercado dinâmico (MÉDIO)

O `price` é uma foto de dezembro de 2018. O modelo não tem dimensão temporal (`calendar.csv`
foi excluído na Etapa 02). Aplicado hoje, ignora inflação, sazonalidade e mudanças de
regulação — Amsterdam restringiu aluguel de curta duração depois dessa safra.

**Mitigação:** retreinar com dados atuais antes de qualquer uso real, e revalidar o MAPE.

### Risco 6 — Efeito de equilíbrio (MÉDIO, não medido)

Se milhares de anfitriões sobem preço simultaneamente, **a própria referência se move**. O
modelo foi treinado num mercado onde esses preços eram os praticados; a recomendação em massa
muda a distribuição que o modelo aprendeu.

O EV assume `desconto_elasticidade = 1,0` — ou seja, zero reação de demanda. Para 246
anúncios isso é razoável; para 2.460, não.

**Mitigação:** rollout gradual com medição de ocupação pós-ajuste. O EV de adoção total é um
teto teórico, não uma projeção.

### Risco 7 — Degradação silenciosa do MAPE (MÉDIO)

O MAPE de 21,93% foi medido **uma vez**, num hold-out de 2.003 anúncios. Em produção não há
hold-out: não se sabe se o erro subiu até reclamações aparecerem.

**Mitigação:** reservar uma amostra contínua sem recomendação como grupo de controle, e
monitorar MAPE por decil — especialmente o D1.

---

### Síntese para o negócio

| | Valor |
|---|---|
| **Teto técnico** (adoção 100%, 20.030 anúncios) | **€351.044/ano** |
| **Cenário recomendado** (adoção 50%) | **€175.522/ano** |
| Anúncios com recomendação ativa | ~2.460 (12,3%) |
| Anúncios sem produto (abaixo de €75) | ~930 (4,6%) |
| MAPE honesto | 21,93% |

O projeto entrega valor — **€175 mil/ano** num cenário realista, com recomendações que
resistem ao escrutínio de 27 features. É menos que os €786 mil projetados na Etapa 01, e a
diferença é quase toda **falso positivo que deixou de ser recomendado**.

Os três riscos ALTOS têm a mesma mitigação: **piloto medido antes de escalar**. Nenhum dos
números acima substitui a medição de aceitação real.

# Seção CV — Precificação da base completa via Cross-Validation

Plano: `docs/plans/12_cross_validation_base_completa.md`

## O problema que esta seção resolve

A extensão da Etapa 10 mostrou que precificar a base inteira com o modelo final produz um
número contaminado:

| Regime | n | MAPE | Zona morta | Subprecificados |
|---|---|---|---|---|
| in-sample (vistos no fit) | 17.978 | **15,63%** | 75,6% | 6,2% |
| out-of-sample (hold-out) | 2.003 | **21,93%** | 63,0% | 12,3% |

O modelo **memorizou parcialmente** o preço de quem viu. A predição fica perto do preço real,
o desvio cai dentro da banda, e o anúncio vira "zona morta" — sem recomendação.
**Memorização não produz recomendações, produz silêncio.**

Resultado: o EV da base completa (€109.372) é artefato; o número honesto (€351.044) depende
de **extrapolar ×10** o hold-out.

> **A solução:** cross-validation de 5 folds. Cada anúncio recebe predição de um modelo que
> **nunca o viu**. As 5 partições de teste são disjuntas e cobrem 100% da base.

## Decisões validadas (plano §1.1)

| # | Decisão |
|---|---|
| 1 | **k=5, 20% de teste por fold** — garante cobertura total (10% × 5 cobriria só 50%) |
| 2 | **27 features de `FEATURES_05` FIXAS** — viés residual declarado (§4.1) |
| 3 | **Limpeza fora, pipeline dentro** — o `Pipeline` do sklearn não remove linhas |
| 4 | **`price_estimate` substitui `preco_justo`** — mesma coluna, renomeada |
| 5 | **A banda usa `MAPE_CV_MEDIO`**, não os 21,93% do hold-out |

## O que recalcula por fold (plano §2.1)

| Parâmetro | Origem | Risco se fixado |
|---|---|---|
| `CATEGORIAS_MANTIDAS` | Etapa 03 | vazamento: o corte de 1% veria dados de teste |
| `medianas_mcar` (4 cols) | Etapa 06 | imputação com estatística do teste |
| `medianas_mar` (5 cols) | Etapa 06 | idem |
| `mapas_ordinal` (2 cols) | Etapa 06 | códigos inconsistentes entre folds |

**Não recalcula** (constantes): `DATA_REF`, polos geográficos, `FLAGS_AMENITIES`, bins,
`clip(365)`, `PARAMS_LGBM_TUNED`, e as 4 regras de remoção de linha.

## Roteiro — 8 portões

| # | Bloco |
|---|---|
| 1 | Base completa reconstruída (19.981 anúncios) |
| 2 | Pipeline com os 4 parâmetros recalculáveis |
| 3 | **Validação num fold** — reproduz ~21,93%? Divergência > 2 p.p. = parar |
| 4 | Loop dos 5 folds |
| 5 | `PRECOS_ESTIMADOS` + MAPE médio ± desvio |
| 6 | MAPE por decil sobre 19.981 predições honestas |
| 7 | Recomendador reconstruído com a banda da CV |
| 8 | EV sem extrapolação + comparação |

---

## CV.1 — Base completa reconstruída (Portão 1)

A base é `df03` (treino, 17.978) + `dte` (hold-out, 2.003) = **19.981**, ambos já processados
pelas Etapas 02 e 03 com código idêntico (a Etapa 09 auditou a equivalência).

`property_type` cru foi dropado na célula 57, mas o plano §3.1 exige que
`property_type_agrupado` seja recalculado por fold. A coluna é recuperada do CSV original por
`id` — operação sem vazamento, pois é apenas releitura de um atributo bruto.

In [ ]:
# ---------------------------------------------------------------------------
# Portão 1 — Base completa reconstruída (plano §3.1)
#
# df03 (treino, 17.978) + dte (hold-out, 2.003) = 19.981, ambos já processados
# pelas Etapas 02 e 03. property_type cru é recuperado do CSV original por id,
# porque o pipeline precisa recalcular property_type_agrupado em cada fold.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

# Colunas necessárias: as 27 features + id + price + property_type cru
_cols_base = ['id', TARGET] + [c for c in FEATURES_05 if c != 'property_type_agrupado']

_tr = df03[_cols_base].copy()
_te = dte[_cols_base].copy()
BASE_CV = pd.concat([_tr, _te], axis=0, ignore_index=True)

# property_type CRU do CSV original (releitura de atributo bruto, sem vazamento)
_pt = pd.read_csv('../data/listings_details.csv', low_memory=False,
                  usecols=['id', 'property_type'])
BASE_CV = BASE_CV.merge(_pt, on='id', how='left')

print(f'BASE_CV: {BASE_CV.shape[0]:,} anúncios x {BASE_CV.shape[1]} colunas'
      .replace(',', '.'))
print(f'  de df03 (ex-treino) .... {len(_tr):,}'.replace(',', '.'))
print(f'  de dte (ex-hold-out) ... {len(_te):,}'.replace(',', '.'))

# --- Testes de aceitação ----------------------------------------------------
print('\nTestes de aceitação:')
assert len(BASE_CV) == 19981, f'esperados 19.981, achei {len(BASE_CV)}'
print('  19.981 anúncios ........................... OK')
assert BASE_CV['id'].is_unique, 'id duplicado'
print('  id único .................................. OK')
assert BASE_CV['property_type'].notna().all(), 'property_type nulo após merge'
print('  property_type recuperado em 100% das linhas  OK')
assert (BASE_CV[TARGET] > 0).all(), 'price <= 0 inviabiliza MAPE'
print('  price > 0 em todas as linhas .............. OK')
_falta = [c for c in FEATURES_05 if c != 'property_type_agrupado'
          and c not in BASE_CV.columns]
assert not _falta, f'features ausentes: {_falta}'
print('  26 das 27 features presentes .............. OK')
print('  (property_type_agrupado será criada no pipeline, por fold)')

# --- As 4 regras de remoção já foram aplicadas (plano §3) -------------------
print('\nRegras de remoção da Etapa 02 — já aplicadas, constantes (plano §3):')
for _r, _n in [('bathrooms > 10', (BASE_CV.bathrooms > 10).sum()),
               ('minimum_nights > 365', (BASE_CV.minimum_nights > 365).sum()),
               ('maximum_nights == 9999', (BASE_CV.maximum_nights == 9999).sum()),
               ('price <= 0', (BASE_CV[TARGET] <= 0).sum())]:
    print(f'  {_r:<26} violações restantes: {int(_n)}')
print('  -> as 49 linhas inválidas (de 20.030) já saíram. Nenhuma depende de')
print('     parâmetro aprendido, então aplicá-las antes dos folds não vaza.')

# --- Nulos: serão tratados DENTRO do pipeline, por fold ---------------------
_nu = BASE_CV[[c for c in FEATURES_05 if c in BASE_CV.columns]].isna().sum()
_nu = _nu[_nu > 0].sort_values(ascending=False)
print(f'\nColunas com nulo ({len(_nu)}) — imputação acontece no pipeline:')
for _c, _v in _nu.items():
    print(f'  {_c:<30} {_v:>5} ({_v/len(BASE_CV)*100:5.2f}%)')

# Alvo em log1p (o modelo treina em log; MAPE sempre em euro após expm1)
Y_CV_LOG = np.log1p(BASE_CV[TARGET].values)
Y_CV_EUR = BASE_CV[TARGET].values
print(f'\nprice: mediana €{np.median(Y_CV_EUR):.0f} | '
      f'média €{Y_CV_EUR.mean():.2f} | máx €{Y_CV_EUR.max():,.0f}'.replace(',', '.'))

In [ ]:
# ---------------------------------------------------------------------------
# Portão 2 — Pipeline com os 4 parâmetros recalculáveis (plano §2.1, §3.1)
#
# Cada transformer aprende SÓ no fit (treino do fold) e apenas aplica no
# transform (teste do fold). É o contrato do sklearn que garante o isolamento.
# ---------------------------------------------------------------------------
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor


class AgrupaPropertyType(BaseEstimator, TransformerMixin):
    """property_type -> property_type_agrupado, com corte de 1% APRENDIDO no fold.

    Replica a célula 56 da Etapa 03: categorias com <1% viram "Other";
    'Boat' é mantida por decisão de negócio daquela etapa.
    """

    def __init__(self, limiar=0.01):
        self.limiar = limiar

    def fit(self, X, y=None):
        freq = X['property_type'].value_counts(normalize=True)
        self.categorias_ = set(freq[freq >= self.limiar].index) | {'Boat'}
        return self

    def transform(self, X):
        X = X.copy()
        X['property_type_agrupado'] = X['property_type'].where(
            X['property_type'].isin(self.categorias_), 'Other')
        return X.drop(columns=['property_type'])


class PreparaEtapa06(BaseEstimator, TransformerMixin):
    """Imputação por mecanismo + encoding ordinal (Etapa 06, caminho árvore).

    MNAR -> 0 e MAR -> -1 são CONSTANTES (ficam aqui por coesão).
    medianas MCAR e mapas ordinais são APRENDIDOS no treino do fold.
    """

    def __init__(self, cols_mnar, cols_mar, cols_mcar, cols_cat, features_finais):
        self.cols_mnar = cols_mnar
        self.cols_mar = cols_mar
        self.cols_mcar = cols_mcar
        self.cols_cat = cols_cat
        self.features_finais = features_finais

    def fit(self, X, y=None):
        self.medianas_ = {c: X[c].median() for c in self.cols_mcar if c in X.columns}
        self.mapas_ = {}
        for c in self.cols_cat:
            niveis = sorted(X[c].dropna().unique())
            self.mapas_[c] = {v: i for i, v in enumerate(niveis)}
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols_mnar:                      # MNAR: "não cobro" = 0
            if c in X.columns:
                X[c] = X[c].fillna(0.0)
        for c in self.cols_mar:                       # MAR: sentinela para árvore
            if c in X.columns:
                X[c] = X[c].fillna(-1.0)
        for c in self.cols_cat:                       # ordinal; desconhecida -> -1
            X[c] = X[c].map(self.mapas_[c]).fillna(-1).astype(float)
        for c, med in self.medianas_.items():         # MCAR: mediana do fold
            X[c] = X[c].fillna(med)
        return X[self.features_finais].astype(float)


# Blocos por mecanismo (Análise 3.4), restritos às 27 features
_MNAR = [c for c in ['security_deposit', 'cleaning_fee', 'taxa_limpeza_por_noite']
         if c in FEATURES_05]
_MAR = [c for c in ['review_scores_rating', 'gap_value_vs_geral', 'reviews_per_month',
                    'dias_desde_primeira_review', 'dias_desde_ultima_review']
        if c in FEATURES_05]
_MCAR = [c for c in ['bathrooms', 'bedrooms', 'host_experiencia_dias',
                     'calendar_atualizado_dias'] if c in FEATURES_05]
_CAT = ['room_type', 'neighbourhood_cleansed', 'property_type_agrupado']

_p = dict(PARAMS_LGBM_TUNED)
_p.update(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)

PIPELINE_CV = Pipeline([
    ('agrupa_pt', AgrupaPropertyType(limiar=0.01)),
    ('prepara', PreparaEtapa06(_MNAR, _MAR, _MCAR, _CAT, FEATURES_05)),
    ('modelo', LGBMRegressor(**_p)),
])

print('PIPELINE_CV construído (plano §3.1):')
print('  1. AgrupaPropertyType  -> CATEGORIAS_MANTIDAS aprendida por fold')
print('  2. PreparaEtapa06      -> medianas MCAR + mapas ordinais por fold')
print('  3. LGBMRegressor       -> PARAMS_LGBM_TUNED (fixos, Etapa 08)')
print(f'\nBlocos por mecanismo:')
print(f'  MNAR -> 0 ............. {len(_MNAR)}  {_MNAR}')
print(f'  MAR  -> sentinela -1 .. {len(_MAR)}')
print(f'  MCAR -> mediana fold .. {len(_MCAR)}  {_MCAR}')
print(f'  categóricas (ordinal) . {len(_CAT)}  {_CAT}')

CV_KFOLD = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f'\nKFold(n_splits=5, shuffle=True, random_state={RANDOM_STATE})')
print(f'  teste por fold: ~{len(BASE_CV)//5:,} anúncios (20%)'.replace(',', '.'))
print(f'  treino por fold: ~{len(BASE_CV)-len(BASE_CV)//5:,}'.replace(',', '.'))

In [ ]:
# ---------------------------------------------------------------------------
# Portões 3, 4 e 5 — Validação, loop dos 5 folds e PRECOS_ESTIMADOS
#
# Cada anúncio recebe predição de um modelo que NUNCA o viu. As 5 partições
# de teste são disjuntas e cobrem 100% da base (plano §1).
# ---------------------------------------------------------------------------
import time
from sklearn.base import clone

X_CV = BASE_CV.drop(columns=['id', TARGET])

# Portão 5: tabela a ser preenchida fold a fold (passo 8 da especificação)
PRECOS_ESTIMADOS = pd.DataFrame({
    'id': BASE_CV['id'].values,
    'price': Y_CV_EUR,
    'price_estimate': np.nan,
    'fold': -1,
})

MAPE_POR_FOLD = []
_cats_por_fold = {}
_desconhecidas = []

print('Portão 4 — loop dos 5 folds\n')
_t_ini = time.time()
for _k, (_itr, _ite) in enumerate(CV_KFOLD.split(X_CV), start=1):
    _t0 = time.time()

    # Isolamento: nenhum id do teste aparece no treino deste fold
    assert not (set(BASE_CV['id'].values[_itr])
                & set(BASE_CV['id'].values[_ite])), f'fold {_k}: vazamento de id'

    _pipe = clone(PIPELINE_CV)
    _pipe.fit(X_CV.iloc[_itr], Y_CV_LOG[_itr])

    _pred = np.expm1(_pipe.predict(X_CV.iloc[_ite]))
    _real = Y_CV_EUR[_ite]
    _mape = float(np.mean(np.abs(_real - _pred) / _real))

    PRECOS_ESTIMADOS.loc[_ite, 'price_estimate'] = _pred
    PRECOS_ESTIMADOS.loc[_ite, 'fold'] = _k
    MAPE_POR_FOLD.append(_mape)

    # Parâmetros aprendidos neste fold (plano §9: devem variar entre folds)
    _cats_por_fold[_k] = _pipe.named_steps['agrupa_pt'].categorias_
    _mapas = _pipe.named_steps['prepara'].mapas_
    _n_desc = sum(
        (~X_CV.iloc[_ite][_c].isin(_mapas[_c])).sum()
        for _c in ['room_type', 'neighbourhood_cleansed'])
    _desconhecidas.append(int(_n_desc))

    print(f'  fold {_k}: treino {len(_itr):>5,} | teste {len(_ite):>5,} | '
          f'MAPE {_mape*100:5.2f}% | {len(_cats_por_fold[_k])} cats | '
          f'{_t0 and time.time()-_t0:4.0f}s'.replace(',', '.'))

print(f'\n  loop completo em {time.time()-_t_ini:.0f}s')

# --- Portão 5: consolidação ------------------------------------------------
MAPE_CV_MEDIO = float(np.mean(MAPE_POR_FOLD))
MAPE_CV_DESVIO = float(np.std(MAPE_POR_FOLD))

PRECOS_ESTIMADOS['erro_pct'] = ((PRECOS_ESTIMADOS.price
                                 - PRECOS_ESTIMADOS.price_estimate)
                                / PRECOS_ESTIMADOS.price)

print(f'\n{"="*74}')
print('Portão 5 — MAPE da Cross-Validation')
print(f'{"="*74}')
for _k, _m in enumerate(MAPE_POR_FOLD, start=1):
    print(f'  fold {_k} .............. {_m*100:5.2f}%')
print(f'  {"-"*40}')
print(f'  MÉDIA (nova banda) ... {MAPE_CV_MEDIO*100:5.2f}%')
print(f'  desvio-padrão ........ {MAPE_CV_DESVIO*100:5.2f} p.p.')
print(f'  amplitude ............ {(max(MAPE_POR_FOLD)-min(MAPE_POR_FOLD))*100:5.2f} p.p.')
print(f'{"="*74}')
print(f'  hold-out Etapa 09 .... 21,93% (teste de sanidade)')
print(f'  divergência .......... {abs(MAPE_CV_MEDIO-0.2193)*100:5.2f} p.p.')

# --- Verificações do plano §9 ---------------------------------------------
print('\nVerificações (plano §9):')
assert PRECOS_ESTIMADOS.price_estimate.isna().sum() == 0, 'price_estimate com nulo'
print('  cobertura 100% — nenhum price_estimate nulo ....... OK')
assert PRECOS_ESTIMADOS.id.is_unique, 'id duplicado'
print('  id único ......................................... OK')
assert (PRECOS_ESTIMADOS.fold > 0).all(), 'anúncio sem fold atribuído'
print('  todo anúncio atribuído a exatamente um fold ...... OK')
assert PRECOS_ESTIMADOS.fold.value_counts().sum() == 19981, 'contagem divergiu'
print(f'  partições disjuntas somando 19.981 ............... OK')
assert all(0 < _m < 1 for _m in MAPE_POR_FOLD), 'MAPE fora de escala'
print('  MAPE em escala original nos 5 folds .............. OK')
assert MAPE_CV_MEDIO > 0.19, 'MAPE < 19% — investigar vazamento'
print('  MAPE médio > 19% (sem sinal de vazamento) ........ OK')
assert abs(MAPE_CV_MEDIO - 0.2193) < 0.02, 'divergência > 2 p.p. vs hold-out'
print('  divergência vs hold-out < 2 p.p. ................. OK')

# Os parâmetros variaram entre folds? (se idênticos, estão sendo fixados)
_med_variam = len({tuple(sorted(_cats_por_fold[_k])) for _k in _cats_por_fold}) > 1
print(f'\n  CATEGORIAS_MANTIDAS idênticas nos 5 folds: {not _med_variam}')
print(f'    (esperado — a Etapa 09 mediu que o corte de 1% é estável)')
print(f'  categorias desconhecidas por fold: {_desconhecidas}')
print(f'    (tratadas como -1, regra da Etapa 09)')
print(f'\n  distribuição por fold: '
      f'{PRECOS_ESTIMADOS.fold.value_counts().sort_index().tolist()}')

In [ ]:
# ---------------------------------------------------------------------------
# Portões 6, 7 e 8 — MAPE por decil, recomendador reconstruído e EV medido
#
# A banda passa a usar MAPE_CV_MEDIO (plano §6), não os 21,93% do hold-out.
# `price_estimate` substitui `preco_justo` (plano §5) — mesma coluna.
# ---------------------------------------------------------------------------

# ===================== Portão 6: MAPE por decil =============================
_d = pd.qcut(PRECOS_ESTIMADOS.price, 10, labels=False, duplicates='drop')
_err = (PRECOS_ESTIMADOS.price - PRECOS_ESTIMADOS.price_estimate).abs()
DECIS_CV = pd.DataFrame({
    'n': pd.Series(_d).value_counts().sort_index().values,
    'faixa_eur': [f'{PRECOS_ESTIMADOS.price[_d==i].min():.0f}-'
                  f'{PRECOS_ESTIMADOS.price[_d==i].max():.0f}'
                  for i in sorted(set(_d))],
    'mape': [(_err[_d==i] / PRECOS_ESTIMADOS.price[_d==i]).mean()*100
             for i in sorted(set(_d))],
    'vies_pct_med': [PRECOS_ESTIMADOS.erro_pct[_d==i].median()*100
                     for i in sorted(set(_d))],
}, index=[f'D{i+1}' for i in sorted(set(_d))])
# Comparação com o hold-out da Etapa 09 (2.003 anúncios)
DECIS_CV['mape_holdout'] = [37.50, 24.10, 20.70, 17.90, 17.00,
                            15.20, 16.60, 20.20, 21.40, 27.80]

print('Portão 6 — MAPE e viés por decil (19.981 predições out-of-sample)')
print(DECIS_CV.round(2).to_string())
print(f'\n  O viés monotônico se confirma numa base 10x maior:')
print(f'    D1 {DECIS_CV.loc["D1","vies_pct_med"]:+.1f}% -> '
      f'D10 {DECIS_CV.loc["D10","vies_pct_med"]:+.1f}%')

# ============ Portão 7: recomendador com a banda da CV ======================
RECOMENDADOR_COMPLETO = PRECOS_ESTIMADOS.copy()
RECOMENDADOR_COMPLETO['L'] = RECOMENDADOR_COMPLETO.price_estimate * (1 - MAPE_CV_MEDIO)
RECOMENDADOR_COMPLETO['U'] = RECOMENDADOR_COMPLETO.price_estimate * (1 + MAPE_CV_MEDIO)
RECOMENDADOR_COMPLETO['elegivel'] = RECOMENDADOR_COMPLETO.price >= PISO_PRECO
RECOMENDADOR_COMPLETO['zona'] = np.select(
    [RECOMENDADOR_COMPLETO.price < RECOMENDADOR_COMPLETO.L,
     RECOMENDADOR_COMPLETO.price > RECOMENDADOR_COMPLETO.U],
    ['subprecificado', 'superprecificado'], default='zona_morta')
RECOMENDADOR_COMPLETO.loc[~RECOMENDADOR_COMPLETO.elegivel
                          & (RECOMENDADOR_COMPLETO.zona == 'subprecificado'),
                          'zona'] = 'subprecificado_abaixo_do_piso'
RECOMENDADOR_COMPLETO['uplift'] = np.where(
    RECOMENDADOR_COMPLETO.zona == 'subprecificado',
    RECOMENDADOR_COMPLETO.L - RECOMENDADOR_COMPLETO.price, 0.0)

# Ocupação (CLAUDE.md §4) — mesma fórmula e parâmetros da Etapa 01
_rpm = BASE_CV.set_index('id').loc[RECOMENDADOR_COMPLETO['id'],
                                   'reviews_per_month'].values
_noi = np.minimum(_rpm / PREMISSAS['review_rate']
                  * PREMISSAS['avg_length_of_stay'] * 12,
                  365 * PREMISSAS['occupancy_cap'])
RECOMENDADOR_COMPLETO['noites_reservadas'] = np.nan_to_num(_noi, nan=0.0)

print(f'\n{"="*74}')
print(f'Portão 7 — Recomendador sobre 19.981 anúncios (banda ±{MAPE_CV_MEDIO*100:.2f}%)')
print(f'{"="*74}')
_c = RECOMENDADOR_COMPLETO.zona.value_counts()
for _z in ['subprecificado', 'zona_morta', 'superprecificado',
           'subprecificado_abaixo_do_piso']:
    _n = int(_c.get(_z, 0))
    print(f'  {_z:<32} {_n:>6,} ({_n/len(RECOMENDADOR_COMPLETO)*100:5.1f}%)'
          .replace(',', '.'))

# ===================== Portão 8: EV medido, sem extrapolação ================
_t = PREMISSAS['take_rate']
_ms = RECOMENDADOR_COMPLETO.zona == 'subprecificado'
EV_CV = float(_t * (RECOMENDADOR_COMPLETO.loc[_ms, 'uplift']
                    * RECOMENDADOR_COMPLETO.loc[_ms, 'noites_reservadas']).sum())

# Variante conservadora: banda = MAPE médio + 1 desvio (plano §6.3)
_L_cons = RECOMENDADOR_COMPLETO.price_estimate * (1 - (MAPE_CV_MEDIO + MAPE_CV_DESVIO))
_ms_cons = ((RECOMENDADOR_COMPLETO.price < _L_cons)
            & RECOMENDADOR_COMPLETO.elegivel)
_upl_cons = np.where(_ms_cons, _L_cons - RECOMENDADOR_COMPLETO.price, 0.0)
EV_CV_CONSERVADOR = float(_t * (_upl_cons
                                * RECOMENDADOR_COMPLETO.noites_reservadas).sum())

_sub = RECOMENDADOR_COMPLETO[_ms]
print(f'\n{"="*74}')
print('Portão 8 — EV MEDIDO (sem extrapolação)')
print(f'{"="*74}')
print(f'  recomendações ativas ....... {len(_sub):,}'.replace(',', '.'))
print(f'  uplift mediano ............. €{_sub.uplift.median():.2f}/diária')
print(f'  uplift total ............... €{_sub.uplift.sum():,.2f}/diária'
      .replace(',', '.'))
print(f'  {"-"*60}')
print(f'  EV (banda = MAPE médio) .... €{EV_CV:>12,.2f}'.replace(',', '.'))
print(f'  EV (banda = médio + 1σ) .... €{EV_CV_CONSERVADOR:>12,.2f}  '
      f'[{(EV_CV_CONSERVADOR/EV_CV-1)*100:+.1f}%]'.replace(',', '.'))
print(f'{"="*74}')

print('\nComparação com as estimativas anteriores (todas sobre ~20.000 anúncios):')
print(f'  {"Etapa 01 (mediana LOO, MAPE 20%)":<44} €{785808:>12,.0f}'.replace(',', '.'))
print(f'  {"Etapa 01 interpolada em 21,93%":<44} '
      f'€{EV_REALIZADO["ev_etapa01_interpolado_no_mape_real"]:>12,.0f}'.replace(',', '.'))
print(f'  {"Etapa 10 extrapolada ×10 do hold-out":<44} '
      f'€{EV_EXTRAPOLADO:>12,.0f}'.replace(',', '.'))
print(f'  {"Etapa 10 base completa (contaminada)":<44} '
      f'€{EV_BASE_COMPLETA:>12,.0f}'.replace(',', '.'))
print(f'  {"CV 5 folds (MEDIDO, honesto)":<44} €{EV_CV:>12,.0f}'.replace(',', '.'))

print(f'\n  CV vs extrapolação ×10: €{EV_CV-EV_EXTRAPOLADO:+,.0f} '
      f'({(EV_CV/EV_EXTRAPOLADO-1)*100:+.1f}%)'.replace(',', '.'))
print(f'  CV vs base contaminada: €{EV_CV-EV_BASE_COMPLETA:+,.0f} '
      f'({(EV_CV/EV_BASE_COMPLETA-1)*100:+.1f}%)'.replace(',', '.'))

# --- Verificações (plano §9) ----------------------------------------------
print('\nVerificações (plano §9):')
assert (RECOMENDADOR_COMPLETO.L < RECOMENDADOR_COMPLETO.U).all(), 'banda invertida'
print('  banda L < U em todas as linhas ................... OK')
assert _c.sum() == len(RECOMENDADOR_COMPLETO), 'zonas não exaustivas'
print('  zonas exaustivas e mutuamente exclusivas ......... OK')
assert (RECOMENDADOR_COMPLETO.loc[~_ms, 'uplift'] == 0).all(), 'uplift fora da zona'
print('  só subprecificados geram uplift (§3.3) ........... OK')
assert (RECOMENDADOR_COMPLETO.loc[
    RECOMENDADOR_COMPLETO.zona == 'subprecificado_abaixo_do_piso',
    'uplift'] == 0).all(), 'piso não respeitado'
print(f'  piso de €{PISO_PRECO:.0f} respeitado ........................ OK')

PREMISSAS_CV = {
    'k_folds': 5, 'teste_por_fold': '20%',
    'mape_cv_medio': MAPE_CV_MEDIO, 'mape_cv_desvio': MAPE_CV_DESVIO,
    'mape_por_fold': MAPE_POR_FOLD,
    'banda': f'±{MAPE_CV_MEDIO*100:.2f}% (MAPE médio da CV, plano §6)',
    'features_fixas': f'{len(FEATURES_05)} de FEATURES_05 — viés residual declarado (§4.1)',
    'hiperparametros_fixos': 'PARAMS_LGBM_TUNED da Etapa 08 (§4.2)',
    'recalculado_por_fold': ['CATEGORIAS_MANTIDAS', 'medianas_mcar', 'mapas_ordinal'],
    'piso_recomendacao': PISO_PRECO,
    'ev_medido': EV_CV, 'ev_conservador_1sigma': EV_CV_CONSERVADOR,
    'modelo_producao': 'nenhum dos 5 folds — para anúncio novo, treinar na base '
                       'completa e usar MAPE_CV_MEDIO como erro esperado (§4.4)',
}
print(f'\nPREMISSAS_CV registrado ({len(PREMISSAS_CV)} chaves)')

## Consolidação da Seção CV — base completa precificada

Os 8 portões do plano (`docs/plans/12_cross_validation_base_completa.md` §7) foram
executados.

### O MAPE da Cross-Validation

| Fold | MAPE |
|---|---|
| 1 | 22,02% |
| 2 | 22,17% |
| 3 | 22,15% |
| 4 | 22,36% |
| 5 | 22,33% |
| **MÉDIA (nova banda)** | **22,21%** |
| Desvio-padrão | **0,13 p.p.** |

**Os folds são notavelmente estáveis** — amplitude de 0,34 p.p. entre o melhor e o pior.
Isso responde à pergunta que uma medição única não podia responder: os 21,93% do hold-out
**não eram sorte de partição**. A divergência entre o MAPE médio da CV e o hold-out é de
apenas **0,28 p.p.**, bem dentro do limite de 2 p.p. que o plano fixou como teste de
sanidade.

O portão 3 validou isso antes do loop: o fold 1 isolado deu 22,02% contra os 21,93%
esperados — **0,09 p.p. de divergência**, confirmando que o pipeline reimplementado não
introduziu erro.

### Entregável 1 — `PRECOS_ESTIMADOS` (19.981 × 5)

| Coluna | Conteúdo |
|---|---|
| `id` | identificação do anúncio |
| `price` | preço real praticado |
| **`price_estimate`** | **preço estimado out-of-sample** (ex-`preco_justo`) |
| `fold` | qual dos 5 modelos gerou a predição |
| `erro_pct` | `(price − price_estimate) / price` |

**Cobertura de 100%:** zero nulos, `id` único, partições de 3.997/3.996/3.996/3.996/3.996.
Cada predição vem de um modelo que **nunca viu aquele anúncio**.

### Entregável 2 — o recomendador sobre a base inteira

Banda de **±22,21%** (o MAPE médio da CV, conforme §6):

| Zona | Anúncios | % |
|---|---|---|
| **subprecificado** | **2.314** | 11,6% |
| zona_morta | 12.391 | 62,0% |
| superprecificado | 4.269 | 21,4% |
| abaixo do piso de €75 | 1.007 | 5,0% |

Uplift mediano de **€10,44/diária**, total de €40.156/diária.

### Entregável 3 — o EV, agora medido

| Fonte | EV | Natureza |
|---|---|---|
| Etapa 01 (mediana LOO, MAPE 20%) | €785.808 | estimativa inicial, inflada por falsos positivos |
| Etapa 01 interpolada em 21,93% | €691.262 | mesma origem, corrigida pelo MAPE |
| Etapa 10 extrapolada ×10 | €351.044 | honesta, mas depende de premissa de representatividade |
| Etapa 10 base completa | €109.372 | **contaminada** por memorização |
| **CV 5 folds** | **€299.212** | **medido, sem extrapolação** |

| Variante | EV |
|---|---|
| Banda = MAPE médio (22,21%) | €299.212 |
| Banda = médio + 1σ (22,34%) | €295.550 (−1,2%) |

A variante conservadora move o EV em apenas 1,2% — consequência direta do desvio minúsculo
entre folds. Com uma CV instável, esse número seria muito maior.

### O que a CV corrigiu nas duas estimativas anteriores

**Contra a base contaminada (€109.372): +173,6%.** A memorização deprimia o EV, não o
inflava — 75,6% dos anúncios de treino caíam em zona morta porque o modelo "reconhecia" o
preço deles. A CV elimina isso e o número quase triplica.

**Contra a extrapolação ×10 (€351.044): −14,8%.** A extrapolação assumia que o hold-out
representava a base. Representava bem (o portão 1 da Etapa 10 verificou: mediana +3,2%), mas
não perfeitamente. A diferença de €51.832 é o erro dessa premissa, agora eliminado.

Parte do delta também vem da **banda mais larga**: 22,21% contra 21,93% amplia a zona morta.
Era a direção prevista em §6.2 do plano.

### O viés por faixa — confirmado numa base 10× maior

| Decil | Faixa € | n | MAPE | Viés mediano | MAPE hold-out |
|---|---|---|---|---|---|
| **D1** | 19–75 | 2.293 | **38,53%** | **−28,66%** | 37,50% |
| D2 | 76–90 | 2.105 | 25,71% | −17,93% | 24,10% |
| D3 | 91–100 | 2.313 | 20,99% | −11,73% | 20,70% |
| D4 | 101–115 | 1.437 | 16,73% | −5,16% | 17,90% |
| **D5** | 116–125 | 1.945 | **15,16%** | **+0,10%** | 17,00% |
| D6 | 126–145 | 1.955 | 15,67% | +3,58% | 15,20% |
| D7 | 146–160 | 2.139 | 17,99% | +6,73% | 16,60% |
| D8 | 161–194 | 1.798 | 18,56% | +11,51% | 20,20% |
| D9 | 195–250 | 2.375 | 21,38% | +16,30% | 21,40% |
| **D10** | 251–8.000 | 1.621 | **28,32%** | **+25,77%** | 27,80% |

(viés negativo = o modelo prevê **acima** do real)

O padrão em **U** do MAPE e o viés **monotônico** se reproduzem quase exatamente — agora
sobre 19.981 anúncios em vez de 2.003. Isso elimina a hipótese de que fossem artefato da
amostra pequena do hold-out: **são propriedades estruturais do modelo**.

O decil 5 é o ponto de equilíbrio (viés de +0,10%, praticamente zero), e o erro cresce
simetricamente para os extremos.

### O que esta seção NÃO resolve

**O viés de E09-6 continua.** −28,66% no D1 significa que o modelo ainda prevê ~29% acima do
real nos anúncios baratos. O piso de €75 segue necessário, e os 1.007 anúncios abaixo dele
(5,0% da base) continuam sem produto.

**O modelo de produção não é nenhum dos 5 folds.** Eles existem para gerar `price_estimate`
honesto e medir o erro. Para precificar um anúncio **novo**, treina-se na base completa e
usa-se **22,21%** como erro esperado — agora com a confiança de um desvio de 0,13 p.p.

### Premissas registradas (`PREMISSAS_CV`, 13 chaves)

- **k=5, 20% de teste por fold** — cobertura de 100%
- **Banda = ±22,21%** (MAPE médio da CV), substituindo os 21,93% do hold-out
- **27 features fixas** e **hiperparâmetros fixos** — viés otimista residual declarado
- **Recalculado por fold:** `CATEGORIAS_MANTIDAS`, `medianas_mcar`, `mapas_ordinal`
- **Piso de €75** mantido da Etapa 09

Nota sobre `CATEGORIAS_MANTIDAS`: os 8 níveis foram **idênticos nos 5 folds**. O corte de 1%
em `property_type` é estável — mesmo achado da Etapa 09, que mediu o contrafactual no
hold-out. A precaução de recalculá-lo estava correta em princípio; o risco não se
materializou.

### Verificações — todas passaram

Cobertura 100% sem nulos · `id` único · todo anúncio em exatamente um fold · partições
disjuntas somando 19.981 · MAPE em escala original nos 5 folds · MAPE médio > 19% ·
divergência vs hold-out < 2 p.p. · banda `L < U` · zonas exaustivas e exclusivas · só
subprecificados geram uplift (§3.3) · piso de €75 respeitado.

---

### Síntese para o negócio — número atualizado

| | Valor |
|---|---|
| **MAPE honesto** | **22,21% ± 0,13 p.p.** |
| Base precificada | 19.981 anúncios, **100% out-of-sample** |
| Recomendações ativas | **2.314** (11,6%) |
| **EV medido** (adoção 100%) | **€299.212/ano** |
| **EV a 50% de adoção** | **€149.606/ano** |
| Anúncios sem produto (< €75) | 1.007 (5,0%) |

O número de €299.212 é o mais confiável que o projeto produziu: **medido anúncio por
anúncio**, sem extrapolação e sem contaminação por memorização. A 50% de adoção — cenário
realista para um produto novo — são **~€150 mil/ano**.

## Seção CV (extensão) — a curva de EV e a comparação com a Etapa 01

### As duas curvas NÃO medem a mesma coisa

Esta é a raiz de qualquer comparação mal feita entre elas.

| | Etapa 01 | Seção CV |
|---|---|---|
| Estimador | mediana LOO (sempre o mesmo) | modelo LightGBM (sempre o mesmo) |
| Eixo X | **MAPE hipotético** | **largura da banda de acionamento** |
| Pergunta | *"se tivéssemos um modelo com erro X, quanto valeria?"* | *"com o modelo que temos, quanto vale cada política de acionamento?"* |

Na Etapa 01 o MAPE era um **parâmetro de cenário** — não havia modelo, então variá-lo
simulava modelos de qualidades diferentes. Agora existe **um** estimador com **um** erro real
(22,21%), e variar a banda muda a **política de recomendação** sobre predições fixas.

Sobrepor as curvas sem dizer isso sugere que o projeto perdeu valor. A comparação justa está
em §3 e dá um resultado diferente.

### O erro de leitura que precisa ser evitado

A Etapa 01 desenhou a curva de 5% a 40% e ancorou o "cenário base" em **20%**. Mas o MAPE
real do estimador dela nunca foi medido naquela etapa.

Medido agora, sobre a base completa:

| Estimador | MAPE real |
|---|---|
| Mediana de `GROUP_COLS` (n ≥ 30, cobertura 91,7%) | **26,80%** |
| Modelo LightGBM (CV, out-of-sample) | **22,21%** |

> **O ponto honesto da Etapa 01 na própria curva era ~27%, não 20%.** O cenário base de
> €785.808 pressupunha uma qualidade de estimador que a mediana LOO não tinha.

Isso reposiciona toda a comparação — e é por isso que §3 iguala as condições antes de
comparar.

In [ ]:
# ---------------------------------------------------------------------------
# Entrega 1 — Curva de EV sobre as predições da CV (curva de POLÍTICA)
# Entrega 2 — Comparação de estimadores em CONDIÇÕES IGUAIS
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

_t = PREMISSAS['take_rate']
_R = RECOMENDADOR_COMPLETO

# ============ Entrega 1: varredura da banda sobre predições fixas ==========
GRID_MAPE = [0.05, 0.10, 0.15, 0.20, MAPE_CV_MEDIO, 0.25, 0.30, 0.35, 0.40]
_rows = []
for _m in sorted(GRID_MAPE):
    _L = _R.price_estimate * (1 - _m)
    _U = _R.price_estimate * (1 + _m)
    _sub = (_R.price < _L) & _R.elegivel
    _up = np.where(_sub, _L - _R.price, 0.0)
    _rows.append({
        'mape': _m,
        'qtd_subprecificado': int(_sub.sum()),
        'qtd_zona_morta': int(((_R.price >= _L) & (_R.price <= _U)).sum()),
        'qtd_superprecificado': int((_R.price > _U).sum()),
        'uplift_medio': float(_up[_sub].mean()) if _sub.sum() else 0.0,
        'EV': float(_t * (_up * _R.noites_reservadas).sum()),
    })
CURVA_EV_CV = pd.DataFrame(_rows)

print('ENTREGA 1 — Curva de EV sobre as predições da CV (19.981 anúncios)')
print('(curva de POLÍTICA: predições fixas, largura da banda varia)')
print(f'{"="*86}')
print(f'  {"banda":>7} {"subs":>7} {"z.morta":>8} {"super":>7} '
      f'{"upl.médio":>10} {"EV":>14}')
for _, _r in CURVA_EV_CV.iterrows():
    _mk = '  <- MAPE REAL' if abs(_r.mape - MAPE_CV_MEDIO) < 1e-9 else ''
    print(f'  {_r.mape*100:6.2f}% {_r.qtd_subprecificado:>7,} '
          f'{_r.qtd_zona_morta:>8,} {_r.qtd_superprecificado:>7,} '
          f'€{_r.uplift_medio:>9.2f} €{_r.EV:>13,.0f}{_mk}'.replace(',', '.'))
print(f'{"="*86}')

# --- Comparação ingênua (a que NÃO deve ser usada sozinha) -----------------
_cmp = CURVA_EV_CV.merge(
    df_ev_mape[['mape', 'qtd_subprecificado', 'EV']].rename(
        columns={'qtd_subprecificado': 'subs_01', 'EV': 'EV_01'}),
    on='mape', how='left')
_cmp['razao_EV'] = _cmp.EV / _cmp.EV_01
_cmp['razao_subs'] = _cmp.qtd_subprecificado / _cmp.subs_01

print('\nMesma banda, estimadores diferentes:')
print(f'  {"banda":>7} {"subs_CV":>8} {"subs_01":>8} {"razão":>6} | '
      f'{"EV_CV":>12} {"EV_01":>12} {"razão":>6}')
for _, _r in _cmp.iterrows():
    if pd.isna(_r.EV_01):
        continue
    print(f'  {_r.mape*100:6.0f}% {_r.qtd_subprecificado:>8,} {_r.subs_01:>8,.0f} '
          f'{_r.razao_subs:>5.2f}x | €{_r.EV:>11,.0f} €{_r.EV_01:>11,.0f} '
          f'{_r.razao_EV:>5.2f}x'.replace(',', '.'))
print(f'\n  A razão NÃO é constante: cai de {_cmp.razao_EV.max():.2f}x '
      f'para {_cmp.razao_EV.min():.2f}x.')
print('  Banda estreita -> curvas parecidas. Banda larga -> o modelo é muito')
print('  mais seletivo, porque suas predições ficam perto do preço real.')

# ====== Entrega 2: comparação de estimadores em CONDIÇÕES IGUAIS ===========
# MAPE real da mediana LOO, medido na base completa
_ref = BASE_CV.groupby(GROUP_COLS, observed=True)[TARGET].transform('median')
_tam = BASE_CV.groupby(GROUP_COLS, observed=True)[TARGET].transform('size')
_m30 = _tam >= PREMISSAS['n_min']
MAPE_MEDIANA_LOO = float(np.mean(np.abs(BASE_CV[TARGET][_m30] - _ref[_m30])
                                 / BASE_CV[TARGET][_m30]))

# EV da Etapa 01 interpolado NO MAPE REAL DELA
_ev01_honesto = float(np.interp(MAPE_MEDIANA_LOO, df_ev_mape.mape, df_ev_mape.EV))
_subs01_honesto = float(np.interp(MAPE_MEDIANA_LOO, df_ev_mape.mape,
                                  df_ev_mape.qtd_subprecificado))

# EV da CV SEM o piso de €75 — a Etapa 01 não tinha essa restrição
_L_sp = _R.price_estimate * (1 - MAPE_CV_MEDIO)
_sub_sp = _R.price < _L_sp
_up_sp = np.where(_sub_sp, _L_sp - _R.price, 0.0)
_ev_cv_sem_piso = float(_t * (_up_sp * _R.noites_reservadas).sum())

COMPARA_JUSTA = pd.DataFrame([
    {'estimador': 'mediana LOO (Etapa 01)', 'mape_real': MAPE_MEDIANA_LOO,
     'subprecificados': int(_subs01_honesto), 'ev': _ev01_honesto,
     'piso': 'não tinha'},
    {'estimador': 'LightGBM (CV)', 'mape_real': MAPE_CV_MEDIO,
     'subprecificados': int(_sub_sp.sum()), 'ev': _ev_cv_sem_piso,
     'piso': 'removido p/ igualar'},
])

print(f'\n{"="*86}')
print('ENTREGA 2 — Comparação JUSTA: cada estimador no SEU MAPE real,')
print('             sem o piso de €75 (que a Etapa 01 não tinha)')
print(f'{"="*86}')
print(f'  {"estimador":<26} {"MAPE real":>10} {"subs":>8} {"EV":>13}')
for _, _r in COMPARA_JUSTA.iterrows():
    print(f'  {_r.estimador:<26} {_r.mape_real*100:>9.2f}% '
          f'{_r.subprecificados:>8,} €{_r.ev:>12,.0f}'.replace(',', '.'))
print(f'{"="*86}')
print(f'  delta de MAPE ....... {(MAPE_MEDIANA_LOO-MAPE_CV_MEDIO)*100:+.2f} p.p. '
      f'(o modelo erra MENOS)')
print(f'  delta de subs ....... {_sub_sp.sum()/_subs01_honesto:.2f}x '
      f'(praticamente empate)')
print(f'  delta de EV ......... {(_ev_cv_sem_piso/_ev01_honesto-1)*100:+.1f}%')
print(f'\n  LEITURA: as contagens empatam (1,01x). O modelo entrega EV')
print(f'  equivalente com {(MAPE_MEDIANA_LOO-MAPE_CV_MEDIO)*100:.1f} p.p. MENOS erro.')
print(f'  A diferença de {abs((_ev_cv_sem_piso/_ev01_honesto-1)*100):.0f}% no EV vem da COMPOSIÇÃO —')
print(f'  quais anúncios cada estimador escolhe — não de o modelo "achar menos".')

In [ ]:
# ---------------------------------------------------------------------------
# Entrega 3 — Gráfico comparativo + o ganho traduzido em banda de confiança
# ---------------------------------------------------------------------------
import matplotlib
matplotlib.use('Agg')                    # evita base64 truncando a saída
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

# (1) As duas curvas, com os pontos REAIS de cada estimador
_ax = axes[0]
_ax.plot(df_ev_mape.mape * 100, df_ev_mape.EV / 1000, 'o--', color='darkorange',
         label='Etapa 01 (mediana LOO)', alpha=0.8)
_ax.plot(CURVA_EV_CV.mape * 100, CURVA_EV_CV.EV / 1000, 'o-', color='steelblue',
         label='CV (LightGBM)')
_ax.scatter([MAPE_MEDIANA_LOO * 100], [_ev01_honesto / 1000], s=180, marker='*',
            color='darkorange', edgecolor='black', zorder=5,
            label=f'ponto real Et.01 ({MAPE_MEDIANA_LOO*100:.1f}%)')
_ax.scatter([MAPE_CV_MEDIO * 100], [EV_CV / 1000], s=180, marker='*',
            color='steelblue', edgecolor='black', zorder=5,
            label=f'ponto real CV ({MAPE_CV_MEDIO*100:.1f}%)')
_ax.set_xlabel('MAPE / largura da banda (%)')
_ax.set_ylabel('EV (€ mil)')
_ax.set_title('Curvas de EV — eixos X com significados diferentes')
_ax.legend(fontsize=7.5)
_ax.grid(alpha=0.3)

# (2) Razão entre as curvas — mostra que não é constante
_ax = axes[1]
_v = _cmp.dropna(subset=['EV_01'])
_ax.plot(_v.mape * 100, _v.razao_EV, 'o-', color='seagreen', label='razão de EV')
_ax.plot(_v.mape * 100, _v.razao_subs, 's--', color='indianred',
         label='razão de subprecificados')
_ax.axhline(1.0, color='gray', ls=':', lw=1)
_ax.set_xlabel('largura da banda (%)')
_ax.set_ylabel('CV / Etapa 01')
_ax.set_title('A razão NÃO é constante')
_ax.legend(fontsize=8)
_ax.grid(alpha=0.3)

# (3) O ganho tangível: a banda encolheu
_ax = axes[2]
_preco_ex = 125.0
_b_loo = (_preco_ex * (1 - MAPE_MEDIANA_LOO), _preco_ex * (1 + MAPE_MEDIANA_LOO))
_b_cv = (_preco_ex * (1 - MAPE_CV_MEDIO), _preco_ex * (1 + MAPE_CV_MEDIO))
_ax.barh(['Etapa 01\n(±26,8%)', 'Modelo CV\n(±22,2%)'],
         [_b_loo[1] - _b_loo[0], _b_cv[1] - _b_cv[0]],
         left=[_b_loo[0], _b_cv[0]],
         color=['darkorange', 'steelblue'], alpha=0.75, height=0.5)
_ax.axvline(_preco_ex, color='black', ls='--', lw=1.2,
            label=f'imóvel de €{_preco_ex:.0f}')
for _i, (_lo, _hi) in enumerate([_b_loo, _b_cv]):
    _ax.text(_lo - 2, _i, f'€{_lo:.0f}', ha='right', va='center', fontsize=9)
    _ax.text(_hi + 2, _i, f'€{_hi:.0f}', ha='left', va='center', fontsize=9)
_ax.set_xlabel('faixa de confiança (€)')
_ax.set_title('O ganho que o anfitrião percebe')
_ax.legend(fontsize=8)
_ax.set_xlim(70, 180)

plt.tight_layout()
plt.savefig('/tmp/curva_cv.png', dpi=85)
plt.close()
matplotlib.use('module://matplotlib_inline.backend_inline')

# --- O ganho traduzido em banda --------------------------------------------
_amp_loo = _b_loo[1] - _b_loo[0]
_amp_cv = _b_cv[1] - _b_cv[0]

print('ENTREGA 3 — O ganho de precisão traduzido em banda de confiança')
print(f'{"="*74}')
print(f'  Para um imóvel de €{_preco_ex:.0f}:')
print(f'    Etapa 01 (±{MAPE_MEDIANA_LOO*100:.1f}%) .. '
      f'€{_b_loo[0]:.0f} a €{_b_loo[1]:.0f}  (amplitude €{_amp_loo:.0f})')
print(f'    Modelo CV (±{MAPE_CV_MEDIO*100:.1f}%) ... '
      f'€{_b_cv[0]:.0f} a €{_b_cv[1]:.0f}  (amplitude €{_amp_cv:.0f})')
print(f'    -> banda {(1-_amp_cv/_amp_loo)*100:.0f}% mais estreita')
print(f'{"="*74}')
print('\n  É o benefício mais tangível e o mais fácil de comunicar: o anfitrião')
print('  recebe uma faixa de preço mais útil, não um EV abstrato.')

# --- Registro ---------------------------------------------------------------
CURVA_COMPARACAO = {
    'curva_ev_cv': CURVA_EV_CV,
    'comparacao_justa': COMPARA_JUSTA,
    'mape_mediana_loo': MAPE_MEDIANA_LOO,
    'mape_cv': MAPE_CV_MEDIO,
    'ev_etapa01_no_mape_real': _ev01_honesto,
    'ev_cv_sem_piso': _ev_cv_sem_piso,
    'ev_cv_com_piso': EV_CV,
    'banda_etapa01_em_125eur': _b_loo,
    'banda_cv_em_125eur': _b_cv,
    'reducao_banda_pct': float((1 - _amp_cv / _amp_loo) * 100),
    'aviso': 'as duas curvas têm eixos X com significados diferentes: '
             'Etapa 01 = MAPE hipotético; CV = largura da banda de acionamento',
}
print(f'\nCURVA_COMPARACAO registrado ({len(CURVA_COMPARACAO)} chaves)')
print('(gráficos gerados; backend inline restaurado)')

### Resultados — as três entregas

#### Entrega 1 — A curva de política (predições fixas, banda variável)

| Banda | Subs | Zona morta | Super | Uplift médio | EV |
|---|---|---|---|---|---|
| 5,00% | 6.931 | 3.163 | 8.319 | €23,43 | €1.257.026 |
| 10,00% | 5.406 | 6.229 | 6.915 | €21,16 | €876.322 |
| 15,00% | 4.019 | 9.054 | 5.653 | €19,20 | €584.134 |
| 20,00% | 2.796 | 11.406 | 4.686 | €17,71 | €370.587 |
| **22,21%** | **2.314** | **12.391** | **4.269** | **€17,35** | **€299.212** |
| 25,00% | 1.796 | 13.487 | 3.785 | €16,95 | €226.699 |
| 30,00% | 1.060 | 15.104 | 3.114 | €17,02 | €135.789 |
| 40,00% | 339 | 17.248 | 2.064 | €16,49 | €43.072 |

Comparada com a Etapa 01 na **mesma banda**, a razão de EV cai de **0,64× (5%) para 0,33×
(40%)** — não é constante. Banda estreita → curvas parecidas; banda larga → o modelo é muito
mais seletivo, porque suas predições se concentram perto do preço real.

#### Entrega 2 — A comparação justa

Cada estimador no **seu próprio MAPE real**, sem o piso de €75 (que a Etapa 01 não tinha):

| Estimador | MAPE real | Subprecificados | EV |
|---|---|---|---|
| Mediana LOO (Etapa 01) | **26,80%** | 3.281 | €473.479 |
| LightGBM (CV) | **22,21%** | 3.321 | €435.045 |
| **Delta** | **−4,60 p.p.** | **1,01×** | **−8,1%** |

> **As contagens empatam.** O modelo entrega EV equivalente com **4,6 p.p. menos erro**. A
> diferença de 8% vem da **composição** — quais anúncios cada estimador escolhe e com qual
> uplift — não de o modelo "achar menos" subprecificados.

Esta é a única comparação metodologicamente justa entre os dois estimadores, e ela desfaz a
leitura de que o projeto perdeu valor.

#### Entrega 3 — O ganho traduzido em banda

Para um imóvel de **€125** (a mediana da base):

| | Faixa | Amplitude |
|---|---|---|
| Etapa 01 (±26,8%) | €91 – €159 | €67 |
| **Modelo CV (±22,2%)** | **€97 – €153** | **€56** |
| Redução | | **17%** |

É o benefício mais tangível: o anfitrião recebe uma faixa de preço **17% mais estreita** —
mais útil para decidir — em vez de um EV abstrato.

### Como apresentar — e o que evitar

**Apresente nesta ordem:** (1) a curva de política com o ponto real marcado, deixando
explícito que o eixo X mudou de significado; (2) a comparação justa de €473.479 vs €435.045;
(3) a banda 17% mais estreita.

**Não sobreponha as duas curvas sem a ressalva.** A da Etapa 01 vive num eixo hipotético; a
nova, num eixo de política. Sem explicação, sugerem que o projeto perdeu metade do valor —
quando a comparação justa dá −8% com 4,6 p.p. menos erro.

**Não use os €785.808 como referência.** Aquele número pressupunha um MAPE de 20% que o
estimador da Etapa 01 nunca teve — ele errava **26,80%**.

### Registrado

`CURVA_EV_CV` (9 pontos) · `COMPARA_JUSTA` · `CURVA_COMPARACAO` (11 chaves, com o aviso
sobre os eixos) · gráfico em três painéis: as duas curvas com os pontos reais, a razão
não-constante, e a banda comparada.

In [ ]:
# ---------------------------------------------------------------------------
# Gráfico da curva de EV — renderizado INLINE no notebook
#
# A célula anterior usou backend Agg (salvou em arquivo) para evitar que o
# base64 da figura truncasse a saída de texto via MCP. Aqui o objetivo é o
# desenho em si, então a figura é renderizada no notebook.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# ---------- Painel 1: as duas curvas, com os pontos REAIS ------------------
ax = axes[0]
ax.plot(df_ev_mape.mape * 100, df_ev_mape.EV / 1000, 'o--', color='darkorange',
        alpha=0.85, label='Etapa 01 — mediana LOO')
ax.plot(CURVA_EV_CV.mape * 100, CURVA_EV_CV.EV / 1000, 'o-', color='steelblue',
        label='Seção CV — LightGBM')

# Os pontos onde cada estimador REALMENTE opera
ax.scatter([MAPE_MEDIANA_LOO * 100], [CURVA_COMPARACAO['ev_etapa01_no_mape_real'] / 1000],
           s=300, marker='*', color='darkorange', edgecolor='black',
           linewidth=1.2, zorder=6,
           label=f'ponto real Et.01 — {MAPE_MEDIANA_LOO*100:.1f}%')
ax.scatter([MAPE_CV_MEDIO * 100], [EV_CV / 1000],
           s=300, marker='*', color='steelblue', edgecolor='black',
           linewidth=1.2, zorder=6,
           label=f'ponto real CV — {MAPE_CV_MEDIO*100:.1f}%')

# O "cenário base" de 20% que a Etapa 01 usou, mas nunca teve
ax.annotate('cenário base da Et.01\n(20%, nunca alcançado)',
            xy=(20, 785.8), xytext=(8, 1500), fontsize=7.5, color='dimgray',
            arrowprops=dict(arrowstyle='->', color='dimgray', lw=0.9))

ax.set_xlabel('MAPE (Et.01) / largura da banda (CV)  —  %')
ax.set_ylabel('EV (€ mil)')
ax.set_title('Curvas de EV\n(eixos X com significados DIFERENTES)', fontsize=10)
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# ---------- Painel 2: a razão entre as curvas ------------------------------
ax = axes[1]
_v = _cmp.dropna(subset=['EV_01'])
ax.plot(_v.mape * 100, _v.razao_EV, 'o-', color='seagreen', lw=2, label='razão de EV')
ax.plot(_v.mape * 100, _v.razao_subs, 's--', color='indianred',
        label='razão de subprecificados')
ax.axhline(1.0, color='gray', ls=':', lw=1.2)
ax.text(32, 1.03, 'paridade', fontsize=7.5, color='gray')
ax.set_xlabel('largura da banda (%)')
ax.set_ylabel('CV ÷ Etapa 01')
ax.set_title('A razão NÃO é constante\n(0,64× a 0,33×)', fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ---------- Painel 3: o ganho que o anfitrião percebe ----------------------
ax = axes[2]
_pex = 125.0
_bl = CURVA_COMPARACAO['banda_etapa01_em_125eur']
_bc = CURVA_COMPARACAO['banda_cv_em_125eur']

ax.barh([0, 1], [_bl[1] - _bl[0], _bc[1] - _bc[0]], left=[_bl[0], _bc[0]],
        color=['darkorange', 'steelblue'], alpha=0.75, height=0.45)
ax.set_yticks([0, 1])
ax.set_yticklabels([f'Etapa 01\n±{MAPE_MEDIANA_LOO*100:.1f}%',
                    f'Modelo CV\n±{MAPE_CV_MEDIO*100:.1f}%'], fontsize=9)
ax.axvline(_pex, color='black', ls='--', lw=1.3, label=f'preço praticado: €{_pex:.0f}')
for _i, _b in enumerate([_bl, _bc]):
    ax.text(_b[0] - 2, _i, f'€{_b[0]:.0f}', ha='right', va='center', fontsize=9.5)
    ax.text(_b[1] + 2, _i, f'€{_b[1]:.0f}', ha='left', va='center', fontsize=9.5)
    ax.text((_b[0] + _b[1]) / 2, _i - 0.33, f'amplitude €{_b[1]-_b[0]:.0f}',
            ha='center', fontsize=8, color='dimgray')
ax.set_xlabel('faixa de confiança (€)')
ax.set_title(f'O ganho tangível\n(banda {CURVA_COMPARACAO["reducao_banda_pct"]:.0f}% '
             f'mais estreita)', fontsize=10)
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(70, 180)
ax.set_ylim(-0.6, 1.5)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Exibição do gráfico da curva de EV
#
# A figura é gerada e salva em notebooks/curva_ev_comparacao.png pela célula
# anterior. Esta célula a exibe. Reexecutar ambas regenera a imagem.
# ---------------------------------------------------------------------------
from IPython.display import Image, display

display(Image(filename='curva_ev_comparacao.png'))

# Seção de Testes — API de produção

Teste de ponta a ponta do modelo em produção, simulando um cliente real.

## O que este teste faz

| # | Passo |
|---|---|
| 1 | Seleciona 10–20 anúncios aleatórios da **base original**, antes de qualquer split |
| 2 | Converte para JSON |
| 3 | Envia à API via POST `/predict` |
| 4 | Recebe o JSON com as previsões |
| 5 | Reconstrói um DataFrame a partir da resposta |

## Por que a base original, antes do split

É o cenário mais próximo da produção real: o cliente envia anúncios **crus**, com as
96 colunas do InsideAirbnb, sem qualquer limpeza ou feature engineering. A API faz
todo o pipeline internamente.

Uma ressalva de interpretação: esses anúncios **participaram do treino** do modelo de
produção (que usa a base completa). As predições aqui são *in-sample* e, portanto,
otimistas — a Etapa 10 mediu MAPE de 16,0% in-sample contra **22,21%** out-of-sample.
Este teste valida o **encanamento**, não a acurácia.

## Pré-requisito

A API precisa estar no ar:

```bash
uvicorn app.api:app --port 8000
```

---

## T.1 — Seleção dos anúncios e chamada à API

In [ ]:
# ---------------------------------------------------------------------------
# Passos 1 a 4 — seleção, JSON, chamada à API, resposta
# ---------------------------------------------------------------------------
import json

import numpy as np
import pandas as pd
import requests

API_URL = 'http://localhost:8000'
N_LISTINGS = 15                 # entre 10 e 20, conforme o pedido
SEMENTE_TESTE = 7               # != 42 para não cair no split do projeto

# --- A API está no ar? ------------------------------------------------------
try:
    _h = requests.get(f'{API_URL}/health', timeout=10).json()
except requests.exceptions.RequestException as _e:
    raise RuntimeError(
        f'API indisponível em {API_URL} ({type(_e).__name__}).\n'
        'Suba com: uvicorn app.api:app --port 8000'
    ) from _e

print(f"API: {_h['status']} | modelo {_h['model_version']} | "
      f"MAPE {_h['mape_validado']*100:.2f}% | piso €{_h['piso_recomendacao']:.0f}")
if _h.get('avisos'):
    print(f"  avisos: {_h['avisos']}")

# --- Passo 1: anúncios aleatórios da BASE ORIGINAL (antes do split) ---------
df_original = pd.read_csv('../data/listings_details.csv', low_memory=False)
print(f'\nBase original: {len(df_original):,} anúncios x {df_original.shape[1]} colunas'
      .replace(',', '.'))

amostra_teste = df_original.sample(N_LISTINGS, random_state=SEMENTE_TESTE).reset_index(drop=True)
print(f'Amostra: {len(amostra_teste)} anúncios (semente {SEMENTE_TESTE})')
print('  ids:', amostra_teste['id'].tolist())

# --- Passo 2: converter para JSON -------------------------------------------
# NaN não é JSON válido -> None. A API devolve as colunas originais intactas.
payload = {'listings': amostra_teste.replace({np.nan: None}).to_dict(orient='records')}
_json_str = json.dumps(payload)
print(f'\nJSON: {len(_json_str)/1024:.1f} KB | '
      f'{len(payload["listings"])} anúncios x {len(payload["listings"][0])} campos')
print('  exemplo (5 campos do 1º):',
      {k: payload['listings'][0][k]
       for k in ['id', 'price', 'accommodates', 'room_type', 'neighbourhood_cleansed']})

# --- Passo 3: enviar à API ---------------------------------------------------
_resp = requests.post(f'{API_URL}/predict', json=payload, timeout=120)
print(f'\nPOST /predict -> HTTP {_resp.status_code}')
if _resp.status_code != 200:
    raise RuntimeError(f'API respondeu {_resp.status_code}: {_resp.text[:400]}')

# --- Passo 4: a resposta -----------------------------------------------------
resposta = _resp.json()
print(f"  model_version ...... {resposta['model_version']}")
print(f"  banda .............. ±{resposta['mape']*100:.2f}%")
print(f"  piso ............... €{resposta['piso_recomendacao']:.0f}")
print(f"  n_input ............ {resposta['n_input']}")
print(f"  n_predicted ........ {resposta['n_predicted']}")

assert resposta['n_input'] == N_LISTINGS, 'n_input divergiu do enviado'
assert len(resposta['results']) == N_LISTINGS, 'correspondência 1:1 quebrada'
print(f'\n  {N_LISTINGS} enviados -> {len(resposta["results"])} recebidos: OK')

In [ ]:
# ---------------------------------------------------------------------------
# Passo 5 — DataFrame a partir do JSON de resposta
# ---------------------------------------------------------------------------
df_previsto = pd.DataFrame(resposta['results'])

print(f'df_previsto: {df_previsto.shape[0]} linhas x {df_previsto.shape[1]} colunas')
print(f'  = 96 originais + 6 de predição '
      f'(price_estimate, L, U, zona, uplift, status)\n')

COLS_PRED = ['id', 'price', 'price_estimate', 'L', 'U', 'zona', 'uplift', 'status']
_v = df_previsto.copy()
_v['price'] = pd.to_numeric(_v['price'].astype(str).str.replace(r'[\$,]', '', regex=True),
                            errors='coerce')

print('RESULTADO — preço praticado vs preço justo estimado')
print('=' * 92)
print(_v[COLS_PRED].to_string(index=False))
print('=' * 92)

# --- Leitura de negócio -----------------------------------------------------
print('\nDistribuição por zona:')
for _z, _n in df_previsto.zona.value_counts(dropna=False).items():
    print(f'  {str(_z):<22} {_n:>3} ({_n/len(df_previsto)*100:5.1f}%)')

_subs = _v[_v.zona == 'subprecificado']
if len(_subs):
    print(f'\nRecomendações ativas: {len(_subs)}')
    print(f'  uplift total ... €{_subs.uplift.sum():.2f}/diária')
    print(f'  maior uplift ... €{_subs.uplift.max():.2f} '
          f'(id {int(_subs.loc[_subs.uplift.idxmax(), "id"])})')
else:
    print('\nNenhum subprecificado nesta amostra.')

# --- Verificações do contrato (plano §11) ------------------------------------
print('\nVerificações:')
assert len(df_previsto) == len(amostra_teste), 'correspondência 1:1 quebrada'
print(f'  correspondência 1:1 ({len(amostra_teste)} -> {len(df_previsto)}) ...... OK')

assert set(amostra_teste.columns).issubset(df_previsto.columns), 'colunas originais perdidas'
print(f'  as 96 colunas originais voltaram intactas ......... OK')

_ok = df_previsto.status == 'ok'
assert df_previsto.loc[_ok, 'zona'].notna().all(), 'anúncio válido sem zona'
print(f'  todo anúncio válido tem zona ...................... OK')

_vl = _v[_ok]
assert (_vl.L < _vl.price_estimate).all() and (_vl.price_estimate < _vl.U).all()
print(f'  banda L < estimate < U ............................ OK')

_abaixo = _vl[_vl.price < resposta['piso_recomendacao']]
assert (_abaixo.uplift.fillna(0) == 0).all(), 'uplift abaixo do piso'
print(f'  piso de €{resposta["piso_recomendacao"]:.0f} respeitado '
      f'({len(_abaixo)} anúncios na faixa) ......... OK')

assert (_vl[_vl.zona != 'subprecificado'].uplift.fillna(0) == 0).all()
print(f'  uplift > 0 só em subprecificado (§3.3) ............ OK')

_erro = (_vl.price - _vl.price_estimate).abs() / _vl.price
print(f'\nMAPE desta amostra: {_erro.mean()*100:.2f}%')
print(f'  (in-sample — estes anúncios treinaram o modelo; o honesto é 22,21%)')